# Sakai 2022 / Gravity Spy — Full 224×224

## Notebook 03 — Benchmark e preparazione del training

Questo notebook parte soltanto dopo la chiusura dei contratti dati e modello nei Notebook 01 e 02.

Obiettivi:

1. identificare e congelare l'hardware effettivamente assegnato da Colab;
2. costruire un bootstrap idempotente per il recovery dopo reset del runtime;
3. benchmark fair dei batch candidate `32`, `64`, `128`;
4. scegliere il batch operativo, privilegiando `128` se sostenibile perché è il valore pubblicato da Sakai;
5. implementare checkpoint e resume persistenti;
6. eseguire smoke training controllati;
7. soltanto dopo tutti i gate, avviare la run scientifica ufficiale di 100 epoche.

Nessun risultato del Notebook 02 viene rigenerato o sostituito.

In [1]:
import torch

print("CUDA_AVAILABLE =", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU_NAME =", torch.cuda.get_device_name(0))
    print(
        "GPU_VRAM_GIB =",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / (1024**3),
            3,
        ),
    )
    print(
        "COMPUTE_CAPABILITY =",
        torch.cuda.get_device_capability(0),
    )

CUDA_AVAILABLE = True
GPU_NAME = NVIDIA A100-SXM4-80GB
GPU_VRAM_GIB = 79.251
COMPUTE_CAPABILITY = (8, 0)


## STEP 03-A0 — Identità del runtime

Il Notebook 03 viene eseguito in un nuovo ambiente di esecuzione.

Prima di ricostruire dati o modello vengono congelate le caratteristiche dell'hardware effettivamente disponibile e verificata l'accessibilità delle sorgenti persistenti sotto `PROJECT_ROOT`.

Questo step è esclusivamente read-only:

- nessuna estrazione dei dati;
- nessuna creazione del modello;
- nessun benchmark;
- nessun training.

In [2]:
# ============================================================
# NOTEBOOK 03 — RUNTIME + PERSISTENT STORAGE PREFLIGHT
# STEP 03-A0
#
# Read-only environment inspection.
# ============================================================

from pathlib import Path

import os
import platform
import psutil
import shutil
import tempfile
import torch

try:
    from google.colab import drive
except ImportError:
    drive = None


# ------------------------------------------------------------
# PORTABLE ENVIRONMENT ROOTS
# ------------------------------------------------------------

_in_colab = drive is not None


def _resolve_absolute_root(
    value,
    *,
    variable_name,
):

    root = Path(
        value
    ).expanduser()

    if not root.is_absolute():
        raise ValueError(
            f"{variable_name} must be an absolute path: {value!r}"
        )

    return root.resolve(
        strict=False
    )


_local_runtime_override = os.environ.get(
    "GRAVITYSPY_LOCAL_RUNTIME_ROOT"
)

if _local_runtime_override is not None:

    LOCAL_RUNTIME_ROOT = _resolve_absolute_root(
        _local_runtime_override,
        variable_name=(
            "GRAVITYSPY_LOCAL_RUNTIME_ROOT"
        ),
    )

elif _in_colab:

    LOCAL_RUNTIME_ROOT = _resolve_absolute_root(
        "/content",
        variable_name=(
            "Colab local runtime root"
        ),
    )

else:

    LOCAL_RUNTIME_ROOT = _resolve_absolute_root(
        tempfile.gettempdir(),
        variable_name=(
            "system temporary directory"
        ),
    )


_project_root_override = os.environ.get(
    "GRAVITYSPY_PROJECT_ROOT"
)

if _project_root_override is not None:

    PROJECT_ROOT = _resolve_absolute_root(
        _project_root_override,
        variable_name=(
            "GRAVITYSPY_PROJECT_ROOT"
        ),
    )

elif _in_colab:

    _colab_drive_mount = _resolve_absolute_root(
        "/content/drive",
        variable_name=(
            "Colab Drive mount point"
        ),
    )

    drive.mount(
        str(
            _colab_drive_mount
        ),
        force_remount=False,
    )

    PROJECT_ROOT = (
        _colab_drive_mount
        / "MyDrive"
        / "TESI_SAKAI2022"
    ).resolve(
        strict=False
    )

else:

    raise RuntimeError(
        "GRAVITYSPY_PROJECT_ROOT must be set outside Colab."
    )

SAKAI224_ROOT = (
    PROJECT_ROOT
    / "sakai224_full"
)

TRANSFER_V1 = (
    SAKAI224_ROOT
    / "transfer_v1"
)

NOTEBOOKS_DIR = (
    SAKAI224_ROOT
    / "notebooks"
)


# ------------------------------------------------------------
# CUDA / GPU
# ------------------------------------------------------------

cuda_available = torch.cuda.is_available()

assert cuda_available


gpu_name = torch.cuda.get_device_name(0)

gpu_properties = (
    torch.cuda.get_device_properties(0)
)

gpu_vram_gib = (
    gpu_properties.total_memory
    / (1024 ** 3)
)

compute_capability = (
    torch.cuda.get_device_capability(0)
)


# ------------------------------------------------------------
# SYSTEM MEMORY / TEMPORARY DISK
# ------------------------------------------------------------

ram_gib = (
    psutil.virtual_memory().total
    / (1024 ** 3)
)

content_free_gib = (
    shutil.disk_usage(LOCAL_RUNTIME_ROOT).free
    / (1024 ** 3)
)


# ------------------------------------------------------------
# PERSISTENT SOURCE GATES
# ------------------------------------------------------------

persistent_paths = {
    "PROJECT_ROOT":
        PROJECT_ROOT,

    "SAKAI224_ROOT":
        SAKAI224_ROOT,

    "TRANSFER_V1":
        TRANSFER_V1,

    "NOTEBOOKS_DIR":
        NOTEBOOKS_DIR,
}


persistent_paths_ok = True

for name, path in persistent_paths.items():

    exists = path.is_dir()

    persistent_paths_ok &= exists

    print(
        f"{name:18s} =",
        exists,
    )


assert persistent_paths_ok


# ------------------------------------------------------------
# A100 IDENTITY
# ------------------------------------------------------------

a100_assigned = (
    "A100" in gpu_name.upper()
)

assert a100_assigned


# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

print()
print(
    "=== NOTEBOOK 03 RUNTIME IDENTITY ==="
)

print(
    "PYTHON =",
    platform.python_version(),
)

print(
    "PYTORCH =",
    torch.__version__,
)

print(
    "TORCH_CUDA =",
    torch.version.cuda,
)

print(
    "CUDNN =",
    torch.backends.cudnn.version(),
)

print()

print(
    "GPU_NAME =",
    gpu_name,
)

print(
    "GPU_VRAM_GIB =",
    round(
        gpu_vram_gib,
        3,
    ),
)

print(
    "COMPUTE_CAPABILITY =",
    compute_capability,
)

print(
    "A100_ASSIGNED =",
    a100_assigned,
)

print()

print(
    "RAM_GIB =",
    round(
        ram_gib,
        3,
    ),
)

print(
    "CONTENT_FREE_GIB =",
    round(
        content_free_gib,
        3,
    ),
)

print()

print(
    "PERSISTENT_PATHS_OK =",
    persistent_paths_ok,
)

print(
    "DATA_EXTRACTED =",
    False,
)

print(
    "MODEL_CREATED =",
    False,
)

print(
    "BENCHMARK_STARTED =",
    False,
)

print(
    "OFFICIAL_TRAINING_STARTED =",
    False,
)

print()

print(
    "NOTEBOOK_03_RUNTIME_IDENTITY_PASS =",
    (
        cuda_available
        and a100_assigned
        and persistent_paths_ok
    ),
)

Mounted at /content/drive
PROJECT_ROOT       = True
SAKAI224_ROOT      = True
TRANSFER_V1        = True
NOTEBOOKS_DIR      = True

=== NOTEBOOK 03 RUNTIME IDENTITY ===
PYTHON = 3.13.15
PYTORCH = 2.11.0+cu128
TORCH_CUDA = 12.8
CUDNN = 91900

GPU_NAME = NVIDIA A100-SXM4-80GB
GPU_VRAM_GIB = 79.251
COMPUTE_CAPABILITY = (8, 0)
A100_ASSIGNED = True

RAM_GIB = 167.052
CONTENT_FREE_GIB = 188.491

PERSISTENT_PATHS_OK = True
DATA_EXTRACTED = False
MODEL_CREATED = False
BENCHMARK_STARTED = False
OFFICIAL_TRAINING_STARTED = False

NOTEBOOK_03_RUNTIME_IDENTITY_PASS = True


## STEP 03-A1 — Bootstrap e recovery automatico dei dati

Lo storage locale sotto `LOCAL_RUNTIME_ROOT` è temporaneo e può essere perso quando il runtime viene terminato o sostituito.

Questa sezione rende il Notebook 03 **idempotente rispetto ai reset del runtime**.

Il bootstrap:

- usa `transfer_v1` sotto `PROJECT_ROOT` come sorgente persistente congelata;
- ricrea nello storage locale sotto `LOCAL_RUNTIME_ROOT` soltanto i file mancanti;
- verifica SHA-256 dei sette artefatti locali;
- riutilizza un'estrazione già completa e conforme;
- se trova una directory temporanea parziale/non conforme, elimina esclusivamente quella copia derivata sotto `LOCAL_RUNTIME_ROOT` e la ricostruisce dal TAR verificato;
- verifica nuovamente numero di file e byte delle due sorgenti dati.

Nessun file nello storage persistente del progetto viene modificato.

Il bootstrap non costituisce preprocessing e non modifica dataset, manifest, split o modello neurale.

In [3]:
# ============================================================
# NOTEBOOK 03 — IDEMPOTENT DATA BOOTSTRAP / RECOVERY
# STEP 03-A1
#
# Persistent authority:
#   PROJECT_ROOT / "sakai224_full" / "transfer_v1"
#
# Temporary runtime cache:
#   LOCAL_RUNTIME_ROOT / "sakai224_full_transfer_v1"
#   LOCAL_RUNTIME_ROOT / "sakai224_full_data"
#
# This cell is safe to re-run after a runtime reset.
#
# NO model creation.
# NO benchmark.
# NO training.
# ============================================================

from pathlib import Path

import hashlib
import shutil
import subprocess


# ============================================================
# TEMPORARY LOCAL PATHS
# ============================================================

LOCAL_TRANSFER_DIR = (
    LOCAL_RUNTIME_ROOT
    / "sakai224_full_transfer_v1"
)

LOCAL_DATA_ROOT = (
    LOCAL_RUNTIME_ROOT
    / "sakai224_full_data"
)

LOCAL_OUT272_DIR = (
    LOCAL_DATA_ROOT
    / "out272_gray"
)

LOCAL_CENTRAL224_DIR = (
    LOCAL_DATA_ROOT
    / "tensors_224_central_uint8"
)


# ============================================================
# FROZEN TRANSFER_V1 SHA-256 CONTRACT
#
# These digests identify the exact artifacts previously
# validated for sakai224_full.
# ============================================================

EXPECTED_TRANSFER_SHA256 = {
    "sakai224_full_out272_gray.tar":
        "70c766c3652f586c8da4c8cadf134a8ce83cfd07c8cd10545367f380017ddab2",

    "sakai224_full_central224.tar":
        "0fad37cdf4ea056e2c3abfa2d1be73b9d9bd55e644d54a58013950697cf4d4b6",

    "manifest_events_272.csv":
        "620960e6b35da31bc4cf1a8a6da8679027d378e1d4b2ac73c5267c1c5311ae82",

    "manifest_tensors_224_central.csv":
        "94b7e2e051bd32de72dd2cb6a5fb7eda9bcaec123cbf3a07d0d901bd19779afa",

    "split_train_indices_seed123.csv":
        "8c7a513b98dbe996261bc80965500ab15ab77f498ec9440858003152e42f926f",

    "split_val_indices_seed123.csv":
        "e63109c88df7b3ee97317f94791f26df3fcaf8b937921cfb27967d01c2546467",

    "transfer_manifest_sha256.txt":
        "59a04c1098ec4465c1a9864e7bc61181c7981e316c8bc78adbf4c18b78da962f",
}


# ============================================================
# FROZEN EXTRACTED PAYLOAD CONTRACT
# ============================================================

EXPECTED_OUT272_FILE_COUNT = 34_332
EXPECTED_OUT272_BYTES = 1_144_867_540

EXPECTED_CENTRAL224_FILE_COUNT = 8_583
EXPECTED_CENTRAL224_BYTES = 1_722_642_432


# ============================================================
# HELPERS
# ============================================================

def sha256_file_streaming(
    path,
    chunk_size=8 * 1024 * 1024,
):
    """
    Compute SHA-256 without loading a multi-GB file into RAM.
    """
    digest = hashlib.sha256()

    with path.open("rb") as handle:

        while True:

            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def audit_payload_directory(
    directory,
    expected_file_count,
    expected_bytes,
):
    """
    Validate one extracted temporary dataset by exact
    file count and total payload bytes.

    Returns:
        valid, actual_file_count, actual_bytes
    """

    if not directory.is_dir():
        return False, 0, 0

    files = [
        path
        for path in directory.rglob("*")
        if path.is_file()
    ]

    actual_file_count = len(
        files
    )

    actual_bytes = sum(
        path.stat().st_size
        for path in files
    )

    valid = (
        actual_file_count
            == expected_file_count

        and actual_bytes
            == expected_bytes
    )

    return (
        valid,
        actual_file_count,
        actual_bytes,
    )


def restore_local_transfer_file(
    filename,
    expected_sha256,
):
    """
    Restore one transfer_v1 artifact into local runtime storage.

    Existing valid local files are reused.

    An invalid local file is safe to replace because
    Local runtime storage is only a temporary cache; the persistent
    Persistent project source remains untouched.
    """

    source = (
        TRANSFER_V1
        / filename
    )

    destination = (
        LOCAL_TRANSFER_DIR
        / filename
    )

    if not source.is_file():

        raise RuntimeError(
            "Missing persistent transfer source: "
            f"{source}"
        )


    if destination.is_file():

        current_sha = (
            sha256_file_streaming(
                destination
            )
        )

        if current_sha == expected_sha256:

            return (
                "REUSED_VERIFIED",
                True,
            )

        # A corrupt/partial local runtime copy has no scientific
        # authority and can safely be rebuilt.
        destination.unlink()

        action = "RECOPIED_INVALID"

    else:

        action = "COPIED"


    shutil.copy2(
        source,
        destination,
    )


    final_sha = (
        sha256_file_streaming(
            destination
        )
    )

    file_ok = (
        final_sha
        == expected_sha256
    )

    if not file_ok:

        raise RuntimeError(
            "SHA-256 mismatch after local restore: "
            f"{filename}"
        )

    return (
        action,
        file_ok,
    )


def restore_extracted_payload(
    archive_path,
    target_directory,
    expected_file_count,
    expected_bytes,
):
    """
    Ensure that one extracted dataset exists and is complete.

    Valid existing data are reused.

    A partial/invalid temporary directory is deleted and
    recreated exclusively from the verified local TAR.
    """

    (
        valid_before,
        count_before,
        bytes_before,
    ) = audit_payload_directory(
        target_directory,
        expected_file_count,
        expected_bytes,
    )


    if valid_before:

        return {
            "action":
                "REUSED_VERIFIED",

            "file_count":
                count_before,

            "total_bytes":
                bytes_before,

            "valid":
                True,
        }


    # --------------------------------------------------------
    # PARTIAL / STALE TEMPORARY EXTRACTION
    #
    # Only the known derived directory under LOCAL_RUNTIME_ROOT is
    # removed. Persistent project data are never touched.
    # --------------------------------------------------------

    if target_directory.exists():

        assert (
            target_directory.parent
            == LOCAL_DATA_ROOT
        )

        shutil.rmtree(
            target_directory
        )

        extraction_action = (
            "REBUILT_INVALID"
        )

    else:

        extraction_action = (
            "EXTRACTED"
        )


    LOCAL_DATA_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )


    subprocess.run(
        [
            "tar",
            "-xf",
            str(archive_path),
            "-C",
            str(LOCAL_DATA_ROOT),
        ],
        check=True,
    )


    (
        valid_after,
        count_after,
        bytes_after,
    ) = audit_payload_directory(
        target_directory,
        expected_file_count,
        expected_bytes,
    )


    if not valid_after:

        raise RuntimeError(
            "Extracted payload failed audit: "
            f"{target_directory}"
        )


    return {
        "action":
            extraction_action,

        "file_count":
            count_after,

        "total_bytes":
            bytes_after,

        "valid":
            True,
    }


# ============================================================
# STORAGE PREFLIGHT
#
# Calculate a conservative upper bound for a completely fresh
# runtime:
#
#   local copy of transfer_v1
#   + both extracted datasets
#   + 1 GiB safety margin
# ============================================================

persistent_transfer_bytes = sum(
    (
        TRANSFER_V1
        / filename
    ).stat().st_size

    for filename
    in EXPECTED_TRANSFER_SHA256
)

fresh_runtime_required_bytes = (
    persistent_transfer_bytes
    + EXPECTED_OUT272_BYTES
    + EXPECTED_CENTRAL224_BYTES
    + 1 * 1024**3
)

free_bytes = (
    shutil.disk_usage(
        LOCAL_RUNTIME_ROOT
    ).free
)

storage_gate = (
    free_bytes
    >= fresh_runtime_required_bytes
)


print(
    "=== BOOTSTRAP STORAGE PREFLIGHT ==="
)

print(
    "CONTENT_FREE_GIB =",
    round(
        free_bytes / (1024**3),
        3,
    ),
)

print(
    "FRESH_RUNTIME_REQUIRED_GIB =",
    round(
        fresh_runtime_required_bytes
        / (1024**3),
        3,
    ),
)

print(
    "STORAGE_GATE =",
    storage_gate,
)


assert storage_gate


# ============================================================
# RESTORE / VERIFY LOCAL TRANSFER CACHE
# ============================================================

LOCAL_TRANSFER_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


transfer_all_ok = True


print()
print(
    "=== LOCAL TRANSFER CACHE ==="
)


for filename, expected_sha in (
    EXPECTED_TRANSFER_SHA256.items()
):

    action, file_ok = (
        restore_local_transfer_file(
            filename,
            expected_sha,
        )
    )

    transfer_all_ok &= file_ok

    print(
        f"{filename:42s}",
        action,
        "SHA256_OK =",
        file_ok,
    )


assert transfer_all_ok


# ============================================================
# RESTORE / VERIFY OUT272_GRAY
# ============================================================

out272_result = restore_extracted_payload(
    archive_path=(
        LOCAL_TRANSFER_DIR
        / "sakai224_full_out272_gray.tar"
    ),

    target_directory=(
        LOCAL_OUT272_DIR
    ),

    expected_file_count=(
        EXPECTED_OUT272_FILE_COUNT
    ),

    expected_bytes=(
        EXPECTED_OUT272_BYTES
    ),
)


# ============================================================
# RESTORE / VERIFY CENTRAL224
# ============================================================

central224_result = restore_extracted_payload(
    archive_path=(
        LOCAL_TRANSFER_DIR
        / "sakai224_full_central224.tar"
    ),

    target_directory=(
        LOCAL_CENTRAL224_DIR
    ),

    expected_file_count=(
        EXPECTED_CENTRAL224_FILE_COUNT
    ),

    expected_bytes=(
        EXPECTED_CENTRAL224_BYTES
    ),
)


# ============================================================
# FINAL LOCAL DATA CONTRACT
# ============================================================

local_data_ready = (
    transfer_all_ok

    and out272_result[
        "valid"
    ]

    and central224_result[
        "valid"
    ]
)


assert local_data_ready


# ============================================================
# REPORT
# ============================================================

print()
print(
    "=== OUT272_GRAY ==="
)

print(
    "ACTION =",
    out272_result[
        "action"
    ],
)

print(
    "FILE_COUNT =",
    out272_result[
        "file_count"
    ],
)

print(
    "TOTAL_BYTES =",
    out272_result[
        "total_bytes"
    ],
)

print(
    "PAYLOAD_OK =",
    out272_result[
        "valid"
    ],
)


print()
print(
    "=== CENTRAL224 ==="
)

print(
    "ACTION =",
    central224_result[
        "action"
    ],
)

print(
    "FILE_COUNT =",
    central224_result[
        "file_count"
    ],
)

print(
    "TOTAL_BYTES =",
    central224_result[
        "total_bytes"
    ],
)

print(
    "PAYLOAD_OK =",
    central224_result[
        "valid"
    ],
)


print()
print(
    "TRANSFER_CACHE_OK =",
    transfer_all_ok,
)

print(
    "LOCAL_DATA_READY =",
    local_data_ready,
)

print(
    "DRIVE_MODIFIED =",
    False,
)

print(
    "MODEL_CREATED =",
    False,
)

print(
    "BENCHMARK_STARTED =",
    False,
)

print(
    "OFFICIAL_TRAINING_STARTED =",
    False,
)


print()
print(
    "NOTEBOOK_03_DATA_BOOTSTRAP_PASS =",
    (
        storage_gate
        and transfer_all_ok
        and local_data_ready
    ),
)

=== BOOTSTRAP STORAGE PREFLIGHT ===
CONTENT_FREE_GIB = 188.489
FRESH_RUNTIME_REQUIRED_GIB = 6.373
STORAGE_GATE = True

=== LOCAL TRANSFER CACHE ===
sakai224_full_out272_gray.tar              COPIED SHA256_OK = True
sakai224_full_central224.tar               COPIED SHA256_OK = True
manifest_events_272.csv                    COPIED SHA256_OK = True
manifest_tensors_224_central.csv           COPIED SHA256_OK = True
split_train_indices_seed123.csv            COPIED SHA256_OK = True
split_val_indices_seed123.csv              COPIED SHA256_OK = True
transfer_manifest_sha256.txt               COPIED SHA256_OK = True

=== OUT272_GRAY ===
ACTION = EXTRACTED
FILE_COUNT = 34332
TOTAL_BYTES = 1144867540
PAYLOAD_OK = True

=== CENTRAL224 ===
ACTION = EXTRACTED
FILE_COUNT = 8583
TOTAL_BYTES = 1722642432
PAYLOAD_OK = True

TRANSFER_CACHE_OK = True
LOCAL_DATA_READY = True
DRIVE_MODIFIED = False
MODEL_CREATED = False
BENCHMARK_STARTED = False
OFFICIAL_TRAINING_STARTED = False

NOTEBOOK_03_DATA_BOOTSTRA

## STEP 03-A2A — Ricostruzione del contratto del modello

Il Notebook 02 ha già validato architettura, inizializzazione, forward, loss, backward e Adam.

Nel Notebook 03 tali contratti non vengono ridefiniti scientificamente: vengono **ricostruiti operativamente** per benchmark e training.

Hard gate principale:

`PROJECT_PARAMETER_SHA256 = 1df6f7d32b4d0705c273b5bfae7c3218f72ff7cd1d160560b497afb3f235859b`

Se il fingerprint non coincide, il benchmark non può iniziare.

Questo step:

- crea il modello esclusivamente su CPU;
- applica l'inizializzazione PROJECT già verificata contro il C;
- verifica 16.524.548 parametri e 28 tensori parametro;
- verifica il fingerprint congelato;
- non esegue forward, backward o optimizer.

In [4]:
# ============================================================
# NOTEBOOK 03 — MODEL CONTRACT RECONSTRUCTION
# STEP 03-A2A
#
# Rebuild the already validated Notebook 02 model contract.
#
# NO forward.
# NO backward.
# NO optimizer.
# NO benchmark.
# NO training.
# ============================================================

from dataclasses import dataclass
import hashlib
import math

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# FROZEN FULL-224 CONTRACT
# ============================================================

@dataclass(frozen=True)
class Sakai224FullSpec:
    input_channels: int = 4
    input_side: int = 224
    latent_dim: int = 512

    bn_eps: float = 1.0e-5
    bn_momentum: float = 0.1

    beta_kl: float = 1.0


SPEC = Sakai224FullSpec()

EXPECTED_PARAMETER_COUNT = 16_524_548

PROJECT_INITIALIZATION_SEED = 123

FROZEN_INIT_SHA256 = (
    "1df6f7d32b4d0705c273b5bfae7c321"
    "8f72ff7cd1d160560b497afb3f235859b"
)


# ============================================================
# ENCODER
#
# PAPER architecture:
#   4x224x224
#   -> 64x112x112
#   -> MaxPool -> 64x56x56
#   -> 128x28x28
#   -> 256x14x14
#   -> 512x7x7
#   -> global average pooling
#   -> FC 1024
#   -> mu 512 + logvar 512
# ============================================================

class Sakai224FullEncoder(nn.Module):

    def __init__(self, spec):
        super().__init__()

        self.conv1 = nn.Conv2d(
            4, 64,
            kernel_size=3,
            stride=2,
            padding=1,
            bias=False,
        )
        self.bn1 = nn.BatchNorm2d(
            64,
            eps=spec.bn_eps,
            momentum=spec.bn_momentum,
        )

        self.pool = nn.MaxPool2d(
            kernel_size=2,
            stride=2,
        )

        self.conv2 = nn.Conv2d(
            64, 128,
            kernel_size=3,
            stride=2,
            padding=1,
            bias=False,
        )
        self.bn2 = nn.BatchNorm2d(
            128,
            eps=spec.bn_eps,
            momentum=spec.bn_momentum,
        )

        self.conv3 = nn.Conv2d(
            128, 256,
            kernel_size=3,
            stride=2,
            padding=1,
            bias=False,
        )
        self.bn3 = nn.BatchNorm2d(
            256,
            eps=spec.bn_eps,
            momentum=spec.bn_momentum,
        )

        self.conv4 = nn.Conv2d(
            256, 512,
            kernel_size=3,
            stride=2,
            padding=1,
            bias=False,
        )
        self.bn4 = nn.BatchNorm2d(
            512,
            eps=spec.bn_eps,
            momentum=spec.bn_momentum,
        )

        self.fc = nn.Linear(
            512,
            1024,
            bias=True,
        )

    def forward(self, x):

        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)

        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))

        # Global average pooling 7x7 -> 1x1.
        x = x.mean(dim=(2, 3))

        packed = self.fc(x)

        mu = packed[:, :512]
        logvar = packed[:, 512:]

        return mu, logvar


# ============================================================
# DECODER
#
# 512
# -> FC 512*7*7
# -> BN + ReLU
# -> nearest 7->14->28
# -> Conv 512->256
# -> nearest 56
# -> Conv 256->128
# -> nearest 112
# -> Conv 128->64
# -> nearest 224
# -> linear output logits 64->4
#
# The final linear-logit choice is PROJECT and is paired with
# BCE-with-logits; it resolves the ambiguity in Sakai Fig. 8.
# ============================================================

class Sakai224FullDecoder(nn.Module):

    def __init__(self, spec):
        super().__init__()

        self.fc = nn.Linear(
            512,
            512 * 7 * 7,
            bias=False,
        )

        self.bn_fc = nn.BatchNorm1d(
            512 * 7 * 7,
            eps=spec.bn_eps,
            momentum=spec.bn_momentum,
        )

        self.conv1 = nn.Conv2d(
            512, 256,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.bn1 = nn.BatchNorm2d(
            256,
            eps=spec.bn_eps,
            momentum=spec.bn_momentum,
        )

        self.conv2 = nn.Conv2d(
            256, 128,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.bn2 = nn.BatchNorm2d(
            128,
            eps=spec.bn_eps,
            momentum=spec.bn_momentum,
        )

        self.conv3 = nn.Conv2d(
            128, 64,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.bn3 = nn.BatchNorm2d(
            64,
            eps=spec.bn_eps,
            momentum=spec.bn_momentum,
        )

        self.out_conv = nn.Conv2d(
            64, 4,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=True,
        )

    def forward(self, z):

        x = self.fc(z)
        x = F.relu(self.bn_fc(x))

        x = x.reshape(
            z.shape[0],
            512,
            7,
            7,
        )

        x = F.interpolate(
            x,
            size=(14, 14),
            mode="nearest",
        )

        x = F.interpolate(
            x,
            size=(28, 28),
            mode="nearest",
        )

        x = F.relu(self.bn1(self.conv1(x)))

        x = F.interpolate(
            x,
            size=(56, 56),
            mode="nearest",
        )

        x = F.relu(self.bn2(self.conv2(x)))

        x = F.interpolate(
            x,
            size=(112, 112),
            mode="nearest",
        )

        x = F.relu(self.bn3(self.conv3(x)))

        x = F.interpolate(
            x,
            size=(224, 224),
            mode="nearest",
        )

        # Linear logits: no sigmoid inside the model.
        return self.out_conv(x)


# ============================================================
# COMPLETE VAE
#
# RNG deliberately remains external to the model.
# ============================================================

class Sakai224FullVAE(nn.Module):

    def __init__(self, spec):
        super().__init__()

        self.encoder = Sakai224FullEncoder(spec)
        self.decoder = Sakai224FullDecoder(spec)

    def forward(
        self,
        x,
        epsilon=None,
        mean_mode=False,
    ):
        mu, logvar = self.encoder(x)

        if mean_mode:
            z = mu

        else:
            if epsilon is None:
                raise ValueError(
                    "epsilon is required in stochastic mode"
                )

            if epsilon.shape != mu.shape:
                raise ValueError(
                    "epsilon shape must equal mu shape"
                )

            z = (
                mu
                + torch.exp(0.5 * logvar)
                * epsilon
            )

        logits = self.decoder(z)

        return logits, mu, logvar, z


# ============================================================
# PROJECT RNG USED BY INITIALIZATION
#
# Same stateless uint64 contract already tested Python <-> C
# in Notebook 02.
# ============================================================

MASK64 = (1 << 64) - 1

RNG_DOMAIN_INITIALIZATION = 1

MIX_GAMMA = 0x9E3779B97F4A7C15
MIX_MUL1 = 0xBF58476D1CE4E5B9
MIX_MUL2 = 0x94D049BB133111EB
DERIVE_XOR = 0xD1B54A32D192ED03


def c_mix64(value):
    value = (
        value + MIX_GAMMA
    ) & MASK64

    value = (
        (value ^ (value >> 30))
        * MIX_MUL1
    ) & MASK64

    value = (
        (value ^ (value >> 27))
        * MIX_MUL2
    ) & MASK64

    return (
        value ^ (value >> 31)
    ) & MASK64


def c_derive_seed(
    base_seed,
    domain,
    epoch,
    manifest_index,
    stream_index,
):
    state = c_mix64(
        base_seed ^ DERIVE_XOR
    )

    state = c_mix64(
        state ^ int(domain)
    )
    state = c_mix64(
        state ^ int(epoch)
    )
    state = c_mix64(
        state ^ int(manifest_index)
    )
    state = c_mix64(
        state ^ int(stream_index)
    )

    return state


# ============================================================
# EXACT PROJECT-C INITIALIZATION GENERATOR
# ============================================================

def project_c_uniform_values(
    count,
    base_seed,
    parameter_key,
    bound,
):
    """
    Generate the exact float32 uniform stream used by the
    verified PROJECT C initializer for one parameter tensor.
    """

    derived_seed = c_derive_seed(
        base_seed,
        RNG_DOMAIN_INITIALIZATION,
        0,
        0,
        parameter_key,
    )

    indices = np.arange(
        1,
        count + 1,
        dtype=np.uint64,
    )

    values = (
        np.uint64(derived_seed)
        + np.uint64(MIX_GAMMA)
        * indices
    )

    values ^= values >> np.uint64(30)
    values *= np.uint64(MIX_MUL1)

    values ^= values >> np.uint64(27)
    values *= np.uint64(MIX_MUL2)

    values ^= values >> np.uint64(31)

    # C uniform_f32 uses the top 24 bits.
    uniform = (
        (values >> np.uint64(40))
        .astype(np.uint32)
        .astype(np.float32)
        * np.float32(1.0 / 16777216.0)
    )

    # C computes the affine transformation in double and then
    # stores the result as float32.
    result = (
        -float(bound)
        + 2.0
        * float(bound)
        * uniform.astype(np.float64)
    ).astype(np.float32)

    return result


def python_he_values(
    count,
    seed,
    parameter_key,
    fan_in,
):
    bound = math.sqrt(
        6.0 / float(fan_in)
    )

    return project_c_uniform_values(
        count,
        seed,
        parameter_key,
        bound,
    )


def python_xavier_values(
    count,
    seed,
    parameter_key,
    fan_in,
    fan_out,
):
    bound = math.sqrt(
        6.0
        / float(fan_in + fan_out)
    )

    return project_c_uniform_values(
        count,
        seed,
        parameter_key,
        bound,
    )


# ============================================================
# STABLE C PARAMETER-KEY PLAN: 1..28
# ============================================================

def build_project_parameter_plan(model):
    return [
        (1,  model.encoder.conv1.weight, "he",      4*3*3,   None),
        (2,  model.encoder.bn1.weight,   "one",     None,    None),
        (3,  model.encoder.bn1.bias,     "zero",    None,    None),

        (4,  model.encoder.conv2.weight, "he",      64*3*3,  None),
        (5,  model.encoder.bn2.weight,   "one",     None,    None),
        (6,  model.encoder.bn2.bias,     "zero",    None,    None),

        (7,  model.encoder.conv3.weight, "he",      128*3*3, None),
        (8,  model.encoder.bn3.weight,   "one",     None,    None),
        (9,  model.encoder.bn3.bias,     "zero",    None,    None),

        (10, model.encoder.conv4.weight, "he",      256*3*3, None),
        (11, model.encoder.bn4.weight,   "one",     None,    None),
        (12, model.encoder.bn4.bias,     "zero",    None,    None),

        (13, model.encoder.fc.weight,    "xavier",  512,     1024),
        (14, model.encoder.fc.bias,      "zero",    None,    None),

        (15, model.decoder.fc.weight,    "he",      512,     None),
        (16, model.decoder.bn_fc.weight, "one",     None,    None),
        (17, model.decoder.bn_fc.bias,   "zero",    None,    None),

        (18, model.decoder.conv1.weight, "he",      512*3*3, None),
        (19, model.decoder.bn1.weight,   "one",     None,    None),
        (20, model.decoder.bn1.bias,     "zero",    None,    None),

        (21, model.decoder.conv2.weight, "he",      256*3*3, None),
        (22, model.decoder.bn2.weight,   "one",     None,    None),
        (23, model.decoder.bn2.bias,     "zero",    None,    None),

        (24, model.decoder.conv3.weight, "he",      128*3*3, None),
        (25, model.decoder.bn3.weight,   "one",     None,    None),
        (26, model.decoder.bn3.bias,     "zero",    None,    None),

        (27, model.decoder.out_conv.weight, "xavier", 64*3*3, 4*3*3),
        (28, model.decoder.out_conv.bias,   "zero",   None,   None),
    ]


def get_project_batchnorm_layers(model):
    return [
        model.encoder.bn1,
        model.encoder.bn2,
        model.encoder.bn3,
        model.encoder.bn4,
        model.decoder.bn_fc,
        model.decoder.bn1,
        model.decoder.bn2,
        model.decoder.bn3,
    ]


def initialize_project_c_exact_v1(
    model,
    seed=PROJECT_INITIALIZATION_SEED,
):
    """
    Apply the exact initialization policy already validated
    against the current C implementation in Notebook 02.
    """

    assert (
        next(model.parameters()).device.type
        == "cpu"
    )

    plan = build_project_parameter_plan(
        model
    )

    assert [
        row[0]
        for row in plan
    ] == list(range(1, 29))

    with torch.no_grad():

        for (
            key,
            parameter,
            kind,
            fan_in,
            fan_out,
        ) in plan:

            if kind == "zero":
                parameter.zero_()

            elif kind == "one":
                parameter.fill_(1.0)

            elif kind == "he":

                values = python_he_values(
                    parameter.numel(),
                    seed,
                    key,
                    fan_in,
                )

                parameter.copy_(
                    torch.from_numpy(
                        values.reshape(
                            tuple(parameter.shape)
                        )
                    )
                )

            elif kind == "xavier":

                values = python_xavier_values(
                    parameter.numel(),
                    seed,
                    key,
                    fan_in,
                    fan_out,
                )

                parameter.copy_(
                    torch.from_numpy(
                        values.reshape(
                            tuple(parameter.shape)
                        )
                    )
                )

            else:
                raise RuntimeError(
                    f"Unknown initializer: {kind}"
                )

        # Initial BatchNorm running state.
        for bn in get_project_batchnorm_layers(
            model
        ):
            bn.running_mean.zero_()
            bn.running_var.fill_(1.0)
            bn.num_batches_tracked.zero_()


def project_parameter_sha256(model):
    """
    Canonical SHA-256 of raw float32 parameter bytes in
    stable C parameter-key order.
    """

    digest = hashlib.sha256()

    for (
        key,
        parameter,
        kind,
        fan_in,
        fan_out,
    ) in build_project_parameter_plan(
        model
    ):

        array = (
            parameter
            .detach()
            .cpu()
            .contiguous()
            .numpy()
        )

        digest.update(
            array.tobytes(
                order="C"
            )
        )

    return digest.hexdigest()


# ============================================================
# VAE LOSS CONTRACT
# ============================================================

def sakai_vae_loss(
    logits,
    targets,
    mu,
    logvar,
    beta_kl=1.0,
):
    """
    PROJECT loss matched to the validated C contract:

      BCE-with-logits:
          sum over channels/pixels/batch

      KL:
          0.5 * sum(mu^2 + exp(logvar) - 1 - logvar)

      optimized loss:
          (reconstruction_sum + KL_sum) / batch_size
    """

    batch_size = logits.shape[0]

    reconstruction_sum = (
        F.binary_cross_entropy_with_logits(
            logits,
            targets,
            reduction="sum",
        )
    )

    kl_sum = (
        0.5
        * torch.sum(
            mu.square()
            + torch.expm1(logvar)
            - logvar
        )
    )

    total_sum = (
        reconstruction_sum
        + beta_kl * kl_sum
    )

    loss = (
        total_sum
        / batch_size
    )

    return {
        "loss":
            loss,

        "reconstruction_sum":
            reconstruction_sum,

        "reconstruction_mean_per_sample":
            reconstruction_sum
            / batch_size,

        "kl_sum":
            kl_sum,

        "kl_mean_per_sample":
            kl_sum
            / batch_size,

        "batch_size":
            batch_size,

        "beta_kl":
            beta_kl,
    }


# ============================================================
# BUILD CPU REFERENCE MODEL + HARD GATES
# ============================================================

model_reference = Sakai224FullVAE(
    SPEC
)

initialize_project_c_exact_v1(
    model_reference
)


parameter_tensor_count = len(
    list(
        model_reference.parameters()
    )
)

trainable_parameter_count = sum(
    parameter.numel()
    for parameter
    in model_reference.parameters()
    if parameter.requires_grad
)

reference_sha256 = (
    project_parameter_sha256(
        model_reference
    )
)


parameter_tensor_gate = (
    parameter_tensor_count == 28
)

parameter_count_gate = (
    trainable_parameter_count
    == EXPECTED_PARAMETER_COUNT
)

fingerprint_gate = (
    reference_sha256
    == FROZEN_INIT_SHA256
)


assert parameter_tensor_gate
assert parameter_count_gate
assert fingerprint_gate


print(
    "=== NOTEBOOK 03 MODEL CONTRACT ==="
)

print(
    "PARAMETER_TENSORS =",
    parameter_tensor_count,
)

print(
    "TRAINABLE_PARAMETERS =",
    trainable_parameter_count,
)

print(
    "EXPECTED_PARAMETERS =",
    EXPECTED_PARAMETER_COUNT,
)

print()

print(
    "PROJECT_PARAMETER_SHA256 =",
    reference_sha256,
)

print(
    "FROZEN_SHA256_MATCH =",
    fingerprint_gate,
)

print()

print(
    "MODEL_DEVICE =",
    next(
        model_reference.parameters()
    ).device,
)

print(
    "FORWARD_EXECUTED =",
    False,
)

print(
    "BACKWARD_EXECUTED =",
    False,
)

print(
    "OPTIMIZER_EXECUTED =",
    False,
)

print(
    "BENCHMARK_STARTED =",
    False,
)

print(
    "OFFICIAL_TRAINING_STARTED =",
    False,
)

print()

print(
    "NOTEBOOK_03_MODEL_CONTRACT_PASS =",
    (
        parameter_tensor_gate
        and parameter_count_gate
        and fingerprint_gate
    ),
)

=== NOTEBOOK 03 MODEL CONTRACT ===
PARAMETER_TENSORS = 28
TRAINABLE_PARAMETERS = 16524548
EXPECTED_PARAMETERS = 16524548

PROJECT_PARAMETER_SHA256 = 1df6f7d32b4d0705c273b5bfae7c3218f72ff7cd1d160560b497afb3f235859b
FROZEN_SHA256_MATCH = True

MODEL_DEVICE = cpu
FORWARD_EXECUTED = False
BACKWARD_EXECUTED = False
OPTIMIZER_EXECUTED = False
BENCHMARK_STARTED = False
OFFICIAL_TRAINING_STARTED = False

NOTEBOOK_03_MODEL_CONTRACT_PASS = True


## STEP 03-A2B — Ricostruzione del contratto dati e RNG

Il Notebook 03 ricostruisce il contratto operativo già validato nei Notebook 01 e 02.

Vengono verificati nuovamente:

- full split `6866 train / 1717 validation`;
- augmentation stateless PROJECT con seed `789`;
- shuffle deterministico PROJECT con seed `101112`;
- epsilon latente stateless con seed `456`;
- primo batch train dell'epoch 0;
- loader della validation centrale.

Hard gate congelato per il primo batch train:

- posizioni split: `[752, 2659, 4819, 4649]`
- `manifest_index`: `[2676, 6250, 2396, 5602]`
- `crop_left`: `[20, 15, 19, 12]`

Questo step non esegue il VAE e non avvia il benchmark o il training.

In [5]:
# ============================================================
# NOTEBOOK 03 — DATA / SHUFFLE / LATENT RNG CONTRACT
# STEP 03-A2B
#
# Reconstructs contracts already validated in Notebook 01/02.
#
# NO model forward.
# NO benchmark.
# NO training.
# ============================================================

import csv
import math

from PIL import Image

import numpy as np
import torch

from torch.utils.data import (
    Dataset,
    DataLoader,
    Sampler,
)


# ============================================================
# LOCAL DATA / METADATA
# ============================================================

DATA_DIR = LOCAL_DATA_ROOT

OUT272_DIR = LOCAL_OUT272_DIR
CENTRAL224_DIR = LOCAL_CENTRAL224_DIR

METADATA_DIR = LOCAL_TRANSFER_DIR


def read_csv_dicts(path):
    """Read one frozen PROJECT CSV without changing row order."""

    with path.open(
        "r",
        encoding="utf-8",
        newline="",
    ) as handle:

        return list(
            csv.DictReader(handle)
        )


event_rows = read_csv_dicts(
    METADATA_DIR
    / "manifest_events_272.csv"
)

tensor_rows = read_csv_dicts(
    METADATA_DIR
    / "manifest_tensors_224_central.csv"
)

train_rows = read_csv_dicts(
    METADATA_DIR
    / "split_train_indices_seed123.csv"
)

val_rows = read_csv_dicts(
    METADATA_DIR
    / "split_val_indices_seed123.csv"
)


# ------------------------------------------------------------
# FULL SPLIT CONTRACT
# ------------------------------------------------------------

train_indices = {
    int(row["manifest_index"])
    for row in train_rows
}

val_indices = {
    int(row["manifest_index"])
    for row in val_rows
}


split_contract_pass = (
    len(event_rows) == 8583
    and len(tensor_rows) == 8583
    and len(train_rows) == 6866
    and len(val_rows) == 1717
    and len(train_indices) == 6866
    and len(val_indices) == 1717
    and not (
        train_indices
        & val_indices
    )
    and (
        train_indices
        | val_indices
    ) == set(range(8583))
)

assert split_contract_pass


# ============================================================
# STATELESS TRAIN AUGMENTATION
# ============================================================

GS_AUGMENT_MODE_LEGACY_TEXT = 0
GS_AUGMENT_MODE_SYMMETRIC_FIGURE = 1

GS_AUGMENT_STATELESS_V1_DOMAIN_TAG = (
    0x4753415547563121
)

AUGMENTATION_SEED = 789


def c_u64(value):
    return value & MASK64


def gs_augment_stateless_mix64(value):

    value = c_u64(
        value
        + SPLITMIX_GAMMA
    )

    value = c_u64(
        (
            value
            ^ (value >> 30)
        )
        * SPLITMIX_MUL1
    )

    value = c_u64(
        (
            value
            ^ (value >> 27)
        )
        * SPLITMIX_MUL2
    )

    return c_u64(
        value
        ^ (value >> 31)
    )


def gs_augment_stateless_value(
    base_seed,
    epoch,
    manifest_index,
    mode,
    stream_index,
):

    h = gs_augment_stateless_mix64(
        GS_AUGMENT_STATELESS_V1_DOMAIN_TAG
    )

    h = gs_augment_stateless_mix64(
        h ^ c_u64(base_seed)
    )

    h = gs_augment_stateless_mix64(
        h ^ c_u64(epoch)
    )

    h = gs_augment_stateless_mix64(
        h ^ c_u64(manifest_index)
    )

    h = gs_augment_stateless_mix64(
        h ^ c_u64(mode)
    )

    return gs_augment_stateless_mix64(
        h ^ c_u64(stream_index)
    )


def gs_augment_crop_left_for_event_mode(
    base_seed,
    epoch,
    manifest_index,
    mode,
):

    if mode == GS_AUGMENT_MODE_LEGACY_TEXT:
        min_crop, max_crop = 0, 24

    elif mode == GS_AUGMENT_MODE_SYMMETRIC_FIGURE:
        min_crop, max_crop = 0, 48

    else:
        raise ValueError(
            "Unsupported augmentation mode"
        )

    crop_range = (
        max_crop
        - min_crop
        + 1
    )

    threshold = (
        c_u64(
            0 - crop_range
        )
        % crop_range
    )

    stream_index = 0

    while True:

        value = gs_augment_stateless_value(
            base_seed,
            epoch,
            manifest_index,
            mode,
            stream_index,
        )

        if value >= threshold:
            return (
                min_crop
                + value % crop_range
            )

        stream_index += 1


# Frozen C golden vectors.
augmentation_golden = [
    (0, 0, 0, 0, 7),
    (123, 0, 7663, 1, 36),
    (789, 79, 4224, 1, 25),
    (456, 3, 42, 0, 0),
    (MASK64, MASK64, MASK64, 1, 40),
]


augmentation_golden_pass = all(
    gs_augment_crop_left_for_event_mode(
        seed,
        epoch,
        manifest_index,
        mode,
    ) == expected

    for (
        seed,
        epoch,
        manifest_index,
        mode,
        expected,
    ) in augmentation_golden
)

assert augmentation_golden_pass


# ============================================================
# REAL EVENT LOADERS
# ============================================================

VIEW_COLUMNS = [
    "img_0_5_272",
    "img_1_0_272",
    "img_2_0_272",
    "img_4_0_272",
]


def load_train_event_224(
    manifest_index,
    epoch,
    augmentation_seed=AUGMENTATION_SEED,
):
    """
    Training path:
      4 grayscale 224x272 PNG
      -> shared stateless horizontal crop
      -> float32 (4,224,224) in [0,1].
    """

    event = event_rows[
        manifest_index
    ]

    crop_left = (
        gs_augment_crop_left_for_event_mode(
            augmentation_seed,
            epoch,
            manifest_index,
            GS_AUGMENT_MODE_SYMMETRIC_FIGURE,
        )
    )

    channels = []

    for column in VIEW_COLUMNS:

        path = (
            DATA_DIR
            / event[column]
        )

        with Image.open(path) as image:

            if image.mode != "L":
                raise RuntimeError(
                    f"Non-grayscale PNG: {path}"
                )

            array = np.asarray(
                image,
                dtype=np.uint8,
            ).copy()

        if array.shape != (
            224,
            272,
        ):
            raise RuntimeError(
                f"Unexpected PNG shape: {array.shape}"
            )

        channels.append(
            array[
                :,
                crop_left:
                crop_left + 224
            ]
        )

    tensor_u8 = np.stack(
        channels,
        axis=0,
    )

    tensor = torch.from_numpy(
        tensor_u8.astype(
            np.float32
        )
        / np.float32(255.0)
    )

    return (
        tensor,
        crop_left,
        event["event_id"],
    )


def load_validation_event_224(
    manifest_index,
):
    """
    Validation path:
      frozen central uint8 tensor
      -> float32 (4,224,224) in [0,1].
    """

    row = tensor_rows[
        manifest_index
    ]

    path = (
        DATA_DIR
        / row["tensor_path"]
    )

    values = np.fromfile(
        path,
        dtype=np.uint8,
    )

    if values.size != (
        4 * 224 * 224
    ):
        raise RuntimeError(
            "Invalid central tensor size"
        )

    values = values.reshape(
        4,
        224,
        224,
    )

    tensor = torch.from_numpy(
        values.astype(
            np.float32
        )
        / np.float32(255.0)
    )

    return (
        tensor,
        row["event_id"],
        int(row["crop_left"]),
    )


# ============================================================
# PYTORCH DATASET
# ============================================================

class GravitySpy224Dataset(Dataset):

    def __init__(
        self,
        split_rows,
        split_name,
        epoch=0,
        augmentation_seed=AUGMENTATION_SEED,
    ):

        if split_name not in (
            "train",
            "validation",
        ):
            raise ValueError(
                "Invalid split_name"
            )

        self.split_rows = split_rows
        self.split_name = split_name
        self.epoch = int(epoch)
        self.augmentation_seed = int(
            augmentation_seed
        )

    def __len__(self):
        return len(
            self.split_rows
        )

    def set_epoch(
        self,
        epoch,
    ):
        self.epoch = int(epoch)

    def __getitem__(
        self,
        index,
    ):

        row = self.split_rows[
            index
        ]

        manifest_index = int(
            row["manifest_index"]
        )

        if self.split_name == "train":

            (
                tensor,
                crop_left,
                event_id,
            ) = load_train_event_224(
                manifest_index,
                self.epoch,
                self.augmentation_seed,
            )

        else:

            (
                tensor,
                event_id,
                crop_left,
            ) = load_validation_event_224(
                manifest_index
            )

        return {
            "x":
                tensor,

            "manifest_index":
                manifest_index,

            "event_id":
                event_id,

            "crop_left":
                crop_left,
        }


train_dataset = GravitySpy224Dataset(
    train_rows,
    "train",
    epoch=0,
)

val_dataset = GravitySpy224Dataset(
    val_rows,
    "validation",
)


assert len(train_dataset) == 6866
assert len(val_dataset) == 1717


# ============================================================
# SHUFFLE PROJECT
# ============================================================

RNG_DOMAIN_SHUFFLE = 2
RNG_DOMAIN_LATENT_TRAIN = 4
RNG_DOMAIN_LATENT_VALIDATION = 5

PROJECT_SHUFFLE_SEED = 101112
PROJECT_LATENT_SEED = 456


def c_next_u64(state):
    """Exact PROJECT/C SplitMix64 next-value operation."""

    state = (
        state + SPLITMIX_GAMMA
    ) & MASK64

    value = state

    value = (
        (
            value
            ^ (value >> 30)
        )
        * SPLITMIX_MUL1
    ) & MASK64

    value = (
        (
            value
            ^ (value >> 27)
        )
        * SPLITMIX_MUL2
    ) & MASK64

    value = (
        value
        ^ (value >> 31)
    ) & MASK64

    return state, value


def project_shuffle_bounded_u64(
    state,
    upper_bound_exclusive,
):

    threshold = (
        (
            -upper_bound_exclusive
        )
        & MASK64
    ) % upper_bound_exclusive

    while True:

        state, random_value = (
            c_next_u64(state)
        )

        if random_value >= threshold:

            return (
                state,
                random_value
                % upper_bound_exclusive,
            )


def project_shuffle_permutation(
    count,
    base_seed,
    epoch,
):

    if count == 0:
        return []

    state = c_derive_seed(
        base_seed,
        RNG_DOMAIN_SHUFFLE,
        epoch,
        0,
        0,
    )

    permutation = list(
        range(count)
    )

    for index in range(
        count - 1,
        0,
        -1,
    ):

        state, selected = (
            project_shuffle_bounded_u64(
                state,
                index + 1,
            )
        )

        (
            permutation[index],
            permutation[selected],
        ) = (
            permutation[selected],
            permutation[index],
        )

    return permutation


shuffle_golden_pass = (
    project_shuffle_permutation(
        8,
        101112,
        0,
    ) == [0, 5, 4, 7, 1, 3, 6, 2]

    and project_shuffle_permutation(
        8,
        101112,
        1,
    ) == [2, 3, 1, 4, 0, 5, 6, 7]

    and project_shuffle_permutation(
        8,
        202224,
        0,
    ) == [3, 6, 4, 2, 1, 7, 5, 0]

    and project_shuffle_permutation(
        12,
        101112,
        7,
    ) == [8, 11, 10, 6, 2, 0, 9, 4, 5, 1, 7, 3]
)

assert shuffle_golden_pass


class ProjectEpochSampler(Sampler):

    def __init__(
        self,
        count,
        base_seed=PROJECT_SHUFFLE_SEED,
        epoch=0,
    ):

        self.count = int(count)
        self.base_seed = int(base_seed)
        self.epoch = int(epoch)

    def set_epoch(
        self,
        epoch,
    ):
        self.epoch = int(epoch)

    def __iter__(self):

        return iter(
            project_shuffle_permutation(
                self.count,
                self.base_seed,
                self.epoch,
            )
        )

    def __len__(self):
        return self.count


train_sampler = ProjectEpochSampler(
    len(train_dataset),
    PROJECT_SHUFFLE_SEED,
    epoch=0,
)


def set_project_train_epoch(
    epoch,
):
    """
    The same epoch controls both shuffle and augmentation.
    """

    train_sampler.set_epoch(
        epoch
    )

    train_dataset.set_epoch(
        epoch
    )


# ============================================================
# LATENT EPSILON PROJECT
# ============================================================

TWO_PI = (
    6.283185307179586476925286766559
)


def python_latent_epsilon(
    base_seed,
    domain,
    epoch,
    manifest_index,
    count,
):
    """
    Stateless standard-normal epsilon.

    Box-Muller semantics already checked Python <-> C
    in Notebook 02.
    """

    state = c_derive_seed(
        base_seed,
        domain,
        epoch,
        manifest_index,
        0,
    )

    output = np.empty(
        count,
        dtype=np.float32,
    )

    for index in range(count):

        state, random1 = c_next_u64(
            state
        )

        state, random2 = c_next_u64(
            state
        )

        uniform1 = (
            float(random1 >> 12)
            + 0.5
        ) / 4503599627370496.0

        uniform2 = (
            float(random2 >> 12)
            + 0.5
        ) / 4503599627370496.0

        radius = math.sqrt(
            -2.0
            * math.log(
                uniform1
            )
        )

        angle = (
            TWO_PI
            * uniform2
        )

        output[index] = np.float32(
            radius
            * math.cos(angle)
        )

    return output


# Seed-derivation golden values from the current C tests.
latent_seed_gate = (
    c_derive_seed(
        456,
        RNG_DOMAIN_LATENT_TRAIN,
        3,
        42,
        0,
    ) == 0xBFF9489E190BCDD4

    and c_derive_seed(
        456,
        RNG_DOMAIN_LATENT_VALIDATION,
        0,
        42,
        0,
    ) == 0xB849E85BFE7894E7
)

assert latent_seed_gate


# ============================================================
# FIRST REAL TRAIN BATCH — FROZEN AUDIT
# ============================================================

set_project_train_epoch(0)

train_loader_gate = DataLoader(
    train_dataset,
    batch_size=4,
    sampler=train_sampler,
    num_workers=0,
    drop_last=False,
)

val_loader_gate = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    drop_last=False,
)


train_batch_gate = next(
    iter(train_loader_gate)
)

val_batch_gate = next(
    iter(val_loader_gate)
)


expected_positions = [
    752,
    2659,
    4819,
    4649,
]

expected_manifests = [
    2676,
    6250,
    2396,
    5602,
]

expected_crops = [
    20,
    15,
    19,
    12,
]


first_train_batch_pass = (
    project_shuffle_permutation(
        6866,
        PROJECT_SHUFFLE_SEED,
        0,
    )[:4] == expected_positions

    and train_batch_gate[
        "manifest_index"
    ].tolist() == expected_manifests

    and train_batch_gate[
        "crop_left"
    ].tolist() == expected_crops

    and tuple(
        train_batch_gate["x"].shape
    ) == (4, 4, 224, 224)
)


first_val_batch_pass = (
    val_batch_gate[
        "manifest_index"
    ].tolist()
    == [6864, 1488, 8384, 6585]

    and val_batch_gate[
        "crop_left"
    ].tolist()
    == [24, 24, 24, 24]

    and tuple(
        val_batch_gate["x"].shape
    ) == (4, 4, 224, 224)
)


assert first_train_batch_pass
assert first_val_batch_pass


# ------------------------------------------------------------
# EPSILON GATE ON THE SAME FIRST TRAIN BATCH
# ------------------------------------------------------------

epsilon_a = np.stack(
    [
        python_latent_epsilon(
            PROJECT_LATENT_SEED,
            RNG_DOMAIN_LATENT_TRAIN,
            0,
            manifest_index,
            SPEC.latent_dim,
        )
        for manifest_index
        in expected_manifests
    ]
)

epsilon_b = np.stack(
    [
        python_latent_epsilon(
            PROJECT_LATENT_SEED,
            RNG_DOMAIN_LATENT_TRAIN,
            0,
            manifest_index,
            SPEC.latent_dim,
        )
        for manifest_index
        in expected_manifests
    ]
)


latent_epsilon_gate = (
    epsilon_a.shape
        == (4, 512)

    and np.isfinite(
        epsilon_a
    ).all()

    and np.array_equal(
        epsilon_a.view(np.uint32),
        epsilon_b.view(np.uint32),
    )
)


assert latent_epsilon_gate


# ============================================================
# FINAL REPORT
# ============================================================

print(
    "=== NOTEBOOK 03 DATA / RNG CONTRACT ==="
)

print(
    "SPLIT_CONTRACT_PASS =",
    split_contract_pass,
)

print(
    "AUGMENTATION_GOLDEN_PASS =",
    augmentation_golden_pass,
)

print(
    "SHUFFLE_GOLDEN_PASS =",
    shuffle_golden_pass,
)

print(
    "LATENT_SEED_GATE =",
    latent_seed_gate,
)

print()

print(
    "FIRST_TRAIN_POSITIONS =",
    expected_positions,
)

print(
    "FIRST_TRAIN_MANIFESTS =",
    train_batch_gate[
        "manifest_index"
    ].tolist(),
)

print(
    "FIRST_TRAIN_CROPS =",
    train_batch_gate[
        "crop_left"
    ].tolist(),
)

print(
    "FIRST_TRAIN_BATCH_PASS =",
    first_train_batch_pass,
)

print()

print(
    "FIRST_VAL_MANIFESTS =",
    val_batch_gate[
        "manifest_index"
    ].tolist(),
)

print(
    "FIRST_VAL_CROPS =",
    val_batch_gate[
        "crop_left"
    ].tolist(),
)

print(
    "FIRST_VAL_BATCH_PASS =",
    first_val_batch_pass,
)

print()

print(
    "LATENT_EPSILON_SHAPE =",
    epsilon_a.shape,
)

print(
    "LATENT_EPSILON_REPEATABLE =",
    latent_epsilon_gate,
)

print()

print(
    "MODEL_FORWARD_EXECUTED =",
    False,
)

print(
    "BENCHMARK_STARTED =",
    False,
)

print(
    "OFFICIAL_TRAINING_STARTED =",
    False,
)

print()

print(
    "NOTEBOOK_03_DATA_RNG_CONTRACT_PASS =",
    (
        split_contract_pass
        and augmentation_golden_pass
        and shuffle_golden_pass
        and latent_seed_gate
        and first_train_batch_pass
        and first_val_batch_pass
        and latent_epsilon_gate
    ),
)

In [6]:
# ============================================================
# NOTEBOOK 03 — DATA / SHUFFLE / LATENT RNG CONTRACT
# STEP 03-A2B
#
# Reconstructs contracts already validated in Notebook 01/02.
#
# NO model forward.
# NO benchmark.
# NO training.
# ============================================================

import csv
import math

from PIL import Image

import numpy as np
import torch

from torch.utils.data import (
    Dataset,
    DataLoader,
    Sampler,
)


# ============================================================
# LOCAL DATA / METADATA
# ============================================================

DATA_DIR = LOCAL_DATA_ROOT

OUT272_DIR = LOCAL_OUT272_DIR
CENTRAL224_DIR = LOCAL_CENTRAL224_DIR

METADATA_DIR = LOCAL_TRANSFER_DIR


def read_csv_dicts(path):
    """Read one frozen PROJECT CSV without changing row order."""

    with path.open(
        "r",
        encoding="utf-8",
        newline="",
    ) as handle:

        return list(
            csv.DictReader(handle)
        )


event_rows = read_csv_dicts(
    METADATA_DIR
    / "manifest_events_272.csv"
)

tensor_rows = read_csv_dicts(
    METADATA_DIR
    / "manifest_tensors_224_central.csv"
)

train_rows = read_csv_dicts(
    METADATA_DIR
    / "split_train_indices_seed123.csv"
)

val_rows = read_csv_dicts(
    METADATA_DIR
    / "split_val_indices_seed123.csv"
)


# ------------------------------------------------------------
# FULL SPLIT CONTRACT
# ------------------------------------------------------------

train_indices = {
    int(row["manifest_index"])
    for row in train_rows
}

val_indices = {
    int(row["manifest_index"])
    for row in val_rows
}


split_contract_pass = (
    len(event_rows) == 8583
    and len(tensor_rows) == 8583
    and len(train_rows) == 6866
    and len(val_rows) == 1717
    and len(train_indices) == 6866
    and len(val_indices) == 1717
    and not (
        train_indices
        & val_indices
    )
    and (
        train_indices
        | val_indices
    ) == set(range(8583))
)

assert split_contract_pass


# ============================================================
# STATELESS TRAIN AUGMENTATION
# ============================================================

GS_AUGMENT_MODE_LEGACY_TEXT = 0
GS_AUGMENT_MODE_SYMMETRIC_FIGURE = 1

GS_AUGMENT_STATELESS_V1_DOMAIN_TAG = (
    0x4753415547563121
)

AUGMENTATION_SEED = 789


def c_u64(value):
    return value & MASK64


def gs_augment_stateless_mix64(value):

    value = c_u64(
        value
        + SPLITMIX_GAMMA
    )

    value = c_u64(
        (
            value
            ^ (value >> 30)
        )
        * SPLITMIX_MUL1
    )

    value = c_u64(
        (
            value
            ^ (value >> 27)
        )
        * SPLITMIX_MUL2
    )

    return c_u64(
        value
        ^ (value >> 31)
    )


def gs_augment_stateless_value(
    base_seed,
    epoch,
    manifest_index,
    mode,
    stream_index,
):

    h = gs_augment_stateless_mix64(
        GS_AUGMENT_STATELESS_V1_DOMAIN_TAG
    )

    h = gs_augment_stateless_mix64(
        h ^ c_u64(base_seed)
    )

    h = gs_augment_stateless_mix64(
        h ^ c_u64(epoch)
    )

    h = gs_augment_stateless_mix64(
        h ^ c_u64(manifest_index)
    )

    h = gs_augment_stateless_mix64(
        h ^ c_u64(mode)
    )

    return gs_augment_stateless_mix64(
        h ^ c_u64(stream_index)
    )


def gs_augment_crop_left_for_event_mode(
    base_seed,
    epoch,
    manifest_index,
    mode,
):

    if mode == GS_AUGMENT_MODE_LEGACY_TEXT:
        min_crop, max_crop = 0, 24

    elif mode == GS_AUGMENT_MODE_SYMMETRIC_FIGURE:
        min_crop, max_crop = 0, 48

    else:
        raise ValueError(
            "Unsupported augmentation mode"
        )

    crop_range = (
        max_crop
        - min_crop
        + 1
    )

    threshold = (
        c_u64(
            0 - crop_range
        )
        % crop_range
    )

    stream_index = 0

    while True:

        value = gs_augment_stateless_value(
            base_seed,
            epoch,
            manifest_index,
            mode,
            stream_index,
        )

        if value >= threshold:
            return (
                min_crop
                + value % crop_range
            )

        stream_index += 1


# Frozen C golden vectors.
augmentation_golden = [
    (0, 0, 0, 0, 7),
    (123, 0, 7663, 1, 36),
    (789, 79, 4224, 1, 25),
    (456, 3, 42, 0, 0),
    (MASK64, MASK64, MASK64, 1, 40),
]


augmentation_golden_pass = all(
    gs_augment_crop_left_for_event_mode(
        seed,
        epoch,
        manifest_index,
        mode,
    ) == expected

    for (
        seed,
        epoch,
        manifest_index,
        mode,
        expected,
    ) in augmentation_golden
)

assert augmentation_golden_pass


# ============================================================
# REAL EVENT LOADERS
# ============================================================

VIEW_COLUMNS = [
    "img_0_5_272",
    "img_1_0_272",
    "img_2_0_272",
    "img_4_0_272",
]


def load_train_event_224(
    manifest_index,
    epoch,
    augmentation_seed=AUGMENTATION_SEED,
):
    """
    Training path:
      4 grayscale 224x272 PNG
      -> shared stateless horizontal crop
      -> float32 (4,224,224) in [0,1].
    """

    event = event_rows[
        manifest_index
    ]

    crop_left = (
        gs_augment_crop_left_for_event_mode(
            augmentation_seed,
            epoch,
            manifest_index,
            GS_AUGMENT_MODE_SYMMETRIC_FIGURE,
        )
    )

    channels = []

    for column in VIEW_COLUMNS:

        path = (
            DATA_DIR
            / event[column]
        )

        with Image.open(path) as image:

            if image.mode != "L":
                raise RuntimeError(
                    f"Non-grayscale PNG: {path}"
                )

            array = np.asarray(
                image,
                dtype=np.uint8,
            ).copy()

        if array.shape != (
            224,
            272,
        ):
            raise RuntimeError(
                f"Unexpected PNG shape: {array.shape}"
            )

        channels.append(
            array[
                :,
                crop_left:
                crop_left + 224
            ]
        )

    tensor_u8 = np.stack(
        channels,
        axis=0,
    )

    tensor = torch.from_numpy(
        tensor_u8.astype(
            np.float32
        )
        / np.float32(255.0)
    )

    return (
        tensor,
        crop_left,
        event["event_id"],
    )


def load_validation_event_224(
    manifest_index,
):
    """
    Validation path:
      frozen central uint8 tensor
      -> float32 (4,224,224) in [0,1].
    """

    row = tensor_rows[
        manifest_index
    ]

    path = (
        DATA_DIR
        / row["tensor_path"]
    )

    values = np.fromfile(
        path,
        dtype=np.uint8,
    )

    if values.size != (
        4 * 224 * 224
    ):
        raise RuntimeError(
            "Invalid central tensor size"
        )

    values = values.reshape(
        4,
        224,
        224,
    )

    tensor = torch.from_numpy(
        values.astype(
            np.float32
        )
        / np.float32(255.0)
    )

    return (
        tensor,
        row["event_id"],
        int(row["crop_left"]),
    )


# ============================================================
# PYTORCH DATASET
# ============================================================

class GravitySpy224Dataset(Dataset):

    def __init__(
        self,
        split_rows,
        split_name,
        epoch=0,
        augmentation_seed=AUGMENTATION_SEED,
    ):

        if split_name not in (
            "train",
            "validation",
        ):
            raise ValueError(
                "Invalid split_name"
            )

        self.split_rows = split_rows
        self.split_name = split_name
        self.epoch = int(epoch)
        self.augmentation_seed = int(
            augmentation_seed
        )

    def __len__(self):
        return len(
            self.split_rows
        )

    def set_epoch(
        self,
        epoch,
    ):
        self.epoch = int(epoch)

    def __getitem__(
        self,
        index,
    ):

        row = self.split_rows[
            index
        ]

        manifest_index = int(
            row["manifest_index"]
        )

        if self.split_name == "train":

            (
                tensor,
                crop_left,
                event_id,
            ) = load_train_event_224(
                manifest_index,
                self.epoch,
                self.augmentation_seed,
            )

        else:

            (
                tensor,
                event_id,
                crop_left,
            ) = load_validation_event_224(
                manifest_index
            )

        return {
            "x":
                tensor,

            "manifest_index":
                manifest_index,

            "event_id":
                event_id,

            "crop_left":
                crop_left,
        }


train_dataset = GravitySpy224Dataset(
    train_rows,
    "train",
    epoch=0,
)

val_dataset = GravitySpy224Dataset(
    val_rows,
    "validation",
)


assert len(train_dataset) == 6866
assert len(val_dataset) == 1717


# ============================================================
# SHUFFLE PROJECT
# ============================================================

RNG_DOMAIN_SHUFFLE = 2
RNG_DOMAIN_LATENT_TRAIN = 4
RNG_DOMAIN_LATENT_VALIDATION = 5

PROJECT_SHUFFLE_SEED = 101112
PROJECT_LATENT_SEED = 456


def c_next_u64(state):
    """Exact PROJECT/C SplitMix64 next-value operation."""

    state = (
        state + SPLITMIX_GAMMA
    ) & MASK64

    value = state

    value = (
        (
            value
            ^ (value >> 30)
        )
        * SPLITMIX_MUL1
    ) & MASK64

    value = (
        (
            value
            ^ (value >> 27)
        )
        * SPLITMIX_MUL2
    ) & MASK64

    value = (
        value
        ^ (value >> 31)
    ) & MASK64

    return state, value


def project_shuffle_bounded_u64(
    state,
    upper_bound_exclusive,
):

    threshold = (
        (
            -upper_bound_exclusive
        )
        & MASK64
    ) % upper_bound_exclusive

    while True:

        state, random_value = (
            c_next_u64(state)
        )

        if random_value >= threshold:

            return (
                state,
                random_value
                % upper_bound_exclusive,
            )


def project_shuffle_permutation(
    count,
    base_seed,
    epoch,
):

    if count == 0:
        return []

    state = c_derive_seed(
        base_seed,
        RNG_DOMAIN_SHUFFLE,
        epoch,
        0,
        0,
    )

    permutation = list(
        range(count)
    )

    for index in range(
        count - 1,
        0,
        -1,
    ):

        state, selected = (
            project_shuffle_bounded_u64(
                state,
                index + 1,
            )
        )

        (
            permutation[index],
            permutation[selected],
        ) = (
            permutation[selected],
            permutation[index],
        )

    return permutation


shuffle_golden_pass = (
    project_shuffle_permutation(
        8,
        101112,
        0,
    ) == [0, 5, 4, 7, 1, 3, 6, 2]

    and project_shuffle_permutation(
        8,
        101112,
        1,
    ) == [2, 3, 1, 4, 0, 5, 6, 7]

    and project_shuffle_permutation(
        8,
        202224,
        0,
    ) == [3, 6, 4, 2, 1, 7, 5, 0]

    and project_shuffle_permutation(
        12,
        101112,
        7,
    ) == [8, 11, 10, 6, 2, 0, 9, 4, 5, 1, 7, 3]
)

assert shuffle_golden_pass


class ProjectEpochSampler(Sampler):

    def __init__(
        self,
        count,
        base_seed=PROJECT_SHUFFLE_SEED,
        epoch=0,
    ):

        self.count = int(count)
        self.base_seed = int(base_seed)
        self.epoch = int(epoch)

    def set_epoch(
        self,
        epoch,
    ):
        self.epoch = int(epoch)

    def __iter__(self):

        return iter(
            project_shuffle_permutation(
                self.count,
                self.base_seed,
                self.epoch,
            )
        )

    def __len__(self):
        return self.count


train_sampler = ProjectEpochSampler(
    len(train_dataset),
    PROJECT_SHUFFLE_SEED,
    epoch=0,
)


def set_project_train_epoch(
    epoch,
):
    """
    The same epoch controls both shuffle and augmentation.
    """

    train_sampler.set_epoch(
        epoch
    )

    train_dataset.set_epoch(
        epoch
    )


# ============================================================
# LATENT EPSILON PROJECT
# ============================================================

TWO_PI = (
    6.283185307179586476925286766559
)


def python_latent_epsilon(
    base_seed,
    domain,
    epoch,
    manifest_index,
    count,
):
    """
    Stateless standard-normal epsilon.

    Box-Muller semantics already checked Python <-> C
    in Notebook 02.
    """

    state = c_derive_seed(
        base_seed,
        domain,
        epoch,
        manifest_index,
        0,
    )

    output = np.empty(
        count,
        dtype=np.float32,
    )

    for index in range(count):

        state, random1 = c_next_u64(
            state
        )

        state, random2 = c_next_u64(
            state
        )

        uniform1 = (
            float(random1 >> 12)
            + 0.5
        ) / 4503599627370496.0

        uniform2 = (
            float(random2 >> 12)
            + 0.5
        ) / 4503599627370496.0

        radius = math.sqrt(
            -2.0
            * math.log(
                uniform1
            )
        )

        angle = (
            TWO_PI
            * uniform2
        )

        output[index] = np.float32(
            radius
            * math.cos(angle)
        )

    return output


# Seed-derivation golden values from the current C tests.
latent_seed_gate = (
    c_derive_seed(
        456,
        RNG_DOMAIN_LATENT_TRAIN,
        3,
        42,
        0,
    ) == 0xBFF9489E190BCDD4

    and c_derive_seed(
        456,
        RNG_DOMAIN_LATENT_VALIDATION,
        0,
        42,
        0,
    ) == 0xB849E85BFE7894E7
)

assert latent_seed_gate


# ============================================================
# FIRST REAL TRAIN BATCH — FROZEN AUDIT
# ============================================================

set_project_train_epoch(0)

train_loader_gate = DataLoader(
    train_dataset,
    batch_size=4,
    sampler=train_sampler,
    num_workers=0,
    drop_last=False,
)

val_loader_gate = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    drop_last=False,
)


train_batch_gate = next(
    iter(train_loader_gate)
)

val_batch_gate = next(
    iter(val_loader_gate)
)


expected_positions = [
    752,
    2659,
    4819,
    4649,
]

expected_manifests = [
    2676,
    6250,
    2396,
    5602,
]

expected_crops = [
    20,
    15,
    19,
    12,
]


first_train_batch_pass = (
    project_shuffle_permutation(
        6866,
        PROJECT_SHUFFLE_SEED,
        0,
    )[:4] == expected_positions

    and train_batch_gate[
        "manifest_index"
    ].tolist() == expected_manifests

    and train_batch_gate[
        "crop_left"
    ].tolist() == expected_crops

    and tuple(
        train_batch_gate["x"].shape
    ) == (4, 4, 224, 224)
)


first_val_batch_pass = (
    val_batch_gate[
        "manifest_index"
    ].tolist()
    == [6864, 1488, 8384, 6585]

    and val_batch_gate[
        "crop_left"
    ].tolist()
    == [24, 24, 24, 24]

    and tuple(
        val_batch_gate["x"].shape
    ) == (4, 4, 224, 224)
)


assert first_train_batch_pass
assert first_val_batch_pass


# ------------------------------------------------------------
# EPSILON GATE ON THE SAME FIRST TRAIN BATCH
# ------------------------------------------------------------

epsilon_a = np.stack(
    [
        python_latent_epsilon(
            PROJECT_LATENT_SEED,
            RNG_DOMAIN_LATENT_TRAIN,
            0,
            manifest_index,
            SPEC.latent_dim,
        )
        for manifest_index
        in expected_manifests
    ]
)

epsilon_b = np.stack(
    [
        python_latent_epsilon(
            PROJECT_LATENT_SEED,
            RNG_DOMAIN_LATENT_TRAIN,
            0,
            manifest_index,
            SPEC.latent_dim,
        )
        for manifest_index
        in expected_manifests
    ]
)


latent_epsilon_gate = (
    epsilon_a.shape
        == (4, 512)

    and np.isfinite(
        epsilon_a
    ).all()

    and np.array_equal(
        epsilon_a.view(np.uint32),
        epsilon_b.view(np.uint32),
    )
)


assert latent_epsilon_gate


# ============================================================
# FINAL REPORT
# ============================================================

print(
    "=== NOTEBOOK 03 DATA / RNG CONTRACT ==="
)

print(
    "SPLIT_CONTRACT_PASS =",
    split_contract_pass,
)

print(
    "AUGMENTATION_GOLDEN_PASS =",
    augmentation_golden_pass,
)

print(
    "SHUFFLE_GOLDEN_PASS =",
    shuffle_golden_pass,
)

print(
    "LATENT_SEED_GATE =",
    latent_seed_gate,
)

print()

print(
    "FIRST_TRAIN_POSITIONS =",
    expected_positions,
)

print(
    "FIRST_TRAIN_MANIFESTS =",
    train_batch_gate[
        "manifest_index"
    ].tolist(),
)

print(
    "FIRST_TRAIN_CROPS =",
    train_batch_gate[
        "crop_left"
    ].tolist(),
)

print(
    "FIRST_TRAIN_BATCH_PASS =",
    first_train_batch_pass,
)

print()

print(
    "FIRST_VAL_MANIFESTS =",
    val_batch_gate[
        "manifest_index"
    ].tolist(),
)

print(
    "FIRST_VAL_CROPS =",
    val_batch_gate[
        "crop_left"
    ].tolist(),
)

print(
    "FIRST_VAL_BATCH_PASS =",
    first_val_batch_pass,
)

print()

print(
    "LATENT_EPSILON_SHAPE =",
    epsilon_a.shape,
)

print(
    "LATENT_EPSILON_REPEATABLE =",
    latent_epsilon_gate,
)

print()

print(
    "MODEL_FORWARD_EXECUTED =",
    False,
)

print(
    "BENCHMARK_STARTED =",
    False,
)

print(
    "OFFICIAL_TRAINING_STARTED =",
    False,
)

print()

print(
    "NOTEBOOK_03_DATA_RNG_CONTRACT_PASS =",
    (
        split_contract_pass
        and augmentation_golden_pass
        and shuffle_golden_pass
        and latent_seed_gate
        and first_train_batch_pass
        and first_val_batch_pass
        and latent_epsilon_gate
    ),
)

In [7]:
# ------------------------------------------------------------
# SPLITMIX64 CONSTANTS
#
# Same constants already used by the PROJECT RNG in STEP 03-A2A.
# They are repeated here deliberately so this data/RNG contract
# remains explicit and readable.
# ------------------------------------------------------------

SPLITMIX_GAMMA = 0x9E3779B97F4A7C15
SPLITMIX_MUL1 = 0xBF58476D1CE4E5B9
SPLITMIX_MUL2 = 0x94D049BB133111EB

# Guard against accidental divergence from the model/RNG
# constants reconstructed in STEP 03-A2A.
assert SPLITMIX_GAMMA == MIX_GAMMA
assert SPLITMIX_MUL1 == MIX_MUL1
assert SPLITMIX_MUL2 == MIX_MUL2

In [8]:
# ============================================================
# NOTEBOOK 03 — DATA / SHUFFLE / LATENT RNG CONTRACT
# STEP 03-A2B
#
# Reconstructs contracts already validated in Notebook 01/02.
#
# NO model forward.
# NO benchmark.
# NO training.
# ============================================================

import csv
import math

from PIL import Image

import numpy as np
import torch

from torch.utils.data import (
    Dataset,
    DataLoader,
    Sampler,
)


# ============================================================
# LOCAL DATA / METADATA
# ============================================================

DATA_DIR = LOCAL_DATA_ROOT

OUT272_DIR = LOCAL_OUT272_DIR
CENTRAL224_DIR = LOCAL_CENTRAL224_DIR

METADATA_DIR = LOCAL_TRANSFER_DIR


def read_csv_dicts(path):
    """Read one frozen PROJECT CSV without changing row order."""

    with path.open(
        "r",
        encoding="utf-8",
        newline="",
    ) as handle:

        return list(
            csv.DictReader(handle)
        )


event_rows = read_csv_dicts(
    METADATA_DIR
    / "manifest_events_272.csv"
)

tensor_rows = read_csv_dicts(
    METADATA_DIR
    / "manifest_tensors_224_central.csv"
)

train_rows = read_csv_dicts(
    METADATA_DIR
    / "split_train_indices_seed123.csv"
)

val_rows = read_csv_dicts(
    METADATA_DIR
    / "split_val_indices_seed123.csv"
)


# ------------------------------------------------------------
# FULL SPLIT CONTRACT
# ------------------------------------------------------------

train_indices = {
    int(row["manifest_index"])
    for row in train_rows
}

val_indices = {
    int(row["manifest_index"])
    for row in val_rows
}


split_contract_pass = (
    len(event_rows) == 8583
    and len(tensor_rows) == 8583
    and len(train_rows) == 6866
    and len(val_rows) == 1717
    and len(train_indices) == 6866
    and len(val_indices) == 1717
    and not (
        train_indices
        & val_indices
    )
    and (
        train_indices
        | val_indices
    ) == set(range(8583))
)

assert split_contract_pass


# ============================================================
# STATELESS TRAIN AUGMENTATION
# ============================================================

GS_AUGMENT_MODE_LEGACY_TEXT = 0
GS_AUGMENT_MODE_SYMMETRIC_FIGURE = 1

GS_AUGMENT_STATELESS_V1_DOMAIN_TAG = (
    0x4753415547563121
)

AUGMENTATION_SEED = 789


# ------------------------------------------------------------
# SPLITMIX64 CONSTANTS
#
# Same constants already used by the PROJECT RNG in STEP 03-A2A.
# ------------------------------------------------------------

SPLITMIX_GAMMA = 0x9E3779B97F4A7C15
SPLITMIX_MUL1 = 0xBF58476D1CE4E5B9
SPLITMIX_MUL2 = 0x94D049BB133111EB

assert SPLITMIX_GAMMA == MIX_GAMMA
assert SPLITMIX_MUL1 == MIX_MUL1
assert SPLITMIX_MUL2 == MIX_MUL2


def c_u64(value):
    return value & MASK64

def gs_augment_stateless_mix64(value):

    value = c_u64(
        value
        + SPLITMIX_GAMMA
    )

    value = c_u64(
        (
            value
            ^ (value >> 30)
        )
        * SPLITMIX_MUL1
    )

    value = c_u64(
        (
            value
            ^ (value >> 27)
        )
        * SPLITMIX_MUL2
    )

    return c_u64(
        value
        ^ (value >> 31)
    )


def gs_augment_stateless_value(
    base_seed,
    epoch,
    manifest_index,
    mode,
    stream_index,
):

    h = gs_augment_stateless_mix64(
        GS_AUGMENT_STATELESS_V1_DOMAIN_TAG
    )

    h = gs_augment_stateless_mix64(
        h ^ c_u64(base_seed)
    )

    h = gs_augment_stateless_mix64(
        h ^ c_u64(epoch)
    )

    h = gs_augment_stateless_mix64(
        h ^ c_u64(manifest_index)
    )

    h = gs_augment_stateless_mix64(
        h ^ c_u64(mode)
    )

    return gs_augment_stateless_mix64(
        h ^ c_u64(stream_index)
    )


def gs_augment_crop_left_for_event_mode(
    base_seed,
    epoch,
    manifest_index,
    mode,
):

    if mode == GS_AUGMENT_MODE_LEGACY_TEXT:
        min_crop, max_crop = 0, 24

    elif mode == GS_AUGMENT_MODE_SYMMETRIC_FIGURE:
        min_crop, max_crop = 0, 48

    else:
        raise ValueError(
            "Unsupported augmentation mode"
        )

    crop_range = (
        max_crop
        - min_crop
        + 1
    )

    threshold = (
        c_u64(
            0 - crop_range
        )
        % crop_range
    )

    stream_index = 0

    while True:

        value = gs_augment_stateless_value(
            base_seed,
            epoch,
            manifest_index,
            mode,
            stream_index,
        )

        if value >= threshold:
            return (
                min_crop
                + value % crop_range
            )

        stream_index += 1


# Frozen C golden vectors.
augmentation_golden = [
    (0, 0, 0, 0, 7),
    (123, 0, 7663, 1, 36),
    (789, 79, 4224, 1, 25),
    (456, 3, 42, 0, 0),
    (MASK64, MASK64, MASK64, 1, 40),
]


augmentation_golden_pass = all(
    gs_augment_crop_left_for_event_mode(
        seed,
        epoch,
        manifest_index,
        mode,
    ) == expected

    for (
        seed,
        epoch,
        manifest_index,
        mode,
        expected,
    ) in augmentation_golden
)

assert augmentation_golden_pass


# ============================================================
# REAL EVENT LOADERS
# ============================================================

VIEW_COLUMNS = [
    "img_0_5_272",
    "img_1_0_272",
    "img_2_0_272",
    "img_4_0_272",
]


def load_train_event_224(
    manifest_index,
    epoch,
    augmentation_seed=AUGMENTATION_SEED,
):
    """
    Training path:
      4 grayscale 224x272 PNG
      -> shared stateless horizontal crop
      -> float32 (4,224,224) in [0,1].
    """

    event = event_rows[
        manifest_index
    ]

    crop_left = (
        gs_augment_crop_left_for_event_mode(
            augmentation_seed,
            epoch,
            manifest_index,
            GS_AUGMENT_MODE_SYMMETRIC_FIGURE,
        )
    )

    channels = []

    for column in VIEW_COLUMNS:

        path = (
            DATA_DIR
            / event[column]
        )

        with Image.open(path) as image:

            if image.mode != "L":
                raise RuntimeError(
                    f"Non-grayscale PNG: {path}"
                )

            array = np.asarray(
                image,
                dtype=np.uint8,
            ).copy()

        if array.shape != (
            224,
            272,
        ):
            raise RuntimeError(
                f"Unexpected PNG shape: {array.shape}"
            )

        channels.append(
            array[
                :,
                crop_left:
                crop_left + 224
            ]
        )

    tensor_u8 = np.stack(
        channels,
        axis=0,
    )

    tensor = torch.from_numpy(
        tensor_u8.astype(
            np.float32
        )
        / np.float32(255.0)
    )

    return (
        tensor,
        crop_left,
        event["event_id"],
    )


def load_validation_event_224(
    manifest_index,
):
    """
    Validation path:
      frozen central uint8 tensor
      -> float32 (4,224,224) in [0,1].
    """

    row = tensor_rows[
        manifest_index
    ]

    path = (
        DATA_DIR
        / row["tensor_path"]
    )

    values = np.fromfile(
        path,
        dtype=np.uint8,
    )

    if values.size != (
        4 * 224 * 224
    ):
        raise RuntimeError(
            "Invalid central tensor size"
        )

    values = values.reshape(
        4,
        224,
        224,
    )

    tensor = torch.from_numpy(
        values.astype(
            np.float32
        )
        / np.float32(255.0)
    )

    return (
        tensor,
        row["event_id"],
        int(row["crop_left"]),
    )


# ============================================================
# PYTORCH DATASET
# ============================================================

class GravitySpy224Dataset(Dataset):

    def __init__(
        self,
        split_rows,
        split_name,
        epoch=0,
        augmentation_seed=AUGMENTATION_SEED,
    ):

        if split_name not in (
            "train",
            "validation",
        ):
            raise ValueError(
                "Invalid split_name"
            )

        self.split_rows = split_rows
        self.split_name = split_name
        self.epoch = int(epoch)
        self.augmentation_seed = int(
            augmentation_seed
        )

    def __len__(self):
        return len(
            self.split_rows
        )

    def set_epoch(
        self,
        epoch,
    ):
        self.epoch = int(epoch)

    def __getitem__(
        self,
        index,
    ):

        row = self.split_rows[
            index
        ]

        manifest_index = int(
            row["manifest_index"]
        )

        if self.split_name == "train":

            (
                tensor,
                crop_left,
                event_id,
            ) = load_train_event_224(
                manifest_index,
                self.epoch,
                self.augmentation_seed,
            )

        else:

            (
                tensor,
                event_id,
                crop_left,
            ) = load_validation_event_224(
                manifest_index
            )

        return {
            "x":
                tensor,

            "manifest_index":
                manifest_index,

            "event_id":
                event_id,

            "crop_left":
                crop_left,
        }


train_dataset = GravitySpy224Dataset(
    train_rows,
    "train",
    epoch=0,
)

val_dataset = GravitySpy224Dataset(
    val_rows,
    "validation",
)


assert len(train_dataset) == 6866
assert len(val_dataset) == 1717


# ============================================================
# SHUFFLE PROJECT
# ============================================================

RNG_DOMAIN_SHUFFLE = 2
RNG_DOMAIN_LATENT_TRAIN = 4
RNG_DOMAIN_LATENT_VALIDATION = 5

PROJECT_SHUFFLE_SEED = 101112
PROJECT_LATENT_SEED = 456


def c_next_u64(state):
    """Exact PROJECT/C SplitMix64 next-value operation."""

    state = (
        state + SPLITMIX_GAMMA
    ) & MASK64

    value = state

    value = (
        (
            value
            ^ (value >> 30)
        )
        * SPLITMIX_MUL1
    ) & MASK64

    value = (
        (
            value
            ^ (value >> 27)
        )
        * SPLITMIX_MUL2
    ) & MASK64

    value = (
        value
        ^ (value >> 31)
    ) & MASK64

    return state, value


def project_shuffle_bounded_u64(
    state,
    upper_bound_exclusive,
):

    threshold = (
        (
            -upper_bound_exclusive
        )
        & MASK64
    ) % upper_bound_exclusive

    while True:

        state, random_value = (
            c_next_u64(state)
        )

        if random_value >= threshold:

            return (
                state,
                random_value
                % upper_bound_exclusive,
            )


def project_shuffle_permutation(
    count,
    base_seed,
    epoch,
):

    if count == 0:
        return []

    state = c_derive_seed(
        base_seed,
        RNG_DOMAIN_SHUFFLE,
        epoch,
        0,
        0,
    )

    permutation = list(
        range(count)
    )

    for index in range(
        count - 1,
        0,
        -1,
    ):

        state, selected = (
            project_shuffle_bounded_u64(
                state,
                index + 1,
            )
        )

        (
            permutation[index],
            permutation[selected],
        ) = (
            permutation[selected],
            permutation[index],
        )

    return permutation


shuffle_golden_pass = (
    project_shuffle_permutation(
        8,
        101112,
        0,
    ) == [0, 5, 4, 7, 1, 3, 6, 2]

    and project_shuffle_permutation(
        8,
        101112,
        1,
    ) == [2, 3, 1, 4, 0, 5, 6, 7]

    and project_shuffle_permutation(
        8,
        202224,
        0,
    ) == [3, 6, 4, 2, 1, 7, 5, 0]

    and project_shuffle_permutation(
        12,
        101112,
        7,
    ) == [8, 11, 10, 6, 2, 0, 9, 4, 5, 1, 7, 3]
)

assert shuffle_golden_pass


class ProjectEpochSampler(Sampler):

    def __init__(
        self,
        count,
        base_seed=PROJECT_SHUFFLE_SEED,
        epoch=0,
    ):

        self.count = int(count)
        self.base_seed = int(base_seed)
        self.epoch = int(epoch)

    def set_epoch(
        self,
        epoch,
    ):
        self.epoch = int(epoch)

    def __iter__(self):

        return iter(
            project_shuffle_permutation(
                self.count,
                self.base_seed,
                self.epoch,
            )
        )

    def __len__(self):
        return self.count


train_sampler = ProjectEpochSampler(
    len(train_dataset),
    PROJECT_SHUFFLE_SEED,
    epoch=0,
)


def set_project_train_epoch(
    epoch,
):
    """
    The same epoch controls both shuffle and augmentation.
    """

    train_sampler.set_epoch(
        epoch
    )

    train_dataset.set_epoch(
        epoch
    )


# ============================================================
# LATENT EPSILON PROJECT
# ============================================================

TWO_PI = (
    6.283185307179586476925286766559
)


def python_latent_epsilon(
    base_seed,
    domain,
    epoch,
    manifest_index,
    count,
):
    """
    Stateless standard-normal epsilon.

    Box-Muller semantics already checked Python <-> C
    in Notebook 02.
    """

    state = c_derive_seed(
        base_seed,
        domain,
        epoch,
        manifest_index,
        0,
    )

    output = np.empty(
        count,
        dtype=np.float32,
    )

    for index in range(count):

        state, random1 = c_next_u64(
            state
        )

        state, random2 = c_next_u64(
            state
        )

        uniform1 = (
            float(random1 >> 12)
            + 0.5
        ) / 4503599627370496.0

        uniform2 = (
            float(random2 >> 12)
            + 0.5
        ) / 4503599627370496.0

        radius = math.sqrt(
            -2.0
            * math.log(
                uniform1
            )
        )

        angle = (
            TWO_PI
            * uniform2
        )

        output[index] = np.float32(
            radius
            * math.cos(angle)
        )

    return output


# Seed-derivation golden values from the current C tests.
latent_seed_gate = (
    c_derive_seed(
        456,
        RNG_DOMAIN_LATENT_TRAIN,
        3,
        42,
        0,
    ) == 0xBFF9489E190BCDD4

    and c_derive_seed(
        456,
        RNG_DOMAIN_LATENT_VALIDATION,
        0,
        42,
        0,
    ) == 0xB849E85BFE7894E7
)

assert latent_seed_gate


# ============================================================
# FIRST REAL TRAIN BATCH — FROZEN AUDIT
# ============================================================

set_project_train_epoch(0)

train_loader_gate = DataLoader(
    train_dataset,
    batch_size=4,
    sampler=train_sampler,
    num_workers=0,
    drop_last=False,
)

val_loader_gate = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    drop_last=False,
)


train_batch_gate = next(
    iter(train_loader_gate)
)

val_batch_gate = next(
    iter(val_loader_gate)
)


expected_positions = [
    752,
    2659,
    4819,
    4649,
]

expected_manifests = [
    2676,
    6250,
    2396,
    5602,
]

expected_crops = [
    20,
    15,
    19,
    12,
]


first_train_batch_pass = (
    project_shuffle_permutation(
        6866,
        PROJECT_SHUFFLE_SEED,
        0,
    )[:4] == expected_positions

    and train_batch_gate[
        "manifest_index"
    ].tolist() == expected_manifests

    and train_batch_gate[
        "crop_left"
    ].tolist() == expected_crops

    and tuple(
        train_batch_gate["x"].shape
    ) == (4, 4, 224, 224)
)


first_val_batch_pass = (
    val_batch_gate[
        "manifest_index"
    ].tolist()
    == [6864, 1488, 8384, 6585]

    and val_batch_gate[
        "crop_left"
    ].tolist()
    == [24, 24, 24, 24]

    and tuple(
        val_batch_gate["x"].shape
    ) == (4, 4, 224, 224)
)


assert first_train_batch_pass
assert first_val_batch_pass


# ------------------------------------------------------------
# EPSILON GATE ON THE SAME FIRST TRAIN BATCH
# ------------------------------------------------------------

epsilon_a = np.stack(
    [
        python_latent_epsilon(
            PROJECT_LATENT_SEED,
            RNG_DOMAIN_LATENT_TRAIN,
            0,
            manifest_index,
            SPEC.latent_dim,
        )
        for manifest_index
        in expected_manifests
    ]
)

epsilon_b = np.stack(
    [
        python_latent_epsilon(
            PROJECT_LATENT_SEED,
            RNG_DOMAIN_LATENT_TRAIN,
            0,
            manifest_index,
            SPEC.latent_dim,
        )
        for manifest_index
        in expected_manifests
    ]
)


latent_epsilon_gate = (
    epsilon_a.shape
        == (4, 512)

    and np.isfinite(
        epsilon_a
    ).all()

    and np.array_equal(
        epsilon_a.view(np.uint32),
        epsilon_b.view(np.uint32),
    )
)


assert latent_epsilon_gate


# ============================================================
# FINAL REPORT
# ============================================================

print(
    "=== NOTEBOOK 03 DATA / RNG CONTRACT ==="
)

print(
    "SPLIT_CONTRACT_PASS =",
    split_contract_pass,
)

print(
    "AUGMENTATION_GOLDEN_PASS =",
    augmentation_golden_pass,
)

print(
    "SHUFFLE_GOLDEN_PASS =",
    shuffle_golden_pass,
)

print(
    "LATENT_SEED_GATE =",
    latent_seed_gate,
)

print()

print(
    "FIRST_TRAIN_POSITIONS =",
    expected_positions,
)

print(
    "FIRST_TRAIN_MANIFESTS =",
    train_batch_gate[
        "manifest_index"
    ].tolist(),
)

print(
    "FIRST_TRAIN_CROPS =",
    train_batch_gate[
        "crop_left"
    ].tolist(),
)

print(
    "FIRST_TRAIN_BATCH_PASS =",
    first_train_batch_pass,
)

print()

print(
    "FIRST_VAL_MANIFESTS =",
    val_batch_gate[
        "manifest_index"
    ].tolist(),
)

print(
    "FIRST_VAL_CROPS =",
    val_batch_gate[
        "crop_left"
    ].tolist(),
)

print(
    "FIRST_VAL_BATCH_PASS =",
    first_val_batch_pass,
)

print()

print(
    "LATENT_EPSILON_SHAPE =",
    epsilon_a.shape,
)

print(
    "LATENT_EPSILON_REPEATABLE =",
    latent_epsilon_gate,
)

print()

print(
    "MODEL_FORWARD_EXECUTED =",
    False,
)

print(
    "BENCHMARK_STARTED =",
    False,
)

print(
    "OFFICIAL_TRAINING_STARTED =",
    False,
)

print()

print(
    "NOTEBOOK_03_DATA_RNG_CONTRACT_PASS =",
    (
        split_contract_pass
        and augmentation_golden_pass
        and shuffle_golden_pass
        and latent_seed_gate
        and first_train_batch_pass
        and first_val_batch_pass
        and latent_epsilon_gate
    ),
)

=== NOTEBOOK 03 DATA / RNG CONTRACT ===
SPLIT_CONTRACT_PASS = True
AUGMENTATION_GOLDEN_PASS = True
SHUFFLE_GOLDEN_PASS = True
LATENT_SEED_GATE = True

FIRST_TRAIN_POSITIONS = [752, 2659, 4819, 4649]
FIRST_TRAIN_MANIFESTS = [2676, 6250, 2396, 5602]
FIRST_TRAIN_CROPS = [20, 15, 19, 12]
FIRST_TRAIN_BATCH_PASS = True

FIRST_VAL_MANIFESTS = [6864, 1488, 8384, 6585]
FIRST_VAL_CROPS = [24, 24, 24, 24]
FIRST_VAL_BATCH_PASS = True

LATENT_EPSILON_SHAPE = (4, 512)
LATENT_EPSILON_REPEATABLE = True

MODEL_FORWARD_EXECUTED = False
BENCHMARK_STARTED = False
OFFICIAL_TRAINING_STARTED = False

NOTEBOOK_03_DATA_RNG_CONTRACT_PASS = True


## STEP 03-B0 — Pool congelato per il benchmark

Il benchmark dei batch size deve confrontare configurazioni differenti sugli stessi dati e nello stesso ordine.

Viene quindi preparato un pool fisso costituito dai primi 128 eventi del train nell'ordine PROJECT dell'epoch 0.

Per ciascun evento vengono congelati:

- posizione nello split;
- `manifest_index`;
- crop stateless dell'epoch 0;
- tensore reale `4×224×224` float32;
- epsilon latente PROJECT corrispondente.

I candidati `batch=32`, `64` e `128` utilizzeranno tutti questi stessi 128 eventi, nello stesso ordine.

### Precisione numerica del benchmark

Il benchmark viene eseguito in **float32 puro**, senza AMP e senza TF32.

Questa scelta evita di introdurre sul moderno hardware A100 una precisione TensorFloat-32 che non era disponibile sulle RTX 2080 Ti utilizzate da Sakai.

Il benchmark non è training scientifico e non produce checkpoint ufficiali.

In [9]:
# ============================================================
# NOTEBOOK 03 — FROZEN REAL-DATA BENCHMARK POOL
# STEP 03-B0
#
# Prepare exactly 128 real Gravity Spy train samples.
#
# Same samples + same order + same augmentation + same epsilon
# will be used for batch-size candidates 32 / 64 / 128.
#
# NO optimizer step in this cell.
# NO benchmark timing yet.
# NO official training.
# ============================================================

import hashlib
import numpy as np
import torch


# ============================================================
# NUMERICAL POLICY FOR BENCHMARK + FUTURE TRAINING TESTS
#
# Sakai used RTX 2080 Ti hardware, which did not provide the
# A100 TensorFloat-32 execution mode.
#
# Therefore PROJECT benchmark policy remains genuine FP32:
#   - no AMP
#   - no TF32
# ============================================================

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

torch.set_float32_matmul_precision(
    "highest"
)

# Conservative cuDNN policy for reproducibility.
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True


BENCHMARK_DTYPE = torch.float32
BENCHMARK_AMP_ENABLED = False

BENCHMARK_EPOCH = 0
BENCHMARK_POOL_SIZE = 128


# Current runtime must still be the A100 selected for this
# benchmark campaign.
gpu_name_now = torch.cuda.get_device_name(0)

assert "A100" in gpu_name_now.upper()


# ============================================================
# FREEZE FIRST 128 PROJECT-SHUFFLED POSITIONS
# ============================================================

set_project_train_epoch(
    BENCHMARK_EPOCH
)

benchmark_positions = (
    project_shuffle_permutation(
        len(train_dataset),
        PROJECT_SHUFFLE_SEED,
        BENCHMARK_EPOCH,
    )[:BENCHMARK_POOL_SIZE]
)


assert len(
    benchmark_positions
) == BENCHMARK_POOL_SIZE

assert len(
    set(benchmark_positions)
) == BENCHMARK_POOL_SIZE


# ============================================================
# LOAD THE 128 REAL TRAIN EVENTS ON CPU
#
# This loading is intentionally NOT timed.
# Loader performance will be benchmarked separately later.
# ============================================================

benchmark_x_rows = []
benchmark_manifest_indices = []
benchmark_crop_left = []


print(
    "Loading frozen 128-event benchmark pool..."
)


for pool_index, split_position in enumerate(
    benchmark_positions
):

    item = train_dataset[
        split_position
    ]

    benchmark_x_rows.append(
        item["x"]
    )

    benchmark_manifest_indices.append(
        int(
            item["manifest_index"]
        )
    )

    benchmark_crop_left.append(
        int(
            item["crop_left"]
        )
    )


benchmark_x_cpu = torch.stack(
    benchmark_x_rows,
    dim=0,
).contiguous()


# ------------------------------------------------------------
# INPUT CONTRACT
# ------------------------------------------------------------

benchmark_input_gate = (
    tuple(
        benchmark_x_cpu.shape
    ) == (
        128,
        4,
        224,
        224,
    )

    and benchmark_x_cpu.dtype
        == torch.float32

    and torch.isfinite(
        benchmark_x_cpu
    ).all().item()

    and float(
        benchmark_x_cpu.min()
    ) >= 0.0

    and float(
        benchmark_x_cpu.max()
    ) <= 1.0
)


assert benchmark_input_gate


# ============================================================
# EXTERNAL STATELESS LATENT EPSILON
# ============================================================

benchmark_epsilon_np = np.stack(
    [
        python_latent_epsilon(
            PROJECT_LATENT_SEED,
            RNG_DOMAIN_LATENT_TRAIN,
            BENCHMARK_EPOCH,
            manifest_index,
            SPEC.latent_dim,
        )

        for manifest_index
        in benchmark_manifest_indices
    ],
    axis=0,
)


benchmark_epsilon_cpu = (
    torch.from_numpy(
        benchmark_epsilon_np
    )
    .contiguous()
)


benchmark_epsilon_gate = (
    tuple(
        benchmark_epsilon_cpu.shape
    ) == (
        128,
        512,
    )

    and benchmark_epsilon_cpu.dtype
        == torch.float32

    and torch.isfinite(
        benchmark_epsilon_cpu
    ).all().item()
)


assert benchmark_epsilon_gate


# ============================================================
# GOLDEN PREFIX
#
# Must reproduce the already validated first train batch.
# ============================================================

prefix_gate = (
    benchmark_positions[:4]
        == [
            752,
            2659,
            4819,
            4649,
        ]

    and benchmark_manifest_indices[:4]
        == [
            2676,
            6250,
            2396,
            5602,
        ]

    and benchmark_crop_left[:4]
        == [
            20,
            15,
            19,
            12,
        ]
)


assert prefix_gate


# ============================================================
# RUN FINGERPRINTS
#
# These hashes freeze exactly the benchmark input assembled
# in this A100 runtime. They are diagnostic/reproducibility
# fingerprints, not pre-published expected values.
# ============================================================

benchmark_x_sha256 = hashlib.sha256(
    benchmark_x_cpu
    .numpy()
    .tobytes(
        order="C"
    )
).hexdigest()


benchmark_epsilon_sha256 = hashlib.sha256(
    benchmark_epsilon_cpu
    .numpy()
    .tobytes(
        order="C"
    )
).hexdigest()


benchmark_identity_text = (
    ",".join(
        str(value)
        for value
        in benchmark_manifest_indices
    )
    +
    "|"
    +
    ",".join(
        str(value)
        for value
        in benchmark_crop_left
    )
)


benchmark_identity_sha256 = (
    hashlib.sha256(
        benchmark_identity_text.encode(
            "utf-8"
        )
    ).hexdigest()
)


# ============================================================
# VERIFY MODEL REFERENCE IS STILL UNTOUCHED
# ============================================================

reference_model_unchanged = (
    project_parameter_sha256(
        model_reference
    )
    == FROZEN_INIT_SHA256
)


assert reference_model_unchanged


# ============================================================
# REPORT
# ============================================================

print()
print(
    "=== FROZEN BENCHMARK POOL ==="
)

print(
    "GPU =",
    gpu_name_now,
)

print(
    "POOL_SIZE =",
    BENCHMARK_POOL_SIZE,
)

print(
    "EPOCH =",
    BENCHMARK_EPOCH,
)

print()

print(
    "FIRST_4_POSITIONS =",
    benchmark_positions[:4],
)

print(
    "FIRST_4_MANIFESTS =",
    benchmark_manifest_indices[:4],
)

print(
    "FIRST_4_CROPS =",
    benchmark_crop_left[:4],
)

print(
    "GOLDEN_PREFIX_PASS =",
    prefix_gate,
)

print()

print(
    "INPUT_SHAPE =",
    tuple(
        benchmark_x_cpu.shape
    ),
)

print(
    "INPUT_CONTRACT_PASS =",
    benchmark_input_gate,
)

print(
    "EPSILON_SHAPE =",
    tuple(
        benchmark_epsilon_cpu.shape
    ),
)

print(
    "EPSILON_CONTRACT_PASS =",
    benchmark_epsilon_gate,
)

print()

print(
    "BENCHMARK_INPUT_SHA256 =",
    benchmark_x_sha256,
)

print(
    "BENCHMARK_EPSILON_SHA256 =",
    benchmark_epsilon_sha256,
)

print(
    "BENCHMARK_IDENTITY_SHA256 =",
    benchmark_identity_sha256,
)

print()

print(
    "DTYPE =",
    BENCHMARK_DTYPE,
)

print(
    "AMP_ENABLED =",
    BENCHMARK_AMP_ENABLED,
)

print(
    "CUDA_MATMUL_TF32 =",
    torch.backends.cuda.matmul.allow_tf32,
)

print(
    "CUDNN_TF32 =",
    torch.backends.cudnn.allow_tf32,
)

print(
    "CUDNN_BENCHMARK =",
    torch.backends.cudnn.benchmark,
)

print(
    "CUDNN_DETERMINISTIC =",
    torch.backends.cudnn.deterministic,
)

print()

print(
    "REFERENCE_MODEL_UNCHANGED =",
    reference_model_unchanged,
)

print(
    "BENCHMARK_TIMING_STARTED =",
    False,
)

print(
    "OPTIMIZER_EXECUTED =",
    False,
)

print(
    "OFFICIAL_TRAINING_STARTED =",
    False,
)

print()

print(
    "NOTEBOOK_03_BENCHMARK_POOL_PASS =",
    (
        benchmark_input_gate
        and benchmark_epsilon_gate
        and prefix_gate
        and reference_model_unchanged
    ),
)

Loading frozen 128-event benchmark pool...

=== FROZEN BENCHMARK POOL ===
GPU = NVIDIA A100-SXM4-80GB
POOL_SIZE = 128
EPOCH = 0

FIRST_4_POSITIONS = [752, 2659, 4819, 4649]
FIRST_4_MANIFESTS = [2676, 6250, 2396, 5602]
FIRST_4_CROPS = [20, 15, 19, 12]
GOLDEN_PREFIX_PASS = True

INPUT_SHAPE = (128, 4, 224, 224)
INPUT_CONTRACT_PASS = True
EPSILON_SHAPE = (128, 512)
EPSILON_CONTRACT_PASS = True

BENCHMARK_INPUT_SHA256 = deee45b1010d5cf4180b211565ec9ee1343c0705ffc3dd75eeba0b355b62bab4
BENCHMARK_EPSILON_SHA256 = 0cd2c08b95d2e9fd5b89a5f7e40af2d4cd4b502555919b30792f2558ad5dad97
BENCHMARK_IDENTITY_SHA256 = 71575a928c45b1ecf48bacd1b8bb85d095913f36c7723ebbe17a4e6b22a308d5

DTYPE = torch.float32
AMP_ENABLED = False
CUDA_MATMUL_TF32 = False
CUDNN_TF32 = False
CUDNN_BENCHMARK = False
CUDNN_DETERMINISTIC = True

REFERENCE_MODEL_UNCHANGED = True
BENCHMARK_TIMING_STARTED = False
OPTIMIZER_EXECUTED = False
OFFICIAL_TRAINING_STARTED = False

NOTEBOOK_03_BENCHMARK_POOL_PASS = True


## STEP 03-B1 — Benchmark fair del batch size sulla A100

Vengono confrontati i batch size candidati:

- `32`
- `64`
- `128`

Il valore `128` è il target pubblicato da Sakai e viene quindi preferito se risulta sostenibile.

### Regole del confronto

Per ogni candidato:

- stesso pool congelato di 128 eventi reali;
- stesso ordine degli eventi;
- stessa augmentation già materializzata;
- stesso epsilon stateless;
- stessa inizializzazione PROJECT;
- optimizer Adam nuovo;
- `float32` puro;
- AMP disabilitato;
- TF32 disabilitato;
- un warm-up su modello separato;
- modello di misura ricreato da zero dopo il warm-up;
- forward + loss + backward + Adam reali.

Il benchmark misura esclusivamente la parte neurale sulla GPU: il pool è già materializzato e viene trasferito sulla GPU **prima** del cronometro. Il loader verrà misurato separatamente.

Le tre configurazioni processano lo stesso pool di 128 eventi, ma con il numero naturale di optimizer step:

- batch 32 → 4 step per pool;
- batch 64 → 2 step;
- batch 128 → 1 step.

Questo benchmark serve a scegliere il regime computazionale e non confronta traiettorie di apprendimento.

In [10]:
# ============================================================
# NOTEBOOK 03 — FAIR GPU BATCH BENCHMARK
# STEP 03-B1
#
# Candidates:
#   32 / 64 / 128
#
# Same frozen 128-event real-data pool.
#
# Measures:
#   forward + loss + backward + Adam
#
# EXCLUDES:
#   PNG loading
#   augmentation computation
#   CPU -> GPU transfer
#
# NO official training.
# ============================================================

import gc
import math
import statistics
import time

import torch


BENCHMARK_BATCH_CANDIDATES = [
    32,
    64,
    128,
]

BENCHMARK_MEASURED_REPEATS = 3


# ============================================================
# HARD NUMERICAL POLICY
# ============================================================

assert BENCHMARK_AMP_ENABLED is False
assert torch.backends.cuda.matmul.allow_tf32 is False
assert torch.backends.cudnn.allow_tf32 is False

assert benchmark_x_sha256 == (
    "deee45b1010d5cf4180b211565ec9ee1"
    "343c0705ffc3dd75eeba0b355b62bab4"
)

assert benchmark_epsilon_sha256 == (
    "0cd2c08b95d2e9fd5b89a5f7e40af2"
    "d4cd4b502555919b30792f2558ad5dad97"
)

assert benchmark_identity_sha256 == (
    "71575a928c45b1ecf48bacd1b8bb85d0"
    "95913f36c7723ebbe17a4e6b22a308d5"
)


# ============================================================
# PRELOAD THE FROZEN POOL TO GPU
#
# Transfer is deliberately outside benchmark timing.
# ============================================================

device = torch.device(
    "cuda:0"
)

benchmark_x_gpu = (
    benchmark_x_cpu
    .to(
        device=device,
        dtype=torch.float32,
    )
)

benchmark_epsilon_gpu = (
    benchmark_epsilon_cpu
    .to(
        device=device,
        dtype=torch.float32,
    )
)

torch.cuda.synchronize()


assert tuple(
    benchmark_x_gpu.shape
) == (
    128,
    4,
    224,
    224,
)

assert tuple(
    benchmark_epsilon_gpu.shape
) == (
    128,
    512,
)


# ============================================================
# MODEL / OPTIMIZER FACTORY
#
# Every benchmark candidate starts from exactly the same
# PROJECT parameter initialization and a fresh Adam state.
# ============================================================

def make_benchmark_model_and_optimizer():

    benchmark_model = (
        Sakai224FullVAE(
            SPEC
        )
    )

    # Initialization is performed on CPU exactly as validated.
    initialize_project_c_exact_v1(
        benchmark_model,
        PROJECT_INITIALIZATION_SEED,
    )

    initial_hash = (
        project_parameter_sha256(
            benchmark_model
        )
    )

    if initial_hash != FROZEN_INIT_SHA256:
        raise RuntimeError(
            "Benchmark model initialization mismatch"
        )

    benchmark_model = (
        benchmark_model.to(
            device=device,
            dtype=torch.float32,
        )
    )

    benchmark_model.train()

    optimizer = torch.optim.Adam(
        benchmark_model.parameters(),
        lr=5.0e-4,
        betas=(0.9, 0.999),
        eps=1.0e-8,
        weight_decay=0.0,
        amsgrad=False,
        foreach=False,
        fused=False,
    )

    return (
        benchmark_model,
        optimizer,
    )


# ============================================================
# PROCESS ONE COMPLETE 128-EVENT POOL
# ============================================================

def run_benchmark_pool(
    benchmark_model,
    optimizer,
    batch_size,
):
    """
    Process all 128 frozen events using the requested batch.

    Returns:
        number of optimizer steps,
        final loss.
    """

    step_count = 0
    final_loss = None

    for start in range(
        0,
        BENCHMARK_POOL_SIZE,
        batch_size,
    ):

        end = min(
            start + batch_size,
            BENCHMARK_POOL_SIZE,
        )

        x_batch = benchmark_x_gpu[
            start:end
        ]

        epsilon_batch = (
            benchmark_epsilon_gpu[
                start:end
            ]
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        (
            logits,
            mu,
            logvar,
            z,
        ) = benchmark_model(
            x_batch,
            epsilon=epsilon_batch,
            mean_mode=False,
        )

        stats = sakai_vae_loss(
            logits,
            x_batch,
            mu,
            logvar,
            beta_kl=SPEC.beta_kl,
        )

        loss = stats["loss"]

        if not torch.isfinite(
            loss
        ).item():
            raise RuntimeError(
                "Non-finite benchmark loss"
            )

        loss.backward()

        optimizer.step()

        final_loss = float(
            loss.detach()
        )

        step_count += 1

    return (
        step_count,
        final_loss,
    )


# ============================================================
# ONE CANDIDATE
# ============================================================

def benchmark_candidate(
    batch_size,
):

    # --------------------------------------------------------
    # WARM-UP ON A DISPOSABLE MODEL
    #
    # Kernel/context initialization must not contaminate the
    # measured model state.
    # --------------------------------------------------------

    warmup_model = None
    warmup_optimizer = None

    measured_model = None
    measured_optimizer = None

    try:

        (
            warmup_model,
            warmup_optimizer,
        ) = make_benchmark_model_and_optimizer()

        # One real optimizer step is enough to initialize the
        # relevant CUDA/cuDNN/Adam paths for this batch shape.
        x_warm = benchmark_x_gpu[
            :batch_size
        ]

        epsilon_warm = (
            benchmark_epsilon_gpu[
                :batch_size
            ]
        )

        warmup_optimizer.zero_grad(
            set_to_none=True
        )

        (
            logits,
            mu,
            logvar,
            z,
        ) = warmup_model(
            x_warm,
            epsilon=epsilon_warm,
            mean_mode=False,
        )

        warm_stats = sakai_vae_loss(
            logits,
            x_warm,
            mu,
            logvar,
            beta_kl=SPEC.beta_kl,
        )

        warm_stats[
            "loss"
        ].backward()

        warmup_optimizer.step()

        torch.cuda.synchronize()


        # ----------------------------------------------------
        # DISCARD WARM-UP MODEL
        # ----------------------------------------------------

        del warmup_model
        del warmup_optimizer

        warmup_model = None
        warmup_optimizer = None

        gc.collect()
        torch.cuda.empty_cache()


        # ----------------------------------------------------
        # FRESH MEASURED MODEL
        # ----------------------------------------------------

        (
            measured_model,
            measured_optimizer,
        ) = make_benchmark_model_and_optimizer()


        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()


        wall_times = []
        optimizer_steps = []
        final_losses = []


        # ----------------------------------------------------
        # MEASURE REPEATED COMPLETE POOLS
        # ----------------------------------------------------

        for repeat in range(
            BENCHMARK_MEASURED_REPEATS
        ):

            torch.cuda.synchronize()

            start_time = (
                time.perf_counter()
            )

            (
                step_count,
                final_loss,
            ) = run_benchmark_pool(
                measured_model,
                measured_optimizer,
                batch_size,
            )

            torch.cuda.synchronize()

            elapsed = (
                time.perf_counter()
                - start_time
            )

            wall_times.append(
                elapsed
            )

            optimizer_steps.append(
                step_count
            )

            final_losses.append(
                final_loss
            )


        # ----------------------------------------------------
        # FINAL NUMERIC VALIDITY
        # ----------------------------------------------------

        parameters_finite = all(
            torch.isfinite(
                parameter
            ).all().item()

            for parameter
            in measured_model.parameters()
        )

        if not parameters_finite:
            raise RuntimeError(
                "Non-finite model parameter after benchmark"
            )


        peak_allocated_gib = (
            torch.cuda.max_memory_allocated()
            / (1024 ** 3)
        )

        peak_reserved_gib = (
            torch.cuda.max_memory_reserved()
            / (1024 ** 3)
        )


        median_pool_seconds = (
            statistics.median(
                wall_times
            )
        )

        throughput = (
            BENCHMARK_POOL_SIZE
            / median_pool_seconds
        )


        expected_steps = math.ceil(
            BENCHMARK_POOL_SIZE
            / batch_size
        )


        step_count_gate = all(
            value == expected_steps
            for value
            in optimizer_steps
        )


        assert step_count_gate


        return {
            "batch_size":
                batch_size,

            "status":
                "PASS",

            "median_pool_seconds":
                median_pool_seconds,

            "throughput_samples_per_s":
                throughput,

            "peak_allocated_gib":
                peak_allocated_gib,

            "peak_reserved_gib":
                peak_reserved_gib,

            "optimizer_steps_per_pool":
                expected_steps,

            "wall_times":
                wall_times,

            "final_losses":
                final_losses,

            "parameters_finite":
                parameters_finite,
        }


    except torch.cuda.OutOfMemoryError:

        return {
            "batch_size":
                batch_size,

            "status":
                "OOM",
        }


    finally:

        for object_name in [
            "warmup_model",
            "warmup_optimizer",
            "measured_model",
            "measured_optimizer",
        ]:

            obj = locals().get(
                object_name,
                None,
            )

            if obj is not None:
                del obj

        gc.collect()
        torch.cuda.empty_cache()


# ============================================================
# RUN 32 -> 64 -> 128
# ============================================================

benchmark_results = []


print(
    "=== A100 FAIR BATCH BENCHMARK ==="
)


for batch_size in (
    BENCHMARK_BATCH_CANDIDATES
):

    print()
    print(
        "Running batch",
        batch_size,
        "..."
    )

    result = benchmark_candidate(
        batch_size
    )

    benchmark_results.append(
        result
    )

    print(
        "STATUS =",
        result["status"],
    )

    if result[
        "status"
    ] == "PASS":

        print(
            "MEDIAN_POOL_SECONDS =",
            round(
                result[
                    "median_pool_seconds"
                ],
                4,
            ),
        )

        print(
            "THROUGHPUT_SAMPLES_PER_S =",
            round(
                result[
                    "throughput_samples_per_s"
                ],
                3,
            ),
        )

        print(
            "PEAK_ALLOCATED_GIB =",
            round(
                result[
                    "peak_allocated_gib"
                ],
                3,
            ),
        )

        print(
            "PEAK_RESERVED_GIB =",
            round(
                result[
                    "peak_reserved_gib"
                ],
                3,
            ),
        )

        print(
            "OPTIMIZER_STEPS_PER_POOL =",
            result[
                "optimizer_steps_per_pool"
            ],
        )


# ============================================================
# SUMMARY
# ============================================================

passing_results = [
    result
    for result
    in benchmark_results
    if result["status"] == "PASS"
]


assert passing_results


paper_batch_128_supported = any(
    result[
        "batch_size"
    ] == 128

    and result[
        "status"
    ] == "PASS"

    for result
    in benchmark_results
)


# Fastest measured throughput among candidates.
fastest_result = max(
    passing_results,
    key=lambda item:
        item[
            "throughput_samples_per_s"
        ],
)


# PROJECT decision rule:
# use paper batch=128 when it fits and remains numerically valid.
recommended_batch = (
    128
    if paper_batch_128_supported
    else fastest_result[
        "batch_size"
    ]
)


# Rough neural-only epoch estimate.
recommended_result = next(
    result
    for result
    in passing_results
    if result[
        "batch_size"
    ] == recommended_batch
)


estimated_train_epoch_seconds_neural_only = (
    6866
    / recommended_result[
        "throughput_samples_per_s"
    ]
)


print()
print(
    "=== BENCHMARK SUMMARY ==="
)

for result in benchmark_results:

    print(
        "BATCH",
        result["batch_size"],
        "STATUS",
        result["status"],
    )


print()

print(
    "PAPER_BATCH_128_SUPPORTED =",
    paper_batch_128_supported,
)

print(
    "FASTEST_BATCH =",
    fastest_result[
        "batch_size"
    ],
)

print(
    "RECOMMENDED_BATCH =",
    recommended_batch,
)

print(
    "ESTIMATED_TRAIN_EPOCH_SECONDS_NEURAL_ONLY =",
    round(
        estimated_train_epoch_seconds_neural_only,
        3,
    ),
)

print()

print(
    "OFFICIAL_TRAINING_STARTED =",
    False,
)

print(
    "NOTEBOOK_03_GPU_BENCHMARK_PASS =",
    all(
        result["status"]
        in ("PASS", "OOM")

        for result
        in benchmark_results
    )
)

=== A100 FAIR BATCH BENCHMARK ===

Running batch 32 ...
STATUS = PASS
MEDIAN_POOL_SECONDS = 0.1567
THROUGHPUT_SAMPLES_PER_S = 816.72
PEAK_ALLOCATED_GIB = 3.343
PEAK_RESERVED_GIB = 4.828
OPTIMIZER_STEPS_PER_POOL = 4

Running batch 64 ...
STATUS = PASS
MEDIAN_POOL_SECONDS = 0.1428
THROUGHPUT_SAMPLES_PER_S = 896.402
PEAK_ALLOCATED_GIB = 6.253
PEAK_RESERVED_GIB = 6.932
OPTIMIZER_STEPS_PER_POOL = 2

Running batch 128 ...
STATUS = PASS
MEDIAN_POOL_SECONDS = 0.1361
THROUGHPUT_SAMPLES_PER_S = 940.793
PEAK_ALLOCATED_GIB = 12.078
PEAK_RESERVED_GIB = 17.086
OPTIMIZER_STEPS_PER_POOL = 1

=== BENCHMARK SUMMARY ===
BATCH 32 STATUS PASS
BATCH 64 STATUS PASS
BATCH 128 STATUS PASS

PAPER_BATCH_128_SUPPORTED = True
FASTEST_BATCH = 128
RECOMMENDED_BATCH = 128
ESTIMATED_TRAIN_EPOCH_SECONDS_NEURAL_ONLY = 7.298

OFFICIAL_TRAINING_STARTED = False
NOTEBOOK_03_GPU_BENCHMARK_PASS = True


## STEP 03-B2 — Benchmark del loader Gravity Spy

Il benchmark GPU ha mostrato che `batch=128` è sostenibile e, tra i candidati testati, fornisce il throughput neurale più alto sulla A100.

Prima di congelare il batch operativo è necessario misurare separatamente la pipeline CPU/I/O:

`PNG locali → crop stateless → stack 4 canali → float32 → batch`

Il benchmark usa l'intero full split train di 6.866 eventi con:

- batch size `128`;
- shuffle PROJECT dell'epoch 0;
- augmentation stateless dell'epoch 0;
- `pin_memory=True`;
- `drop_last=False`;
- candidati `num_workers = 0, 2, 4, 8` quando compatibili con le CPU disponibili.

`persistent_workers=False` è intenzionale: il Dataset contiene un `epoch` che cambia a ogni epoca e deve essere correttamente propagato alle copie dei worker.

Prima delle misure viene effettuata una traversata non cronometrata del dataset per ridurre il vantaggio artificiale dovuto alla cache del filesystem per i candidati eseguiti successivamente.

Questo step non esegue il VAE e non modifica alcun parametro.

In [11]:
# ============================================================
# NOTEBOOK 03 — REAL TRAIN LOADER BENCHMARK
# STEP 03-B2
#
# Measures the CPU/I/O side of the real Gravity Spy pipeline.
#
# NO VAE.
# NO backward.
# NO optimizer.
# NO official training.
# ============================================================

import gc
import os
import statistics
import time

import torch

from torch.utils.data import DataLoader


# ============================================================
# BENCHMARK POLICY
# ============================================================

LOADER_BATCH_SIZE = 128
LOADER_BENCHMARK_EPOCH = 0

# Two measured full epochs are enough here because every pass
# already contains all 6,866 training events.
LOADER_MEASURED_REPEATS = 2

LOADER_PIN_MEMORY = True


logical_cpu_count = (
    os.cpu_count()
    or 1
)


# Do not deliberately oversubscribe the runtime.
LOADER_WORKER_CANDIDATES = [
    workers
    for workers
    in [0, 2, 4, 8]
    if workers <= logical_cpu_count
]


assert 0 in LOADER_WORKER_CANDIDATES


EXPECTED_TRAIN_SAMPLES = 6866

EXPECTED_TRAIN_BATCHES = (
    EXPECTED_TRAIN_SAMPLES
    + LOADER_BATCH_SIZE
    - 1
) // LOADER_BATCH_SIZE

EXPECTED_TAIL_BATCH = (
    EXPECTED_TRAIN_SAMPLES
    % LOADER_BATCH_SIZE
)


assert EXPECTED_TAIL_BATCH > 0


# ============================================================
# DATALOADER FACTORY
#
# persistent_workers=False is deliberate.
#
# train_dataset.epoch changes every epoch. With persistent
# worker copies, that mutable value would not automatically
# follow changes made in the parent process.
# ============================================================

def make_train_loader_for_benchmark(
    num_workers,
):

    kwargs = {
        "dataset":
            train_dataset,

        "batch_size":
            LOADER_BATCH_SIZE,

        "sampler":
            train_sampler,

        "num_workers":
            num_workers,

        "pin_memory":
            LOADER_PIN_MEMORY,

        "drop_last":
            False,

        "persistent_workers":
            False,
    }


    # PyTorch permits prefetch_factor only when workers > 0.
    if num_workers > 0:

        kwargs[
            "prefetch_factor"
        ] = 2


    return DataLoader(
        **kwargs
    )


# ============================================================
# PRECONDITIONS
# ============================================================

assert (
    torch.cuda.get_device_name(0)
    == "NVIDIA A100-SXM4-80GB"
)

assert (
    BENCHMARK_BATCH_CANDIDATES
    == [32, 64, 128]
)

assert any(
    result["batch_size"] == 128
    and result["status"] == "PASS"

    for result
    in benchmark_results
)


# ============================================================
# FILESYSTEM CACHE WARM-UP
#
# This pass is NOT measured.
#
# It reads one full epoch so that worker candidates are not
# mainly distinguished by whether Linux happened to cache a
# particular PNG before their turn.
# ============================================================

print(
    "=== FILESYSTEM CACHE WARM-UP ==="
)

set_project_train_epoch(
    LOADER_BENCHMARK_EPOCH
)

cache_warm_loader = (
    make_train_loader_for_benchmark(
        num_workers=0
    )
)


cache_warm_samples = 0

for batch in cache_warm_loader:

    cache_warm_samples += int(
        batch["x"].shape[0]
    )


assert (
    cache_warm_samples
    == EXPECTED_TRAIN_SAMPLES
)


del cache_warm_loader

gc.collect()


print(
    "CACHE_WARM_SAMPLES =",
    cache_warm_samples,
)

print(
    "CACHE_WARM_PASS =",
    True,
)


# ============================================================
# ONE WORKER CONFIGURATION
# ============================================================

def benchmark_loader_workers(
    num_workers,
):

    elapsed_values = []

    first_manifest_reference = None
    first_crop_reference = None

    pin_status = None

    repeat_batch_counts = []
    repeat_sample_counts = []
    repeat_tail_sizes = []


    for repeat in range(
        LOADER_MEASURED_REPEATS
    ):

        # Both PROJECT mechanisms must be restored to the
        # benchmark epoch before each complete traversal.
        set_project_train_epoch(
            LOADER_BENCHMARK_EPOCH
        )


        loader = (
            make_train_loader_for_benchmark(
                num_workers
            )
        )


        sample_count = 0
        batch_count = 0
        last_batch_size = None


        start_time = (
            time.perf_counter()
        )


        for batch_index, batch in enumerate(
            loader
        ):

            x = batch["x"]

            current_batch_size = int(
                x.shape[0]
            )


            # The first batch also acts as an order/augmentation
            # correctness gate for every worker configuration.
            if batch_index == 0:

                manifests = (
                    batch[
                        "manifest_index"
                    ].tolist()
                )

                crops = (
                    batch[
                        "crop_left"
                    ].tolist()
                )


                assert manifests[:4] == [
                    2676,
                    6250,
                    2396,
                    5602,
                ]

                assert crops[:4] == [
                    20,
                    15,
                    19,
                    12,
                ]


                if first_manifest_reference is None:

                    first_manifest_reference = (
                        manifests
                    )

                    first_crop_reference = (
                        crops
                    )

                    pin_status = (
                        x.is_pinned()
                    )

                else:

                    assert (
                        manifests
                        == first_manifest_reference
                    )

                    assert (
                        crops
                        == first_crop_reference
                    )


            sample_count += (
                current_batch_size
            )

            batch_count += 1

            last_batch_size = (
                current_batch_size
            )


        elapsed = (
            time.perf_counter()
            - start_time
        )


        assert (
            sample_count
            == EXPECTED_TRAIN_SAMPLES
        )

        assert (
            batch_count
            == EXPECTED_TRAIN_BATCHES
        )

        assert (
            last_batch_size
            == EXPECTED_TAIL_BATCH
        )


        elapsed_values.append(
            elapsed
        )

        repeat_sample_counts.append(
            sample_count
        )

        repeat_batch_counts.append(
            batch_count
        )

        repeat_tail_sizes.append(
            last_batch_size
        )


        del loader

        gc.collect()


    median_seconds = (
        statistics.median(
            elapsed_values
        )
    )


    throughput = (
        EXPECTED_TRAIN_SAMPLES
        / median_seconds
    )


    return {
        "num_workers":
            num_workers,

        "median_epoch_seconds":
            median_seconds,

        "throughput_samples_per_s":
            throughput,

        "elapsed_values":
            elapsed_values,

        "sample_counts":
            repeat_sample_counts,

        "batch_counts":
            repeat_batch_counts,

        "tail_sizes":
            repeat_tail_sizes,

        "first_manifests":
            first_manifest_reference,

        "first_crops":
            first_crop_reference,

        "first_batch_pinned":
            pin_status,

        "pass":
            True,
    }


# ============================================================
# RUN WORKER CANDIDATES
# ============================================================

loader_benchmark_results = []


print()
print(
    "=== REAL TRAIN LOADER BENCHMARK ==="
)

print(
    "LOGICAL_CPU_COUNT =",
    logical_cpu_count,
)

print(
    "WORKER_CANDIDATES =",
    LOADER_WORKER_CANDIDATES,
)

print(
    "BATCH_SIZE =",
    LOADER_BATCH_SIZE,
)

print(
    "EXPECTED_BATCHES =",
    EXPECTED_TRAIN_BATCHES,
)

print(
    "EXPECTED_TAIL =",
    EXPECTED_TAIL_BATCH,
)


for workers in (
    LOADER_WORKER_CANDIDATES
):

    print()
    print(
        "Running num_workers =",
        workers,
        "..."
    )


    result = (
        benchmark_loader_workers(
            workers
        )
    )


    loader_benchmark_results.append(
        result
    )


    print(
        "MEDIAN_EPOCH_SECONDS =",
        round(
            result[
                "median_epoch_seconds"
            ],
            4,
        ),
    )

    print(
        "THROUGHPUT_SAMPLES_PER_S =",
        round(
            result[
                "throughput_samples_per_s"
            ],
            3,
        ),
    )

    print(
        "FIRST_BATCH_PINNED =",
        result[
            "first_batch_pinned"
        ],
    )

    print(
        "PASS =",
        result[
            "pass"
        ],
    )


# ============================================================
# SELECT PROVISIONAL WORKER COUNT
#
# This is not yet the final training configuration.
# End-to-end timing will confirm it in STEP 03-B3.
# ============================================================

best_loader_result = max(
    loader_benchmark_results,
    key=lambda item:
        item[
            "throughput_samples_per_s"
        ],
)


PROVISIONAL_NUM_WORKERS = (
    best_loader_result[
        "num_workers"
    ]
)


all_loader_gates_pass = all(
    result["pass"]
    and all(
        value
        == EXPECTED_TRAIN_SAMPLES

        for value
        in result[
            "sample_counts"
        ]
    )
    and all(
        value
        == EXPECTED_TRAIN_BATCHES

        for value
        in result[
            "batch_counts"
        ]
    )
    and all(
        value
        == EXPECTED_TAIL_BATCH

        for value
        in result[
            "tail_sizes"
        ]
    )

    for result
    in loader_benchmark_results
)


assert all_loader_gates_pass


# ============================================================
# REPORT
# ============================================================

print()
print(
    "=== LOADER BENCHMARK SUMMARY ==="
)


for result in loader_benchmark_results:

    print(
        "NUM_WORKERS =",
        result[
            "num_workers"
        ],

        "| EPOCH_S =",
        round(
            result[
                "median_epoch_seconds"
            ],
            4,
        ),

        "| SAMPLE_S =",
        round(
            result[
                "throughput_samples_per_s"
            ],
            3,
        ),
    )


print()

print(
    "PROVISIONAL_NUM_WORKERS =",
    PROVISIONAL_NUM_WORKERS,
)

print(
    "PIN_MEMORY =",
    LOADER_PIN_MEMORY,
)

print(
    "PERSISTENT_WORKERS =",
    False,
)

print()

print(
    "TRAIN_SAMPLES_PER_EPOCH =",
    EXPECTED_TRAIN_SAMPLES,
)

print(
    "TRAIN_BATCHES_PER_EPOCH =",
    EXPECTED_TRAIN_BATCHES,
)

print(
    "TAIL_BATCH_SIZE =",
    EXPECTED_TAIL_BATCH,
)

print()

print(
    "MODEL_FORWARD_EXECUTED =",
    False,
)

print(
    "OPTIMIZER_EXECUTED =",
    False,
)

print(
    "OFFICIAL_TRAINING_STARTED =",
    False,
)

print()

print(
    "NOTEBOOK_03_LOADER_BENCHMARK_PASS =",
    all_loader_gates_pass,
)

=== FILESYSTEM CACHE WARM-UP ===
CACHE_WARM_SAMPLES = 6866
CACHE_WARM_PASS = True

=== REAL TRAIN LOADER BENCHMARK ===
LOGICAL_CPU_COUNT = 12
WORKER_CANDIDATES = [0, 2, 4, 8]
BATCH_SIZE = 128
EXPECTED_BATCHES = 54
EXPECTED_TAIL = 82

Running num_workers = 0 ...
MEDIAN_EPOCH_SECONDS = 31.6321
THROUGHPUT_SAMPLES_PER_S = 217.058
FIRST_BATCH_PINNED = True
PASS = True

Running num_workers = 2 ...
MEDIAN_EPOCH_SECONDS = 17.3401
THROUGHPUT_SAMPLES_PER_S = 395.962
FIRST_BATCH_PINNED = True
PASS = True

Running num_workers = 4 ...
MEDIAN_EPOCH_SECONDS = 9.1416
THROUGHPUT_SAMPLES_PER_S = 751.071
FIRST_BATCH_PINNED = True
PASS = True

Running num_workers = 8 ...
MEDIAN_EPOCH_SECONDS = 6.1727
THROUGHPUT_SAMPLES_PER_S = 1112.311
FIRST_BATCH_PINNED = True
PASS = True

=== LOADER BENCHMARK SUMMARY ===
NUM_WORKERS = 0 | EPOCH_S = 31.6321 | SAMPLE_S = 217.058
NUM_WORKERS = 2 | EPOCH_S = 17.3401 | SAMPLE_S = 395.962
NUM_WORKERS = 4 | EPOCH_S = 9.1416 | SAMPLE_S = 751.071
NUM_WORKERS = 8 | EPOCH_S = 6.17

## STEP 03-B3 — Benchmark end-to-end del training reale

I benchmark separati hanno stabilito:

- batch candidato: `128`;
- loader candidato: `num_workers=8`;
- `pin_memory=True`;
- AMP disabilitato;
- TF32 disabilitato.

Ora viene misurato il percorso realmente utilizzato durante il training:

`PNG → augmentation → epsilon stateless → DataLoader → H2D → VAE → loss → backward → Adam`

L'epsilon viene prodotto nei worker del DataLoader, così anche questa parte CPU può essere eseguita in parallelo con il caricamento delle immagini.

Vengono misurate due epoche consecutive su un **modello temporaneo**:

- epoch 0: include anche la prima allocazione degli stati Adam;
- epoch 1: rappresenta meglio il regime steady-state.

Ogni epoch deve contenere:

- `6866` eventi;
- `54` batch;
- `53` batch completi da `128`;
- tail finale da `82`.

Il modello e l'optimizer del benchmark vengono eliminati al termine.  
Nessun checkpoint viene scritto e la run scientifica ufficiale resta non avviata.

In [12]:
# ============================================================
# NOTEBOOK 03 — REAL END-TO-END TRAINING BENCHMARK
# STEP 03-B3
#
# Real pipeline:
#   PNG
#   -> stateless augmentation
#   -> stateless latent epsilon
#   -> pinned DataLoader
#   -> GPU transfer
#   -> forward
#   -> VAE loss
#   -> backward
#   -> Adam
#
# TWO DISPOSABLE benchmark epochs only.
#
# NO checkpoint.
# NO official training.
# ============================================================

import gc
import math
import time

import numpy as np
import torch

from torch.utils.data import (
    Dataset,
    DataLoader,
)


# ============================================================
# FROZEN BENCHMARK CONFIGURATION
# ============================================================

E2E_BATCH_SIZE = 128
E2E_NUM_WORKERS = 8
E2E_PIN_MEMORY = True

E2E_EPOCHS = [
    0,
    1,
]

EXPECTED_TRAIN_SAMPLES = 6866
EXPECTED_TRAIN_BATCHES = 54
EXPECTED_TAIL_BATCH = 82


assert PROVISIONAL_NUM_WORKERS == 8

assert any(
    result["batch_size"] == 128
    and result["status"] == "PASS"

    for result
    in benchmark_results
)


# ============================================================
# REMOVE THE PRELOADED GPU BENCHMARK POOL
#
# It was useful for B1 but is not part of the real training
# pipeline and should not consume VRAM during this benchmark.
# ============================================================

globals().pop(
    "benchmark_x_gpu",
    None,
)

globals().pop(
    "benchmark_epsilon_gpu",
    None,
)

gc.collect()
torch.cuda.empty_cache()


device = torch.device(
    "cuda:0"
)


# ============================================================
# TRAINING DATASET WRAPPER WITH EXTERNAL EPSILON
#
# The core GravitySpy224Dataset remains unchanged.
#
# This thin wrapper adds the PROJECT latent epsilon so that
# PNG loading, augmentation and epsilon generation can all
# happen inside the DataLoader workers.
# ============================================================

class GravitySpy224TrainWithEpsilon(Dataset):

    def __init__(
        self,
        base_dataset,
        epoch=0,
    ):
        self.base_dataset = (
            base_dataset
        )

        self.epoch = int(
            epoch
        )

        self.base_dataset.set_epoch(
            self.epoch
        )

    def __len__(self):
        return len(
            self.base_dataset
        )

    def set_epoch(
        self,
        epoch,
    ):
        self.epoch = int(
            epoch
        )

        self.base_dataset.set_epoch(
            self.epoch
        )

    def __getitem__(
        self,
        index,
    ):

        item = self.base_dataset[
            index
        ]

        manifest_index = int(
            item[
                "manifest_index"
            ]
        )

        epsilon_np = (
            python_latent_epsilon(
                PROJECT_LATENT_SEED,
                RNG_DOMAIN_LATENT_TRAIN,
                self.epoch,
                manifest_index,
                SPEC.latent_dim,
            )
        )

        epsilon = torch.from_numpy(
            epsilon_np
        )

        return {
            "x":
                item["x"],

            "epsilon":
                epsilon,

            "manifest_index":
                manifest_index,

            "event_id":
                item["event_id"],

            "crop_left":
                item["crop_left"],
        }


train_dataset_e2e = (
    GravitySpy224TrainWithEpsilon(
        train_dataset,
        epoch=0,
    )
)


# ============================================================
# SYNCHRONIZE ALL EPOCH-DEPENDENT COMPONENTS
#
#   sampler
#   image augmentation
#   latent epsilon
#
# persistent_workers remains False, therefore each new
# DataLoader iterator receives the newly selected epoch.
# ============================================================

def set_e2e_epoch(
    epoch,
):

    epoch = int(
        epoch
    )

    train_sampler.set_epoch(
        epoch
    )

    train_dataset_e2e.set_epoch(
        epoch
    )


# ============================================================
# REAL TRAIN DATALOADER
# ============================================================

train_loader_e2e = DataLoader(
    train_dataset_e2e,

    batch_size=E2E_BATCH_SIZE,

    sampler=train_sampler,

    num_workers=E2E_NUM_WORKERS,

    pin_memory=E2E_PIN_MEMORY,

    drop_last=False,

    persistent_workers=False,

    prefetch_factor=2,
)


# ============================================================
# FRESH DISPOSABLE MODEL
#
# Same exact PROJECT initialization used everywhere else.
# ============================================================

e2e_model = Sakai224FullVAE(
    SPEC
)

initialize_project_c_exact_v1(
    e2e_model,
    PROJECT_INITIALIZATION_SEED,
)


e2e_initial_sha256 = (
    project_parameter_sha256(
        e2e_model
    )
)


assert (
    e2e_initial_sha256
    == FROZEN_INIT_SHA256
)


e2e_model = e2e_model.to(
    device=device,
    dtype=torch.float32,
)

e2e_model.train()


# ============================================================
# PAPER ADAM
# ============================================================

e2e_optimizer = torch.optim.Adam(
    e2e_model.parameters(),

    lr=5.0e-4,

    betas=(
        0.9,
        0.999,
    ),

    eps=1.0e-8,

    weight_decay=0.0,

    amsgrad=False,

    foreach=False,

    fused=False,
)


# ============================================================
# RUN ONE COMPLETE REAL TRAIN EPOCH
# ============================================================

def run_real_train_epoch(
    epoch,
):

    set_e2e_epoch(
        epoch
    )


    # --------------------------------------------------------
    # EXPECTED FIRST BATCH IDENTITY
    # --------------------------------------------------------

    expected_positions = (
        project_shuffle_permutation(
            EXPECTED_TRAIN_SAMPLES,
            PROJECT_SHUFFLE_SEED,
            epoch,
        )[:4]
    )


    expected_manifests = [
        int(
            train_rows[
                position
            ][
                "manifest_index"
            ]
        )

        for position
        in expected_positions
    ]


    expected_crops = [
        gs_augment_crop_left_for_event_mode(
            AUGMENTATION_SEED,
            epoch,
            manifest_index,
            GS_AUGMENT_MODE_SYMMETRIC_FIGURE,
        )

        for manifest_index
        in expected_manifests
    ]


    # --------------------------------------------------------
    # TIMING
    #
    # Starts before DataLoader iteration, so worker startup is
    # part of the real epoch cost.
    # --------------------------------------------------------

    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    start_time = (
        time.perf_counter()
    )


    sample_count = 0
    batch_count = 0
    tail_batch_size = None

    first_batch_gate = False
    first_epsilon_gate = False

    final_loss = None
    losses_finite = True


    for batch_index, batch in enumerate(
        train_loader_e2e
    ):

        # ----------------------------------------------------
        # FIRST-BATCH REPRODUCIBILITY GATES
        # ----------------------------------------------------

        if batch_index == 0:

            actual_manifests = (
                batch[
                    "manifest_index"
                ].tolist()
            )

            actual_crops = (
                batch[
                    "crop_left"
                ].tolist()
            )


            first_batch_gate = (
                actual_manifests[:4]
                    == expected_manifests

                and actual_crops[:4]
                    == expected_crops
            )


            if not first_batch_gate:

                raise RuntimeError(
                    "First real train batch identity mismatch"
                )


            # Independently regenerate the first four epsilon
            # rows and compare their float32 bit patterns.
            expected_epsilon_np = np.stack(
                [
                    python_latent_epsilon(
                        PROJECT_LATENT_SEED,
                        RNG_DOMAIN_LATENT_TRAIN,
                        epoch,
                        manifest_index,
                        SPEC.latent_dim,
                    )

                    for manifest_index
                    in expected_manifests
                ],
                axis=0,
            )


            actual_epsilon_np = (
                batch[
                    "epsilon"
                ][
                    :4
                ]
                .numpy()
            )


            first_epsilon_gate = (
                np.array_equal(
                    actual_epsilon_np.view(
                        np.uint32
                    ),

                    expected_epsilon_np.view(
                        np.uint32
                    ),
                )
            )


            if not first_epsilon_gate:

                raise RuntimeError(
                    "First batch latent epsilon mismatch"
                )


        # ----------------------------------------------------
        # PINNED CPU -> GPU TRANSFER
        # ----------------------------------------------------

        x_batch = (
            batch["x"]
            .to(
                device=device,
                dtype=torch.float32,
                non_blocking=True,
            )
        )

        epsilon_batch = (
            batch["epsilon"]
            .to(
                device=device,
                dtype=torch.float32,
                non_blocking=True,
            )
        )


        current_batch_size = int(
            x_batch.shape[0]
        )


        # ----------------------------------------------------
        # REAL TRAINING STEP
        # ----------------------------------------------------

        e2e_optimizer.zero_grad(
            set_to_none=True
        )


        (
            logits,
            mu,
            logvar,
            z,
        ) = e2e_model(
            x_batch,
            epsilon=epsilon_batch,
            mean_mode=False,
        )


        stats = sakai_vae_loss(
            logits,
            x_batch,
            mu,
            logvar,
            beta_kl=SPEC.beta_kl,
        )


        loss = stats["loss"]


        if not torch.isfinite(
            loss
        ).item():

            losses_finite = False

            raise RuntimeError(
                "Non-finite end-to-end benchmark loss"
            )


        loss.backward()

        e2e_optimizer.step()


        final_loss = float(
            loss.detach()
        )


        sample_count += (
            current_batch_size
        )

        batch_count += 1

        tail_batch_size = (
            current_batch_size
        )


    # --------------------------------------------------------
    # END TIMING
    # --------------------------------------------------------

    torch.cuda.synchronize()

    elapsed_seconds = (
        time.perf_counter()
        - start_time
    )


    # --------------------------------------------------------
    # HARD EPOCH GATES
    # --------------------------------------------------------

    sample_gate = (
        sample_count
        == EXPECTED_TRAIN_SAMPLES
    )

    batch_gate = (
        batch_count
        == EXPECTED_TRAIN_BATCHES
    )

    tail_gate = (
        tail_batch_size
        == EXPECTED_TAIL_BATCH
    )


    if not (
        sample_gate
        and batch_gate
        and tail_gate
    ):
        raise RuntimeError(
            "Incomplete benchmark epoch"
        )


    # --------------------------------------------------------
    # ADAM GLOBAL STEP
    # --------------------------------------------------------

    expected_global_step = (
        (epoch + 1)
        * EXPECTED_TRAIN_BATCHES
    )


    adam_step_gate = all(
        int(
            e2e_optimizer.state[
                parameter
            ][
                "step"
            ].item()
        )
        == expected_global_step

        for parameter
        in e2e_model.parameters()
    )


    if not adam_step_gate:

        raise RuntimeError(
            "Adam step mismatch"
        )


    # --------------------------------------------------------
    # BATCHNORM COUNTERS
    # --------------------------------------------------------

    bn_step_gate = all(
        int(
            bn.num_batches_tracked.item()
        )
        == expected_global_step

        for bn
        in get_project_batchnorm_layers(
            e2e_model
        )
    )


    if not bn_step_gate:

        raise RuntimeError(
            "BatchNorm counter mismatch"
        )


    # --------------------------------------------------------
    # GPU MEMORY
    # --------------------------------------------------------

    peak_allocated_gib = (
        torch.cuda.max_memory_allocated()
        / (1024 ** 3)
    )

    peak_reserved_gib = (
        torch.cuda.max_memory_reserved()
        / (1024 ** 3)
    )


    throughput = (
        sample_count
        / elapsed_seconds
    )


    return {
        "epoch":
            epoch,

        "elapsed_seconds":
            elapsed_seconds,

        "throughput_samples_per_s":
            throughput,

        "sample_count":
            sample_count,

        "batch_count":
            batch_count,

        "tail_batch_size":
            tail_batch_size,

        "final_loss":
            final_loss,

        "losses_finite":
            losses_finite,

        "first_positions":
            expected_positions,

        "first_manifests":
            expected_manifests,

        "first_crops":
            expected_crops,

        "first_batch_gate":
            first_batch_gate,

        "first_epsilon_gate":
            first_epsilon_gate,

        "adam_step":
            expected_global_step,

        "adam_step_gate":
            adam_step_gate,

        "bn_step_gate":
            bn_step_gate,

        "peak_allocated_gib":
            peak_allocated_gib,

        "peak_reserved_gib":
            peak_reserved_gib,
    }


# ============================================================
# EXECUTE TWO DISPOSABLE REAL EPOCHS
# ============================================================

e2e_epoch_results = []


print(
    "=== REAL END-TO-END TRAINING BENCHMARK ==="
)


for epoch in E2E_EPOCHS:

    print()
    print(
        "Running benchmark epoch",
        epoch,
        "..."
    )


    result = run_real_train_epoch(
        epoch
    )


    e2e_epoch_results.append(
        result
    )


    print(
        "EPOCH_SECONDS =",
        round(
            result[
                "elapsed_seconds"
            ],
            4,
        ),
    )

    print(
        "THROUGHPUT_SAMPLES_PER_S =",
        round(
            result[
                "throughput_samples_per_s"
            ],
            3,
        ),
    )

    print(
        "BATCHES =",
        result[
            "batch_count"
        ],
    )

    print(
        "TAIL =",
        result[
            "tail_batch_size"
        ],
    )

    print(
        "ADAM_STEP =",
        result[
            "adam_step"
        ],
    )

    print(
        "PEAK_ALLOCATED_GIB =",
        round(
            result[
                "peak_allocated_gib"
            ],
            3,
        ),
    )

    print(
        "PEAK_RESERVED_GIB =",
        round(
            result[
                "peak_reserved_gib"
            ],
            3,
        ),
    )


# ============================================================
# FINAL NUMERIC MODEL AUDIT
# ============================================================

updated_parameters_finite = all(
    torch.isfinite(
        parameter
    ).all().item()

    for parameter
    in e2e_model.parameters()
)


assert updated_parameters_finite


post_benchmark_sha256 = (
    project_parameter_sha256(
        e2e_model
    )
)


benchmark_model_changed = (
    post_benchmark_sha256
    != FROZEN_INIT_SHA256
)


assert benchmark_model_changed


all_e2e_gates_pass = all(
    result[
        "sample_count"
    ] == EXPECTED_TRAIN_SAMPLES

    and result[
        "batch_count"
    ] == EXPECTED_TRAIN_BATCHES

    and result[
        "tail_batch_size"
    ] == EXPECTED_TAIL_BATCH

    and result[
        "losses_finite"
    ]

    and result[
        "first_batch_gate"
    ]

    and result[
        "first_epsilon_gate"
    ]

    and result[
        "adam_step_gate"
    ]

    and result[
        "bn_step_gate"
    ]

    for result
    in e2e_epoch_results
)


assert all_e2e_gates_pass


# ============================================================
# COMPARISON WITH ISOLATED BENCHMARKS
# ============================================================

gpu_batch128_result = next(
    result

    for result
    in benchmark_results

    if result[
        "batch_size"
    ] == 128
)


gpu_neural_only_epoch_estimate = (
    EXPECTED_TRAIN_SAMPLES
    / gpu_batch128_result[
        "throughput_samples_per_s"
    ]
)


loader_only_epoch_estimate = (
    best_loader_result[
        "median_epoch_seconds"
    ]
)


steady_state_result = (
    e2e_epoch_results[1]
)


# ============================================================
# REPORT
# ============================================================

print()
print(
    "=== END-TO-END BENCHMARK SUMMARY ==="
)


for result in e2e_epoch_results:

    print(
        "EPOCH",
        result["epoch"],
        "| SEC =",
        round(
            result[
                "elapsed_seconds"
            ],
            4,
        ),
        "| SAMPLE/S =",
        round(
            result[
                "throughput_samples_per_s"
            ],
            3,
        ),
        "| FIRST MANIFESTS =",
        result[
            "first_manifests"
        ],
        "| FIRST CROPS =",
        result[
            "first_crops"
        ],
    )


print()

print(
    "GPU_NEURAL_ONLY_EPOCH_ESTIMATE_S =",
    round(
        gpu_neural_only_epoch_estimate,
        4,
    ),
)

print(
    "LOADER_ONLY_EPOCH_ESTIMATE_S =",
    round(
        loader_only_epoch_estimate,
        4,
    ),
)

print(
    "MEASURED_STEADY_STATE_EPOCH_S =",
    round(
        steady_state_result[
            "elapsed_seconds"
        ],
        4,
    ),
)

print(
    "MEASURED_STEADY_STATE_SAMPLE_S =",
    round(
        steady_state_result[
            "throughput_samples_per_s"
        ],
        3,
    ),
)

print()

print(
    "BATCH_SIZE =",
    E2E_BATCH_SIZE,
)

print(
    "NUM_WORKERS =",
    E2E_NUM_WORKERS,
)

print(
    "PIN_MEMORY =",
    E2E_PIN_MEMORY,
)

print(
    "PERSISTENT_WORKERS =",
    False,
)

print()

print(
    "UPDATED_PARAMETERS_FINITE =",
    updated_parameters_finite,
)

print(
    "DISPOSABLE_MODEL_CHANGED =",
    benchmark_model_changed,
)

print()

print(
    "CHECKPOINT_WRITTEN =",
    False,
)

print(
    "OFFICIAL_TRAINING_STARTED =",
    False,
)

print()

print(
    "NOTEBOOK_03_END_TO_END_BENCHMARK_PASS =",
    (
        all_e2e_gates_pass
        and updated_parameters_finite
        and benchmark_model_changed
    ),
)


# ============================================================
# RELEASE DISPOSABLE BENCHMARK MODEL
# ============================================================

del e2e_model
del e2e_optimizer

gc.collect()
torch.cuda.empty_cache()

=== REAL END-TO-END TRAINING BENCHMARK ===

Running benchmark epoch 0 ...
EPOCH_SECONDS = 9.4459
THROUGHPUT_SAMPLES_PER_S = 726.874
BATCHES = 54
TAIL = 82
ADAM_STEP = 54
PEAK_ALLOCATED_GIB = 11.857
PEAK_RESERVED_GIB = 16.865

Running benchmark epoch 1 ...
EPOCH_SECONDS = 9.4785
THROUGHPUT_SAMPLES_PER_S = 724.374
BATCHES = 54
TAIL = 82
ADAM_STEP = 108
PEAK_ALLOCATED_GIB = 11.857
PEAK_RESERVED_GIB = 16.865

=== END-TO-END BENCHMARK SUMMARY ===
EPOCH 0 | SEC = 9.4459 | SAMPLE/S = 726.874 | FIRST MANIFESTS = [2676, 6250, 2396, 5602] | FIRST CROPS = [20, 15, 19, 12]
EPOCH 1 | SEC = 9.4785 | SAMPLE/S = 724.374 | FIRST MANIFESTS = [6028, 4009, 12, 5170] | FIRST CROPS = [19, 36, 27, 20]

GPU_NEURAL_ONLY_EPOCH_ESTIMATE_S = 7.2981
LOADER_ONLY_EPOCH_ESTIMATE_S = 6.1727
MEASURED_STEADY_STATE_EPOCH_S = 9.4785
MEASURED_STEADY_STATE_SAMPLE_S = 724.374

BATCH_SIZE = 128
NUM_WORKERS = 8
PIN_MEMORY = True
PERSISTENT_WORKERS = False

UPDATED_PARAMETERS_FINITE = True
DISPOSABLE_MODEL_CHANGED = True

CHECK

## STEP 03-B4 — Benchmark della validation completa

Il training end-to-end con batch `128` e `num_workers=8` è risultato stabile sulla A100.

Prima di congelare definitivamente la configurazione viene misurata la validation completa sui 1.717 eventi del full split.

Sono valutati i due percorsi PROJECT:

### STOCHASTIC + EVAL

Utilizzato per la validation deterministica principale:

- tensori centrali `4×224×224`;
- nessuna augmentation;
- epsilon stateless di validation con seed `456`;
- epsilon indipendente dall'epoch;
- BatchNorm in modalità `EVAL`;
- nessun backward.

### MEAN + EVAL

Utilizzato per le diagnostiche deterministiche:

`z = μ`

senza epsilon.

Con batch `128` la validation deve contenere:

- `1717` eventi;
- `14` batch;
- tail finale da `53`.

Il modello utilizzato è temporaneo e non viene addestrato.

In [13]:
# ============================================================
# NOTEBOOK 03 — FULL VALIDATION BENCHMARK
# STEP 03-B4
#
# Measures:
#   STOCHASTIC + EVAL fixed-epsilon validation
#   MEAN + EVAL diagnostic validation
#
# NO backward.
# NO optimizer.
# NO official training.
# ============================================================

import gc
import statistics
import time

import numpy as np
import torch

from torch.utils.data import (
    Dataset,
    DataLoader,
)


# ============================================================
# VALIDATION CONFIGURATION
# ============================================================

VAL_BATCH_SIZE = 128
VAL_NUM_WORKERS = 8
VAL_PIN_MEMORY = True

VAL_MEASURED_REPEATS = 2

EXPECTED_VAL_SAMPLES = 1717
EXPECTED_VAL_BATCHES = 14
EXPECTED_VAL_TAIL = 53


assert len(val_dataset) == EXPECTED_VAL_SAMPLES

assert (
    EXPECTED_VAL_SAMPLES
    // VAL_BATCH_SIZE
    == 13
)

assert (
    EXPECTED_VAL_SAMPLES
    % VAL_BATCH_SIZE
    == EXPECTED_VAL_TAIL
)


# ============================================================
# VALIDATION DATASET + FIXED EPSILON
#
# Validation epsilon uses:
#
#   latent seed = 456
#   RNG domain  = LATENT_VALIDATION
#   epoch       = 0
#   manifest_index
#
# Therefore the same validation event receives the same
# epsilon independently of training epoch.
# ============================================================

class GravitySpy224ValidationWithEpsilon(Dataset):

    def __init__(
        self,
        base_dataset,
    ):
        self.base_dataset = (
            base_dataset
        )

    def __len__(self):
        return len(
            self.base_dataset
        )

    def __getitem__(
        self,
        index,
    ):

        item = self.base_dataset[
            index
        ]

        manifest_index = int(
            item[
                "manifest_index"
            ]
        )

        epsilon_np = python_latent_epsilon(
            PROJECT_LATENT_SEED,
            RNG_DOMAIN_LATENT_VALIDATION,
            0,
            manifest_index,
            SPEC.latent_dim,
        )

        epsilon = torch.from_numpy(
            epsilon_np
        )

        return {
            "x":
                item["x"],

            "epsilon":
                epsilon,

            "manifest_index":
                manifest_index,

            "event_id":
                item["event_id"],

            "crop_left":
                item["crop_left"],
        }


val_dataset_with_epsilon = (
    GravitySpy224ValidationWithEpsilon(
        val_dataset
    )
)


# ============================================================
# DATALOADERS
#
# persistent_workers=False keeps runtime recovery and notebook
# execution simple and matches the conservative PROJECT policy.
# ============================================================

val_loader_stochastic = DataLoader(
    val_dataset_with_epsilon,

    batch_size=VAL_BATCH_SIZE,

    shuffle=False,

    num_workers=VAL_NUM_WORKERS,

    pin_memory=VAL_PIN_MEMORY,

    drop_last=False,

    persistent_workers=False,

    prefetch_factor=2,
)


val_loader_mean = DataLoader(
    val_dataset,

    batch_size=VAL_BATCH_SIZE,

    shuffle=False,

    num_workers=VAL_NUM_WORKERS,

    pin_memory=VAL_PIN_MEMORY,

    drop_last=False,

    persistent_workers=False,

    prefetch_factor=2,
)


# ============================================================
# FRESH DISPOSABLE VALIDATION MODEL
# ============================================================

val_benchmark_model = Sakai224FullVAE(
    SPEC
)

initialize_project_c_exact_v1(
    val_benchmark_model,
    PROJECT_INITIALIZATION_SEED,
)


val_initial_hash = (
    project_parameter_sha256(
        val_benchmark_model
    )
)

assert (
    val_initial_hash
    == FROZEN_INIT_SHA256
)


val_benchmark_model = (
    val_benchmark_model.to(
        device=device,
        dtype=torch.float32,
    )
)

val_benchmark_model.eval()


val_bn_layers = (
    get_project_batchnorm_layers(
        val_benchmark_model
    )
)


def snapshot_validation_bn():

    return [
        (
            bn.running_mean.detach().clone(),
            bn.running_var.detach().clone(),
            bn.num_batches_tracked.detach().clone(),
        )

        for bn
        in val_bn_layers
    ]


bn_before_validation = (
    snapshot_validation_bn()
)


# ============================================================
# ONE WARM-UP BATCH
#
# Outside measured timing.
# ============================================================

warm_batch = next(
    iter(
        val_loader_stochastic
    )
)


x_warm = (
    warm_batch["x"]
    .to(
        device=device,
        dtype=torch.float32,
        non_blocking=True,
    )
)

epsilon_warm = (
    warm_batch["epsilon"]
    .to(
        device=device,
        dtype=torch.float32,
        non_blocking=True,
    )
)


with torch.inference_mode():

    (
        warm_logits,
        warm_mu,
        warm_logvar,
        warm_z,
    ) = val_benchmark_model(
        x_warm,
        epsilon=epsilon_warm,
        mean_mode=False,
    )


torch.cuda.synchronize()


del warm_batch
del x_warm
del epsilon_warm
del warm_logits
del warm_mu
del warm_logvar
del warm_z

gc.collect()
torch.cuda.empty_cache()


# ============================================================
# STOCHASTIC + EVAL BENCHMARK
# ============================================================

stochastic_times = []
stochastic_throughputs = []
stochastic_peaks_allocated = []
stochastic_peaks_reserved = []

stochastic_gates = []


for repeat in range(
    VAL_MEASURED_REPEATS
):

    sample_count = 0
    batch_count = 0
    tail_size = None

    first_batch_gate = False
    first_epsilon_gate = False

    losses_finite = True


    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    start_time = (
        time.perf_counter()
    )


    with torch.inference_mode():

        for batch_index, batch in enumerate(
            val_loader_stochastic
        ):

            # ------------------------------------------------
            # FIRST BATCH IDENTITY
            # ------------------------------------------------

            if batch_index == 0:

                manifests = (
                    batch[
                        "manifest_index"
                    ].tolist()
                )

                crops = (
                    batch[
                        "crop_left"
                    ].tolist()
                )


                first_batch_gate = (
                    manifests[:4]
                    == [
                        6864,
                        1488,
                        8384,
                        6585,
                    ]

                    and crops[:4]
                    == [
                        24,
                        24,
                        24,
                        24,
                    ]
                )


                expected_epsilon_np = np.stack(
                    [
                        python_latent_epsilon(
                            PROJECT_LATENT_SEED,
                            RNG_DOMAIN_LATENT_VALIDATION,
                            0,
                            manifest_index,
                            SPEC.latent_dim,
                        )

                        for manifest_index
                        in manifests[:4]
                    ],
                    axis=0,
                )


                actual_epsilon_np = (
                    batch[
                        "epsilon"
                    ][
                        :4
                    ]
                    .numpy()
                )


                first_epsilon_gate = (
                    np.array_equal(
                        actual_epsilon_np.view(
                            np.uint32
                        ),
                        expected_epsilon_np.view(
                            np.uint32
                        ),
                    )
                )


            # ------------------------------------------------
            # H2D
            # ------------------------------------------------

            x_batch = (
                batch["x"]
                .to(
                    device=device,
                    dtype=torch.float32,
                    non_blocking=True,
                )
            )

            epsilon_batch = (
                batch["epsilon"]
                .to(
                    device=device,
                    dtype=torch.float32,
                    non_blocking=True,
                )
            )


            current_batch_size = int(
                x_batch.shape[0]
            )


            # ------------------------------------------------
            # MODEL
            # ------------------------------------------------

            (
                logits,
                mu,
                logvar,
                z,
            ) = val_benchmark_model(
                x_batch,
                epsilon=epsilon_batch,
                mean_mode=False,
            )


            stats = sakai_vae_loss(
                logits,
                x_batch,
                mu,
                logvar,
                beta_kl=SPEC.beta_kl,
            )


            if not torch.isfinite(
                stats["loss"]
            ).item():

                losses_finite = False

                raise RuntimeError(
                    "Non-finite stochastic validation loss"
                )


            sample_count += (
                current_batch_size
            )

            batch_count += 1

            tail_size = (
                current_batch_size
            )


    torch.cuda.synchronize()


    elapsed = (
        time.perf_counter()
        - start_time
    )


    epoch_gate = (
        sample_count
            == EXPECTED_VAL_SAMPLES

        and batch_count
            == EXPECTED_VAL_BATCHES

        and tail_size
            == EXPECTED_VAL_TAIL

        and first_batch_gate

        and first_epsilon_gate

        and losses_finite
    )


    assert epoch_gate


    stochastic_times.append(
        elapsed
    )

    stochastic_throughputs.append(
        sample_count / elapsed
    )

    stochastic_peaks_allocated.append(
        torch.cuda.max_memory_allocated()
        / (1024 ** 3)
    )

    stochastic_peaks_reserved.append(
        torch.cuda.max_memory_reserved()
        / (1024 ** 3)
    )

    stochastic_gates.append(
        epoch_gate
    )


# ============================================================
# MEAN + EVAL BENCHMARK
#
# One full pass is sufficient because this is a periodic
# diagnostic and contains no epsilon generation.
# ============================================================

mean_sample_count = 0
mean_batch_count = 0
mean_tail_size = None
mean_losses_finite = True


torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()

mean_start_time = (
    time.perf_counter()
)


with torch.inference_mode():

    for batch_index, batch in enumerate(
        val_loader_mean
    ):

        x_batch = (
            batch["x"]
            .to(
                device=device,
                dtype=torch.float32,
                non_blocking=True,
            )
        )


        current_batch_size = int(
            x_batch.shape[0]
        )


        (
            logits,
            mu,
            logvar,
            z,
        ) = val_benchmark_model(
            x_batch,
            epsilon=None,
            mean_mode=True,
        )


        if not torch.equal(
            z,
            mu,
        ):
            raise RuntimeError(
                "MEAN validation z != mu"
            )


        stats = sakai_vae_loss(
            logits,
            x_batch,
            mu,
            logvar,
            beta_kl=SPEC.beta_kl,
        )


        if not torch.isfinite(
            stats["loss"]
        ).item():

            mean_losses_finite = False

            raise RuntimeError(
                "Non-finite MEAN validation loss"
            )


        mean_sample_count += (
            current_batch_size
        )

        mean_batch_count += 1

        mean_tail_size = (
            current_batch_size
        )


torch.cuda.synchronize()


mean_elapsed = (
    time.perf_counter()
    - mean_start_time
)


mean_gate = (
    mean_sample_count
        == EXPECTED_VAL_SAMPLES

    and mean_batch_count
        == EXPECTED_VAL_BATCHES

    and mean_tail_size
        == EXPECTED_VAL_TAIL

    and mean_losses_finite
)


assert mean_gate


# ============================================================
# MODEL INVARIANCE
# ============================================================

val_final_hash = (
    project_parameter_sha256(
        val_benchmark_model
    )
)


parameters_unchanged = (
    val_final_hash
    == FROZEN_INIT_SHA256
)


assert parameters_unchanged


bn_after_validation = (
    snapshot_validation_bn()
)


bn_unchanged = all(
    torch.equal(
        before_tensor,
        after_tensor,
    )

    for before, after
    in zip(
        bn_before_validation,
        bn_after_validation,
    )

    for before_tensor, after_tensor
    in zip(
        before,
        after,
    )
)


assert bn_unchanged


# ============================================================
# TIMING SUMMARY
# ============================================================

median_stochastic_seconds = (
    statistics.median(
        stochastic_times
    )
)

median_stochastic_sample_s = (
    EXPECTED_VAL_SAMPLES
    / median_stochastic_seconds
)


# Train steady-state already measured in B3.
steady_train_seconds = (
    e2e_epoch_results[1][
        "elapsed_seconds"
    ]
)


# PROJECT currently performs stochastic validation each epoch
# and a MEAN diagnostic every five epochs.
mean_diagnostic_count_100_epochs = 20


estimated_100_epoch_core_seconds = (
    100
    * (
        steady_train_seconds
        + median_stochastic_seconds
    )

    + mean_diagnostic_count_100_epochs
    * mean_elapsed
)


# This estimate deliberately excludes checkpoint write time,
# startup/bootstrap and any final post-training analyses.
estimated_100_epoch_core_minutes = (
    estimated_100_epoch_core_seconds
    / 60.0
)


# ============================================================
# REPORT
# ============================================================

print(
    "=== FULL VALIDATION BENCHMARK ==="
)

print(
    "VAL_SAMPLES =",
    EXPECTED_VAL_SAMPLES,
)

print(
    "VAL_BATCHES =",
    EXPECTED_VAL_BATCHES,
)

print(
    "VAL_TAIL =",
    EXPECTED_VAL_TAIL,
)

print()

print(
    "STOCHASTIC_TIMES_S =",
    [
        round(value, 4)
        for value
        in stochastic_times
    ],
)

print(
    "STOCHASTIC_MEDIAN_S =",
    round(
        median_stochastic_seconds,
        4,
    ),
)

print(
    "STOCHASTIC_SAMPLE_S =",
    round(
        median_stochastic_sample_s,
        3,
    ),
)

print(
    "STOCHASTIC_PEAK_ALLOCATED_GIB =",
    round(
        max(
            stochastic_peaks_allocated
        ),
        3,
    ),
)

print(
    "STOCHASTIC_PEAK_RESERVED_GIB =",
    round(
        max(
            stochastic_peaks_reserved
        ),
        3,
    ),
)

print()

print(
    "MEAN_EVAL_SECONDS =",
    round(
        mean_elapsed,
        4,
    ),
)

print(
    "MEAN_EVAL_SAMPLE_S =",
    round(
        EXPECTED_VAL_SAMPLES
        / mean_elapsed,
        3,
    ),
)

print()

print(
    "PARAMETERS_UNCHANGED =",
    parameters_unchanged,
)

print(
    "BN_STATE_UNCHANGED =",
    bn_unchanged,
)

print()

print(
    "TRAIN_STEADY_STATE_SECONDS =",
    round(
        steady_train_seconds,
        4,
    ),
)

print(
    "ESTIMATED_100_EPOCH_CORE_MINUTES =",
    round(
        estimated_100_epoch_core_minutes,
        2,
    ),
)

print(
    "ESTIMATE_EXCLUDES_CHECKPOINT_IO =",
    True,
)

print()

print(
    "BACKWARD_EXECUTED =",
    False,
)

print(
    "OPTIMIZER_EXECUTED =",
    False,
)

print(
    "OFFICIAL_TRAINING_STARTED =",
    False,
)

print()

print(
    "NOTEBOOK_03_VALIDATION_BENCHMARK_PASS =",
    (
        all(
            stochastic_gates
        )
        and mean_gate
        and parameters_unchanged
        and bn_unchanged
    ),
)


# ============================================================
# RELEASE TEMPORARY VALIDATION MODEL
# ============================================================

del val_benchmark_model

gc.collect()
torch.cuda.empty_cache()

=== FULL VALIDATION BENCHMARK ===
VAL_SAMPLES = 1717
VAL_BATCHES = 14
VAL_TAIL = 53

STOCHASTIC_TIMES_S = [1.5585, 1.61]
STOCHASTIC_MEDIAN_S = 1.5843
STOCHASTIC_SAMPLE_S = 1083.788
STOCHASTIC_PEAK_ALLOCATED_GIB = 2.911
STOCHASTIC_PEAK_RESERVED_GIB = 5.105

MEAN_EVAL_SECONDS = 1.2533
MEAN_EVAL_SAMPLE_S = 1369.929

PARAMETERS_UNCHANGED = True
BN_STATE_UNCHANGED = True

TRAIN_STEADY_STATE_SECONDS = 9.4785
ESTIMATED_100_EPOCH_CORE_MINUTES = 18.86
ESTIMATE_EXCLUDES_CHECKPOINT_IO = True

BACKWARD_EXECUTED = False
OPTIMIZER_EXECUTED = False
OFFICIAL_TRAINING_STARTED = False

NOTEBOOK_03_VALIDATION_BENCHMARK_PASS = True


## STEP 03-B5 — Contratto congelato della run `sakai224_full`

I benchmark su A100 hanno concluso la fase di scelta della configurazione.

### PAPER — Sakai Case 3

- input `4×224×224`;
- latent dimension `512`;
- Adam;
- learning rate `5×10⁻⁴`;
- `β1 = 0.9`;
- `β2 = 0.999`;
- `ε = 10⁻⁸`;
- minibatch `128`;
- encoder del VAE a **100 epoche** utilizzato successivamente per l'IIC.

### PROJECT — configurazione full congelata

- full split: `6866 train / 1717 validation`;
- batch `128`;
- 100 epoche complete, indicizzate `0...99`;
- nessun early stopping;
- modello scientifico ufficiale: stato dopo epoch `99`, cioè **100 epoche completate**;
- eventuale checkpoint `BEST`: esclusivamente diagnostico;
- validation `STOCHASTIC+EVAL` completa dopo ogni epoca;
- diagnostica `MEAN+EVAL` ogni 5 epoche;
- validation iniziale prima del training per misurare il miglioramento rispetto allo stato iniziale;
- `num_workers=8`, `pin_memory=True`, `persistent_workers=False`;
- float32 puro, AMP e TF32 disabilitati;
- seed PROJECT: initialization `123`, latent `456`, augmentation `789`, shuffle `101112`.

Con batch 128:

- train: `54` optimizer step per epoca;
- validation: `14` batch;
- budget finale: `5400` optimizer step dopo 100 epoche.

Il checkpointing viene definito nello step successivo e non modifica la numerica della run.

In [14]:
# ============================================================
# NOTEBOOK 03 — FROZEN SCIENTIFIC RUN CONTRACT
# STEP 03-B5
#
# Freeze the numerically relevant sakai224_full configuration.
#
# NO model execution.
# NO checkpoint write.
# NO official training.
# ============================================================

import hashlib
import json
import math


# ============================================================
# SCIENTIFIC TRAINING CONTROLLER
# ============================================================

RUN_ID = "sakai224_full_v1"

FROZEN_MAX_EPOCHS = 100

# Python epochs are indexed 0..99.
FROZEN_FIRST_EPOCH_INDEX = 0
FROZEN_FINAL_EPOCH_INDEX = 99

# Sakai uses the encoder at 100 completed epochs.
OFFICIAL_MODEL_SELECTION = "EPOCH_100"

EARLY_STOPPING_ENABLED = False

# BEST remains useful diagnostically, but never replaces
# epoch-100 as the official full-Sakai model.
BEST_CHECKPOINT_ROLE = "DIAGNOSTIC_ONLY"


# ============================================================
# FROZEN DATA / BATCH CONTRACT
# ============================================================

FROZEN_TRAIN_SAMPLES = 6866
FROZEN_VALIDATION_SAMPLES = 1717

FROZEN_BATCH_SIZE = 128

FROZEN_TRAIN_BATCHES = math.ceil(
    FROZEN_TRAIN_SAMPLES
    / FROZEN_BATCH_SIZE
)

FROZEN_VALIDATION_BATCHES = math.ceil(
    FROZEN_VALIDATION_SAMPLES
    / FROZEN_BATCH_SIZE
)

FROZEN_TRAIN_TAIL = (
    FROZEN_TRAIN_SAMPLES
    % FROZEN_BATCH_SIZE
)

FROZEN_VALIDATION_TAIL = (
    FROZEN_VALIDATION_SAMPLES
    % FROZEN_BATCH_SIZE
)

FROZEN_TOTAL_OPTIMIZER_STEPS = (
    FROZEN_MAX_EPOCHS
    * FROZEN_TRAIN_BATCHES
)


assert FROZEN_TRAIN_BATCHES == 54
assert FROZEN_VALIDATION_BATCHES == 14
assert FROZEN_TRAIN_TAIL == 82
assert FROZEN_VALIDATION_TAIL == 53
assert FROZEN_TOTAL_OPTIMIZER_STEPS == 5400


# ============================================================
# VALIDATION POLICY
# ============================================================

INITIAL_VALIDATION_ENABLED = True

STOCHASTIC_VALIDATION_EVERY_EPOCH = True

MEAN_DIAGNOSTIC_EVERY_EPOCHS = 5

# After completed epochs 5,10,...,100.
MEAN_DIAGNOSTIC_EPOCH_INDICES = list(
    range(
        4,
        FROZEN_MAX_EPOCHS,
        MEAN_DIAGNOSTIC_EVERY_EPOCHS,
    )
)

assert len(
    MEAN_DIAGNOSTIC_EPOCH_INDICES
) == 20

assert (
    MEAN_DIAGNOSTIC_EPOCH_INDICES[-1]
    == 99
)


# ============================================================
# COMPUTE / LOADER POLICY
# ============================================================

FROZEN_NUM_WORKERS = 8
FROZEN_PIN_MEMORY = True
FROZEN_PERSISTENT_WORKERS = False

FROZEN_DTYPE = "float32"
FROZEN_AMP_ENABLED = False
FROZEN_TF32_ENABLED = False


# ============================================================
# PAPER / PROJECT OPTIMIZER
# ============================================================

FROZEN_LEARNING_RATE = 5.0e-4

FROZEN_ADAM_BETA1 = 0.9
FROZEN_ADAM_BETA2 = 0.999
FROZEN_ADAM_EPSILON = 1.0e-8

FROZEN_WEIGHT_DECAY = 0.0
FROZEN_AMSGRAD = False


# ============================================================
# PROJECT SEEDS
# ============================================================

FROZEN_INITIALIZATION_SEED = 123
FROZEN_LATENT_SEED = 456
FROZEN_AUGMENTATION_SEED = 789
FROZEN_SHUFFLE_SEED = 101112


# ============================================================
# DATA PROVENANCE
# ============================================================

FROZEN_EVENT_MANIFEST_SHA256 = (
    "620960e6b35da31bc4cf1a8a6da86790"
    "27d378e1d4b2ac73c5267c1c5311ae82"
)

FROZEN_TENSOR_MANIFEST_SHA256 = (
    "94b7e2e051bd32de72dd2cb6a5fb7eda"
    "9bcaec123cbf3a07d0d901bd19779afa"
)

FROZEN_TRAIN_SPLIT_SHA256 = (
    "8c7a513b98dbe996261bc80965500ab15"
    "ab77f498ec9440858003152e42f926f"
)

FROZEN_VALIDATION_SPLIT_SHA256 = (
    "e63109c88df7b3ee97317f94791f26df"
    "3fcaf8b937921cfb27967d01c2546467"
)


FROZEN_MODEL_INITIAL_SHA256 = (
    "1df6f7d32b4d0705c273b5bfae7c321"
    "8f72ff7cd1d160560b497afb3f235859b"
)


# C repository used as the frozen PROJECT semantic reference.
C_REFERENCE_COMMIT = (
    "99f95aacca5e2a146585ccac9f7e2e2bc1cc8256"
)


# ============================================================
# CANONICAL SCIENTIFIC CONTRACT
#
# Hardware and checkpoint frequency are deliberately excluded:
# they affect execution/provenance, but not the mathematical
# trajectory when the numerical contract above is unchanged.
# ============================================================

scientific_contract = {

    "contract_version":
        "sakai224_full_scientific_v1",

    "run_id":
        RUN_ID,

    "model": {
        "input":
            [4, 224, 224],

        "encoder_channels":
            [64, 128, 256, 512],

        "latent_dim":
            512,

        "initial_parameter_sha256":
            FROZEN_MODEL_INITIAL_SHA256,
    },

    "data": {
        "dataset":
            "GravitySpy_O1_8583_legacy_luma_v1",

        "train_samples":
            FROZEN_TRAIN_SAMPLES,

        "validation_samples":
            FROZEN_VALIDATION_SAMPLES,

        "event_manifest_sha256":
            FROZEN_EVENT_MANIFEST_SHA256,

        "tensor_manifest_sha256":
            FROZEN_TENSOR_MANIFEST_SHA256,

        "train_split_sha256":
            FROZEN_TRAIN_SPLIT_SHA256,

        "validation_split_sha256":
            FROZEN_VALIDATION_SPLIT_SHA256,
    },

    "training": {
        "epochs":
            FROZEN_MAX_EPOCHS,

        "official_model_selection":
            OFFICIAL_MODEL_SELECTION,

        "early_stopping":
            EARLY_STOPPING_ENABLED,

        "batch_size":
            FROZEN_BATCH_SIZE,

        "total_optimizer_steps":
            FROZEN_TOTAL_OPTIMIZER_STEPS,
    },

    "optimizer": {
        "name":
            "Adam",

        "learning_rate":
            FROZEN_LEARNING_RATE,

        "beta1":
            FROZEN_ADAM_BETA1,

        "beta2":
            FROZEN_ADAM_BETA2,

        "epsilon":
            FROZEN_ADAM_EPSILON,

        "weight_decay":
            FROZEN_WEIGHT_DECAY,

        "amsgrad":
            FROZEN_AMSGRAD,
    },

    "objective": {
        "reconstruction":
            "BCE_WITH_LOGITS_SUM",

        "kl":
            "GAUSSIAN_DIAGONAL_SUM",

        "batch_reduction":
            "MEAN_BATCH_ONLY",

        "beta_kl":
            1.0,
    },

    "rng": {
        "initialization_seed":
            FROZEN_INITIALIZATION_SEED,

        "latent_seed":
            FROZEN_LATENT_SEED,

        "augmentation_seed":
            FROZEN_AUGMENTATION_SEED,

        "shuffle_seed":
            FROZEN_SHUFFLE_SEED,

        "augmentation_policy":
            "STATELESS_SYMMETRIC_FIGURE_CROP_LEFT_0_48",

        "shuffle_policy":
            "STATELESS_FISHER_YATES_SPLIT_POSITIONS",
    },

    "validation": {
        "initial_validation":
            INITIAL_VALIDATION_ENABLED,

        "stochastic_every_epoch":
            STOCHASTIC_VALIDATION_EVERY_EPOCH,

        "mean_every_epochs":
            MEAN_DIAGNOSTIC_EVERY_EPOCHS,

        "official_validation_input":
            "CENTRAL_CROP_LEFT_24",
    },

    "numerics": {
        "dtype":
            FROZEN_DTYPE,

        "amp":
            FROZEN_AMP_ENABLED,

        "tf32":
            FROZEN_TF32_ENABLED,
    },

    "project_reference": {
        "c_repository_commit":
            C_REFERENCE_COMMIT,
    },
}


canonical_contract_json = json.dumps(
    scientific_contract,
    sort_keys=True,
    separators=(",", ":"),
    ensure_ascii=True,
)


SCIENTIFIC_CONTRACT_SHA256 = (
    hashlib.sha256(
        canonical_contract_json.encode(
            "utf-8"
        )
    ).hexdigest()
)


# ============================================================
# HARD GATES AGAINST MEASURED BENCHMARK RESULTS
# ============================================================

benchmark_contract_gate = (
    recommended_batch == 128

    and PROVISIONAL_NUM_WORKERS == 8

    and steady_train_seconds > 0.0

    and median_stochastic_seconds > 0.0

    and mean_elapsed > 0.0
)


assert benchmark_contract_gate


print(
    "=== FROZEN SAKAI224_FULL RUN CONTRACT ==="
)

print(
    "RUN_ID =",
    RUN_ID,
)

print(
    "EPOCHS =",
    FROZEN_MAX_EPOCHS,
)

print(
    "OFFICIAL_MODEL_SELECTION =",
    OFFICIAL_MODEL_SELECTION,
)

print(
    "EARLY_STOPPING_ENABLED =",
    EARLY_STOPPING_ENABLED,
)

print(
    "BEST_CHECKPOINT_ROLE =",
    BEST_CHECKPOINT_ROLE,
)

print()

print(
    "BATCH_SIZE =",
    FROZEN_BATCH_SIZE,
)

print(
    "NUM_WORKERS =",
    FROZEN_NUM_WORKERS,
)

print(
    "TRAIN_BATCHES_PER_EPOCH =",
    FROZEN_TRAIN_BATCHES,
)

print(
    "TRAIN_TAIL =",
    FROZEN_TRAIN_TAIL,
)

print(
    "VALIDATION_BATCHES =",
    FROZEN_VALIDATION_BATCHES,
)

print(
    "VALIDATION_TAIL =",
    FROZEN_VALIDATION_TAIL,
)

print(
    "TOTAL_OPTIMIZER_STEPS =",
    FROZEN_TOTAL_OPTIMIZER_STEPS,
)

print()

print(
    "INITIAL_VALIDATION =",
    INITIAL_VALIDATION_ENABLED,
)

print(
    "MEAN_DIAGNOSTIC_COUNT =",
    len(
        MEAN_DIAGNOSTIC_EPOCH_INDICES
    ),
)

print(
    "FINAL_MEAN_DIAGNOSTIC_EPOCH_INDEX =",
    MEAN_DIAGNOSTIC_EPOCH_INDICES[-1],
)

print()

print(
    "DTYPE =",
    FROZEN_DTYPE,
)

print(
    "AMP_ENABLED =",
    FROZEN_AMP_ENABLED,
)

print(
    "TF32_ENABLED =",
    FROZEN_TF32_ENABLED,
)

print()

print(
    "SCIENTIFIC_CONTRACT_SHA256 =",
    SCIENTIFIC_CONTRACT_SHA256,
)

print()

print(
    "BENCHMARK_CONTRACT_GATE =",
    benchmark_contract_gate,
)

print(
    "CHECKPOINT_IMPLEMENTED =",
    False,
)

print(
    "OFFICIAL_TRAINING_STARTED =",
    False,
)

print()

print(
    "NOTEBOOK_03_FROZEN_RUN_CONTRACT_PASS =",
    benchmark_contract_gate,
)

=== FROZEN SAKAI224_FULL RUN CONTRACT ===
RUN_ID = sakai224_full_v1
EPOCHS = 100
OFFICIAL_MODEL_SELECTION = EPOCH_100
EARLY_STOPPING_ENABLED = False
BEST_CHECKPOINT_ROLE = DIAGNOSTIC_ONLY

BATCH_SIZE = 128
NUM_WORKERS = 8
TRAIN_BATCHES_PER_EPOCH = 54
TRAIN_TAIL = 82
VALIDATION_BATCHES = 14
VALIDATION_TAIL = 53
TOTAL_OPTIMIZER_STEPS = 5400

INITIAL_VALIDATION = True
MEAN_DIAGNOSTIC_COUNT = 20
FINAL_MEAN_DIAGNOSTIC_EPOCH_INDEX = 99

DTYPE = float32
AMP_ENABLED = False
TF32_ENABLED = False

SCIENTIFIC_CONTRACT_SHA256 = c895b626454278f2293e217776c4604d24e03fa78dbaa7ba5049f86f917af949

BENCHMARK_CONTRACT_GATE = True
CHECKPOINT_IMPLEMENTED = False
OFFICIAL_TRAINING_STARTED = False

NOTEBOOK_03_FROZEN_RUN_CONTRACT_PASS = True


## STEP 03-C0 — Checkpoint persistente e round-trip

La run ufficiale deve poter sopravvivere alla perdita completa del runtime Colab.

Il checkpoint PyTorch conserva tutto lo stato necessario a riprendere **al confine tra epoche**:

- parametri del VAE;
- buffer BatchNorm;
- stato completo Adam;
- ultimo epoch completato;
- optimizer step globale;
- stato diagnostico BEST;
- `RUN_ID`;
- SHA-256 del contratto scientifico congelato.

Gli RNG PROJECT non richiedono serializzazione perché sono **stateless** e dipendono da seed, epoch e `manifest_index`.

### Politica atomica

Il salvataggio segue la filosofia già validata nel codice C:

`costruzione → file .tmp → rilettura/verifica → pubblicazione`

Un checkpoint precedente valido non viene sostituito finché il nuovo file temporaneo non è stato verificato.

Questo step utilizza un modello temporaneo e una sola epoca reale di test.  
Non costituisce la run scientifica ufficiale.

In [15]:
# ============================================================
# NOTEBOOK 03 — PERSISTENT CHECKPOINT ROUND-TRIP
# STEP 03-C0
#
# Disposable test:
#   real epoch 0
#   -> stochastic validation
#   -> atomic checkpoint to persistent project storage
#   -> fresh model/optimizer
#   -> load
#   -> exact state comparison
#
# NO official training.
# ============================================================

from pathlib import Path
import gc
import hashlib
import os
import struct

import torch
from torch.utils.data import DataLoader


# ============================================================
# CHECKPOINT FORMAT
# ============================================================

PYTORCH_CHECKPOINT_FORMAT_VERSION = 1

CHECKPOINT_TEST_ROOT = (
    SAKAI224_ROOT
    / "checkpoint_tests"
    / "sakai224_full_roundtrip_v1"
)

CHECKPOINT_TEST_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

CHECKPOINT_TEST_PATH = (
    CHECKPOINT_TEST_ROOT
    / "latest_test.pt"
)


# ============================================================
# GENERIC CANONICAL HASH FOR PYTORCH STATE
#
# Handles:
#   tensors
#   dict
#   list / tuple
#   strings
#   bool / int / float / None
#
# The hash does not depend on CPU vs CUDA placement.
# ============================================================

def canonical_tree_sha256(obj):

    digest = hashlib.sha256()

    def update(value):

        if torch.is_tensor(value):

            tensor = (
                value
                .detach()
                .cpu()
                .contiguous()
            )

            digest.update(b"TENSOR\0")

            digest.update(
                str(tensor.dtype).encode(
                    "utf-8"
                )
            )

            digest.update(b"\0")

            digest.update(
                repr(
                    tuple(tensor.shape)
                ).encode("utf-8")
            )

            digest.update(b"\0")

            digest.update(
                tensor.numpy().tobytes(
                    order="C"
                )
            )

            return


        if isinstance(value, dict):

            digest.update(b"DICT\0")

            for key in sorted(
                value.keys(),
                key=lambda item:
                    (
                        type(item).__name__,
                        repr(item),
                    ),
            ):

                update(key)
                update(value[key])

            return


        if isinstance(
            value,
            (list, tuple),
        ):

            digest.update(
                b"LIST\0"
                if isinstance(value, list)
                else b"TUPLE\0"
            )

            for item in value:
                update(item)

            return


        if value is None:

            digest.update(b"NONE\0")
            return


        if isinstance(value, bool):

            digest.update(
                b"BOOL1\0"
                if value
                else b"BOOL0\0"
            )

            return


        if isinstance(value, int):

            digest.update(b"INT\0")

            digest.update(
                str(value).encode(
                    "ascii"
                )
            )

            digest.update(b"\0")
            return


        if isinstance(value, float):

            digest.update(b"FLOAT64\0")

            digest.update(
                struct.pack(
                    "<d",
                    value,
                )
            )

            return


        if isinstance(value, str):

            encoded = value.encode(
                "utf-8"
            )

            digest.update(b"STR\0")

            digest.update(
                len(encoded).to_bytes(
                    8,
                    "little",
                )
            )

            digest.update(
                encoded
            )

            return


        raise TypeError(
            "Unsupported checkpoint hash type: "
            f"{type(value)}"
        )


    update(obj)

    return digest.hexdigest()


# ============================================================
# CPU SNAPSHOT
#
# Checkpoints are stored device-independently.
# ============================================================

def tree_to_cpu(obj):

    if torch.is_tensor(obj):

        return (
            obj
            .detach()
            .cpu()
            .clone()
        )


    if isinstance(obj, dict):

        return {
            key:
                tree_to_cpu(value)

            for key, value
            in obj.items()
        }


    if isinstance(obj, list):

        return [
            tree_to_cpu(value)
            for value in obj
        ]


    if isinstance(obj, tuple):

        return tuple(
            tree_to_cpu(value)
            for value in obj
        )


    return obj


# ============================================================
# OPTIMIZER GLOBAL STEP
#
# All Adam parameter states must agree.
# ============================================================

def get_adam_global_step(
    optimizer,
):

    steps = []


    for parameter in (
        optimizer.param_groups[0][
            "params"
        ]
    ):

        state = optimizer.state.get(
            parameter,
            None,
        )

        if not state:
            continue

        steps.append(
            int(
                state["step"].item()
            )
        )


    if not steps:
        return 0


    if len(
        set(steps)
    ) != 1:

        raise RuntimeError(
            "Adam parameter steps disagree"
        )


    return steps[0]


# ============================================================
# CHECKPOINT PAYLOAD
# ============================================================

def build_checkpoint_payload(
    model,
    optimizer,
    *,
    kind,
    last_completed_epoch,
    best_defined,
    best_epoch,
    best_validation_total_mean_per_sample,
):

    model_state = tree_to_cpu(
        model.state_dict()
    )

    optimizer_state = tree_to_cpu(
        optimizer.state_dict()
    )


    model_state_sha256 = (
        canonical_tree_sha256(
            model_state
        )
    )

    optimizer_state_sha256 = (
        canonical_tree_sha256(
            optimizer_state
        )
    )


    optimizer_step = (
        get_adam_global_step(
            optimizer
        )
    )


    payload = {

        "format_version":
            PYTORCH_CHECKPOINT_FORMAT_VERSION,

        "kind":
            str(kind),

        "run_id":
            RUN_ID,

        "scientific_contract_sha256":
            SCIENTIFIC_CONTRACT_SHA256,

        "last_completed_epoch":
            int(
                last_completed_epoch
            ),

        "optimizer_step":
            int(
                optimizer_step
            ),

        "best_defined":
            bool(
                best_defined
            ),

        "best_epoch":
            (
                int(best_epoch)
                if best_defined
                else -1
            ),

        "best_validation_total_mean_per_sample":
            (
                float(
                    best_validation_total_mean_per_sample
                )
                if best_defined
                else float("inf")
            ),

        "model_state_sha256":
            model_state_sha256,

        "optimizer_state_sha256":
            optimizer_state_sha256,

        "model_state_dict":
            model_state,

        "optimizer_state_dict":
            optimizer_state,

        # Provenance only; these fields are not the
        # mathematical compatibility authority.
        "provenance": {
            "torch_version":
                str(torch.__version__),

            "cuda_version":
                str(torch.version.cuda),

            "gpu_name":
                str(
                    torch.cuda.get_device_name(
                        0
                    )
                ),

            "c_reference_commit":
                C_REFERENCE_COMMIT,
        },
    }


    return payload


# ============================================================
# PAYLOAD VALIDATION
# ============================================================

def validate_checkpoint_payload(
    payload,
    *,
    expected_kind=None,
):

    required = {
        "format_version",
        "kind",
        "run_id",
        "scientific_contract_sha256",
        "last_completed_epoch",
        "optimizer_step",
        "best_defined",
        "best_epoch",
        "best_validation_total_mean_per_sample",
        "model_state_sha256",
        "optimizer_state_sha256",
        "model_state_dict",
        "optimizer_state_dict",
        "provenance",
    }


    if set(
        payload.keys()
    ) != required:

        raise RuntimeError(
            "Checkpoint field set mismatch"
        )


    if (
        payload[
            "format_version"
        ]
        != PYTORCH_CHECKPOINT_FORMAT_VERSION
    ):

        raise RuntimeError(
            "Checkpoint format version mismatch"
        )


    if payload["run_id"] != RUN_ID:

        raise RuntimeError(
            "Checkpoint RUN_ID mismatch"
        )


    if (
        payload[
            "scientific_contract_sha256"
        ]
        != SCIENTIFIC_CONTRACT_SHA256
    ):

        raise RuntimeError(
            "Scientific contract mismatch"
        )


    if (
        expected_kind is not None
        and payload["kind"]
        != expected_kind
    ):

        raise RuntimeError(
            "Checkpoint kind mismatch"
        )


    model_hash = canonical_tree_sha256(
        payload[
            "model_state_dict"
        ]
    )

    optimizer_hash = canonical_tree_sha256(
        payload[
            "optimizer_state_dict"
        ]
    )


    if (
        model_hash
        != payload[
            "model_state_sha256"
        ]
    ):

        raise RuntimeError(
            "Model checkpoint checksum mismatch"
        )


    if (
        optimizer_hash
        != payload[
            "optimizer_state_sha256"
        ]
    ):

        raise RuntimeError(
            "Optimizer checkpoint checksum mismatch"
        )


    # Epoch-boundary invariant.
    last_epoch = int(
        payload[
            "last_completed_epoch"
        ]
    )

    expected_step = (
        (last_epoch + 1)
        * FROZEN_TRAIN_BATCHES
    )


    if (
        payload[
            "optimizer_step"
        ]
        != expected_step
    ):

        raise RuntimeError(
            "Checkpoint epoch/optimizer-step mismatch"
        )


    return True


# ============================================================
# ATOMIC SAVE
#
# Mirrors the PROJECT C philosophy:
#
#   write .tmp
#   -> flush/fsync
#   -> read back
#   -> validate
#   -> publish via os.replace()
#
# A stale .tmp is NEVER silently overwritten.
# ============================================================

def save_checkpoint_atomic(
    path,
    payload,
):

    path = Path(path)

    temporary_path = Path(
        str(path) + ".tmp"
    )


    if temporary_path.exists():

        raise RuntimeError(
            "STOP: stale checkpoint .tmp exists: "
            f"{temporary_path}"
        )


    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    try:

        with temporary_path.open(
            "xb"
        ) as handle:

            torch.save(
                payload,
                handle,
            )

            handle.flush()

            os.fsync(
                handle.fileno()
            )


        # Independent read-back from the persistent file.
        verification_payload = torch.load(
            temporary_path,
            map_location="cpu",
            weights_only=True,
        )


        validate_checkpoint_payload(
            verification_payload,
            expected_kind=payload[
                "kind"
            ],
        )


        if (
            canonical_tree_sha256(
                verification_payload[
                    "model_state_dict"
                ]
            )
            != payload[
                "model_state_sha256"
            ]
        ):

            raise RuntimeError(
                "Checkpoint model read-back mismatch"
            )


        if (
            canonical_tree_sha256(
                verification_payload[
                    "optimizer_state_dict"
                ]
            )
            != payload[
                "optimizer_state_sha256"
            ]
        ):

            raise RuntimeError(
                "Checkpoint optimizer read-back mismatch"
            )


        # The old published checkpoint remains valid until
        # this operation succeeds.
        os.replace(
            temporary_path,
            path,
        )


    except Exception:

        # Preserve a stale .tmp when failure occurs after it
        # was created. This makes an interrupted publication
        # visible rather than silently hiding it.
        raise


    return path


# ============================================================
# CHECKPOINT LOAD INTO AN ALLOCATED MODEL / OPTIMIZER
# ============================================================

def load_checkpoint_into(
    path,
    model,
    optimizer,
    *,
    expected_kind,
    device,
):

    payload = torch.load(
        path,
        map_location=device,
        weights_only=True,
    )


    validate_checkpoint_payload(
        payload,
        expected_kind=expected_kind,
    )


    model.load_state_dict(
        payload[
            "model_state_dict"
        ],
        strict=True,
    )


    optimizer.load_state_dict(
        payload[
            "optimizer_state_dict"
        ]
    )


    # Validate the actual committed destination objects.
    loaded_model_hash = (
        canonical_tree_sha256(
            model.state_dict()
        )
    )

    loaded_optimizer_hash = (
        canonical_tree_sha256(
            optimizer.state_dict()
        )
    )


    if (
        loaded_model_hash
        != payload[
            "model_state_sha256"
        ]
    ):

        raise RuntimeError(
            "Loaded model differs from checkpoint"
        )


    if (
        loaded_optimizer_hash
        != payload[
            "optimizer_state_sha256"
        ]
    ):

        raise RuntimeError(
            "Loaded optimizer differs from checkpoint"
        )


    return payload


# ============================================================
# DISPOSABLE REAL TRAINING MODEL
# ============================================================

checkpoint_test_model = (
    Sakai224FullVAE(
        SPEC
    )
)

initialize_project_c_exact_v1(
    checkpoint_test_model,
    PROJECT_INITIALIZATION_SEED,
)

assert (
    project_parameter_sha256(
        checkpoint_test_model
    )
    == FROZEN_INIT_SHA256
)


checkpoint_test_model = (
    checkpoint_test_model.to(
        device=device,
        dtype=torch.float32,
    )
)

checkpoint_test_model.train()


checkpoint_test_optimizer = (
    torch.optim.Adam(
        checkpoint_test_model.parameters(),

        lr=FROZEN_LEARNING_RATE,

        betas=(
            FROZEN_ADAM_BETA1,
            FROZEN_ADAM_BETA2,
        ),

        eps=FROZEN_ADAM_EPSILON,

        weight_decay=FROZEN_WEIGHT_DECAY,

        amsgrad=FROZEN_AMSGRAD,

        foreach=False,

        fused=False,
    )
)


# ============================================================
# TEST TRAIN DATASET / LOADER
# ============================================================

checkpoint_train_dataset = (
    GravitySpy224TrainWithEpsilon(
        train_dataset,
        epoch=0,
    )
)


def make_checkpoint_train_loader(
    epoch,
):

    epoch = int(epoch)

    train_sampler.set_epoch(
        epoch
    )

    checkpoint_train_dataset.set_epoch(
        epoch
    )


    return DataLoader(
        checkpoint_train_dataset,

        batch_size=FROZEN_BATCH_SIZE,

        sampler=train_sampler,

        num_workers=FROZEN_NUM_WORKERS,

        pin_memory=FROZEN_PIN_MEMORY,

        drop_last=False,

        persistent_workers=False,

        prefetch_factor=2,
    )


# ============================================================
# GENERIC ONE-EPOCH TRAIN HELPER
# ============================================================

def checkpoint_test_train_epoch(
    model,
    optimizer,
    epoch,
):

    model.train()

    loader = (
        make_checkpoint_train_loader(
            epoch
        )
    )


    sample_count = 0
    batch_count = 0


    for batch in loader:

        x_batch = (
            batch["x"]
            .to(
                device=device,
                dtype=torch.float32,
                non_blocking=True,
            )
        )

        epsilon_batch = (
            batch["epsilon"]
            .to(
                device=device,
                dtype=torch.float32,
                non_blocking=True,
            )
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        (
            logits,
            mu,
            logvar,
            z,
        ) = model(
            x_batch,
            epsilon=epsilon_batch,
            mean_mode=False,
        )


        stats = sakai_vae_loss(
            logits,
            x_batch,
            mu,
            logvar,
            beta_kl=SPEC.beta_kl,
        )


        loss = stats["loss"]


        if not torch.isfinite(
            loss
        ).item():

            raise RuntimeError(
                "Non-finite checkpoint-test train loss"
            )


        loss.backward()

        optimizer.step()


        sample_count += int(
            x_batch.shape[0]
        )

        batch_count += 1


    torch.cuda.synchronize()


    assert (
        sample_count
        == FROZEN_TRAIN_SAMPLES
    )

    assert (
        batch_count
        == FROZEN_TRAIN_BATCHES
    )


# ============================================================
# STOCHASTIC VALIDATION HELPER
# ============================================================

def checkpoint_test_validation(
    model,
):

    model.eval()

    total_sum = 0.0
    sample_count = 0


    with torch.inference_mode():

        for batch in (
            val_loader_stochastic
        ):

            x_batch = (
                batch["x"]
                .to(
                    device=device,
                    dtype=torch.float32,
                    non_blocking=True,
                )
            )

            epsilon_batch = (
                batch["epsilon"]
                .to(
                    device=device,
                    dtype=torch.float32,
                    non_blocking=True,
                )
            )


            (
                logits,
                mu,
                logvar,
                z,
            ) = model(
                x_batch,
                epsilon=epsilon_batch,
                mean_mode=False,
            )


            stats = sakai_vae_loss(
                logits,
                x_batch,
                mu,
                logvar,
                beta_kl=SPEC.beta_kl,
            )


            batch_total = (
                stats[
                    "reconstruction_sum"
                ]
                + stats["kl_sum"]
            )


            total_sum += float(
                batch_total
            )

            sample_count += int(
                x_batch.shape[0]
            )


    model.train()


    assert (
        sample_count
        == FROZEN_VALIDATION_SAMPLES
    )


    return (
        total_sum
        / sample_count
    )


# ============================================================
# TRAIN EXACTLY EPOCH 0
# ============================================================

print(
    "=== CHECKPOINT ROUND-TRIP TEST ==="
)

print(
    "Training disposable epoch 0..."
)


checkpoint_test_train_epoch(
    checkpoint_test_model,
    checkpoint_test_optimizer,
    epoch=0,
)


checkpoint_validation_mean = (
    checkpoint_test_validation(
        checkpoint_test_model
    )
)


assert (
    get_adam_global_step(
        checkpoint_test_optimizer
    )
    == 54
)


# ============================================================
# STATE BEFORE SAVE
# ============================================================

model_hash_before_save = (
    canonical_tree_sha256(
        checkpoint_test_model.state_dict()
    )
)

optimizer_hash_before_save = (
    canonical_tree_sha256(
        checkpoint_test_optimizer.state_dict()
    )
)


# ============================================================
# BUILD + SAVE LATEST TEST CHECKPOINT
# ============================================================

checkpoint_payload = (
    build_checkpoint_payload(
        checkpoint_test_model,
        checkpoint_test_optimizer,

        kind="LATEST",

        last_completed_epoch=0,

        best_defined=True,

        best_epoch=0,

        best_validation_total_mean_per_sample=(
            checkpoint_validation_mean
        ),
    )
)


assert validate_checkpoint_payload(
    checkpoint_payload,
    expected_kind="LATEST",
)


save_checkpoint_atomic(
    CHECKPOINT_TEST_PATH,
    checkpoint_payload,
)


assert CHECKPOINT_TEST_PATH.is_file()


checkpoint_file_sha256 = (
    sha256_file_streaming(
        CHECKPOINT_TEST_PATH
    )
)


# ============================================================
# FRESH MODEL / OPTIMIZER FOR LOAD TEST
# ============================================================

checkpoint_loaded_model = (
    Sakai224FullVAE(
        SPEC
    )
)

initialize_project_c_exact_v1(
    checkpoint_loaded_model,
    PROJECT_INITIALIZATION_SEED,
)


checkpoint_loaded_model = (
    checkpoint_loaded_model.to(
        device=device,
        dtype=torch.float32,
    )
)


checkpoint_loaded_optimizer = (
    torch.optim.Adam(
        checkpoint_loaded_model.parameters(),

        lr=FROZEN_LEARNING_RATE,

        betas=(
            FROZEN_ADAM_BETA1,
            FROZEN_ADAM_BETA2,
        ),

        eps=FROZEN_ADAM_EPSILON,

        weight_decay=FROZEN_WEIGHT_DECAY,

        amsgrad=FROZEN_AMSGRAD,

        foreach=False,

        fused=False,
    )
)


loaded_payload = load_checkpoint_into(
    CHECKPOINT_TEST_PATH,

    checkpoint_loaded_model,

    checkpoint_loaded_optimizer,

    expected_kind="LATEST",

    device=device,
)


# ============================================================
# EXACT ROUND-TRIP GATES
# ============================================================

loaded_model_hash = (
    canonical_tree_sha256(
        checkpoint_loaded_model.state_dict()
    )
)

loaded_optimizer_hash = (
    canonical_tree_sha256(
        checkpoint_loaded_optimizer.state_dict()
    )
)


model_roundtrip_exact = (
    loaded_model_hash
    == model_hash_before_save
)

optimizer_roundtrip_exact = (
    loaded_optimizer_hash
    == optimizer_hash_before_save
)


metadata_roundtrip_pass = (
    loaded_payload[
        "last_completed_epoch"
    ] == 0

    and loaded_payload[
        "optimizer_step"
    ] == 54

    and loaded_payload[
        "best_defined"
    ] is True

    and loaded_payload[
        "best_epoch"
    ] == 0

    and loaded_payload[
        "best_validation_total_mean_per_sample"
    ] == checkpoint_validation_mean

    and loaded_payload[
        "scientific_contract_sha256"
    ] == SCIENTIFIC_CONTRACT_SHA256
)


bn_counters_roundtrip = all(
    int(
        bn.num_batches_tracked.item()
    ) == 54

    for bn
    in get_project_batchnorm_layers(
        checkpoint_loaded_model
    )
)


assert model_roundtrip_exact
assert optimizer_roundtrip_exact
assert metadata_roundtrip_pass
assert bn_counters_roundtrip


# ============================================================
# MAIN CPU REFERENCE MUST REMAIN UNTOUCHED
# ============================================================

reference_model_unchanged = (
    project_parameter_sha256(
        model_reference
    )
    == FROZEN_INIT_SHA256
)


assert reference_model_unchanged


# ============================================================
# REPORT
# ============================================================

print()
print(
    "=== CHECKPOINT ROUND-TRIP AUDIT ==="
)

print(
    "CHECKPOINT_PATH =",
    CHECKPOINT_TEST_PATH,
)

print(
    "CHECKPOINT_FILE_BYTES =",
    CHECKPOINT_TEST_PATH.stat().st_size,
)

print(
    "CHECKPOINT_FILE_SHA256 =",
    checkpoint_file_sha256,
)

print()

print(
    "SCIENTIFIC_CONTRACT_SHA256 =",
    loaded_payload[
        "scientific_contract_sha256"
    ],
)

print(
    "LAST_COMPLETED_EPOCH =",
    loaded_payload[
        "last_completed_epoch"
    ],
)

print(
    "OPTIMIZER_STEP =",
    loaded_payload[
        "optimizer_step"
    ],
)

print(
    "BEST_EPOCH =",
    loaded_payload[
        "best_epoch"
    ],
)

print(
    "BEST_VALIDATION_TOTAL_MEAN =",
    loaded_payload[
        "best_validation_total_mean_per_sample"
    ],
)

print()

print(
    "MODEL_ROUNDTRIP_EXACT =",
    model_roundtrip_exact,
)

print(
    "OPTIMIZER_ROUNDTRIP_EXACT =",
    optimizer_roundtrip_exact,
)

print(
    "METADATA_ROUNDTRIP_PASS =",
    metadata_roundtrip_pass,
)

print(
    "BN_COUNTERS_ROUNDTRIP =",
    bn_counters_roundtrip,
)

print()

print(
    "LATEST_IS_RESUME_AUTHORITY =",
    True,
)

print(
    "CHECKPOINT_BOUNDARY =",
    "EPOCH",
)

print(
    "RNG_STATE_SERIALIZED =",
    False,
)

print(
    "REFERENCE_MODEL_UNCHANGED =",
    reference_model_unchanged,
)

print()

print(
    "OFFICIAL_TRAINING_STARTED =",
    False,
)

print()

print(
    "NOTEBOOK_03_CHECKPOINT_ROUNDTRIP_PASS =",
    (
        model_roundtrip_exact
        and optimizer_roundtrip_exact
        and metadata_roundtrip_pass
        and bn_counters_roundtrip
        and reference_model_unchanged
    ),
)

=== CHECKPOINT ROUND-TRIP TEST ===
Training disposable epoch 0...

=== CHECKPOINT ROUND-TRIP AUDIT ===
CHECKPOINT_PATH = /content/drive/MyDrive/TESI_SAKAI2022/sakai224_full/checkpoint_tests/sakai224_full_roundtrip_v1/latest_test.pt
CHECKPOINT_FILE_BYTES = 198545775
CHECKPOINT_FILE_SHA256 = b536387a0e8ac19922ee9f2ff410cfb57aa5146290733131806ac890650540e3

SCIENTIFIC_CONTRACT_SHA256 = c895b626454278f2293e217776c4604d24e03fa78dbaa7ba5049f86f917af949
LAST_COMPLETED_EPOCH = 0
OPTIMIZER_STEP = 54
BEST_EPOCH = 0
BEST_VALIDATION_TOTAL_MEAN = 93118.81974373908

MODEL_ROUNDTRIP_EXACT = True
OPTIMIZER_ROUNDTRIP_EXACT = True
METADATA_ROUNDTRIP_PASS = True
BN_COUNTERS_ROUNDTRIP = True

LATEST_IS_RESUME_AUTHORITY = True
CHECKPOINT_BOUNDARY = EPOCH
RNG_STATE_SERIALIZED = False
REFERENCE_MODEL_UNCHANGED = True

OFFICIAL_TRAINING_STARTED = False

NOTEBOOK_03_CHECKPOINT_ROUNDTRIP_PASS = True


## STEP 03-C1 — Equivalenza reale del resume

Il round-trip del checkpoint ha dimostrato che lo stato persistente può essere salvato e ricaricato esattamente.

Questo step verifica una proprietà più forte: **continuare dal checkpoint deve essere equivalente a continuare senza interruzione**.

Vengono confrontati due rami:

- **UNINTERRUPTED**: modello e optimizer rimasti in memoria dopo epoch 0;
- **RESUMED**: modello e optimizer ricostruiti dal checkpoint `LATEST` di epoch 0.

Entrambi eseguono esattamente epoch 1 usando:

- stesso shuffle PROJECT;
- stessa augmentation stateless;
- stessi `manifest_index`;
- stesso epsilon latente stateless;
- stessa precisione float32;
- stessa configurazione Adam.

Al termine devono coincidere esattamente:

- stato completo del modello, inclusi i buffer BatchNorm;
- stato completo dell'optimizer Adam;
- optimizer step globale `108`;
- contatori BatchNorm `108`;
- validation stocastica successiva.

Questo test simula, a livello di stato persistente, una perdita del runtime tra epoch 0 ed epoch 1.

La verifica bitwise riguarda l'ambiente numerico corrente. Un eventuale resume futuro su hardware/software differente verrà registrato nella provenance e non verrà automaticamente descritto come bitwise-identico.

In [16]:
# ============================================================
# NOTEBOOK 03 — SAVE/LOAD/RESUME EQUIVALENCE
# STEP 03-C1
#
# Branch A:
#   uninterrupted model after epoch 0
#
# Branch B:
#   fresh model loaded from epoch-0 LATEST checkpoint
#
# Both execute the exact same epoch 1.
#
# NO official training.
# ============================================================

import gc
import hashlib

import numpy as np
import torch


# ============================================================
# PRECONDITIONS
# ============================================================

RESUME_TEST_EPOCH = 1

EXPECTED_START_STEP = 54
EXPECTED_END_STEP = 108

EXPECTED_EPOCH1_MANIFEST_PREFIX = [
    6028,
    4009,
    12,
    5170,
]

EXPECTED_EPOCH1_CROP_PREFIX = [
    19,
    36,
    27,
    20,
]


assert (
    loaded_payload[
        "last_completed_epoch"
    ] == 0
)

assert (
    loaded_payload[
        "last_completed_epoch"
    ] + 1
    == RESUME_TEST_EPOCH
)

assert (
    get_adam_global_step(
        checkpoint_test_optimizer
    )
    == EXPECTED_START_STEP
)

assert (
    get_adam_global_step(
        checkpoint_loaded_optimizer
    )
    == EXPECTED_START_STEP
)

# Keep the numerical policy already frozen for this run.
assert (
    torch.backends.cuda.matmul.allow_tf32
    is False
)

assert (
    torch.backends.cudnn.allow_tf32
    is False
)

assert (
    torch.backends.cudnn.deterministic
    is True
)


# ============================================================
# IDENTIFY THE TWO BRANCHES
# ============================================================

uninterrupted_model = (
    checkpoint_test_model
)

uninterrupted_optimizer = (
    checkpoint_test_optimizer
)


resumed_model = (
    checkpoint_loaded_model
)

resumed_optimizer = (
    checkpoint_loaded_optimizer
)


# ============================================================
# STARTING STATE MUST ALREADY BE IDENTICAL
# ============================================================

uninterrupted_start_model_hash = (
    canonical_tree_sha256(
        uninterrupted_model.state_dict()
    )
)

resumed_start_model_hash = (
    canonical_tree_sha256(
        resumed_model.state_dict()
    )
)


uninterrupted_start_optimizer_hash = (
    canonical_tree_sha256(
        uninterrupted_optimizer.state_dict()
    )
)

resumed_start_optimizer_hash = (
    canonical_tree_sha256(
        resumed_optimizer.state_dict()
    )
)


start_model_exact = (
    uninterrupted_start_model_hash
    == resumed_start_model_hash
)

start_optimizer_exact = (
    uninterrupted_start_optimizer_hash
    == resumed_start_optimizer_hash
)


assert start_model_exact
assert start_optimizer_exact


# ============================================================
# ONE RESUME-AUDIT EPOCH
#
# Besides training, this records the identity of the first
# real batch so we can verify that both branches received
# exactly the same PROJECT inputs.
# ============================================================

def run_resume_audit_epoch(
    model,
    optimizer,
    epoch,
):

    model.train()

    loader = (
        make_checkpoint_train_loader(
            epoch
        )
    )


    sample_count = 0
    batch_count = 0
    tail_size = None

    total_objective_sum = 0.0

    first_manifests = None
    first_crops = None
    first_epsilon_sha256 = None


    for batch_index, batch in enumerate(
        loader
    ):

        # ----------------------------------------------------
        # INPUT IDENTITY BEFORE GPU TRANSFER
        # ----------------------------------------------------

        if batch_index == 0:

            first_manifests = (
                batch[
                    "manifest_index"
                ][:4].tolist()
            )

            first_crops = (
                batch[
                    "crop_left"
                ][:4].tolist()
            )


            epsilon_prefix = (
                batch[
                    "epsilon"
                ][:4]
                .contiguous()
                .numpy()
            )


            first_epsilon_sha256 = (
                hashlib.sha256(
                    epsilon_prefix.tobytes(
                        order="C"
                    )
                ).hexdigest()
            )


            if (
                first_manifests
                != EXPECTED_EPOCH1_MANIFEST_PREFIX
            ):
                raise RuntimeError(
                    "Epoch-1 manifest prefix mismatch"
                )


            if (
                first_crops
                != EXPECTED_EPOCH1_CROP_PREFIX
            ):
                raise RuntimeError(
                    "Epoch-1 crop prefix mismatch"
                )


        # ----------------------------------------------------
        # H2D
        # ----------------------------------------------------

        x_batch = (
            batch["x"]
            .to(
                device=device,
                dtype=torch.float32,
                non_blocking=True,
            )
        )

        epsilon_batch = (
            batch["epsilon"]
            .to(
                device=device,
                dtype=torch.float32,
                non_blocking=True,
            )
        )


        current_batch_size = int(
            x_batch.shape[0]
        )


        # ----------------------------------------------------
        # TRAIN STEP
        # ----------------------------------------------------

        optimizer.zero_grad(
            set_to_none=True
        )


        (
            logits,
            mu,
            logvar,
            z,
        ) = model(
            x_batch,
            epsilon=epsilon_batch,
            mean_mode=False,
        )


        stats = sakai_vae_loss(
            logits,
            x_batch,
            mu,
            logvar,
            beta_kl=SPEC.beta_kl,
        )


        loss = stats["loss"]


        if not torch.isfinite(
            loss
        ).item():

            raise RuntimeError(
                "Non-finite resume-test loss"
            )


        loss.backward()

        optimizer.step()


        batch_total = (
            stats[
                "reconstruction_sum"
            ]
            + stats["kl_sum"]
        )


        total_objective_sum += float(
            batch_total.detach()
        )


        sample_count += (
            current_batch_size
        )

        batch_count += 1

        tail_size = (
            current_batch_size
        )


    torch.cuda.synchronize()


    # --------------------------------------------------------
    # HARD EPOCH GATES
    # --------------------------------------------------------

    assert (
        sample_count
        == FROZEN_TRAIN_SAMPLES
    )

    assert (
        batch_count
        == FROZEN_TRAIN_BATCHES
    )

    assert (
        tail_size
        == FROZEN_TRAIN_TAIL
    )

    assert (
        get_adam_global_step(
            optimizer
        )
        == EXPECTED_END_STEP
    )


    bn_step_gate = all(
        int(
            bn.num_batches_tracked.item()
        )
        == EXPECTED_END_STEP

        for bn
        in get_project_batchnorm_layers(
            model
        )
    )


    assert bn_step_gate


    return {
        "sample_count":
            sample_count,

        "batch_count":
            batch_count,

        "tail_size":
            tail_size,

        "train_total_mean_per_sample":
            (
                total_objective_sum
                / sample_count
            ),

        "first_manifests":
            first_manifests,

        "first_crops":
            first_crops,

        "first_epsilon_sha256":
            first_epsilon_sha256,

        "model_state_sha256":
            canonical_tree_sha256(
                model.state_dict()
            ),

        "optimizer_state_sha256":
            canonical_tree_sha256(
                optimizer.state_dict()
            ),

        "optimizer_step":
            get_adam_global_step(
                optimizer
            ),

        "bn_step_gate":
            bn_step_gate,
    }


# ============================================================
# BRANCH A — UNINTERRUPTED
# ============================================================

print(
    "=== RESUME EQUIVALENCE TEST ==="
)

print(
    "Running uninterrupted epoch 1..."
)


uninterrupted_result = (
    run_resume_audit_epoch(
        uninterrupted_model,
        uninterrupted_optimizer,
        RESUME_TEST_EPOCH,
    )
)


# Free cached workspaces before the second branch.
gc.collect()
torch.cuda.empty_cache()


# ============================================================
# BRANCH B — RESUMED FROM CHECKPOINT
# ============================================================

print(
    "Running resumed epoch 1..."
)


resumed_result = (
    run_resume_audit_epoch(
        resumed_model,
        resumed_optimizer,
        RESUME_TEST_EPOCH,
    )
)


# ============================================================
# INPUT TRAJECTORY MUST MATCH
# ============================================================

first_batch_identity_exact = (
    uninterrupted_result[
        "first_manifests"
    ]
    == resumed_result[
        "first_manifests"
    ]

    and uninterrupted_result[
        "first_crops"
    ]
    == resumed_result[
        "first_crops"
    ]

    and uninterrupted_result[
        "first_epsilon_sha256"
    ]
    == resumed_result[
        "first_epsilon_sha256"
    ]
)


assert first_batch_identity_exact


# ============================================================
# FINAL MODEL / OPTIMIZER STATE MUST MATCH EXACTLY
# ============================================================

final_model_exact = (
    uninterrupted_result[
        "model_state_sha256"
    ]
    == resumed_result[
        "model_state_sha256"
    ]
)


final_optimizer_exact = (
    uninterrupted_result[
        "optimizer_state_sha256"
    ]
    == resumed_result[
        "optimizer_state_sha256"
    ]
)


train_metric_exact = (
    uninterrupted_result[
        "train_total_mean_per_sample"
    ]
    == resumed_result[
        "train_total_mean_per_sample"
    ]
)


assert final_model_exact
assert final_optimizer_exact
assert train_metric_exact


# ============================================================
# VALIDATION AFTER RESUME
#
# Same fixed validation epsilon.
# No state update in EVAL mode.
# ============================================================

uninterrupted_validation = (
    checkpoint_test_validation(
        uninterrupted_model
    )
)


resumed_validation = (
    checkpoint_test_validation(
        resumed_model
    )
)


validation_exact = (
    uninterrupted_validation
    == resumed_validation
)


assert validation_exact


# Validation must not alter persistent model state.
uninterrupted_post_validation_hash = (
    canonical_tree_sha256(
        uninterrupted_model.state_dict()
    )
)

resumed_post_validation_hash = (
    canonical_tree_sha256(
        resumed_model.state_dict()
    )
)


validation_state_unchanged = (
    uninterrupted_post_validation_hash
    == uninterrupted_result[
        "model_state_sha256"
    ]

    and resumed_post_validation_hash
    == resumed_result[
        "model_state_sha256"
    ]
)


assert validation_state_unchanged


# ============================================================
# FINAL CONTROL-STATE GATES
# ============================================================

optimizer_steps_exact = (
    uninterrupted_result[
        "optimizer_step"
    ] == EXPECTED_END_STEP

    and resumed_result[
        "optimizer_step"
    ] == EXPECTED_END_STEP
)


bn_steps_exact = (
    uninterrupted_result[
        "bn_step_gate"
    ]

    and resumed_result[
        "bn_step_gate"
    ]
)


assert optimizer_steps_exact
assert bn_steps_exact


# ============================================================
# CPU REFERENCE MODEL STILL UNTOUCHED
# ============================================================

reference_model_unchanged = (
    project_parameter_sha256(
        model_reference
    )
    == FROZEN_INIT_SHA256
)


assert reference_model_unchanged


# ============================================================
# REPORT
# ============================================================

print()
print(
    "=== SAVE → LOAD → RESUME AUDIT ==="
)

print(
    "RESUME_FROM_LAST_COMPLETED_EPOCH =",
    loaded_payload[
        "last_completed_epoch"
    ],
)

print(
    "RESUMED_EPOCH =",
    RESUME_TEST_EPOCH,
)

print()

print(
    "START_MODEL_STATE_EXACT =",
    start_model_exact,
)

print(
    "START_OPTIMIZER_STATE_EXACT =",
    start_optimizer_exact,
)

print()

print(
    "FIRST_MANIFESTS =",
    resumed_result[
        "first_manifests"
    ],
)

print(
    "FIRST_CROPS =",
    resumed_result[
        "first_crops"
    ],
)

print(
    "FIRST_EPSILON_SHA256 =",
    resumed_result[
        "first_epsilon_sha256"
    ],
)

print(
    "FIRST_BATCH_IDENTITY_EXACT =",
    first_batch_identity_exact,
)

print()

print(
    "UNINTERRUPTED_TRAIN_TOTAL_MEAN =",
    uninterrupted_result[
        "train_total_mean_per_sample"
    ],
)

print(
    "RESUMED_TRAIN_TOTAL_MEAN =",
    resumed_result[
        "train_total_mean_per_sample"
    ],
)

print(
    "TRAIN_METRIC_EXACT =",
    train_metric_exact,
)

print()

print(
    "FINAL_MODEL_STATE_EXACT =",
    final_model_exact,
)

print(
    "FINAL_OPTIMIZER_STATE_EXACT =",
    final_optimizer_exact,
)

print(
    "OPTIMIZER_STEPS_EXACT =",
    optimizer_steps_exact,
)

print(
    "BN_STEPS_EXACT =",
    bn_steps_exact,
)

print()

print(
    "UNINTERRUPTED_VALIDATION =",
    uninterrupted_validation,
)

print(
    "RESUMED_VALIDATION =",
    resumed_validation,
)

print(
    "VALIDATION_EXACT =",
    validation_exact,
)

print(
    "VALIDATION_STATE_UNCHANGED =",
    validation_state_unchanged,
)

print()

print(
    "FINAL_MODEL_STATE_SHA256 =",
    resumed_result[
        "model_state_sha256"
    ],
)

print(
    "FINAL_OPTIMIZER_STATE_SHA256 =",
    resumed_result[
        "optimizer_state_sha256"
    ],
)

print()

print(
    "EXPECTED_OPTIMIZER_STEP =",
    EXPECTED_END_STEP,
)

print(
    "REFERENCE_MODEL_UNCHANGED =",
    reference_model_unchanged,
)

print()

print(
    "BITWISE_SCOPE =",
    "CURRENT_A100_PYTORCH_CUDA_ENVIRONMENT",
)

print(
    "OFFICIAL_TRAINING_STARTED =",
    False,
)

print()

print(
    "NOTEBOOK_03_RESUME_EQUIVALENCE_PASS =",
    (
        start_model_exact
        and start_optimizer_exact
        and first_batch_identity_exact
        and train_metric_exact
        and final_model_exact
        and final_optimizer_exact
        and optimizer_steps_exact
        and bn_steps_exact
        and validation_exact
        and validation_state_unchanged
        and reference_model_unchanged
    ),
)

=== RESUME EQUIVALENCE TEST ===
Running uninterrupted epoch 1...
Running resumed epoch 1...

=== SAVE → LOAD → RESUME AUDIT ===
RESUME_FROM_LAST_COMPLETED_EPOCH = 0
RESUMED_EPOCH = 1

START_MODEL_STATE_EXACT = True
START_OPTIMIZER_STATE_EXACT = True

FIRST_MANIFESTS = [6028, 4009, 12, 5170]
FIRST_CROPS = [19, 36, 27, 20]
FIRST_EPSILON_SHA256 = 03a85fd5fddec071d58eb27bfe1149e4e7e0bddec5f007e6874ca5df9bdc36ea
FIRST_BATCH_IDENTITY_EXACT = True

UNINTERRUPTED_TRAIN_TOTAL_MEAN = 92505.70892805127
RESUMED_TRAIN_TOTAL_MEAN = 92505.70892805127
TRAIN_METRIC_EXACT = True

FINAL_MODEL_STATE_EXACT = True
FINAL_OPTIMIZER_STATE_EXACT = True
OPTIMIZER_STEPS_EXACT = True
BN_STEPS_EXACT = True

UNINTERRUPTED_VALIDATION = 91962.25567850903
RESUMED_VALIDATION = 91962.25567850903
VALIDATION_EXACT = True
VALIDATION_STATE_UNCHANGED = True

FINAL_MODEL_STATE_SHA256 = dc5f99b2806978baba38e033f1470df1d1d74a4de64b58e76b03f426fc4e7cae
FINAL_OPTIMIZER_STATE_SHA256 = e172c276a7a810e6bebbcf68ee6910958808a5b8fd28415

## STEP 03-C2 — Struttura persistente della run ufficiale

Viene ora preparata la directory persistente della run `sakai224_full_v1`.

La directory rimane con suffisso `.partial` fino al completamento delle 100 epoche.

### Artefatti principali

- `run_config.json`: contratto scientifico e provenance iniziale;
- `metrics.csv`: ledger delle metriche;
- `latest.pt`: unica autorità per il resume;
- `best.pt`: checkpoint diagnostico della minima validation stocastica;
- `final.pt`: creato esclusivamente dopo la centesima epoca;
- `runtime_sessions.jsonl`: provenance dei runtime Colab usati.

### Ordine di commit di ogni epoca

1. train completo;
2. validation completa;
3. eventuale MEAN diagnostic;
4. scrittura e flush delle metriche;
5. pubblicazione atomica di `latest.pt`;
6. eventuale aggiornamento di `best.pt`.

Di conseguenza, dopo una disconnessione le metriche possono al massimo trovarsi **più avanti di LATEST**. Il resume dovrà riconciliarle con l'ultimo checkpoint valido.

Il suffisso `.partial` verrà rimosso soltanto dopo:

- epoch 99 completata;
- optimizer step 5400;
- validation finale;
- `latest.pt` valido;
- `final.pt` valido.

Questo step crea soltanto l'infrastruttura persistente.  
Il training ufficiale resta non avviato.

In [17]:
# ============================================================
# NOTEBOOK 03 — OFFICIAL RUN PERSISTENT SKELETON
# STEP 03-C2
#
# Creates:
#   official .partial directory
#   run_config.json
#   metrics.csv header
#   runtime_sessions.jsonl
#
# NO model forward.
# NO optimizer.
# NO checkpoint yet.
# NO official training.
# ============================================================

from pathlib import Path

import csv
import datetime as dt
import hashlib
import json
import os
import platform

import torch


# ============================================================
# OFFICIAL OUTPUT PATHS
# ============================================================

OFFICIAL_RUNS_ROOT = (
    SAKAI224_ROOT
    / "runs"
)

OFFICIAL_FINAL_DIR = (
    OFFICIAL_RUNS_ROOT
    / RUN_ID
)

OFFICIAL_PARTIAL_DIR = (
    OFFICIAL_RUNS_ROOT
    / f"{RUN_ID}.partial"
)


OFFICIAL_RUN_CONFIG_PATH = (
    OFFICIAL_PARTIAL_DIR
    / "run_config.json"
)

OFFICIAL_METRICS_PATH = (
    OFFICIAL_PARTIAL_DIR
    / "metrics.csv"
)

OFFICIAL_RUNTIME_SESSIONS_PATH = (
    OFFICIAL_PARTIAL_DIR
    / "runtime_sessions.jsonl"
)

OFFICIAL_LATEST_PATH = (
    OFFICIAL_PARTIAL_DIR
    / "latest.pt"
)

OFFICIAL_BEST_PATH = (
    OFFICIAL_PARTIAL_DIR
    / "best.pt"
)

OFFICIAL_FINAL_CHECKPOINT_PATH = (
    OFFICIAL_PARTIAL_DIR
    / "final.pt"
)


# ============================================================
# SAFETY GATE
#
# We do not silently reuse an existing official directory.
# ============================================================

OFFICIAL_RUNS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


final_exists_before = (
    OFFICIAL_FINAL_DIR.exists()
)

partial_exists_before = (
    OFFICIAL_PARTIAL_DIR.exists()
)


if final_exists_before:

    raise RuntimeError(
        "STOP: official FINAL directory already exists: "
        f"{OFFICIAL_FINAL_DIR}"
    )


if partial_exists_before:

    raise RuntimeError(
        "STOP: official PARTIAL directory already exists. "
        "Do not overwrite it; an existing run may need resume: "
        f"{OFFICIAL_PARTIAL_DIR}"
    )


# ============================================================
# ATOMIC SMALL-TEXT WRITER
#
# Used for run configuration.
# Existing files are never silently overwritten.
# ============================================================

def write_text_exclusive_atomic(
    path,
    text,
):

    path = Path(path)

    temporary_path = Path(
        str(path) + ".tmp"
    )


    if path.exists():
        raise RuntimeError(
            f"Destination already exists: {path}"
        )


    if temporary_path.exists():
        raise RuntimeError(
            f"Stale temporary file exists: {temporary_path}"
        )


    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    encoded = text.encode(
        "utf-8"
    )


    with temporary_path.open(
        "xb"
    ) as handle:

        handle.write(
            encoded
        )

        handle.flush()

        os.fsync(
            handle.fileno()
        )


    # Independent byte-for-byte verification.
    read_back = (
        temporary_path
        .read_bytes()
    )


    if read_back != encoded:

        raise RuntimeError(
            "Atomic text read-back mismatch"
        )


    os.replace(
        temporary_path,
        path,
    )


# ============================================================
# METRICS SCHEMA
#
# All loss totals are aggregated over samples.
#
# epoch = -1 is reserved for initial validation.
# ============================================================

OFFICIAL_METRICS_COLUMNS = [

    "phase",

    "epoch",

    "completed_epochs",

    "optimizer_step",

    "sample_count",

    "batch_count",

    "total_sum",

    "total_mean_per_sample",

    "reconstruction_sum",

    "reconstruction_mean_per_sample",

    "reconstruction_mean_per_input_value",

    "kl_sum",

    "kl_mean_per_sample",

    "kl_mean_per_latent_value",

    "wall_seconds",
]


# ============================================================
# RUN CONFIG
#
# Scientific identity is separated from runtime provenance.
# ============================================================

run_config = {

    "format_version":
        1,

    "run_id":
        RUN_ID,

    "status":
        "PREPARED_NOT_STARTED",

    "scientific_contract_sha256":
        SCIENTIFIC_CONTRACT_SHA256,

    "scientific_contract":
        scientific_contract,

    "controller": {

        "first_epoch_index":
            FROZEN_FIRST_EPOCH_INDEX,

        "final_epoch_index":
            FROZEN_FINAL_EPOCH_INDEX,

        "max_epochs":
            FROZEN_MAX_EPOCHS,

        "official_model_selection":
            OFFICIAL_MODEL_SELECTION,

        "early_stopping_enabled":
            EARLY_STOPPING_ENABLED,

        "best_checkpoint_role":
            BEST_CHECKPOINT_ROLE,

        "latest_is_resume_authority":
            True,

        "checkpoint_boundary":
            "EPOCH",

        "latest_every_epoch":
            True,

        "best_policy":
            "STRICT_MIN_VALIDATION_TOTAL_MEAN_PER_SAMPLE",

        "best_tie_break":
            "EARLIEST_EPOCH",

        "mean_diagnostic_every_completed_epochs":
            MEAN_DIAGNOSTIC_EVERY_EPOCHS,

        "metrics_commit_order":
            "METRICS_THEN_LATEST",

        "resume_metrics_policy":
            "RECONCILE_TO_LATEST",
    },

    "loader": {

        "batch_size":
            FROZEN_BATCH_SIZE,

        "num_workers":
            FROZEN_NUM_WORKERS,

        "pin_memory":
            FROZEN_PIN_MEMORY,

        "persistent_workers":
            FROZEN_PERSISTENT_WORKERS,

        "drop_last":
            False,

        "prefetch_factor":
            2,
    },

    "expected_counts": {

        "train_samples":
            FROZEN_TRAIN_SAMPLES,

        "validation_samples":
            FROZEN_VALIDATION_SAMPLES,

        "train_batches_per_epoch":
            FROZEN_TRAIN_BATCHES,

        "validation_batches":
            FROZEN_VALIDATION_BATCHES,

        "train_tail":
            FROZEN_TRAIN_TAIL,

        "validation_tail":
            FROZEN_VALIDATION_TAIL,

        "final_optimizer_step":
            FROZEN_TOTAL_OPTIMIZER_STEPS,
    },

    "checkpoint": {

        "format":
            "PYTORCH_PROJECT_V1",

        "checkpoint_format_version":
            PYTORCH_CHECKPOINT_FORMAT_VERSION,

        "latest":
            "latest.pt",

        "best":
            "best.pt",

        "final":
            "final.pt",

        "temporary_suffix":
            ".tmp",

        "rng_state_serialized":
            False,

        "rng_reason":
            "PROJECT_RNG_IS_STATELESS",
    },

    "metrics": {

        "file":
            "metrics.csv",

        "columns":
            OFFICIAL_METRICS_COLUMNS,

        "initial_phases": [
            "validation_stochastic_initial",
            "validation_mean_initial",
        ],

        "epoch_phases": [
            "train_stochastic",
            "validation_stochastic_post_epoch",
            "validation_mean_diagnostic",
        ],

        "final_phase":
            "validation_mean_final",
    },

    "project_reference": {

        "c_repository_commit":
            C_REFERENCE_COMMIT,
    },
}


# ============================================================
# CONFIG FINGERPRINT
#
# Different from SCIENTIFIC_CONTRACT_SHA256:
# this includes operational controller details too.
# ============================================================

canonical_run_config_json = json.dumps(
    run_config,
    sort_keys=True,
    separators=(",", ":"),
    ensure_ascii=True,
)


RUN_CONFIG_SHA256 = (
    hashlib.sha256(
        canonical_run_config_json.encode(
            "utf-8"
        )
    ).hexdigest()
)


run_config[
    "run_config_sha256"
] = RUN_CONFIG_SHA256


pretty_run_config_json = (
    json.dumps(
        run_config,
        indent=2,
        sort_keys=True,
        ensure_ascii=True,
    )
    + "\n"
)


# ============================================================
# CREATE OFFICIAL .partial DIRECTORY
# ============================================================

OFFICIAL_PARTIAL_DIR.mkdir(
    parents=False,
    exist_ok=False,
)


# ============================================================
# WRITE run_config.json
# ============================================================

write_text_exclusive_atomic(
    OFFICIAL_RUN_CONFIG_PATH,
    pretty_run_config_json,
)


# ============================================================
# CREATE metrics.csv EXCLUSIVELY
# ============================================================

with OFFICIAL_METRICS_PATH.open(
    "x",
    newline="",
    encoding="utf-8",
) as handle:

    writer = csv.DictWriter(
        handle,
        fieldnames=OFFICIAL_METRICS_COLUMNS,
    )

    writer.writeheader()

    handle.flush()

    os.fsync(
        handle.fileno()
    )


# ============================================================
# RECORD RUNTIME SESSION PROVENANCE
#
# Hardware is provenance, NOT scientific compatibility.
# ============================================================

runtime_session = {

    "timestamp_utc":
        dt.datetime.now(
            dt.timezone.utc
        ).isoformat(),

    "purpose":
        "OFFICIAL_RUN_PREPARATION",

    "python_version":
        platform.python_version(),

    "torch_version":
        str(torch.__version__),

    "cuda_version":
        str(torch.version.cuda),

    "cudnn_version":
        int(
            torch.backends.cudnn.version()
        ),

    "gpu_name":
        torch.cuda.get_device_name(
            0
        ),

    "gpu_compute_capability":
        list(
            torch.cuda.get_device_capability(
                0
            )
        ),

    "amp_enabled":
        FROZEN_AMP_ENABLED,

    "tf32_enabled":
        FROZEN_TF32_ENABLED,

    "scientific_contract_sha256":
        SCIENTIFIC_CONTRACT_SHA256,

    "run_config_sha256":
        RUN_CONFIG_SHA256,
}


with OFFICIAL_RUNTIME_SESSIONS_PATH.open(
    "x",
    encoding="utf-8",
) as handle:

    handle.write(
        json.dumps(
            runtime_session,
            sort_keys=True,
            ensure_ascii=True,
        )
        + "\n"
    )

    handle.flush()

    os.fsync(
        handle.fileno()
    )


# ============================================================
# READ-BACK AUDIT
# ============================================================

loaded_run_config = json.loads(
    OFFICIAL_RUN_CONFIG_PATH.read_text(
        encoding="utf-8"
    )
)


config_identity_gate = (
    loaded_run_config[
        "run_id"
    ] == RUN_ID

    and loaded_run_config[
        "scientific_contract_sha256"
    ] == SCIENTIFIC_CONTRACT_SHA256

    and loaded_run_config[
        "run_config_sha256"
    ] == RUN_CONFIG_SHA256

    and loaded_run_config[
        "controller"
    ][
        "final_epoch_index"
    ] == 99

    and loaded_run_config[
        "expected_counts"
    ][
        "final_optimizer_step"
    ] == 5400
)


assert config_identity_gate


with OFFICIAL_METRICS_PATH.open(
    "r",
    newline="",
    encoding="utf-8",
) as handle:

    reader = csv.reader(
        handle
    )

    rows = list(
        reader
    )


metrics_initial_gate = (
    len(rows) == 1

    and rows[0]
        == OFFICIAL_METRICS_COLUMNS
)


assert metrics_initial_gate


runtime_lines = (
    OFFICIAL_RUNTIME_SESSIONS_PATH
    .read_text(
        encoding="utf-8"
    )
    .splitlines()
)


runtime_provenance_gate = (
    len(runtime_lines) == 1

    and json.loads(
        runtime_lines[0]
    )[
        "scientific_contract_sha256"
    ] == SCIENTIFIC_CONTRACT_SHA256
)


assert runtime_provenance_gate


# ============================================================
# CHECKPOINTS MUST NOT EXIST YET
# ============================================================

checkpoint_absence_gate = (
    not OFFICIAL_LATEST_PATH.exists()

    and not OFFICIAL_BEST_PATH.exists()

    and not OFFICIAL_FINAL_CHECKPOINT_PATH.exists()

    and not Path(
        str(
            OFFICIAL_LATEST_PATH
        ) + ".tmp"
    ).exists()

    and not Path(
        str(
            OFFICIAL_BEST_PATH
        ) + ".tmp"
    ).exists()

    and not Path(
        str(
            OFFICIAL_FINAL_CHECKPOINT_PATH
        ) + ".tmp"
    ).exists()
)


assert checkpoint_absence_gate


# ============================================================
# DIRECTORY ALLOWLIST
# ============================================================

actual_names = sorted(
    path.name

    for path
    in OFFICIAL_PARTIAL_DIR.iterdir()
)


expected_names = sorted([
    "metrics.csv",
    "run_config.json",
    "runtime_sessions.jsonl",
])


directory_allowlist_gate = (
    actual_names
    == expected_names
)


assert directory_allowlist_gate


# ============================================================
# REPORT
# ============================================================

print(
    "=== OFFICIAL RUN PERSISTENT SKELETON ==="
)

print(
    "PARTIAL_DIRECTORY =",
    OFFICIAL_PARTIAL_DIR,
)

print(
    "FINAL_DIRECTORY =",
    OFFICIAL_FINAL_DIR,
)

print()

print(
    "SCIENTIFIC_CONTRACT_SHA256 =",
    SCIENTIFIC_CONTRACT_SHA256,
)

print(
    "RUN_CONFIG_SHA256 =",
    RUN_CONFIG_SHA256,
)

print()

print(
    "DIRECTORY_CONTENTS =",
    actual_names,
)

print()

print(
    "CONFIG_IDENTITY_GATE =",
    config_identity_gate,
)

print(
    "METRICS_INITIAL_GATE =",
    metrics_initial_gate,
)

print(
    "RUNTIME_PROVENANCE_GATE =",
    runtime_provenance_gate,
)

print(
    "CHECKPOINT_ABSENCE_GATE =",
    checkpoint_absence_gate,
)

print(
    "DIRECTORY_ALLOWLIST_GATE =",
    directory_allowlist_gate,
)

print()

print(
    "LATEST_EXISTS =",
    OFFICIAL_LATEST_PATH.exists(),
)

print(
    "BEST_EXISTS =",
    OFFICIAL_BEST_PATH.exists(),
)

print(
    "FINAL_EXISTS =",
    OFFICIAL_FINAL_CHECKPOINT_PATH.exists(),
)

print()

print(
    "MODEL_FORWARD_EXECUTED =",
    False,
)

print(
    "OPTIMIZER_EXECUTED =",
    False,
)

print(
    "OFFICIAL_TRAINING_STARTED =",
    False,
)

print()

print(
    "NOTEBOOK_03_OFFICIAL_SKELETON_PASS =",
    (
        config_identity_gate
        and metrics_initial_gate
        and runtime_provenance_gate
        and checkpoint_absence_gate
        and directory_allowlist_gate
    ),
)

=== OFFICIAL RUN PERSISTENT SKELETON ===
PARTIAL_DIRECTORY = /content/drive/MyDrive/TESI_SAKAI2022/sakai224_full/runs/sakai224_full_v1.partial
FINAL_DIRECTORY = /content/drive/MyDrive/TESI_SAKAI2022/sakai224_full/runs/sakai224_full_v1

SCIENTIFIC_CONTRACT_SHA256 = c895b626454278f2293e217776c4604d24e03fa78dbaa7ba5049f86f917af949
RUN_CONFIG_SHA256 = cb2160294fa2ebe82a84d689ebd0cb9f2fea0264713011eee65413d0d1a9a879

DIRECTORY_CONTENTS = ['metrics.csv', 'run_config.json', 'runtime_sessions.jsonl']

CONFIG_IDENTITY_GATE = True
METRICS_INITIAL_GATE = True
RUNTIME_PROVENANCE_GATE = True
CHECKPOINT_ABSENCE_GATE = True
DIRECTORY_ALLOWLIST_GATE = True

LATEST_EXISTS = False
BEST_EXISTS = False
FINAL_EXISTS = False

MODEL_FORWARD_EXECUTED = False
OPTIMIZER_EXECUTED = False
OFFICIAL_TRAINING_STARTED = False

NOTEBOOK_03_OFFICIAL_SKELETON_PASS = True


## STEP 03-C3 — Stato ufficiale pre-training

La directory ufficiale è stata preparata ma nessun training è ancora iniziato.

Prima di epoch 0 viene congelato uno stato persistente pre-training:

- modello inizializzato con seed PROJECT `123`;
- stato Adam ancora vuoto;
- optimizer step `0`;
- `last_completed_epoch = -1`;
- nessun BEST ancora definito.

Questo `LATEST` rappresenta il confine immediatamente precedente a epoch 0.

Successivamente vengono misurate le validation iniziali ufficiali:

1. `validation_stochastic_initial`
   - validation centrale;
   - BatchNorm in `EVAL`;
   - epsilon validation stateless fisso;

2. `validation_mean_initial`
   - stesso input centrale;
   - `z = μ`;
   - nessun epsilon.

Le validation non devono modificare parametri o stato BatchNorm.

Il checkpoint BEST non viene creato dalla validation iniziale: la competizione BEST inizierà soltanto dalle validation post-epoch.

**Nessun backward e nessun optimizer step vengono eseguiti in questo step.**

In [18]:
# ============================================================
# NOTEBOOK 03 — OFFICIAL PRE-TRAIN STATE
# STEP 03-C3
#
# Creates the first official persistent recovery anchor:
#
#   latest.pt
#   last_completed_epoch = -1
#   optimizer_step       = 0
#
# Then records:
#
#   validation_stochastic_initial
#   validation_mean_initial
#
# NO backward.
# NO optimizer step.
# NO training epoch.
# ============================================================

import csv
import gc
import time

import torch


# ============================================================
# EXCLUSIVE PRECONDITIONS
#
# This cell is intentionally NOT rerunnable.
# ============================================================

if OFFICIAL_LATEST_PATH.exists():
    raise RuntimeError(
        "STOP: official latest.pt already exists. "
        "Do not rerun STEP 03-C3."
    )

if OFFICIAL_BEST_PATH.exists():
    raise RuntimeError(
        "STOP: unexpected official best.pt already exists."
    )

if OFFICIAL_FINAL_CHECKPOINT_PATH.exists():
    raise RuntimeError(
        "STOP: unexpected official final.pt already exists."
    )


# metrics.csv must still contain only its header.
with OFFICIAL_METRICS_PATH.open(
    "r",
    newline="",
    encoding="utf-8",
) as handle:

    existing_metric_rows = list(
        csv.reader(handle)
    )


if (
    len(existing_metric_rows) != 1
    or existing_metric_rows[0]
        != OFFICIAL_METRICS_COLUMNS
):
    raise RuntimeError(
        "STOP: metrics.csv is not in the pristine C2 state."
    )


# ============================================================
# FRESH OFFICIAL MODEL
#
# This is NOT any of the disposable benchmark/test models.
# ============================================================

official_model = Sakai224FullVAE(
    SPEC
)

initialize_project_c_exact_v1(
    official_model,
    FROZEN_INITIALIZATION_SEED,
)


official_parameter_init_sha256 = (
    project_parameter_sha256(
        official_model
    )
)


assert (
    official_parameter_init_sha256
    == FROZEN_MODEL_INITIAL_SHA256
)


official_model = official_model.to(
    device=device,
    dtype=torch.float32,
)


# ============================================================
# FRESH OFFICIAL ADAM
# ============================================================

official_optimizer = torch.optim.Adam(
    official_model.parameters(),

    lr=FROZEN_LEARNING_RATE,

    betas=(
        FROZEN_ADAM_BETA1,
        FROZEN_ADAM_BETA2,
    ),

    eps=FROZEN_ADAM_EPSILON,

    weight_decay=FROZEN_WEIGHT_DECAY,

    amsgrad=FROZEN_AMSGRAD,

    foreach=False,

    fused=False,
)


assert (
    get_adam_global_step(
        official_optimizer
    )
    == 0
)


# ============================================================
# FREEZE INITIAL FULL MODEL STATE
#
# Includes parameters + BatchNorm buffers.
# ============================================================

OFFICIAL_INITIAL_MODEL_STATE_SHA256 = (
    canonical_tree_sha256(
        official_model.state_dict()
    )
)


OFFICIAL_INITIAL_OPTIMIZER_STATE_SHA256 = (
    canonical_tree_sha256(
        official_optimizer.state_dict()
    )
)


# ============================================================
# PRE-TRAIN LATEST CHECKPOINT
#
# last_completed_epoch = -1 means:
#
#   zero completed epochs
#   next epoch = 0
#   optimizer step = 0
#
# BEST is deliberately undefined.
# ============================================================

official_pretrain_payload = (
    build_checkpoint_payload(
        official_model,
        official_optimizer,

        kind="LATEST",

        last_completed_epoch=-1,

        best_defined=False,

        best_epoch=-1,

        best_validation_total_mean_per_sample=float(
            "inf"
        ),
    )
)


assert (
    official_pretrain_payload[
        "optimizer_step"
    ] == 0
)

assert (
    official_pretrain_payload[
        "last_completed_epoch"
    ] == -1
)

assert (
    official_pretrain_payload[
        "best_defined"
    ] is False
)


assert validate_checkpoint_payload(
    official_pretrain_payload,
    expected_kind="LATEST",
)


save_checkpoint_atomic(
    OFFICIAL_LATEST_PATH,
    official_pretrain_payload,
)


assert OFFICIAL_LATEST_PATH.is_file()


OFFICIAL_PRETRAIN_LATEST_FILE_SHA256 = (
    sha256_file_streaming(
        OFFICIAL_LATEST_PATH
    )
)


# ============================================================
# VERIFY PUBLISHED RECOVERY ANCHOR
# ============================================================

published_pretrain_payload = torch.load(
    OFFICIAL_LATEST_PATH,
    map_location="cpu",
    weights_only=True,
)


assert validate_checkpoint_payload(
    published_pretrain_payload,
    expected_kind="LATEST",
)


pretrain_checkpoint_gate = (
    published_pretrain_payload[
        "last_completed_epoch"
    ] == -1

    and published_pretrain_payload[
        "optimizer_step"
    ] == 0

    and published_pretrain_payload[
        "best_defined"
    ] is False

    and published_pretrain_payload[
        "model_state_sha256"
    ] == OFFICIAL_INITIAL_MODEL_STATE_SHA256

    and published_pretrain_payload[
        "optimizer_state_sha256"
    ] == OFFICIAL_INITIAL_OPTIMIZER_STATE_SHA256

    and published_pretrain_payload[
        "scientific_contract_sha256"
    ] == SCIENTIFIC_CONTRACT_SHA256
)


assert pretrain_checkpoint_gate


# ============================================================
# FULL VALIDATION AGGREGATOR
#
# Accumulation between batches is performed in Python float
# (double precision), while neural computation remains FP32.
# ============================================================

def run_official_validation_aggregate(
    model,
    *,
    mode,
):

    if mode not in (
        "stochastic",
        "mean",
    ):
        raise ValueError(
            f"Unsupported validation mode: {mode}"
        )


    loader = (
        val_loader_stochastic
        if mode == "stochastic"
        else val_loader_mean
    )


    model.eval()


    sample_count = 0
    batch_count = 0
    tail_size = None

    reconstruction_sum = 0.0
    kl_sum = 0.0

    first_batch_gate = False


    torch.cuda.synchronize()

    start_time = time.perf_counter()


    with torch.inference_mode():

        for batch_index, batch in enumerate(
            loader
        ):

            if batch_index == 0:

                manifests = (
                    batch[
                        "manifest_index"
                    ][:4].tolist()
                )

                crops = (
                    batch[
                        "crop_left"
                    ][:4].tolist()
                )


                first_batch_gate = (
                    manifests
                    == [
                        6864,
                        1488,
                        8384,
                        6585,
                    ]

                    and crops
                    == [
                        24,
                        24,
                        24,
                        24,
                    ]
                )


                if not first_batch_gate:
                    raise RuntimeError(
                        "Official validation first-batch mismatch"
                    )


            x_batch = (
                batch["x"]
                .to(
                    device=device,
                    dtype=torch.float32,
                    non_blocking=True,
                )
            )


            if mode == "stochastic":

                epsilon_batch = (
                    batch["epsilon"]
                    .to(
                        device=device,
                        dtype=torch.float32,
                        non_blocking=True,
                    )
                )

                (
                    logits,
                    mu,
                    logvar,
                    z,
                ) = model(
                    x_batch,
                    epsilon=epsilon_batch,
                    mean_mode=False,
                )

            else:

                (
                    logits,
                    mu,
                    logvar,
                    z,
                ) = model(
                    x_batch,
                    epsilon=None,
                    mean_mode=True,
                )


                if not torch.equal(
                    z,
                    mu,
                ):
                    raise RuntimeError(
                        "Official MEAN validation z != mu"
                    )


            stats = sakai_vae_loss(
                logits,
                x_batch,
                mu,
                logvar,
                beta_kl=SPEC.beta_kl,
            )


            if not torch.isfinite(
                stats["loss"]
            ).item():

                raise RuntimeError(
                    "Non-finite official validation loss"
                )


            reconstruction_sum += float(
                stats[
                    "reconstruction_sum"
                ].detach()
            )

            kl_sum += float(
                stats[
                    "kl_sum"
                ].detach()
            )


            current_batch_size = int(
                x_batch.shape[0]
            )


            sample_count += (
                current_batch_size
            )

            batch_count += 1

            tail_size = (
                current_batch_size
            )


    torch.cuda.synchronize()


    wall_seconds = (
        time.perf_counter()
        - start_time
    )


    assert (
        sample_count
        == FROZEN_VALIDATION_SAMPLES
    )

    assert (
        batch_count
        == FROZEN_VALIDATION_BATCHES
    )

    assert (
        tail_size
        == FROZEN_VALIDATION_TAIL
    )

    assert first_batch_gate


    total_sum = (
        reconstruction_sum
        + kl_sum
    )


    input_value_count = (
        sample_count
        * 4
        * 224
        * 224
    )


    latent_value_count = (
        sample_count
        * SPEC.latent_dim
    )


    return {

        "sample_count":
            sample_count,

        "batch_count":
            batch_count,

        "total_sum":
            total_sum,

        "total_mean_per_sample":
            total_sum
            / sample_count,

        "reconstruction_sum":
            reconstruction_sum,

        "reconstruction_mean_per_sample":
            reconstruction_sum
            / sample_count,

        "reconstruction_mean_per_input_value":
            reconstruction_sum
            / input_value_count,

        "kl_sum":
            kl_sum,

        "kl_mean_per_sample":
            kl_sum
            / sample_count,

        "kl_mean_per_latent_value":
            kl_sum
            / latent_value_count,

        "wall_seconds":
            wall_seconds,
    }


# ============================================================
# MODEL STATE MUST NOT CHANGE DURING VALIDATION
# ============================================================

model_state_before_initial_validation = (
    canonical_tree_sha256(
        official_model.state_dict()
    )
)


assert (
    model_state_before_initial_validation
    == OFFICIAL_INITIAL_MODEL_STATE_SHA256
)


# ============================================================
# OFFICIAL INITIAL STOCHASTIC VALIDATION
# ============================================================

print(
    "=== OFFICIAL PRE-TRAIN INITIALIZATION ==="
)

print(
    "Running official initial stochastic validation..."
)


official_initial_stochastic = (
    run_official_validation_aggregate(
        official_model,
        mode="stochastic",
    )
)


# ============================================================
# OFFICIAL INITIAL MEAN VALIDATION
# ============================================================

print(
    "Running official initial MEAN validation..."
)


official_initial_mean = (
    run_official_validation_aggregate(
        official_model,
        mode="mean",
    )
)


# ============================================================
# VALIDATION MUST BE STATE-PRESERVING
# ============================================================

model_state_after_initial_validation = (
    canonical_tree_sha256(
        official_model.state_dict()
    )
)


initial_validation_state_unchanged = (
    model_state_after_initial_validation
    == OFFICIAL_INITIAL_MODEL_STATE_SHA256
)


optimizer_still_step_zero = (
    get_adam_global_step(
        official_optimizer
    )
    == 0
)


assert initial_validation_state_unchanged
assert optimizer_still_step_zero


# ============================================================
# METRIC ROW BUILDER
# ============================================================

def make_official_metric_row(
    phase,
    epoch,
    completed_epochs,
    optimizer_step,
    aggregate,
):

    return {

        "phase":
            phase,

        "epoch":
            int(epoch),

        "completed_epochs":
            int(completed_epochs),

        "optimizer_step":
            int(optimizer_step),

        "sample_count":
            int(
                aggregate[
                    "sample_count"
                ]
            ),

        "batch_count":
            int(
                aggregate[
                    "batch_count"
                ]
            ),

        "total_sum":
            repr(
                aggregate[
                    "total_sum"
                ]
            ),

        "total_mean_per_sample":
            repr(
                aggregate[
                    "total_mean_per_sample"
                ]
            ),

        "reconstruction_sum":
            repr(
                aggregate[
                    "reconstruction_sum"
                ]
            ),

        "reconstruction_mean_per_sample":
            repr(
                aggregate[
                    "reconstruction_mean_per_sample"
                ]
            ),

        "reconstruction_mean_per_input_value":
            repr(
                aggregate[
                    "reconstruction_mean_per_input_value"
                ]
            ),

        "kl_sum":
            repr(
                aggregate[
                    "kl_sum"
                ]
            ),

        "kl_mean_per_sample":
            repr(
                aggregate[
                    "kl_mean_per_sample"
                ]
            ),

        "kl_mean_per_latent_value":
            repr(
                aggregate[
                    "kl_mean_per_latent_value"
                ]
            ),

        "wall_seconds":
            repr(
                aggregate[
                    "wall_seconds"
                ]
            ),
    }


# ============================================================
# APPEND + FSYNC
#
# If a future interruption happens after a metric write but
# before a later checkpoint, recovery will reconcile the
# metrics ledger against LATEST.
# ============================================================

def append_official_metric_row(
    row,
):

    with OFFICIAL_METRICS_PATH.open(
        "a",
        newline="",
        encoding="utf-8",
    ) as handle:

        writer = csv.DictWriter(
            handle,
            fieldnames=OFFICIAL_METRICS_COLUMNS,
        )

        writer.writerow(
            row
        )

        handle.flush()

        os.fsync(
            handle.fileno()
        )


# ============================================================
# WRITE THE TWO INITIAL METRICS
# ============================================================

initial_stochastic_row = (
    make_official_metric_row(
        "validation_stochastic_initial",

        epoch=-1,

        completed_epochs=0,

        optimizer_step=0,

        aggregate=official_initial_stochastic,
    )
)


initial_mean_row = (
    make_official_metric_row(
        "validation_mean_initial",

        epoch=-1,

        completed_epochs=0,

        optimizer_step=0,

        aggregate=official_initial_mean,
    )
)


append_official_metric_row(
    initial_stochastic_row
)

append_official_metric_row(
    initial_mean_row
)


# ============================================================
# READ-BACK METRICS AUDIT
# ============================================================

with OFFICIAL_METRICS_PATH.open(
    "r",
    newline="",
    encoding="utf-8",
) as handle:

    official_metric_rows = list(
        csv.DictReader(
            handle
        )
    )


initial_metric_ledger_gate = (
    len(
        official_metric_rows
    ) == 2

    and official_metric_rows[0][
        "phase"
    ] == "validation_stochastic_initial"

    and official_metric_rows[1][
        "phase"
    ] == "validation_mean_initial"

    and int(
        official_metric_rows[0][
            "optimizer_step"
        ]
    ) == 0

    and int(
        official_metric_rows[1][
            "optimizer_step"
        ]
    ) == 0

    and int(
        official_metric_rows[0][
            "completed_epochs"
        ]
    ) == 0

    and int(
        official_metric_rows[1][
            "completed_epochs"
        ]
    ) == 0
)


assert initial_metric_ledger_gate


# ============================================================
# OFFICIAL FILE ALLOWLIST AFTER INITIALIZATION
# ============================================================

official_names_after_c3 = sorted(
    path.name

    for path
    in OFFICIAL_PARTIAL_DIR.iterdir()
)


expected_names_after_c3 = sorted([
    "latest.pt",
    "metrics.csv",
    "run_config.json",
    "runtime_sessions.jsonl",
])


official_allowlist_after_c3 = (
    official_names_after_c3
    == expected_names_after_c3
)


assert official_allowlist_after_c3


# BEST and FINAL still forbidden.
assert not OFFICIAL_BEST_PATH.exists()
assert not OFFICIAL_FINAL_CHECKPOINT_PATH.exists()


# ============================================================
# LEAVE OFFICIAL MODEL READY FOR EPOCH 0
# ============================================================

official_model.train()


# ============================================================
# REPORT
# ============================================================

print()
print(
    "=== OFFICIAL PRE-TRAIN AUDIT ==="
)

print(
    "LATEST_PATH =",
    OFFICIAL_LATEST_PATH,
)

print(
    "LATEST_FILE_BYTES =",
    OFFICIAL_LATEST_PATH.stat().st_size,
)

print(
    "LATEST_FILE_SHA256 =",
    OFFICIAL_PRETRAIN_LATEST_FILE_SHA256,
)

print()

print(
    "LAST_COMPLETED_EPOCH =",
    published_pretrain_payload[
        "last_completed_epoch"
    ],
)

print(
    "NEXT_EPOCH =",
    published_pretrain_payload[
        "last_completed_epoch"
    ] + 1,
)

print(
    "OPTIMIZER_STEP =",
    published_pretrain_payload[
        "optimizer_step"
    ],
)

print(
    "BEST_DEFINED =",
    published_pretrain_payload[
        "best_defined"
    ],
)

print()

print(
    "INITIAL_STOCHASTIC_TOTAL_MEAN =",
    official_initial_stochastic[
        "total_mean_per_sample"
    ],
)

print(
    "INITIAL_STOCHASTIC_RECON_MEAN =",
    official_initial_stochastic[
        "reconstruction_mean_per_sample"
    ],
)

print(
    "INITIAL_STOCHASTIC_KL_MEAN =",
    official_initial_stochastic[
        "kl_mean_per_sample"
    ],
)

print()

print(
    "INITIAL_MEAN_TOTAL_MEAN =",
    official_initial_mean[
        "total_mean_per_sample"
    ],
)

print(
    "INITIAL_MEAN_RECON_MEAN =",
    official_initial_mean[
        "reconstruction_mean_per_sample"
    ],
)

print(
    "INITIAL_MEAN_KL_MEAN =",
    official_initial_mean[
        "kl_mean_per_sample"
    ],
)

print()

print(
    "INITIAL_VALIDATION_STATE_UNCHANGED =",
    initial_validation_state_unchanged,
)

print(
    "OPTIMIZER_STILL_STEP_ZERO =",
    optimizer_still_step_zero,
)

print(
    "INITIAL_METRIC_LEDGER_GATE =",
    initial_metric_ledger_gate,
)

print(
    "PRETRAIN_CHECKPOINT_GATE =",
    pretrain_checkpoint_gate,
)

print()

print(
    "DIRECTORY_CONTENTS =",
    official_names_after_c3,
)

print(
    "BEST_EXISTS =",
    OFFICIAL_BEST_PATH.exists(),
)

print(
    "FINAL_EXISTS =",
    OFFICIAL_FINAL_CHECKPOINT_PATH.exists(),
)

print()

print(
    "BACKWARD_EXECUTED =",
    False,
)

print(
    "OPTIMIZER_STEP_EXECUTED =",
    False,
)

print(
    "OFFICIAL_TRAINING_STARTED =",
    False,
)

print()

print(
    "NOTEBOOK_03_OFFICIAL_PRETRAIN_PASS =",
    (
        pretrain_checkpoint_gate
        and initial_validation_state_unchanged
        and optimizer_still_step_zero
        and initial_metric_ledger_gate
        and official_allowlist_after_c3
        and not OFFICIAL_BEST_PATH.exists()
        and not OFFICIAL_FINAL_CHECKPOINT_PATH.exists()
    ),
)

=== OFFICIAL PRE-TRAIN INITIALIZATION ===
Running official initial stochastic validation...
Running official initial MEAN validation...

=== OFFICIAL PRE-TRAIN AUDIT ===
LATEST_PATH = /content/drive/MyDrive/TESI_SAKAI2022/sakai224_full/runs/sakai224_full_v1.partial/latest.pt
LATEST_FILE_BYTES = 66326791
LATEST_FILE_SHA256 = de9609a7065acf4e52a3a80312576706c045e4a250b029b2d0deb2bc109b1650

LAST_COMPLETED_EPOCH = -1
NEXT_EPOCH = 0
OPTIMIZER_STEP = 0
BEST_DEFINED = False

INITIAL_STOCHASTIC_TOTAL_MEAN = 158220.786210004
INITIAL_STOCHASTIC_RECON_MEAN = 158213.43156668608
INITIAL_STOCHASTIC_KL_MEAN = 7.354643317940767

INITIAL_MEAN_TOTAL_MEAN = 138338.72301839074
INITIAL_MEAN_RECON_MEAN = 138331.3683750728
INITIAL_MEAN_KL_MEAN = 7.354643317940767

INITIAL_VALIDATION_STATE_UNCHANGED = True
OPTIMIZER_STILL_STEP_ZERO = True
INITIAL_METRIC_LEDGER_GATE = True
PRETRAIN_CHECKPOINT_GATE = True

DIRECTORY_CONTENTS = ['latest.pt', 'metrics.csv', 'run_config.json', 'runtime_sessions.jsonl']
BEST_EXIST

## STEP 03-C4 — Audit read-only del resume ufficiale

La run ufficiale possiede ora un checkpoint `LATEST` pre-training valido.

Questo step simula il bootstrap di un runtime nuovo:

1. legge nuovamente `run_config.json` dallo storage persistente del progetto;
2. ricostruisce e verifica il suo SHA-256;
3. legge e valida `metrics.csv`;
4. carica `latest.pt` in un nuovo VAE e in un nuovo optimizer Adam;
5. determina esclusivamente da `LATEST` il prossimo epoch;
6. confronta il ledger delle metriche con lo stato persistente;
7. verifica la coerenza di `BEST`;
8. controlla l'assenza di file temporanei incompleti.

Per lo stato corrente deve risultare:

`last_completed_epoch = -1 → next_epoch = 0`

Le due righe iniziali di validation devono essere conservate ma non definiscono un BEST.

Questo è un audit **read-only** della directory ufficiale:

- nessun file viene scritto;
- nessun forward viene eseguito;
- nessun backward viene eseguito;
- nessun optimizer step viene eseguito.

In [19]:
# ============================================================
# NOTEBOOK 03 — OFFICIAL RESUME PREFLIGHT
# STEP 03-C4
#
# Read-only simulation of a fresh runtime recovering
# the official run exclusively from persistent project state.
#
# READS:
#   run_config.json
#   metrics.csv
#   latest.pt
#   optional best.pt
#
# WRITES NOTHING.
# ============================================================

import csv
import hashlib
import json
import math
from pathlib import Path

import torch


# ============================================================
# HARD DIRECTORY STATE
# ============================================================

assert OFFICIAL_PARTIAL_DIR.is_dir()
assert not OFFICIAL_FINAL_DIR.exists()

assert OFFICIAL_RUN_CONFIG_PATH.is_file()
assert OFFICIAL_METRICS_PATH.is_file()
assert OFFICIAL_RUNTIME_SESSIONS_PATH.is_file()
assert OFFICIAL_LATEST_PATH.is_file()


# ============================================================
# NO STALE TEMPORARY FILES
# ============================================================

stale_temporary_files = sorted(
    path.name

    for path in OFFICIAL_PARTIAL_DIR.iterdir()

    if (
        path.name.endswith(".tmp")
        or path.name.endswith(".reconcile.tmp")
    )
)


stale_tmp_gate = (
    len(stale_temporary_files) == 0
)


assert stale_tmp_gate


# ============================================================
# RELOAD run_config.json FROM PERSISTENT STORAGE
# ============================================================

with OFFICIAL_RUN_CONFIG_PATH.open(
    "r",
    encoding="utf-8",
) as handle:

    resume_run_config = json.load(
        handle
    )


stored_run_config_sha256 = (
    resume_run_config[
        "run_config_sha256"
    ]
)


# The stored SHA was computed before inserting the SHA field
# itself into the JSON object.
run_config_without_sha = dict(
    resume_run_config
)

del run_config_without_sha[
    "run_config_sha256"
]


reconstructed_run_config_json = (
    json.dumps(
        run_config_without_sha,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=True,
    )
)


reconstructed_run_config_sha256 = (
    hashlib.sha256(
        reconstructed_run_config_json.encode(
            "utf-8"
        )
    ).hexdigest()
)


run_config_hash_gate = (
    reconstructed_run_config_sha256
    == stored_run_config_sha256
    == RUN_CONFIG_SHA256
)


assert run_config_hash_gate


# ============================================================
# RECONSTRUCT SCIENTIFIC CONTRACT SHA FROM DISK
# ============================================================

disk_scientific_contract_json = (
    json.dumps(
        resume_run_config[
            "scientific_contract"
        ],
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=True,
    )
)


disk_scientific_contract_sha256 = (
    hashlib.sha256(
        disk_scientific_contract_json.encode(
            "utf-8"
        )
    ).hexdigest()
)


scientific_contract_disk_gate = (
    disk_scientific_contract_sha256
    == SCIENTIFIC_CONTRACT_SHA256

    and resume_run_config[
        "scientific_contract_sha256"
    ] == SCIENTIFIC_CONTRACT_SHA256
)


assert scientific_contract_disk_gate


# ============================================================
# RELOAD LATEST INTO COMPLETELY FRESH OBJECTS
# ============================================================

resume_audit_model = Sakai224FullVAE(
    SPEC
)

initialize_project_c_exact_v1(
    resume_audit_model,
    FROZEN_INITIALIZATION_SEED,
)


resume_audit_model = (
    resume_audit_model.to(
        device=device,
        dtype=torch.float32,
    )
)


resume_audit_optimizer = (
    torch.optim.Adam(
        resume_audit_model.parameters(),

        lr=FROZEN_LEARNING_RATE,

        betas=(
            FROZEN_ADAM_BETA1,
            FROZEN_ADAM_BETA2,
        ),

        eps=FROZEN_ADAM_EPSILON,

        weight_decay=FROZEN_WEIGHT_DECAY,

        amsgrad=FROZEN_AMSGRAD,

        foreach=False,

        fused=False,
    )
)


resume_latest_payload = (
    load_checkpoint_into(
        OFFICIAL_LATEST_PATH,

        resume_audit_model,

        resume_audit_optimizer,

        expected_kind="LATEST",

        device=device,
    )
)


# ============================================================
# LATEST IS THE ONLY RESUME AUTHORITY
# ============================================================

resume_last_completed_epoch = int(
    resume_latest_payload[
        "last_completed_epoch"
    ]
)


resume_next_epoch = (
    resume_last_completed_epoch
    + 1
)


resume_optimizer_step = (
    get_adam_global_step(
        resume_audit_optimizer
    )
)


expected_resume_optimizer_step = (
    resume_next_epoch
    * FROZEN_TRAIN_BATCHES
)


latest_controller_gate = (
    resume_optimizer_step
        == expected_resume_optimizer_step

    and resume_latest_payload[
        "optimizer_step"
    ] == resume_optimizer_step

    and -1
        <= resume_last_completed_epoch
        <= FROZEN_FINAL_EPOCH_INDEX

    and 0
        <= resume_next_epoch
        <= FROZEN_MAX_EPOCHS
)


assert latest_controller_gate


# ============================================================
# STRICT METRICS LEDGER AUDIT
# ============================================================

with OFFICIAL_METRICS_PATH.open(
    "r",
    newline="",
    encoding="utf-8",
) as handle:

    metric_reader = csv.DictReader(
        handle
    )

    resume_metric_rows = list(
        metric_reader
    )

    metric_fieldnames = (
        metric_reader.fieldnames
    )


assert (
    metric_fieldnames
    == OFFICIAL_METRICS_COLUMNS
)


def metric_int(
    row,
    key,
):
    return int(
        row[key]
    )


def metric_float(
    row,
    key,
):
    value = float(
        row[key]
    )

    if not math.isfinite(
        value
    ):
        raise RuntimeError(
            f"Non-finite metric: {key}"
        )

    return value


def validate_metric_numeric_payload(
    row,
):

    numeric_float_fields = [
        "total_sum",
        "total_mean_per_sample",
        "reconstruction_sum",
        "reconstruction_mean_per_sample",
        "reconstruction_mean_per_input_value",
        "kl_sum",
        "kl_mean_per_sample",
        "kl_mean_per_latent_value",
        "wall_seconds",
    ]


    for key in numeric_float_fields:

        value = metric_float(
            row,
            key,
        )

        if value < 0.0:
            raise RuntimeError(
                f"Negative metric value: {key}"
            )


    if metric_int(
        row,
        "sample_count",
    ) <= 0:

        raise RuntimeError(
            "Invalid sample_count"
        )


    if metric_int(
        row,
        "batch_count",
    ) <= 0:

        raise RuntimeError(
            "Invalid batch_count"
        )


# ------------------------------------------------------------
# Initial validation rows
# ------------------------------------------------------------

if len(
    resume_metric_rows
) < 2:

    raise RuntimeError(
        "Initial validation rows are missing"
    )


initial_stochastic_disk = (
    resume_metric_rows[0]
)

initial_mean_disk = (
    resume_metric_rows[1]
)


initial_rows_gate = (

    initial_stochastic_disk[
        "phase"
    ] == "validation_stochastic_initial"

    and initial_mean_disk[
        "phase"
    ] == "validation_mean_initial"

    and metric_int(
        initial_stochastic_disk,
        "epoch",
    ) == -1

    and metric_int(
        initial_mean_disk,
        "epoch",
    ) == -1

    and metric_int(
        initial_stochastic_disk,
        "completed_epochs",
    ) == 0

    and metric_int(
        initial_mean_disk,
        "completed_epochs",
    ) == 0

    and metric_int(
        initial_stochastic_disk,
        "optimizer_step",
    ) == 0

    and metric_int(
        initial_mean_disk,
        "optimizer_step",
    ) == 0

    and metric_int(
        initial_stochastic_disk,
        "sample_count",
    ) == FROZEN_VALIDATION_SAMPLES

    and metric_int(
        initial_mean_disk,
        "sample_count",
    ) == FROZEN_VALIDATION_SAMPLES

    and metric_int(
        initial_stochastic_disk,
        "batch_count",
    ) == FROZEN_VALIDATION_BATCHES

    and metric_int(
        initial_mean_disk,
        "batch_count",
    ) == FROZEN_VALIDATION_BATCHES
)


assert initial_rows_gate


validate_metric_numeric_payload(
    initial_stochastic_disk
)

validate_metric_numeric_payload(
    initial_mean_disk
)


# ============================================================
# VALIDATE ALL COMPLETED EPOCH ROWS
#
# Expected order for every epoch:
#
#   train_stochastic
#   validation_stochastic_post_epoch
#   [validation_mean_diagnostic every 5 completed epochs]
#
# An optional validation_mean_final is allowed only after
# epoch 99.
# ============================================================

cursor = 2

validation_means_from_ledger = []


for epoch in range(
    resume_last_completed_epoch + 1
):

    expected_step = (
        (epoch + 1)
        * FROZEN_TRAIN_BATCHES
    )


    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    if cursor >= len(
        resume_metric_rows
    ):
        raise RuntimeError(
            f"Missing train metrics for epoch {epoch}"
        )


    train_row = (
        resume_metric_rows[
            cursor
        ]
    )

    cursor += 1


    train_gate = (

        train_row[
            "phase"
        ] == "train_stochastic"

        and metric_int(
            train_row,
            "epoch",
        ) == epoch

        and metric_int(
            train_row,
            "completed_epochs",
        ) == epoch + 1

        and metric_int(
            train_row,
            "optimizer_step",
        ) == expected_step

        and metric_int(
            train_row,
            "sample_count",
        ) == FROZEN_TRAIN_SAMPLES

        and metric_int(
            train_row,
            "batch_count",
        ) == FROZEN_TRAIN_BATCHES
    )


    assert train_gate

    validate_metric_numeric_payload(
        train_row
    )


    # --------------------------------------------------------
    # STOCHASTIC VALIDATION
    # --------------------------------------------------------

    if cursor >= len(
        resume_metric_rows
    ):
        raise RuntimeError(
            f"Missing validation metrics for epoch {epoch}"
        )


    validation_row = (
        resume_metric_rows[
            cursor
        ]
    )

    cursor += 1


    validation_gate = (

        validation_row[
            "phase"
        ] == "validation_stochastic_post_epoch"

        and metric_int(
            validation_row,
            "epoch",
        ) == epoch

        and metric_int(
            validation_row,
            "completed_epochs",
        ) == epoch + 1

        and metric_int(
            validation_row,
            "optimizer_step",
        ) == expected_step

        and metric_int(
            validation_row,
            "sample_count",
        ) == FROZEN_VALIDATION_SAMPLES

        and metric_int(
            validation_row,
            "batch_count",
        ) == FROZEN_VALIDATION_BATCHES
    )


    assert validation_gate

    validate_metric_numeric_payload(
        validation_row
    )


    validation_means_from_ledger.append(
        metric_float(
            validation_row,
            "total_mean_per_sample",
        )
    )


    # --------------------------------------------------------
    # PERIODIC MEAN DIAGNOSTIC
    # --------------------------------------------------------

    mean_required = (
        (epoch + 1)
        % MEAN_DIAGNOSTIC_EVERY_EPOCHS
        == 0
    )


    if mean_required:

        if cursor >= len(
            resume_metric_rows
        ):
            raise RuntimeError(
                f"Missing MEAN diagnostic for epoch {epoch}"
            )


        mean_row = (
            resume_metric_rows[
                cursor
            ]
        )

        cursor += 1


        mean_gate = (

            mean_row[
                "phase"
            ] == "validation_mean_diagnostic"

            and metric_int(
                mean_row,
                "epoch",
            ) == epoch

            and metric_int(
                mean_row,
                "completed_epochs",
            ) == epoch + 1

            and metric_int(
                mean_row,
                "optimizer_step",
            ) == expected_step

            and metric_int(
                mean_row,
                "sample_count",
            ) == FROZEN_VALIDATION_SAMPLES

            and metric_int(
                mean_row,
                "batch_count",
            ) == FROZEN_VALIDATION_BATCHES
        )


        assert mean_gate

        validate_metric_numeric_payload(
            mean_row
        )


# ============================================================
# OPTIONAL FINAL ROW
# ============================================================

final_metric_row_present = False


if cursor < len(
    resume_metric_rows
):

    candidate_final = (
        resume_metric_rows[
            cursor
        ]
    )


    if (
        candidate_final[
            "phase"
        ] == "validation_mean_final"
    ):

        final_metric_row_present = True


        final_metric_gate = (

            resume_last_completed_epoch
            == FROZEN_FINAL_EPOCH_INDEX

            and metric_int(
                candidate_final,
                "epoch",
            ) == FROZEN_FINAL_EPOCH_INDEX

            and metric_int(
                candidate_final,
                "completed_epochs",
            ) == FROZEN_MAX_EPOCHS

            and metric_int(
                candidate_final,
                "optimizer_step",
            ) == FROZEN_TOTAL_OPTIMIZER_STEPS

            and metric_int(
                candidate_final,
                "sample_count",
            ) == FROZEN_VALIDATION_SAMPLES

            and metric_int(
                candidate_final,
                "batch_count",
            ) == FROZEN_VALIDATION_BATCHES
        )


        assert final_metric_gate

        validate_metric_numeric_payload(
            candidate_final
        )

        cursor += 1


# No rows are allowed beyond the state supported by LATEST.
metrics_have_no_future_rows = (
    cursor
    == len(
        resume_metric_rows
    )
)


assert metrics_have_no_future_rows


# ============================================================
# RECONSTRUCT BEST FROM THE LEDGER
#
# Strict '<' means earliest epoch wins ties.
# Initial validation does NOT participate.
# ============================================================

if validation_means_from_ledger:

    reconstructed_best_epoch = min(
        range(
            len(
                validation_means_from_ledger
            )
        ),
        key=lambda epoch:
            validation_means_from_ledger[
                epoch
            ],
    )


    reconstructed_best_validation = (
        validation_means_from_ledger[
            reconstructed_best_epoch
        ]
    )


    reconstructed_best_defined = True

else:

    reconstructed_best_epoch = -1

    reconstructed_best_validation = float(
        "inf"
    )

    reconstructed_best_defined = False


best_metadata_gate = (

    resume_latest_payload[
        "best_defined"
    ] == reconstructed_best_defined
)


if reconstructed_best_defined:

    best_metadata_gate = (
        best_metadata_gate

        and resume_latest_payload[
            "best_epoch"
        ] == reconstructed_best_epoch

        and resume_latest_payload[
            "best_validation_total_mean_per_sample"
        ] == reconstructed_best_validation
    )


assert best_metadata_gate


# ============================================================
# BEST CHECKPOINT DISK POLICY
# ============================================================

if not reconstructed_best_defined:

    best_checkpoint_gate = (
        not OFFICIAL_BEST_PATH.exists()
    )

else:

    best_checkpoint_gate = (
        OFFICIAL_BEST_PATH.is_file()
    )


assert best_checkpoint_gate


# ============================================================
# FRESHLY LOADED STATE AUDIT
# ============================================================

loaded_model_state_hash = (
    canonical_tree_sha256(
        resume_audit_model.state_dict()
    )
)


loaded_optimizer_state_hash = (
    canonical_tree_sha256(
        resume_audit_optimizer.state_dict()
    )
)


loaded_state_gate = (

    loaded_model_state_hash
    == resume_latest_payload[
        "model_state_sha256"
    ]

    and loaded_optimizer_state_hash
    == resume_latest_payload[
        "optimizer_state_sha256"
    ]
)


assert loaded_state_gate


# Current pre-training state has a stronger invariant.
pretrain_specific_gate = True


if resume_last_completed_epoch == -1:

    pretrain_specific_gate = (

        resume_next_epoch == 0

        and resume_optimizer_step == 0

        and not reconstructed_best_defined

        and len(
            resume_metric_rows
        ) == 2

        and loaded_model_state_hash
            == OFFICIAL_INITIAL_MODEL_STATE_SHA256

        and loaded_optimizer_state_hash
            == OFFICIAL_INITIAL_OPTIMIZER_STATE_SHA256
    )


assert pretrain_specific_gate


# ============================================================
# OFFICIAL DIRECTORY ALLOWLIST
# ============================================================

allowed_official_names = {
    "run_config.json",
    "metrics.csv",
    "runtime_sessions.jsonl",
    "latest.pt",
}


if reconstructed_best_defined:

    allowed_official_names.add(
        "best.pt"
    )


if final_metric_row_present:

    # final.pt may be produced during terminal finalization.
    allowed_official_names.add(
        "final.pt"
    )


actual_official_names = {
    path.name

    for path
    in OFFICIAL_PARTIAL_DIR.iterdir()
}


directory_state_gate = (
    actual_official_names
    <= allowed_official_names

    and {
        "run_config.json",
        "metrics.csv",
        "runtime_sessions.jsonl",
        "latest.pt",
    }
    <= actual_official_names
)


assert directory_state_gate


# ============================================================
# FINAL RESUME DECISION
# ============================================================

resume_preflight_pass = (

    stale_tmp_gate
    and run_config_hash_gate
    and scientific_contract_disk_gate
    and latest_controller_gate
    and initial_rows_gate
    and metrics_have_no_future_rows
    and best_metadata_gate
    and best_checkpoint_gate
    and loaded_state_gate
    and pretrain_specific_gate
    and directory_state_gate
)


assert resume_preflight_pass


# ============================================================
# REPORT
# ============================================================

print(
    "=== OFFICIAL RESUME READ-ONLY PREFLIGHT ==="
)

print(
    "RUN_ID =",
    resume_run_config[
        "run_id"
    ],
)

print()

print(
    "RUN_CONFIG_SHA256_DISK =",
    reconstructed_run_config_sha256,
)

print(
    "SCIENTIFIC_CONTRACT_SHA256_DISK =",
    disk_scientific_contract_sha256,
)

print(
    "RUN_CONFIG_HASH_GATE =",
    run_config_hash_gate,
)

print(
    "SCIENTIFIC_CONTRACT_DISK_GATE =",
    scientific_contract_disk_gate,
)

print()

print(
    "LAST_COMPLETED_EPOCH =",
    resume_last_completed_epoch,
)

print(
    "RESUME_NEXT_EPOCH =",
    resume_next_epoch,
)

print(
    "OPTIMIZER_STEP =",
    resume_optimizer_step,
)

print()

print(
    "METRIC_ROWS =",
    len(
        resume_metric_rows
    ),
)

print(
    "METRICS_HAVE_NO_FUTURE_ROWS =",
    metrics_have_no_future_rows,
)

print()

print(
    "BEST_DEFINED =",
    reconstructed_best_defined,
)

print(
    "BEST_EPOCH =",
    reconstructed_best_epoch,
)

print(
    "BEST_CHECKPOINT_GATE =",
    best_checkpoint_gate,
)

print()

print(
    "STALE_TEMP_FILES =",
    stale_temporary_files,
)

print(
    "LOADED_STATE_GATE =",
    loaded_state_gate,
)

print(
    "PRETRAIN_SPECIFIC_GATE =",
    pretrain_specific_gate,
)

print(
    "DIRECTORY_STATE_GATE =",
    directory_state_gate,
)

print()

print(
    "MODEL_FORWARD_EXECUTED =",
    False,
)

print(
    "BACKWARD_EXECUTED =",
    False,
)

print(
    "OPTIMIZER_STEP_EXECUTED =",
    False,
)

print(
    "OFFICIAL_FILES_MODIFIED =",
    False,
)

print(
    "OFFICIAL_TRAINING_STARTED =",
    False,
)

print()

print(
    "NOTEBOOK_03_OFFICIAL_RESUME_PREFLIGHT_PASS =",
    resume_preflight_pass,
)


# ============================================================
# RELEASE THE READ-ONLY RESUME COPY
#
# The actual official_model created in C3 remains untouched.
# ============================================================

del resume_audit_model
del resume_audit_optimizer

torch.cuda.empty_cache()

=== OFFICIAL RESUME READ-ONLY PREFLIGHT ===
RUN_ID = sakai224_full_v1

RUN_CONFIG_SHA256_DISK = cb2160294fa2ebe82a84d689ebd0cb9f2fea0264713011eee65413d0d1a9a879
SCIENTIFIC_CONTRACT_SHA256_DISK = c895b626454278f2293e217776c4604d24e03fa78dbaa7ba5049f86f917af949
RUN_CONFIG_HASH_GATE = True
SCIENTIFIC_CONTRACT_DISK_GATE = True

LAST_COMPLETED_EPOCH = -1
RESUME_NEXT_EPOCH = 0
OPTIMIZER_STEP = 0

METRIC_ROWS = 2
METRICS_HAVE_NO_FUTURE_ROWS = True

BEST_DEFINED = False
BEST_EPOCH = -1
BEST_CHECKPOINT_GATE = True

STALE_TEMP_FILES = []
LOADED_STATE_GATE = True
PRETRAIN_SPECIFIC_GATE = True
DIRECTORY_STATE_GATE = True

MODEL_FORWARD_EXECUTED = False
BACKWARD_EXECUTED = False
OPTIMIZER_STEP_EXECUTED = False
OFFICIAL_FILES_MODIFIED = False
OFFICIAL_TRAINING_STARTED = False

NOTEBOOK_03_OFFICIAL_RESUME_PREFLIGHT_PASS = True


## STEP 03-C5 — Test locale della riconciliazione `metrics → LATEST`

Il protocollo della run scrive le metriche prima di pubblicare il nuovo checkpoint `LATEST`.

Questo ordine protegge il checkpoint precedente, ma crea intenzionalmente una possibile situazione dopo una disconnessione:

- le metriche possono essere già state scritte;
- `LATEST` può ancora riferirsi all'epoch precedente.

In questo caso **LATEST è l'autorità di resume**.

La riconciliazione deve quindi:

1. conservare tutte le metriche confermate da `LATEST`;
2. eliminare esclusivamente le righe future;
3. tollerare e rimuovere una eventuale ultima riga parziale prodotta da un'interruzione durante la scrittura;
4. ricostruire il BEST dalle sole validation confermate;
5. rifiutare incoerenze all'interno della parte già confermata.

Il test viene eseguito in una directory temporanea sotto `LOCAL_RUNTIME_ROOT` e non modifica alcun artefatto ufficiale.

Sono simulati tre casi:

- ledger già perfettamente allineato a `LATEST`;
- epoch 0 scritto nelle metriche ma non ancora confermato dal checkpoint pre-training;
- epoch 0 confermato, seguito da una riga incompleta dell'epoch 1.

La stessa funzione di riconciliazione verrà poi usata dal controller ufficiale.

In [20]:
# ============================================================
# NOTEBOOK 03 — METRICS/LATEST RECONCILIATION TEST
# STEP 03-C5
#
# Entire test runs under LOCAL_RUNTIME_ROOT.
#
# OFFICIAL PERSISTENT FILES ARE READ ONLY.
#
# NO model forward.
# NO backward.
# NO optimizer.
# ============================================================

from pathlib import Path

import csv
import hashlib
import io
import math
import os
import shutil


# ============================================================
# LOCAL TEMPORARY TEST DIRECTORY
# ============================================================

RECONCILE_TEST_ROOT = (
    LOCAL_RUNTIME_ROOT
    / "sakai224_full_metrics_reconcile_test"
)


if RECONCILE_TEST_ROOT.exists():

    shutil.rmtree(
        RECONCILE_TEST_ROOT
    )


RECONCILE_TEST_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)


# ============================================================
# FREEZE OFFICIAL METRICS HASH BEFORE THE TEST
# ============================================================

official_metrics_sha_before_c5 = (
    sha256_file_streaming(
        OFFICIAL_METRICS_PATH
    )
)


official_metrics_bytes = (
    OFFICIAL_METRICS_PATH
    .read_bytes()
)


# ============================================================
# STRICT CSV LINE PARSER
# ============================================================

def parse_metric_csv_line(
    line_bytes,
):

    if not line_bytes.endswith(
        b"\n"
    ):
        raise RuntimeError(
            "Metric line is not complete"
        )


    text = (
        line_bytes
        .decode("utf-8")
        .rstrip("\r\n")
    )


    values = next(
        csv.reader(
            [text]
        )
    )


    if len(values) != len(
        OFFICIAL_METRICS_COLUMNS
    ):

        raise RuntimeError(
            "Metric column-count mismatch"
        )


    return dict(
        zip(
            OFFICIAL_METRICS_COLUMNS,
            values,
        )
    )


# ============================================================
# COMMON NUMERIC VALIDATION
# ============================================================

METRIC_FLOAT_COLUMNS = [

    "total_sum",

    "total_mean_per_sample",

    "reconstruction_sum",

    "reconstruction_mean_per_sample",

    "reconstruction_mean_per_input_value",

    "kl_sum",

    "kl_mean_per_sample",

    "kl_mean_per_latent_value",

    "wall_seconds",
]


def metric_row_int(
    row,
    key,
):

    return int(
        row[key]
    )


def metric_row_float(
    row,
    key,
):

    value = float(
        row[key]
    )


    if not math.isfinite(
        value
    ):

        raise RuntimeError(
            f"Non-finite metric field: {key}"
        )


    return value


def validate_metric_row_numeric(
    row,
):

    for key in (
        METRIC_FLOAT_COLUMNS
    ):

        value = metric_row_float(
            row,
            key,
        )


        if value < 0.0:

            raise RuntimeError(
                f"Negative metric field: {key}"
            )


    if metric_row_int(
        row,
        "sample_count",
    ) <= 0:

        raise RuntimeError(
            "Invalid metric sample_count"
        )


    if metric_row_int(
        row,
        "batch_count",
    ) <= 0:

        raise RuntimeError(
            "Invalid metric batch_count"
        )


    return True


# ============================================================
# EXACT EXPECTED-ROW VALIDATION
# ============================================================

def validate_expected_metric_row(
    row,
    *,
    phase,
    epoch,
    completed_epochs,
    optimizer_step,
    sample_count,
    batch_count,
):

    validate_metric_row_numeric(
        row
    )


    gate = (

        row["phase"]
            == phase

        and metric_row_int(
            row,
            "epoch",
        ) == epoch

        and metric_row_int(
            row,
            "completed_epochs",
        ) == completed_epochs

        and metric_row_int(
            row,
            "optimizer_step",
        ) == optimizer_step

        and metric_row_int(
            row,
            "sample_count",
        ) == sample_count

        and metric_row_int(
            row,
            "batch_count",
        ) == batch_count
    )


    if not gate:

        raise RuntimeError(
            "Committed metric row does not match "
            f"expected phase={phase}, epoch={epoch}"
        )


    return True


# ============================================================
# VALIDATION OF A FULL FUTURE ROW
#
# Future rows may be discarded, but if they are complete they
# must still be structurally valid. Corrupted committed-looking
# garbage is never silently ignored.
# ============================================================

def validate_future_metric_row(
    row,
    *,
    latest_epoch,
):

    validate_metric_row_numeric(
        row
    )


    phase = row[
        "phase"
    ]

    epoch = metric_row_int(
        row,
        "epoch",
    )


    if not (
        0 <= epoch
        < FROZEN_MAX_EPOCHS
    ):

        raise RuntimeError(
            "Invalid future metric epoch"
        )


    if epoch <= latest_epoch:

        raise RuntimeError(
            "Trailing row is not actually future state"
        )


    expected_step = (
        (epoch + 1)
        * FROZEN_TRAIN_BATCHES
    )


    if metric_row_int(
        row,
        "optimizer_step",
    ) != expected_step:

        raise RuntimeError(
            "Future metric optimizer-step mismatch"
        )


    if metric_row_int(
        row,
        "completed_epochs",
    ) != epoch + 1:

        raise RuntimeError(
            "Future metric completed_epochs mismatch"
        )


    if phase == "train_stochastic":

        expected_samples = (
            FROZEN_TRAIN_SAMPLES
        )

        expected_batches = (
            FROZEN_TRAIN_BATCHES
        )


    elif phase in (
        "validation_stochastic_post_epoch",
        "validation_mean_diagnostic",
        "validation_mean_final",
    ):

        expected_samples = (
            FROZEN_VALIDATION_SAMPLES
        )

        expected_batches = (
            FROZEN_VALIDATION_BATCHES
        )


    else:

        raise RuntimeError(
            "Unknown future metric phase"
        )


    if metric_row_int(
        row,
        "sample_count",
    ) != expected_samples:

        raise RuntimeError(
            "Future metric sample-count mismatch"
        )


    if metric_row_int(
        row,
        "batch_count",
    ) != expected_batches:

        raise RuntimeError(
            "Future metric batch-count mismatch"
        )


    if (
        phase
        == "validation_mean_diagnostic"
        and (
            (epoch + 1)
            % MEAN_DIAGNOSTIC_EVERY_EPOCHS
        ) != 0
    ):

        raise RuntimeError(
            "Unexpected future MEAN diagnostic"
        )


    if (
        phase
        == "validation_mean_final"
        and epoch
        != FROZEN_FINAL_EPOCH_INDEX
    ):

        raise RuntimeError(
            "Unexpected future FINAL metric"
        )


    return True


# ============================================================
# INSPECT ONE METRICS FILE AGAINST LATEST
#
# Returns the exact byte prefix that is supported by LATEST.
# It performs NO write.
# ============================================================

def inspect_metrics_against_latest(
    raw_bytes,
    latest_metadata,
):

    lines = raw_bytes.splitlines(
        keepends=True
    )


    if not lines:

        raise RuntimeError(
            "Empty metrics file"
        )


    if not lines[0].endswith(
        b"\n"
    ):

        raise RuntimeError(
            "Incomplete metrics header"
        )


    header = (
        lines[0]
        .decode("utf-8")
        .rstrip("\r\n")
    )


    parsed_header = next(
        csv.reader(
            [header]
        )
    )


    if parsed_header != (
        OFFICIAL_METRICS_COLUMNS
    ):

        raise RuntimeError(
            "Metrics header mismatch"
        )


    latest_epoch = int(
        latest_metadata[
            "last_completed_epoch"
        ]
    )


    if not (
        -1 <= latest_epoch
        <= FROZEN_FINAL_EPOCH_INDEX
    ):

        raise RuntimeError(
            "Invalid LATEST epoch"
        )


    expected_optimizer_step = (
        (latest_epoch + 1)
        * FROZEN_TRAIN_BATCHES
    )


    if int(
        latest_metadata[
            "optimizer_step"
        ]
    ) != expected_optimizer_step:

        raise RuntimeError(
            "LATEST optimizer-step mismatch"
        )


    cursor = 1

    kept_lines = [
        lines[0]
    ]


    validation_means = []


    # --------------------------------------------------------
    # REQUIRED INITIAL STOCHASTIC VALIDATION
    # --------------------------------------------------------

    if cursor >= len(lines):

        raise RuntimeError(
            "Missing initial stochastic validation"
        )


    row = parse_metric_csv_line(
        lines[cursor]
    )


    validate_expected_metric_row(
        row,

        phase=(
            "validation_stochastic_initial"
        ),

        epoch=-1,

        completed_epochs=0,

        optimizer_step=0,

        sample_count=(
            FROZEN_VALIDATION_SAMPLES
        ),

        batch_count=(
            FROZEN_VALIDATION_BATCHES
        ),
    )


    kept_lines.append(
        lines[cursor]
    )

    cursor += 1


    # --------------------------------------------------------
    # REQUIRED INITIAL MEAN VALIDATION
    # --------------------------------------------------------

    if cursor >= len(lines):

        raise RuntimeError(
            "Missing initial MEAN validation"
        )


    row = parse_metric_csv_line(
        lines[cursor]
    )


    validate_expected_metric_row(
        row,

        phase=(
            "validation_mean_initial"
        ),

        epoch=-1,

        completed_epochs=0,

        optimizer_step=0,

        sample_count=(
            FROZEN_VALIDATION_SAMPLES
        ),

        batch_count=(
            FROZEN_VALIDATION_BATCHES
        ),
    )


    kept_lines.append(
        lines[cursor]
    )

    cursor += 1


    # --------------------------------------------------------
    # REQUIRED COMMITTED EPOCHS
    # --------------------------------------------------------

    for epoch in range(
        latest_epoch + 1
    ):

        step = (
            (epoch + 1)
            * FROZEN_TRAIN_BATCHES
        )


        # TRAIN
        if cursor >= len(lines):

            raise RuntimeError(
                f"Missing committed train row for epoch {epoch}"
            )


        train_row = parse_metric_csv_line(
            lines[cursor]
        )


        validate_expected_metric_row(
            train_row,

            phase="train_stochastic",

            epoch=epoch,

            completed_epochs=(
                epoch + 1
            ),

            optimizer_step=step,

            sample_count=(
                FROZEN_TRAIN_SAMPLES
            ),

            batch_count=(
                FROZEN_TRAIN_BATCHES
            ),
        )


        kept_lines.append(
            lines[cursor]
        )

        cursor += 1


        # STOCHASTIC VALIDATION
        if cursor >= len(lines):

            raise RuntimeError(
                f"Missing committed validation row for epoch {epoch}"
            )


        validation_row = (
            parse_metric_csv_line(
                lines[cursor]
            )
        )


        validate_expected_metric_row(
            validation_row,

            phase=(
                "validation_stochastic_post_epoch"
            ),

            epoch=epoch,

            completed_epochs=(
                epoch + 1
            ),

            optimizer_step=step,

            sample_count=(
                FROZEN_VALIDATION_SAMPLES
            ),

            batch_count=(
                FROZEN_VALIDATION_BATCHES
            ),
        )


        validation_means.append(
            metric_row_float(
                validation_row,
                "total_mean_per_sample",
            )
        )


        kept_lines.append(
            lines[cursor]
        )

        cursor += 1


        # PERIODIC MEAN
        if (
            (epoch + 1)
            % MEAN_DIAGNOSTIC_EVERY_EPOCHS
        ) == 0:

            if cursor >= len(lines):

                raise RuntimeError(
                    f"Missing committed MEAN row for epoch {epoch}"
                )


            mean_row = (
                parse_metric_csv_line(
                    lines[cursor]
                )
            )


            validate_expected_metric_row(
                mean_row,

                phase=(
                    "validation_mean_diagnostic"
                ),

                epoch=epoch,

                completed_epochs=(
                    epoch + 1
                ),

                optimizer_step=step,

                sample_count=(
                    FROZEN_VALIDATION_SAMPLES
                ),

                batch_count=(
                    FROZEN_VALIDATION_BATCHES
                ),
            )


            kept_lines.append(
                lines[cursor]
            )

            cursor += 1


    # --------------------------------------------------------
    # OPTIONAL TERMINAL FINAL-MEAN ROW
    #
    # It is supported only after epoch 99 is already LATEST.
    # --------------------------------------------------------

    final_row_present = False


    if (
        latest_epoch
        == FROZEN_FINAL_EPOCH_INDEX

        and cursor < len(lines)

        and lines[cursor].endswith(
            b"\n"
        )
    ):

        candidate = (
            parse_metric_csv_line(
                lines[cursor]
            )
        )


        if (
            candidate["phase"]
            == "validation_mean_final"
        ):

            validate_expected_metric_row(
                candidate,

                phase=(
                    "validation_mean_final"
                ),

                epoch=(
                    FROZEN_FINAL_EPOCH_INDEX
                ),

                completed_epochs=(
                    FROZEN_MAX_EPOCHS
                ),

                optimizer_step=(
                    FROZEN_TOTAL_OPTIMIZER_STEPS
                ),

                sample_count=(
                    FROZEN_VALIDATION_SAMPLES
                ),

                batch_count=(
                    FROZEN_VALIDATION_BATCHES
                ),
            )


            kept_lines.append(
                lines[cursor]
            )

            cursor += 1

            final_row_present = True


    # --------------------------------------------------------
    # EVERYTHING AFTER THE COMMITTED PREFIX MUST BE FUTURE.
    #
    # A single incomplete final line is accepted as evidence
    # of interruption during append and is discarded.
    # --------------------------------------------------------

    future_full_rows = 0
    partial_tail_dropped = False


    while cursor < len(lines):

        line = lines[cursor]


        if not line.endswith(
            b"\n"
        ):

            if cursor != (
                len(lines) - 1
            ):

                raise RuntimeError(
                    "Incomplete metric line before EOF"
                )


            partial_tail_dropped = True

            cursor += 1

            break


        future_row = (
            parse_metric_csv_line(
                line
            )
        )


        validate_future_metric_row(
            future_row,
            latest_epoch=latest_epoch,
        )


        future_full_rows += 1

        cursor += 1


    if cursor != len(lines):

        raise RuntimeError(
            "Unexpected bytes after metrics audit"
        )


    # --------------------------------------------------------
    # RECONSTRUCT BEST FROM COMMITTED VALIDATIONS ONLY
    # --------------------------------------------------------

    if validation_means:

        reconstructed_best_defined = (
            True
        )

        reconstructed_best_epoch = 0

        reconstructed_best_value = (
            validation_means[0]
        )


        for epoch, value in enumerate(
            validation_means[1:],
            start=1,
        ):

            # Strict '<':
            # earliest epoch wins exact ties.
            if (
                value
                < reconstructed_best_value
            ):

                reconstructed_best_value = (
                    value
                )

                reconstructed_best_epoch = (
                    epoch
                )


    else:

        reconstructed_best_defined = (
            False
        )

        reconstructed_best_epoch = -1

        reconstructed_best_value = float(
            "inf"
        )


    if (
        bool(
            latest_metadata[
                "best_defined"
            ]
        )
        != reconstructed_best_defined
    ):

        raise RuntimeError(
            "LATEST BEST-defined state disagrees with metrics"
        )


    if reconstructed_best_defined:

        if (
            int(
                latest_metadata[
                    "best_epoch"
                ]
            )
            != reconstructed_best_epoch
        ):

            raise RuntimeError(
                "LATEST BEST epoch disagrees with metrics"
            )


        if (
            float(
                latest_metadata[
                    "best_validation_total_mean_per_sample"
                ]
            )
            != reconstructed_best_value
        ):

            raise RuntimeError(
                "LATEST BEST value disagrees with metrics"
            )


    kept_bytes = b"".join(
        kept_lines
    )


    return {

        "latest_epoch":
            latest_epoch,

        "expected_optimizer_step":
            expected_optimizer_step,

        "kept_bytes":
            kept_bytes,

        "committed_metric_rows":
            len(
                kept_lines
            ) - 1,

        "future_full_rows":
            future_full_rows,

        "partial_tail_dropped":
            partial_tail_dropped,

        "needs_reconciliation":
            (
                kept_bytes
                != raw_bytes
            ),

        "best_defined":
            reconstructed_best_defined,

        "best_epoch":
            reconstructed_best_epoch,

        "best_validation":
            reconstructed_best_value,

        "final_row_present":
            final_row_present,
    }


# ============================================================
# ATOMIC RECONCILIATION
#
#   inspect
#   -> write .reconcile.tmp
#   -> fsync
#   -> verify resulting prefix
#   -> atomic replace
# ============================================================

def reconcile_metrics_to_latest_atomic(
    metrics_path,
    latest_metadata,
):

    metrics_path = Path(
        metrics_path
    )


    temporary_path = Path(
        str(
            metrics_path
        )
        + ".reconcile.tmp"
    )


    if temporary_path.exists():

        raise RuntimeError(
            "Stale .reconcile.tmp exists"
        )


    raw_before = (
        metrics_path
        .read_bytes()
    )


    audit = (
        inspect_metrics_against_latest(
            raw_before,
            latest_metadata,
        )
    )


    if not audit[
        "needs_reconciliation"
    ]:

        return {
            **audit,

            "reconciled":
                False,

            "sha256_after":
                hashlib.sha256(
                    raw_before
                ).hexdigest(),
        }


    with temporary_path.open(
        "xb"
    ) as handle:

        handle.write(
            audit[
                "kept_bytes"
            ]
        )

        handle.flush()

        os.fsync(
            handle.fileno()
        )


    temporary_bytes = (
        temporary_path
        .read_bytes()
    )


    if (
        temporary_bytes
        != audit[
            "kept_bytes"
        ]
    ):

        raise RuntimeError(
            "Reconciliation read-back mismatch"
        )


    # The proposed repaired file must itself be a perfectly
    # aligned ledger: no future rows and no partial tail.
    verification = (
        inspect_metrics_against_latest(
            temporary_bytes,
            latest_metadata,
        )
    )


    if (
        verification[
            "needs_reconciliation"
        ]

        or verification[
            "future_full_rows"
        ] != 0

        or verification[
            "partial_tail_dropped"
        ]
    ):

        raise RuntimeError(
            "Reconciled metrics failed independent verification"
        )


    os.replace(
        temporary_path,
        metrics_path,
    )


    final_bytes = (
        metrics_path
        .read_bytes()
    )


    if (
        final_bytes
        != audit[
            "kept_bytes"
        ]
    ):

        raise RuntimeError(
            "Published reconciled metrics mismatch"
        )


    return {
        **audit,

        "reconciled":
            True,

        "sha256_after":
            hashlib.sha256(
                final_bytes
            ).hexdigest(),
    }


# ============================================================
# SYNTHETIC METRIC ROW GENERATOR FOR LOCAL TESTS
# ============================================================

def make_synthetic_metric_row(
    *,
    phase,
    epoch,
    completed_epochs,
    optimizer_step,
    sample_count,
    batch_count,
    total_mean,
):

    reconstruction_mean = (
        float(total_mean)
        - 10.0
    )

    kl_mean = 10.0


    reconstruction_sum = (
        reconstruction_mean
        * sample_count
    )

    kl_sum = (
        kl_mean
        * sample_count
    )

    total_sum = (
        reconstruction_sum
        + kl_sum
    )


    return {

        "phase":
            phase,

        "epoch":
            str(epoch),

        "completed_epochs":
            str(
                completed_epochs
            ),

        "optimizer_step":
            str(
                optimizer_step
            ),

        "sample_count":
            str(
                sample_count
            ),

        "batch_count":
            str(
                batch_count
            ),

        "total_sum":
            repr(
                total_sum
            ),

        "total_mean_per_sample":
            repr(
                float(
                    total_mean
                )
            ),

        "reconstruction_sum":
            repr(
                reconstruction_sum
            ),

        "reconstruction_mean_per_sample":
            repr(
                reconstruction_mean
            ),

        "reconstruction_mean_per_input_value":
            repr(
                reconstruction_sum
                / (
                    sample_count
                    * 4
                    * 224
                    * 224
                )
            ),

        "kl_sum":
            repr(
                kl_sum
            ),

        "kl_mean_per_sample":
            repr(
                kl_mean
            ),

        "kl_mean_per_latent_value":
            repr(
                kl_sum
                / (
                    sample_count
                    * SPEC.latent_dim
                )
            ),

        "wall_seconds":
            repr(
                1.0
            ),
    }


def csv_rows_to_bytes(
    rows,
):

    stream = io.StringIO(
        newline=""
    )


    writer = csv.DictWriter(
        stream,
        fieldnames=(
            OFFICIAL_METRICS_COLUMNS
        ),
    )


    for row in rows:

        writer.writerow(
            row
        )


    return (
        stream
        .getvalue()
        .encode("utf-8")
    )


# ============================================================
# SYNTHETIC EPOCH-0 ROWS
# ============================================================

synthetic_epoch0_train = (
    make_synthetic_metric_row(

        phase="train_stochastic",

        epoch=0,

        completed_epochs=1,

        optimizer_step=54,

        sample_count=(
            FROZEN_TRAIN_SAMPLES
        ),

        batch_count=(
            FROZEN_TRAIN_BATCHES
        ),

        total_mean=130000.0,
    )
)


synthetic_epoch0_validation = (
    make_synthetic_metric_row(

        phase=(
            "validation_stochastic_post_epoch"
        ),

        epoch=0,

        completed_epochs=1,

        optimizer_step=54,

        sample_count=(
            FROZEN_VALIDATION_SAMPLES
        ),

        batch_count=(
            FROZEN_VALIDATION_BATCHES
        ),

        total_mean=120000.0,
    )
)


epoch0_bytes = (
    csv_rows_to_bytes([
        synthetic_epoch0_train,
        synthetic_epoch0_validation,
    ])
)


# ============================================================
# MINIMAL CHECKPOINT METADATA FOR TESTING
# ============================================================

pretrain_latest_metadata = {

    "last_completed_epoch":
        -1,

    "optimizer_step":
        0,

    "best_defined":
        False,

    "best_epoch":
        -1,

    "best_validation_total_mean_per_sample":
        float("inf"),
}


epoch0_latest_metadata = {

    "last_completed_epoch":
        0,

    "optimizer_step":
        54,

    "best_defined":
        True,

    "best_epoch":
        0,

    "best_validation_total_mean_per_sample":
        120000.0,
}


# ============================================================
# SCENARIO 1
#
# Current pristine state already agrees with LATEST=-1.
# Must be a true no-op.
# ============================================================

scenario1_path = (
    RECONCILE_TEST_ROOT
    / "scenario1_exact.csv"
)


scenario1_path.write_bytes(
    official_metrics_bytes
)


scenario1_before = (
    scenario1_path
    .read_bytes()
)


scenario1_result = (
    reconcile_metrics_to_latest_atomic(
        scenario1_path,
        pretrain_latest_metadata,
    )
)


scenario1_after = (
    scenario1_path
    .read_bytes()
)


scenario1_pass = (

    scenario1_result[
        "reconciled"
    ] is False

    and scenario1_before
        == scenario1_after

    and scenario1_result[
        "committed_metric_rows"
    ] == 2

    and scenario1_result[
        "best_defined"
    ] is False
)


assert scenario1_pass


# ============================================================
# SCENARIO 2 — WORST COMMIT WINDOW
#
# Epoch-0 train + validation are already fully written,
# but LATEST still says -1.
#
# Both rows must disappear.
# ============================================================

scenario2_path = (
    RECONCILE_TEST_ROOT
    / "scenario2_future_epoch0.csv"
)


scenario2_path.write_bytes(
    official_metrics_bytes
    + epoch0_bytes
)


scenario2_result = (
    reconcile_metrics_to_latest_atomic(
        scenario2_path,
        pretrain_latest_metadata,
    )
)


scenario2_after = (
    scenario2_path
    .read_bytes()
)


scenario2_pass = (

    scenario2_result[
        "reconciled"
    ] is True

    and scenario2_result[
        "future_full_rows"
    ] == 2

    and scenario2_result[
        "partial_tail_dropped"
    ] is False

    and scenario2_after
        == official_metrics_bytes

    and scenario2_result[
        "best_defined"
    ] is False
)


assert scenario2_pass


# ============================================================
# SCENARIO 3
#
# LATEST confirms epoch 0 and therefore its train/validation
# rows must survive.
#
# A partially-written epoch-1 row must be discarded.
#
# This also verifies BEST reconstruction for committed epoch 0.
# ============================================================

scenario3_path = (
    RECONCILE_TEST_ROOT
    / "scenario3_partial_epoch1.csv"
)


partial_epoch1_bytes = (
    b"train_stochastic,1,2,108,6866"
)


scenario3_committed_bytes = (
    official_metrics_bytes
    + epoch0_bytes
)


scenario3_path.write_bytes(
    scenario3_committed_bytes
    + partial_epoch1_bytes
)


scenario3_result = (
    reconcile_metrics_to_latest_atomic(
        scenario3_path,
        epoch0_latest_metadata,
    )
)


scenario3_after = (
    scenario3_path
    .read_bytes()
)


scenario3_pass = (

    scenario3_result[
        "reconciled"
    ] is True

    and scenario3_result[
        "future_full_rows"
    ] == 0

    and scenario3_result[
        "partial_tail_dropped"
    ] is True

    and scenario3_after
        == scenario3_committed_bytes

    and scenario3_result[
        "best_defined"
    ] is True

    and scenario3_result[
        "best_epoch"
    ] == 0

    and scenario3_result[
        "best_validation"
    ] == 120000.0
)


assert scenario3_pass


# ============================================================
# OFFICIAL PERSISTENT DIRECTORY MUST BE BITWISE UNCHANGED
# ============================================================

official_metrics_sha_after_c5 = (
    sha256_file_streaming(
        OFFICIAL_METRICS_PATH
    )
)


official_metrics_unchanged = (
    official_metrics_sha_before_c5
    == official_metrics_sha_after_c5
)


assert official_metrics_unchanged


# Official checkpoint must also still be the pretrain anchor.
official_latest_sha_after_c5 = (
    sha256_file_streaming(
        OFFICIAL_LATEST_PATH
    )
)


official_latest_unchanged = (
    official_latest_sha_after_c5
    == OFFICIAL_PRETRAIN_LATEST_FILE_SHA256
)


assert official_latest_unchanged


# ============================================================
# FINAL REPORT
# ============================================================

print(
    "=== LOCAL METRICS → LATEST RECONCILIATION TEST ==="
)

print()

print(
    "SCENARIO_1_ALREADY_ALIGNED_PASS =",
    scenario1_pass,
)

print(
    "SCENARIO_1_RECONCILED =",
    scenario1_result[
        "reconciled"
    ],
)

print()

print(
    "SCENARIO_2_FUTURE_EPOCH0_PASS =",
    scenario2_pass,
)

print(
    "SCENARIO_2_FUTURE_ROWS_DROPPED =",
    scenario2_result[
        "future_full_rows"
    ],
)

print()

print(
    "SCENARIO_3_PARTIAL_TAIL_PASS =",
    scenario3_pass,
)

print(
    "SCENARIO_3_PARTIAL_TAIL_DROPPED =",
    scenario3_result[
        "partial_tail_dropped"
    ],
)

print(
    "SCENARIO_3_RECONSTRUCTED_BEST_EPOCH =",
    scenario3_result[
        "best_epoch"
    ],
)

print()

print(
    "OFFICIAL_METRICS_SHA_BEFORE =",
    official_metrics_sha_before_c5,
)

print(
    "OFFICIAL_METRICS_SHA_AFTER =",
    official_metrics_sha_after_c5,
)

print(
    "OFFICIAL_METRICS_UNCHANGED =",
    official_metrics_unchanged,
)

print(
    "OFFICIAL_LATEST_UNCHANGED =",
    official_latest_unchanged,
)

print()

print(
    "OFFICIAL_FILES_MODIFIED =",
    False,
)

print(
    "MODEL_FORWARD_EXECUTED =",
    False,
)

print(
    "BACKWARD_EXECUTED =",
    False,
)

print(
    "OPTIMIZER_STEP_EXECUTED =",
    False,
)

print(
    "OFFICIAL_TRAINING_STARTED =",
    False,
)

print()

print(
    "NOTEBOOK_03_METRICS_RECONCILIATION_PASS =",
    (
        scenario1_pass
        and scenario2_pass
        and scenario3_pass
        and official_metrics_unchanged
        and official_latest_unchanged
    ),
)

=== LOCAL METRICS → LATEST RECONCILIATION TEST ===

SCENARIO_1_ALREADY_ALIGNED_PASS = True
SCENARIO_1_RECONCILED = False

SCENARIO_2_FUTURE_EPOCH0_PASS = True
SCENARIO_2_FUTURE_ROWS_DROPPED = 2

SCENARIO_3_PARTIAL_TAIL_PASS = True
SCENARIO_3_PARTIAL_TAIL_DROPPED = True
SCENARIO_3_RECONSTRUCTED_BEST_EPOCH = 0

OFFICIAL_METRICS_SHA_BEFORE = 3172643ba06a11abb9d2956a143018c7b96c03ffeccdab21abb8ea07ce5a9f92
OFFICIAL_METRICS_SHA_AFTER = 3172643ba06a11abb9d2956a143018c7b96c03ffeccdab21abb8ea07ce5a9f92
OFFICIAL_METRICS_UNCHANGED = True
OFFICIAL_LATEST_UNCHANGED = True

OFFICIAL_FILES_MODIFIED = False
MODEL_FORWARD_EXECUTED = False
BACKWARD_EXECUTED = False
OPTIMIZER_STEP_EXECUTED = False
OFFICIAL_TRAINING_STARTED = False

NOTEBOOK_03_METRICS_RECONCILIATION_PASS = True


## STEP 03-C6 — Dry run del controller ufficiale

Prima di autorizzare il primo optimizer step ufficiale, il controller viene eseguito in modalità **read-only**.

Partendo esclusivamente dagli artefatti nello storage persistente del progetto deve:

1. verificare `run_config.json`;
2. verificare il contratto scientifico;
3. validare e caricare `LATEST`;
4. confrontare `metrics.csv` con `LATEST`;
5. verificare lo stato di `BEST`;
6. controllare l'assenza di file temporanei;
7. ricostruire il prossimo epoch e il relativo optimizer-step range;
8. produrre il piano residuo fino alla centesima epoca.

Nello stato pre-training attuale il risultato corretto è:

- `last_completed_epoch = -1`;
- `next_epoch = 0`;
- optimizer step iniziale `0`;
- epoch 0 porterà lo step a `54`;
- restano `100` epoche;
- stato terminale atteso: epoch `99`, optimizer step `5400`.

Il dry run non modifica alcun file, non esegue il VAE e non avvia il training.

In [21]:
# ============================================================
# NOTEBOOK 03 — OFFICIAL CONTROLLER DRY RUN
# STEP 03-C6
#
# READ-ONLY.
#
# Determines the exact next action from persistent project state.
#
# NO forward.
# NO backward.
# NO optimizer step.
# NO file write.
# ============================================================

import hashlib
import json
from pathlib import Path

import torch


# ============================================================
# SNAPSHOT OFFICIAL FILES BEFORE DRY RUN
# ============================================================

def official_directory_fingerprint():

    entries = []

    for path in sorted(
        OFFICIAL_PARTIAL_DIR.iterdir(),
        key=lambda p: p.name,
    ):

        if not path.is_file():
            raise RuntimeError(
                f"Unexpected non-file in official run: {path}"
            )

        entries.append({
            "name":
                path.name,

            "bytes":
                path.stat().st_size,

            "sha256":
                sha256_file_streaming(
                    path
                ),
        })


    canonical = json.dumps(
        entries,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=True,
    )


    return {
        "entries":
            entries,

        "sha256":
            hashlib.sha256(
                canonical.encode(
                    "utf-8"
                )
            ).hexdigest(),
    }


official_before_c6 = (
    official_directory_fingerprint()
)


# ============================================================
# HARD ENVIRONMENT / NUMERICAL POLICY
# ============================================================

assert torch.cuda.is_available()

assert (
    torch.backends.cuda.matmul.allow_tf32
    is False
)

assert (
    torch.backends.cudnn.allow_tf32
    is False
)

assert (
    FROZEN_AMP_ENABLED
    is False
)


# ============================================================
# RELOAD run_config.json FROM PERSISTENT STORAGE
# ============================================================

with OFFICIAL_RUN_CONFIG_PATH.open(
    "r",
    encoding="utf-8",
) as handle:

    controller_config = json.load(
        handle
    )


stored_config_sha = (
    controller_config[
        "run_config_sha256"
    ]
)


config_without_sha = dict(
    controller_config
)

del config_without_sha[
    "run_config_sha256"
]


recomputed_config_sha = (
    hashlib.sha256(
        json.dumps(
            config_without_sha,
            sort_keys=True,
            separators=(",", ":"),
            ensure_ascii=True,
        ).encode(
            "utf-8"
        )
    ).hexdigest()
)


config_gate = (
    stored_config_sha
        == RUN_CONFIG_SHA256

    and recomputed_config_sha
        == RUN_CONFIG_SHA256

    and controller_config[
        "run_id"
    ] == RUN_ID

    and controller_config[
        "scientific_contract_sha256"
    ] == SCIENTIFIC_CONTRACT_SHA256
)


assert config_gate


# ============================================================
# STALE TEMPORARY FILE GATE
# ============================================================

controller_stale_tmp = sorted(
    path.name

    for path
    in OFFICIAL_PARTIAL_DIR.iterdir()

    if (
        path.name.endswith(".tmp")
        or path.name.endswith(
            ".reconcile.tmp"
        )
    )
)


stale_tmp_gate = (
    len(
        controller_stale_tmp
    ) == 0
)


assert stale_tmp_gate


# ============================================================
# LOAD LATEST INTO FRESH OBJECTS
#
# This simulates a fresh runtime.
# ============================================================

controller_model = (
    Sakai224FullVAE(
        SPEC
    )
)

initialize_project_c_exact_v1(
    controller_model,
    FROZEN_INITIALIZATION_SEED,
)


controller_model = (
    controller_model.to(
        device=device,
        dtype=torch.float32,
    )
)


controller_optimizer = (
    torch.optim.Adam(
        controller_model.parameters(),

        lr=FROZEN_LEARNING_RATE,

        betas=(
            FROZEN_ADAM_BETA1,
            FROZEN_ADAM_BETA2,
        ),

        eps=FROZEN_ADAM_EPSILON,

        weight_decay=FROZEN_WEIGHT_DECAY,

        amsgrad=FROZEN_AMSGRAD,

        foreach=False,

        fused=False,
    )
)


controller_latest = (
    load_checkpoint_into(
        OFFICIAL_LATEST_PATH,

        controller_model,

        controller_optimizer,

        expected_kind="LATEST",

        device=device,
    )
)


# ============================================================
# DERIVE CONTROLLER POSITION EXCLUSIVELY FROM LATEST
# ============================================================

controller_last_completed_epoch = int(
    controller_latest[
        "last_completed_epoch"
    ]
)


controller_next_epoch = (
    controller_last_completed_epoch
    + 1
)


controller_current_step = (
    get_adam_global_step(
        controller_optimizer
    )
)


expected_current_step = (
    controller_next_epoch
    * FROZEN_TRAIN_BATCHES
)


position_gate = (
    controller_current_step
        == expected_current_step

    and controller_latest[
        "optimizer_step"
    ] == controller_current_step

    and -1
        <= controller_last_completed_epoch
        <= FROZEN_FINAL_EPOCH_INDEX

    and 0
        <= controller_next_epoch
        <= FROZEN_MAX_EPOCHS
)


assert position_gate


# ============================================================
# READ METRICS AND AUDIT THEM AGAINST LATEST
#
# inspect_metrics_against_latest() was validated in STEP C5.
# This invocation performs NO write.
# ============================================================

official_metrics_raw = (
    OFFICIAL_METRICS_PATH
    .read_bytes()
)


controller_metrics_audit = (
    inspect_metrics_against_latest(
        official_metrics_raw,
        controller_latest,
    )
)


metrics_gate = (
    controller_metrics_audit[
        "needs_reconciliation"
    ] is False

    and controller_metrics_audit[
        "future_full_rows"
    ] == 0

    and controller_metrics_audit[
        "partial_tail_dropped"
    ] is False
)


assert metrics_gate


# ============================================================
# BEST CONSISTENCY
# ============================================================

controller_best_defined = bool(
    controller_metrics_audit[
        "best_defined"
    ]
)


controller_best_epoch = int(
    controller_metrics_audit[
        "best_epoch"
    ]
)


if controller_best_defined:

    best_disk_gate = (
        OFFICIAL_BEST_PATH.is_file()

        and controller_latest[
            "best_defined"
        ] is True

        and controller_latest[
            "best_epoch"
        ] == controller_best_epoch
    )

else:

    best_disk_gate = (
        not OFFICIAL_BEST_PATH.exists()

        and controller_latest[
            "best_defined"
        ] is False

        and controller_best_epoch == -1
    )


assert best_disk_gate


# ============================================================
# DETERMINE CONTROLLER MODE
# ============================================================

if controller_next_epoch < (
    FROZEN_MAX_EPOCHS
):

    controller_mode = (
        "TRAIN_NEXT_EPOCH"
    )

elif (
    controller_next_epoch
    == FROZEN_MAX_EPOCHS
):

    controller_mode = (
        "TRAINING_COMPLETE_NEEDS_FINALIZATION"
    )

else:

    raise RuntimeError(
        "Controller next_epoch exceeds frozen limit"
    )


# ============================================================
# NEXT-EPOCH PLAN
# ============================================================

if controller_mode == (
    "TRAIN_NEXT_EPOCH"
):

    planned_epoch = (
        controller_next_epoch
    )

    planned_start_step = (
        controller_current_step
    )

    planned_end_step = (
        planned_start_step
        + FROZEN_TRAIN_BATCHES
    )

    planned_completed_epochs_after = (
        planned_epoch
        + 1
    )

    planned_mean_diagnostic = (
        planned_completed_epochs_after
        % MEAN_DIAGNOSTIC_EVERY_EPOCHS
        == 0
    )

else:

    planned_epoch = None
    planned_start_step = (
        controller_current_step
    )
    planned_end_step = (
        controller_current_step
    )
    planned_completed_epochs_after = (
        FROZEN_MAX_EPOCHS
    )
    planned_mean_diagnostic = False


# ============================================================
# REMAINING WORK
# ============================================================

remaining_epochs = (
    FROZEN_MAX_EPOCHS
    - controller_next_epoch
)


remaining_optimizer_steps = (
    FROZEN_TOTAL_OPTIMIZER_STEPS
    - controller_current_step
)


remaining_stochastic_validations = (
    remaining_epochs
)


remaining_mean_diagnostics = sum(
    1

    for epoch in range(
        controller_next_epoch,
        FROZEN_MAX_EPOCHS,
    )

    if (
        (epoch + 1)
        % MEAN_DIAGNOSTIC_EVERY_EPOCHS
        == 0
    )
)


# ============================================================
# PLAN HARD GATES
# ============================================================

plan_gate = (
    remaining_epochs >= 0

    and remaining_optimizer_steps >= 0

    and (
        controller_current_step
        + remaining_optimizer_steps
        == FROZEN_TOTAL_OPTIMIZER_STEPS
    )

    and (
        controller_next_epoch
        + remaining_epochs
        == FROZEN_MAX_EPOCHS
    )
)


assert plan_gate


# Current pre-training state has an even stronger expected gate.
pretrain_dryrun_gate = True


if (
    controller_last_completed_epoch
    == -1
):

    pretrain_dryrun_gate = (
        controller_mode
            == "TRAIN_NEXT_EPOCH"

        and controller_next_epoch
            == 0

        and controller_current_step
            == 0

        and planned_epoch
            == 0

        and planned_start_step
            == 0

        and planned_end_step
            == 54

        and planned_mean_diagnostic
            is False

        and remaining_epochs
            == 100

        and remaining_optimizer_steps
            == 5400

        and remaining_stochastic_validations
            == 100

        and remaining_mean_diagnostics
            == 20

        and controller_best_defined
            is False
    )


assert pretrain_dryrun_gate


# ============================================================
# NO MODEL EXECUTION OCCURRED
# ============================================================

controller_model_hash_after_load = (
    canonical_tree_sha256(
        controller_model.state_dict()
    )
)


controller_optimizer_hash_after_load = (
    canonical_tree_sha256(
        controller_optimizer.state_dict()
    )
)


loaded_state_gate = (
    controller_model_hash_after_load
        == controller_latest[
            "model_state_sha256"
        ]

    and controller_optimizer_hash_after_load
        == controller_latest[
            "optimizer_state_sha256"
        ]
)


assert loaded_state_gate


# ============================================================
# OFFICIAL PERSISTENT DIRECTORY MUST BE BITWISE UNCHANGED
# ============================================================

official_after_c6 = (
    official_directory_fingerprint()
)


official_files_unchanged = (
    official_before_c6[
        "sha256"
    ]
    == official_after_c6[
        "sha256"
    ]

    and official_before_c6[
        "entries"
    ]
    == official_after_c6[
        "entries"
    ]
)


assert official_files_unchanged


# ============================================================
# FINAL DRY-RUN GATE
# ============================================================

controller_dry_run_pass = (
    config_gate
    and stale_tmp_gate
    and position_gate
    and metrics_gate
    and best_disk_gate
    and plan_gate
    and pretrain_dryrun_gate
    and loaded_state_gate
    and official_files_unchanged
)


assert controller_dry_run_pass


# ============================================================
# REPORT
# ============================================================

print(
    "=== OFFICIAL CONTROLLER DRY RUN ==="
)

print(
    "CONTROLLER_MODE =",
    controller_mode,
)

print()

print(
    "LAST_COMPLETED_EPOCH =",
    controller_last_completed_epoch,
)

print(
    "NEXT_EPOCH =",
    controller_next_epoch,
)

print(
    "CURRENT_OPTIMIZER_STEP =",
    controller_current_step,
)

print()

print(
    "PLANNED_EPOCH =",
    planned_epoch,
)

print(
    "PLANNED_START_STEP =",
    planned_start_step,
)

print(
    "PLANNED_END_STEP =",
    planned_end_step,
)

print(
    "PLANNED_MEAN_DIAGNOSTIC =",
    planned_mean_diagnostic,
)

print()

print(
    "REMAINING_EPOCHS =",
    remaining_epochs,
)

print(
    "REMAINING_OPTIMIZER_STEPS =",
    remaining_optimizer_steps,
)

print(
    "REMAINING_STOCHASTIC_VALIDATIONS =",
    remaining_stochastic_validations,
)

print(
    "REMAINING_MEAN_DIAGNOSTICS =",
    remaining_mean_diagnostics,
)

print()

print(
    "BEST_DEFINED =",
    controller_best_defined,
)

print(
    "BEST_EPOCH =",
    controller_best_epoch,
)

print()

print(
    "METRICS_NEED_RECONCILIATION =",
    controller_metrics_audit[
        "needs_reconciliation"
    ],
)

print(
    "STALE_TEMP_FILES =",
    controller_stale_tmp,
)

print()

print(
    "OFFICIAL_DIRECTORY_SHA_BEFORE =",
    official_before_c6[
        "sha256"
    ],
)

print(
    "OFFICIAL_DIRECTORY_SHA_AFTER =",
    official_after_c6[
        "sha256"
    ],
)

print(
    "OFFICIAL_FILES_UNCHANGED =",
    official_files_unchanged,
)

print()

print(
    "MODEL_FORWARD_EXECUTED =",
    False,
)

print(
    "BACKWARD_EXECUTED =",
    False,
)

print(
    "OPTIMIZER_STEP_EXECUTED =",
    False,
)

print(
    "OFFICIAL_TRAINING_STARTED =",
    False,
)

print()

print(
    "NOTEBOOK_03_CONTROLLER_DRY_RUN_PASS =",
    controller_dry_run_pass,
)


# ============================================================
# RELEASE DRY-RUN OBJECTS
# ============================================================

del controller_model
del controller_optimizer

torch.cuda.empty_cache()

=== OFFICIAL CONTROLLER DRY RUN ===
CONTROLLER_MODE = TRAIN_NEXT_EPOCH

LAST_COMPLETED_EPOCH = -1
NEXT_EPOCH = 0
CURRENT_OPTIMIZER_STEP = 0

PLANNED_EPOCH = 0
PLANNED_START_STEP = 0
PLANNED_END_STEP = 54
PLANNED_MEAN_DIAGNOSTIC = False

REMAINING_EPOCHS = 100
REMAINING_OPTIMIZER_STEPS = 5400
REMAINING_STOCHASTIC_VALIDATIONS = 100
REMAINING_MEAN_DIAGNOSTICS = 20

BEST_DEFINED = False
BEST_EPOCH = -1

METRICS_NEED_RECONCILIATION = False
STALE_TEMP_FILES = []

OFFICIAL_DIRECTORY_SHA_BEFORE = 51375da7040a83dec362f35d4001979db5b094fcadebcd7e1149f4cdd49883e8
OFFICIAL_DIRECTORY_SHA_AFTER = 51375da7040a83dec362f35d4001979db5b094fcadebcd7e1149f4cdd49883e8
OFFICIAL_FILES_UNCHANGED = True

MODEL_FORWARD_EXECUTED = False
BACKWARD_EXECUTED = False
OPTIMIZER_STEP_EXECUTED = False
OFFICIAL_TRAINING_STARTED = False

NOTEBOOK_03_CONTROLLER_DRY_RUN_PASS = True


## STEP 03-C7 — Recovery delle finestre di pubblicazione LATEST/BEST

La pubblicazione di un'epoca segue l'ordine:

`metriche → LATEST → eventuale BEST`

Esistono quindi ulteriori finestre di interruzione che devono essere recuperabili.

### Caso 1 — `.tmp` di LATEST rimasto dopo un'interruzione

Il checkpoint pubblicato `LATEST` rimane l'unica autorità.  
Un eventuale `latest.pt.tmp` non pubblicato deve essere riconosciuto come artefatto incompleto e scartato durante il recovery.

### Caso 2 — LATEST pubblicato, BEST non ancora scritto

Se il nuovo epoch è anche il nuovo BEST, allora:

`best_epoch == last_completed_epoch`

e lo stato BEST è esattamente lo stato contenuto nel nuovo LATEST.

BEST può quindi essere rigenerato in modo deterministico da LATEST.

### Caso 3 — BEST incompleto o non valido

Se LATEST certifica che il BEST coincide con l'ultimo epoch completato, un BEST mancante/non valido può essere sostituito utilizzando LATEST.

Se invece il BEST appartiene a un epoch precedente e il relativo checkpoint è perso, il recovery deve fermarsi: non è lecito inventarlo.

Tutti i test di questo step vengono eseguiti esclusivamente sotto `LOCAL_RUNTIME_ROOT`.

La directory ufficiale sotto `PROJECT_ROOT` deve rimanere byte-identica.

In [22]:
# ============================================================
# NOTEBOOK 03 — CHECKPOINT PUBLICATION RECOVERY
# STEP 03-C7
#
# LOCAL tests only.
#
# Tests crash windows around:
#   LATEST publication
#   BEST publication
#
# OFFICIAL PERSISTENT DIRECTORY MUST REMAIN UNCHANGED.
#
# NO model forward.
# NO backward.
# NO optimizer step.
# ============================================================

from pathlib import Path

import shutil
import torch


# ============================================================
# LOCAL TEST ROOT
# ============================================================

C7_TEST_ROOT = (
    LOCAL_RUNTIME_ROOT
    / "sakai224_full_checkpoint_publication_test"
)


if C7_TEST_ROOT.exists():

    shutil.rmtree(
        C7_TEST_ROOT
    )


C7_TEST_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)


# ============================================================
# FREEZE OFFICIAL DIRECTORY BEFORE TEST
# ============================================================

official_before_c7 = (
    official_directory_fingerprint()
)


# ============================================================
# KNOWN VALID SOURCES
#
# PRETRAIN:
#   official LATEST from STEP C3
#
# EPOCH0:
#   disposable but fully validated epoch-0 LATEST from STEP C0
# ============================================================

C7_PRETRAIN_SOURCE = (
    OFFICIAL_LATEST_PATH
)

C7_EPOCH0_SOURCE = (
    CHECKPOINT_TEST_PATH
)


assert C7_PRETRAIN_SOURCE.is_file()
assert C7_EPOCH0_SOURCE.is_file()


pretrain_source_sha = (
    sha256_file_streaming(
        C7_PRETRAIN_SOURCE
    )
)

epoch0_source_sha = (
    sha256_file_streaming(
        C7_EPOCH0_SOURCE
    )
)


assert pretrain_source_sha == (
    "de9609a7065acf4e52a3a80312576706"
    "c045e4a250b029b2d0deb2bc109b1650"
)

assert epoch0_source_sha == (
    "b536387a0e8ac19922ee9f2ff410cfb5"
    "7aa5146290733131806ac890650540e3"
)


# ============================================================
# CHECKPOINT READ / VALIDATION
# ============================================================

def load_valid_checkpoint_cpu(
    path,
    *,
    expected_kind,
):

    payload = torch.load(
        path,
        map_location="cpu",
        weights_only=True,
    )


    validate_checkpoint_payload(
        payload,
        expected_kind=expected_kind,
    )


    return payload


# ============================================================
# DOES A BEST CHECKPOINT MATCH LATEST METADATA?
# ============================================================

def best_checkpoint_matches_latest(
    best_payload,
    latest_payload,
):

    try:

        validate_checkpoint_payload(
            best_payload,
            expected_kind="BEST",
        )

    except Exception:

        return False


    return (

        best_payload[
            "best_defined"
        ] is True

        and latest_payload[
            "best_defined"
        ] is True

        and best_payload[
            "run_id"
        ] == latest_payload[
            "run_id"
        ]

        and best_payload[
            "scientific_contract_sha256"
        ] == latest_payload[
            "scientific_contract_sha256"
        ]

        # A BEST checkpoint represents the actual epoch where
        # that best state was produced.
        and best_payload[
            "last_completed_epoch"
        ] == best_payload[
            "best_epoch"
        ]

        and best_payload[
            "best_epoch"
        ] == latest_payload[
            "best_epoch"
        ]

        and best_payload[
            "best_validation_total_mean_per_sample"
        ] == latest_payload[
            "best_validation_total_mean_per_sample"
        ]
    )


# ============================================================
# LOCAL RECOVERY FUNCTION
#
# Published LATEST is always authoritative.
#
# Stale .tmp files are explicitly classified as unpublished
# transaction remnants and removed only AFTER published LATEST
# has been successfully validated.
# ============================================================

def recover_checkpoint_publication_state(
    root,
):

    root = Path(root)

    latest_path = (
        root / "latest.pt"
    )

    best_path = (
        root / "best.pt"
    )

    latest_tmp = Path(
        str(latest_path)
        + ".tmp"
    )

    best_tmp = Path(
        str(best_path)
        + ".tmp"
    )


    if not latest_path.is_file():

        raise RuntimeError(
            "Published LATEST is missing"
        )


    # --------------------------------------------------------
    # Validate published authority FIRST.
    # --------------------------------------------------------

    latest_payload = (
        load_valid_checkpoint_cpu(
            latest_path,
            expected_kind="LATEST",
        )
    )


    latest_sha_before = (
        sha256_file_streaming(
            latest_path
        )
    )


    # --------------------------------------------------------
    # Classify/remove stale unpublished LATEST temp.
    # --------------------------------------------------------

    stale_latest_tmp_removed = False
    stale_latest_tmp_valid = None
    stale_latest_tmp_epoch = None


    if latest_tmp.exists():

        try:

            temporary_payload = (
                load_valid_checkpoint_cpu(
                    latest_tmp,
                    expected_kind="LATEST",
                )
            )

            stale_latest_tmp_valid = True

            stale_latest_tmp_epoch = int(
                temporary_payload[
                    "last_completed_epoch"
                ]
            )

        except Exception:

            stale_latest_tmp_valid = False


        latest_tmp.unlink()

        stale_latest_tmp_removed = True


    # --------------------------------------------------------
    # BEST temp is also unpublished.
    # --------------------------------------------------------

    stale_best_tmp_removed = False


    if best_tmp.exists():

        best_tmp.unlink()

        stale_best_tmp_removed = True


    # --------------------------------------------------------
    # No BEST should exist if LATEST says undefined.
    # --------------------------------------------------------

    if not bool(
        latest_payload[
            "best_defined"
        ]
    ):

        if best_path.exists():

            raise RuntimeError(
                "BEST exists although LATEST says BEST undefined"
            )


        best_status = (
            "NOT_DEFINED"
        )

        best_repaired = False

        best_gate = True


    else:

        # ----------------------------------------------------
        # Validate existing BEST, if any.
        # ----------------------------------------------------

        existing_best_valid = False


        if best_path.is_file():

            try:

                best_payload = (
                    load_valid_checkpoint_cpu(
                        best_path,
                        expected_kind="BEST",
                    )
                )


                existing_best_valid = (
                    best_checkpoint_matches_latest(
                        best_payload,
                        latest_payload,
                    )
                )

            except Exception:

                existing_best_valid = False


        if existing_best_valid:

            best_status = (
                "ALREADY_VALID"
            )

            best_repaired = False

            best_gate = True


        else:

            # ------------------------------------------------
            # Safe repair is possible ONLY when the current
            # LATEST state itself is the BEST state.
            # ------------------------------------------------

            repairable_from_latest = (

                int(
                    latest_payload[
                        "best_epoch"
                    ]
                )
                == int(
                    latest_payload[
                        "last_completed_epoch"
                    ]
                )
            )


            if not repairable_from_latest:

                raise RuntimeError(
                    "BEST is missing/invalid and cannot be "
                    "reconstructed from current LATEST"
                )


            repaired_best_payload = dict(
                latest_payload
            )

            repaired_best_payload[
                "kind"
            ] = "BEST"


            validate_checkpoint_payload(
                repaired_best_payload,
                expected_kind="BEST",
            )


            save_checkpoint_atomic(
                best_path,
                repaired_best_payload,
            )


            repaired_best_loaded = (
                load_valid_checkpoint_cpu(
                    best_path,
                    expected_kind="BEST",
                )
            )


            best_gate = (
                best_checkpoint_matches_latest(
                    repaired_best_loaded,
                    latest_payload,
                )
            )


            if not best_gate:

                raise RuntimeError(
                    "Repaired BEST failed verification"
                )


            best_status = (
                "REPAIRED_FROM_LATEST"
            )

            best_repaired = True


    # --------------------------------------------------------
    # LATEST itself must remain bitwise unchanged.
    # --------------------------------------------------------

    latest_sha_after = (
        sha256_file_streaming(
            latest_path
        )
    )


    latest_unchanged = (
        latest_sha_before
        == latest_sha_after
    )


    if not latest_unchanged:

        raise RuntimeError(
            "Recovery unexpectedly changed published LATEST"
        )


    return {

        "latest_epoch":
            int(
                latest_payload[
                    "last_completed_epoch"
                ]
            ),

        "latest_best_defined":
            bool(
                latest_payload[
                    "best_defined"
                ]
            ),

        "latest_best_epoch":
            int(
                latest_payload[
                    "best_epoch"
                ]
            ),

        "stale_latest_tmp_removed":
            stale_latest_tmp_removed,

        "stale_latest_tmp_valid":
            stale_latest_tmp_valid,

        "stale_latest_tmp_epoch":
            stale_latest_tmp_epoch,

        "stale_best_tmp_removed":
            stale_best_tmp_removed,

        "best_status":
            best_status,

        "best_repaired":
            best_repaired,

        "best_gate":
            best_gate,

        "latest_unchanged":
            latest_unchanged,
    }


# ============================================================
# SCENARIO 1
#
# Published LATEST = pretrain epoch -1.
#
# A future epoch-0 latest.pt.tmp exists, simulating a crash
# before atomic publication.
#
# Also create an unrelated best.pt.tmp.
#
# Published pretrain LATEST must remain authoritative.
# ============================================================

scenario1_root = (
    C7_TEST_ROOT
    / "scenario1_stale_latest_tmp"
)

scenario1_root.mkdir()


shutil.copy2(
    C7_PRETRAIN_SOURCE,
    scenario1_root / "latest.pt",
)

shutil.copy2(
    C7_EPOCH0_SOURCE,
    scenario1_root / "latest.pt.tmp",
)

shutil.copy2(
    C7_PRETRAIN_SOURCE,
    scenario1_root / "best.pt.tmp",
)


scenario1_latest_sha_before = (
    sha256_file_streaming(
        scenario1_root
        / "latest.pt"
    )
)


scenario1_result = (
    recover_checkpoint_publication_state(
        scenario1_root
    )
)


scenario1_latest_sha_after = (
    sha256_file_streaming(
        scenario1_root
        / "latest.pt"
    )
)


scenario1_pass = (

    scenario1_result[
        "latest_epoch"
    ] == -1

    and scenario1_result[
        "latest_best_defined"
    ] is False

    and scenario1_result[
        "stale_latest_tmp_removed"
    ] is True

    and scenario1_result[
        "stale_latest_tmp_valid"
    ] is True

    and scenario1_result[
        "stale_latest_tmp_epoch"
    ] == 0

    and scenario1_result[
        "stale_best_tmp_removed"
    ] is True

    and scenario1_result[
        "best_status"
    ] == "NOT_DEFINED"

    and not (
        scenario1_root
        / "best.pt"
    ).exists()

    and not (
        scenario1_root
        / "latest.pt.tmp"
    ).exists()

    and not (
        scenario1_root
        / "best.pt.tmp"
    ).exists()

    and scenario1_latest_sha_before
        == scenario1_latest_sha_after
)


assert scenario1_pass


# ============================================================
# SCENARIO 2
#
# LATEST epoch 0 is already published and says:
#
#   best_defined = True
#   best_epoch   = 0
#
# But BEST does not yet exist.
#
# This is exactly the crash window:
#
#   metrics -> LATEST -> CRASH -> BEST
#
# BEST must be recreated from LATEST.
# ============================================================

scenario2_root = (
    C7_TEST_ROOT
    / "scenario2_missing_best"
)

scenario2_root.mkdir()


shutil.copy2(
    C7_EPOCH0_SOURCE,
    scenario2_root / "latest.pt",
)


scenario2_result = (
    recover_checkpoint_publication_state(
        scenario2_root
    )
)


scenario2_latest = (
    load_valid_checkpoint_cpu(
        scenario2_root
        / "latest.pt",

        expected_kind="LATEST",
    )
)


scenario2_best = (
    load_valid_checkpoint_cpu(
        scenario2_root
        / "best.pt",

        expected_kind="BEST",
    )
)


scenario2_pass = (

    scenario2_result[
        "latest_epoch"
    ] == 0

    and scenario2_result[
        "latest_best_defined"
    ] is True

    and scenario2_result[
        "latest_best_epoch"
    ] == 0

    and scenario2_result[
        "best_status"
    ] == "REPAIRED_FROM_LATEST"

    and scenario2_result[
        "best_repaired"
    ] is True

    and scenario2_result[
        "best_gate"
    ] is True

    and scenario2_best[
        "model_state_sha256"
    ] == scenario2_latest[
        "model_state_sha256"
    ]

    and scenario2_best[
        "optimizer_state_sha256"
    ] == scenario2_latest[
        "optimizer_state_sha256"
    ]
)


assert scenario2_pass


# ============================================================
# SCENARIO 3
#
# LATEST epoch 0 is valid.
#
# BEST exists but is invalid for this transaction because it
# is actually a pretrain LATEST file, plus a stale BEST temp.
#
# Both must be handled and BEST must be republished from
# current LATEST.
# ============================================================

scenario3_root = (
    C7_TEST_ROOT
    / "scenario3_invalid_best"
)

scenario3_root.mkdir()


shutil.copy2(
    C7_EPOCH0_SOURCE,
    scenario3_root / "latest.pt",
)

# Deliberately wrong published BEST.
shutil.copy2(
    C7_PRETRAIN_SOURCE,
    scenario3_root / "best.pt",
)

# Deliberately stale unpublished temp.
shutil.copy2(
    C7_PRETRAIN_SOURCE,
    scenario3_root / "best.pt.tmp",
)


scenario3_result = (
    recover_checkpoint_publication_state(
        scenario3_root
    )
)


scenario3_latest = (
    load_valid_checkpoint_cpu(
        scenario3_root
        / "latest.pt",

        expected_kind="LATEST",
    )
)


scenario3_best = (
    load_valid_checkpoint_cpu(
        scenario3_root
        / "best.pt",

        expected_kind="BEST",
    )
)


scenario3_pass = (

    scenario3_result[
        "latest_epoch"
    ] == 0

    and scenario3_result[
        "stale_best_tmp_removed"
    ] is True

    and scenario3_result[
        "best_status"
    ] == "REPAIRED_FROM_LATEST"

    and scenario3_result[
        "best_repaired"
    ] is True

    and scenario3_result[
        "best_gate"
    ] is True

    and scenario3_best[
        "model_state_sha256"
    ] == scenario3_latest[
        "model_state_sha256"
    ]

    and scenario3_best[
        "optimizer_state_sha256"
    ] == scenario3_latest[
        "optimizer_state_sha256"
    ]

    and not (
        scenario3_root
        / "best.pt.tmp"
    ).exists()
)


assert scenario3_pass


# ============================================================
# OFFICIAL PERSISTENT DIRECTORY MUST REMAIN BITWISE IDENTICAL
# ============================================================

official_after_c7 = (
    official_directory_fingerprint()
)


official_unchanged_c7 = (

    official_before_c7[
        "sha256"
    ]
    == official_after_c7[
        "sha256"
    ]

    and official_before_c7[
        "entries"
    ]
    == official_after_c7[
        "entries"
    ]
)


assert official_unchanged_c7


# ============================================================
# REPORT
# ============================================================

print(
    "=== LOCAL CHECKPOINT PUBLICATION RECOVERY TEST ==="
)

print()

print(
    "SCENARIO_1_STALE_LATEST_TMP_PASS =",
    scenario1_pass,
)

print(
    "SCENARIO_1_TMP_EPOCH =",
    scenario1_result[
        "stale_latest_tmp_epoch"
    ],
)

print(
    "SCENARIO_1_PUBLISHED_LATEST_EPOCH =",
    scenario1_result[
        "latest_epoch"
    ],
)

print()

print(
    "SCENARIO_2_MISSING_BEST_PASS =",
    scenario2_pass,
)

print(
    "SCENARIO_2_BEST_STATUS =",
    scenario2_result[
        "best_status"
    ],
)

print()

print(
    "SCENARIO_3_INVALID_BEST_PASS =",
    scenario3_pass,
)

print(
    "SCENARIO_3_STALE_BEST_TMP_REMOVED =",
    scenario3_result[
        "stale_best_tmp_removed"
    ],
)

print(
    "SCENARIO_3_BEST_STATUS =",
    scenario3_result[
        "best_status"
    ],
)

print()

print(
    "OFFICIAL_DIRECTORY_SHA_BEFORE =",
    official_before_c7[
        "sha256"
    ],
)

print(
    "OFFICIAL_DIRECTORY_SHA_AFTER =",
    official_after_c7[
        "sha256"
    ],
)

print(
    "OFFICIAL_DIRECTORY_UNCHANGED =",
    official_unchanged_c7,
)

print()

print(
    "OFFICIAL_FILES_MODIFIED =",
    False,
)

print(
    "MODEL_FORWARD_EXECUTED =",
    False,
)

print(
    "BACKWARD_EXECUTED =",
    False,
)

print(
    "OPTIMIZER_STEP_EXECUTED =",
    False,
)

print(
    "OFFICIAL_TRAINING_STARTED =",
    False,
)

print()

print(
    "NOTEBOOK_03_CHECKPOINT_PUBLICATION_RECOVERY_PASS =",
    (
        scenario1_pass
        and scenario2_pass
        and scenario3_pass
        and official_unchanged_c7
    ),
)


# ============================================================
# REMOVE LOCAL TEST ARTIFACTS
# ============================================================

shutil.rmtree(
    C7_TEST_ROOT
)


print(
    "LOCAL_TEST_DIRECTORY_REMOVED =",
    not C7_TEST_ROOT.exists(),
)

=== LOCAL CHECKPOINT PUBLICATION RECOVERY TEST ===

SCENARIO_1_STALE_LATEST_TMP_PASS = True
SCENARIO_1_TMP_EPOCH = 0
SCENARIO_1_PUBLISHED_LATEST_EPOCH = -1

SCENARIO_2_MISSING_BEST_PASS = True
SCENARIO_2_BEST_STATUS = REPAIRED_FROM_LATEST

SCENARIO_3_INVALID_BEST_PASS = True
SCENARIO_3_STALE_BEST_TMP_REMOVED = True
SCENARIO_3_BEST_STATUS = REPAIRED_FROM_LATEST

OFFICIAL_DIRECTORY_SHA_BEFORE = 51375da7040a83dec362f35d4001979db5b094fcadebcd7e1149f4cdd49883e8
OFFICIAL_DIRECTORY_SHA_AFTER = 51375da7040a83dec362f35d4001979db5b094fcadebcd7e1149f4cdd49883e8
OFFICIAL_DIRECTORY_UNCHANGED = True

OFFICIAL_FILES_MODIFIED = False
MODEL_FORWARD_EXECUTED = False
BACKWARD_EXECUTED = False
OPTIMIZER_STEP_EXECUTED = False
OFFICIAL_TRAINING_STARTED = False

NOTEBOOK_03_CHECKPOINT_PUBLICATION_RECOVERY_PASS = True
LOCAL_TEST_DIRECTORY_REMOVED = True


## STEP 03-C8 — Test locale della transazione completa di una epoch

Prima di autorizzare epoch 0 della run ufficiale, viene testata integralmente la stessa transazione che verrà usata dal controller reale.

La directory ufficiale pre-training viene copiata sotto `LOCAL_RUNTIME_ROOT`.

Sulla copia locale viene eseguita la transizione:

`epoch -1 → epoch 0`

con:

- full train 6.866 eventi;
- shuffle PROJECT epoch 0;
- augmentation stateless epoch 0;
- epsilon latente stateless;
- batch 128;
- 54 optimizer step;
- validation stocastica completa;
- nessuna diagnostica MEAN, perché epoch 0 non completa un multiplo di 5 epoche;
- append delle metriche;
- pubblicazione atomica di `LATEST`;
- creazione del primo `BEST`.

Poiché epoch 0 è già stato eseguito nel precedente test indipendente C0, il nuovo stato può inoltre essere confrontato con un known-answer reale.

La directory ufficiale sotto `PROJECT_ROOT` deve rimanere byte-identica.

**Questo step è ancora un test e non avvia il training ufficiale.**

In [23]:
# ============================================================
# NOTEBOOK 03 — LOCAL ONE-EPOCH TRANSACTION TEST
# STEP 03-C8
#
# Executes a COMPLETE epoch transaction on a LOCAL COPY:
#
#   pretrain LATEST
#   -> train epoch 0
#   -> stochastic validation
#   -> metrics
#   -> LATEST
#   -> BEST
#
# OFFICIAL PERSISTENT DIRECTORY MUST REMAIN UNCHANGED.
#
# THIS IS NOT OFFICIAL TRAINING.
# ============================================================

from pathlib import Path

import csv
import gc
import shutil
import time

import torch
from torch.utils.data import DataLoader


# ============================================================
# LOCAL TEST ROOT
# ============================================================

C8_TEST_ROOT = (
    LOCAL_RUNTIME_ROOT
    / "sakai224_full_epoch_transaction_test"
)


if C8_TEST_ROOT.exists():

    shutil.rmtree(
        C8_TEST_ROOT
    )


C8_TEST_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)


# ============================================================
# FREEZE OFFICIAL DIRECTORY
# ============================================================

official_before_c8 = (
    official_directory_fingerprint()
)


# ============================================================
# COPY CURRENT OFFICIAL PRE-TRAIN STATE LOCALLY
# ============================================================

for source_path in (
    OFFICIAL_PARTIAL_DIR.iterdir()
):

    if not source_path.is_file():

        raise RuntimeError(
            f"Unexpected official entry: {source_path}"
        )


    shutil.copy2(
        source_path,
        C8_TEST_ROOT
        / source_path.name,
    )


C8_RUN_CONFIG = (
    C8_TEST_ROOT
    / "run_config.json"
)

C8_METRICS = (
    C8_TEST_ROOT
    / "metrics.csv"
)

C8_LATEST = (
    C8_TEST_ROOT
    / "latest.pt"
)

C8_BEST = (
    C8_TEST_ROOT
    / "best.pt"
)

C8_FINAL = (
    C8_TEST_ROOT
    / "final.pt"
)


assert C8_RUN_CONFIG.is_file()
assert C8_METRICS.is_file()
assert C8_LATEST.is_file()

assert not C8_BEST.exists()
assert not C8_FINAL.exists()


# ============================================================
# LOCAL STATE MUST MATCH OFFICIAL PRETRAIN STATE
# ============================================================

local_pretrain_sha = (
    sha256_file_streaming(
        C8_LATEST
    )
)


assert (
    local_pretrain_sha
    == OFFICIAL_PRETRAIN_LATEST_FILE_SHA256
)


# ============================================================
# KNOWN ANSWER FROM THE INDEPENDENT C0 EPOCH-0 TEST
# ============================================================

known_epoch0_payload = (
    load_valid_checkpoint_cpu(
        CHECKPOINT_TEST_PATH,
        expected_kind="LATEST",
    )
)


KNOWN_EPOCH0_MODEL_SHA256 = (
    known_epoch0_payload[
        "model_state_sha256"
    ]
)

KNOWN_EPOCH0_OPTIMIZER_SHA256 = (
    known_epoch0_payload[
        "optimizer_state_sha256"
    ]
)

KNOWN_EPOCH0_VALIDATION_TOTAL_MEAN = (
    known_epoch0_payload[
        "best_validation_total_mean_per_sample"
    ]
)


assert (
    known_epoch0_payload[
        "last_completed_epoch"
    ] == 0
)

assert (
    known_epoch0_payload[
        "optimizer_step"
    ] == 54
)

assert (
    known_epoch0_payload[
        "best_epoch"
    ] == 0
)


# ============================================================
# FRESH MODEL / OPTIMIZER
#
# We reconstruct them exactly as after a runtime restart.
# ============================================================

c8_model = Sakai224FullVAE(
    SPEC
)

initialize_project_c_exact_v1(
    c8_model,
    FROZEN_INITIALIZATION_SEED,
)


c8_model = c8_model.to(
    device=device,
    dtype=torch.float32,
)


c8_optimizer = torch.optim.Adam(
    c8_model.parameters(),

    lr=FROZEN_LEARNING_RATE,

    betas=(
        FROZEN_ADAM_BETA1,
        FROZEN_ADAM_BETA2,
    ),

    eps=FROZEN_ADAM_EPSILON,

    weight_decay=FROZEN_WEIGHT_DECAY,

    amsgrad=FROZEN_AMSGRAD,

    foreach=False,

    fused=False,
)


# ============================================================
# LOAD LOCAL PRETRAIN LATEST
# ============================================================

c8_pretrain_payload = (
    load_checkpoint_into(
        C8_LATEST,

        c8_model,

        c8_optimizer,

        expected_kind="LATEST",

        device=device,
    )
)


assert (
    c8_pretrain_payload[
        "last_completed_epoch"
    ] == -1
)

assert (
    get_adam_global_step(
        c8_optimizer
    ) == 0
)


# ============================================================
# METRICS MUST BE ALIGNED BEFORE EXECUTION
# ============================================================

c8_pre_metrics_audit = (
    inspect_metrics_against_latest(
        C8_METRICS.read_bytes(),
        c8_pretrain_payload,
    )
)


assert (
    c8_pre_metrics_audit[
        "needs_reconciliation"
    ] is False
)

assert (
    c8_pre_metrics_audit[
        "committed_metric_rows"
    ] == 2
)


# ============================================================
# TRAIN DATASET / LOADER FOR THE TRANSACTION
# ============================================================

c8_train_dataset = (
    GravitySpy224TrainWithEpsilon(
        train_dataset,
        epoch=0,
    )
)


def make_c8_train_loader(
    epoch,
):

    epoch = int(epoch)


    train_sampler.set_epoch(
        epoch
    )

    c8_train_dataset.set_epoch(
        epoch
    )


    return DataLoader(
        c8_train_dataset,

        batch_size=FROZEN_BATCH_SIZE,

        sampler=train_sampler,

        num_workers=FROZEN_NUM_WORKERS,

        pin_memory=FROZEN_PIN_MEMORY,

        drop_last=False,

        persistent_workers=False,

        prefetch_factor=2,
    )


# ============================================================
# ONE REAL TRAIN EPOCH WITH FULL METRIC AGGREGATION
#
# The train metric records the objective evaluated on each
# batch BEFORE the corresponding Adam update.
# ============================================================

def run_transaction_train_epoch(
    model,
    optimizer,
    epoch,
):

    model.train()


    loader = (
        make_c8_train_loader(
            epoch
        )
    )


    reconstruction_sum = 0.0
    kl_sum = 0.0

    sample_count = 0
    batch_count = 0
    tail_size = None

    first_batch_gate = False


    torch.cuda.synchronize()

    start_time = (
        time.perf_counter()
    )


    for batch_index, batch in enumerate(
        loader
    ):

        if batch_index == 0:

            first_manifests = (
                batch[
                    "manifest_index"
                ][:4].tolist()
            )

            first_crops = (
                batch[
                    "crop_left"
                ][:4].tolist()
            )


            first_batch_gate = (

                first_manifests
                == [
                    2676,
                    6250,
                    2396,
                    5602,
                ]

                and first_crops
                == [
                    20,
                    15,
                    19,
                    12,
                ]
            )


            if not first_batch_gate:

                raise RuntimeError(
                    "C8 epoch-0 first-batch mismatch"
                )


        x_batch = (
            batch["x"]
            .to(
                device=device,
                dtype=torch.float32,
                non_blocking=True,
            )
        )


        epsilon_batch = (
            batch["epsilon"]
            .to(
                device=device,
                dtype=torch.float32,
                non_blocking=True,
            )
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        (
            logits,
            mu,
            logvar,
            z,
        ) = model(
            x_batch,
            epsilon=epsilon_batch,
            mean_mode=False,
        )


        stats = sakai_vae_loss(
            logits,
            x_batch,
            mu,
            logvar,
            beta_kl=SPEC.beta_kl,
        )


        loss = stats[
            "loss"
        ]


        if not torch.isfinite(
            loss
        ).item():

            raise RuntimeError(
                "Non-finite C8 train loss"
            )


        # Metric is captured before optimizer update.
        reconstruction_sum += float(
            stats[
                "reconstruction_sum"
            ].detach()
        )

        kl_sum += float(
            stats[
                "kl_sum"
            ].detach()
        )


        loss.backward()

        optimizer.step()


        current_batch = int(
            x_batch.shape[0]
        )


        sample_count += (
            current_batch
        )

        batch_count += 1

        tail_size = (
            current_batch
        )


    torch.cuda.synchronize()


    wall_seconds = (
        time.perf_counter()
        - start_time
    )


    assert (
        sample_count
        == FROZEN_TRAIN_SAMPLES
    )

    assert (
        batch_count
        == FROZEN_TRAIN_BATCHES
    )

    assert (
        tail_size
        == FROZEN_TRAIN_TAIL
    )

    assert first_batch_gate


    expected_step = (
        (epoch + 1)
        * FROZEN_TRAIN_BATCHES
    )


    assert (
        get_adam_global_step(
            optimizer
        )
        == expected_step
    )


    assert all(
        int(
            bn.num_batches_tracked.item()
        )
        == expected_step

        for bn
        in get_project_batchnorm_layers(
            model
        )
    )


    total_sum = (
        reconstruction_sum
        + kl_sum
    )


    input_value_count = (
        sample_count
        * 4
        * 224
        * 224
    )


    latent_value_count = (
        sample_count
        * SPEC.latent_dim
    )


    return {

        "sample_count":
            sample_count,

        "batch_count":
            batch_count,

        "total_sum":
            total_sum,

        "total_mean_per_sample":
            total_sum
            / sample_count,

        "reconstruction_sum":
            reconstruction_sum,

        "reconstruction_mean_per_sample":
            reconstruction_sum
            / sample_count,

        "reconstruction_mean_per_input_value":
            reconstruction_sum
            / input_value_count,

        "kl_sum":
            kl_sum,

        "kl_mean_per_sample":
            kl_sum
            / sample_count,

        "kl_mean_per_latent_value":
            kl_sum
            / latent_value_count,

        "wall_seconds":
            wall_seconds,
    }


# ============================================================
# GENERIC LOCAL METRIC APPEND
# ============================================================

def append_metric_row_to_path(
    path,
    row,
):

    with Path(path).open(
        "a",
        newline="",
        encoding="utf-8",
    ) as handle:

        writer = csv.DictWriter(
            handle,
            fieldnames=OFFICIAL_METRICS_COLUMNS,
        )

        writer.writerow(
            row
        )

        handle.flush()

        os.fsync(
            handle.fileno()
        )


# ============================================================
# EXECUTE LOCAL EPOCH 0
# ============================================================

C8_EPOCH = 0


print(
    "=== LOCAL ONE-EPOCH TRANSACTION TEST ==="
)

print(
    "Running local disposable epoch 0..."
)


c8_train_aggregate = (
    run_transaction_train_epoch(
        c8_model,
        c8_optimizer,
        C8_EPOCH,
    )
)


# ============================================================
# STOCHASTIC VALIDATION
# ============================================================

c8_validation_aggregate = (
    run_official_validation_aggregate(
        c8_model,
        mode="stochastic",
    )
)


c8_validation_mean = (
    c8_validation_aggregate[
        "total_mean_per_sample"
    ]
)


# Epoch 0 cannot require periodic MEAN.
c8_mean_required = (
    (
        C8_EPOCH + 1
    )
    % MEAN_DIAGNOSTIC_EVERY_EPOCHS
    == 0
)


assert c8_mean_required is False


# ============================================================
# FIRST POST-EPOCH VALIDATION IS AUTOMATICALLY THE FIRST BEST
# ============================================================

c8_improved = True

c8_best_defined = True
c8_best_epoch = 0

c8_best_validation = (
    c8_validation_mean
)


# ============================================================
# BUILD METRIC ROWS
# ============================================================

c8_train_row = (
    make_official_metric_row(
        "train_stochastic",

        epoch=0,

        completed_epochs=1,

        optimizer_step=54,

        aggregate=c8_train_aggregate,
    )
)


c8_validation_row = (
    make_official_metric_row(
        "validation_stochastic_post_epoch",

        epoch=0,

        completed_epochs=1,

        optimizer_step=54,

        aggregate=c8_validation_aggregate,
    )
)


# ============================================================
# TRANSACTION ORDER
#
#   metrics
#   -> LATEST
#   -> BEST
# ============================================================

append_metric_row_to_path(
    C8_METRICS,
    c8_train_row,
)

append_metric_row_to_path(
    C8_METRICS,
    c8_validation_row,
)


# ------------------------------------------------------------
# Publish LATEST
# ------------------------------------------------------------

c8_latest_payload = (
    build_checkpoint_payload(
        c8_model,
        c8_optimizer,

        kind="LATEST",

        last_completed_epoch=0,

        best_defined=True,

        best_epoch=0,

        best_validation_total_mean_per_sample=(
            c8_best_validation
        ),
    )
)


save_checkpoint_atomic(
    C8_LATEST,
    c8_latest_payload,
)


# ------------------------------------------------------------
# Publish BEST
# ------------------------------------------------------------

c8_best_payload = (
    build_checkpoint_payload(
        c8_model,
        c8_optimizer,

        kind="BEST",

        last_completed_epoch=0,

        best_defined=True,

        best_epoch=0,

        best_validation_total_mean_per_sample=(
            c8_best_validation
        ),
    )
)


save_checkpoint_atomic(
    C8_BEST,
    c8_best_payload,
)


# ============================================================
# INDEPENDENT READ-BACK
# ============================================================

c8_latest_disk = (
    load_valid_checkpoint_cpu(
        C8_LATEST,
        expected_kind="LATEST",
    )
)


c8_best_disk = (
    load_valid_checkpoint_cpu(
        C8_BEST,
        expected_kind="BEST",
    )
)


# ============================================================
# CHECKPOINT CONTROLLER GATES
# ============================================================

c8_checkpoint_gate = (

    c8_latest_disk[
        "last_completed_epoch"
    ] == 0

    and c8_latest_disk[
        "optimizer_step"
    ] == 54

    and c8_latest_disk[
        "best_defined"
    ] is True

    and c8_latest_disk[
        "best_epoch"
    ] == 0

    and c8_latest_disk[
        "best_validation_total_mean_per_sample"
    ] == c8_validation_mean

    and c8_best_disk[
        "last_completed_epoch"
    ] == 0

    and c8_best_disk[
        "best_epoch"
    ] == 0

    and c8_best_disk[
        "model_state_sha256"
    ] == c8_latest_disk[
        "model_state_sha256"
    ]

    and c8_best_disk[
        "optimizer_state_sha256"
    ] == c8_latest_disk[
        "optimizer_state_sha256"
    ]
)


assert c8_checkpoint_gate


# ============================================================
# METRICS ↔ LATEST AUDIT
# ============================================================

c8_metrics_audit = (
    inspect_metrics_against_latest(
        C8_METRICS.read_bytes(),
        c8_latest_disk,
    )
)


c8_metrics_gate = (

    c8_metrics_audit[
        "needs_reconciliation"
    ] is False

    and c8_metrics_audit[
        "future_full_rows"
    ] == 0

    and c8_metrics_audit[
        "partial_tail_dropped"
    ] is False

    and c8_metrics_audit[
        "committed_metric_rows"
    ] == 4

    and c8_metrics_audit[
        "best_defined"
    ] is True

    and c8_metrics_audit[
        "best_epoch"
    ] == 0

    and c8_metrics_audit[
        "best_validation"
    ] == c8_validation_mean
)


assert c8_metrics_gate


# ============================================================
# STRONG KNOWN-ANSWER COMPARISON WITH C0
#
# C0 was an independent real epoch-0 execution.
# Same trajectory in the same frozen numerical environment
# must produce identical persistent state.
# ============================================================

c8_model_known_answer_exact = (

    c8_latest_disk[
        "model_state_sha256"
    ]
    == KNOWN_EPOCH0_MODEL_SHA256
)


c8_optimizer_known_answer_exact = (

    c8_latest_disk[
        "optimizer_state_sha256"
    ]
    == KNOWN_EPOCH0_OPTIMIZER_SHA256
)


c8_validation_known_answer_exact = (

    c8_validation_mean
    == KNOWN_EPOCH0_VALIDATION_TOTAL_MEAN
)


assert c8_model_known_answer_exact
assert c8_optimizer_known_answer_exact
assert c8_validation_known_answer_exact


# ============================================================
# DIRECTORY STATE
# ============================================================

c8_names = sorted(
    path.name

    for path
    in C8_TEST_ROOT.iterdir()
)


expected_c8_names = sorted([
    "best.pt",
    "latest.pt",
    "metrics.csv",
    "run_config.json",
    "runtime_sessions.jsonl",
])


c8_directory_gate = (
    c8_names
    == expected_c8_names
)


assert c8_directory_gate
assert not C8_FINAL.exists()


# ============================================================
# OFFICIAL PERSISTENT DIRECTORY MUST REMAIN BITWISE IDENTICAL
# ============================================================

official_after_c8 = (
    official_directory_fingerprint()
)


official_unchanged_c8 = (

    official_before_c8[
        "sha256"
    ]
    == official_after_c8[
        "sha256"
    ]

    and official_before_c8[
        "entries"
    ]
    == official_after_c8[
        "entries"
    ]
)


assert official_unchanged_c8


# ============================================================
# REPORT
# ============================================================

print()
print(
    "=== LOCAL EPOCH-0 TRANSACTION AUDIT ==="
)

print(
    "TRAIN_TOTAL_MEAN =",
    c8_train_aggregate[
        "total_mean_per_sample"
    ],
)

print(
    "VALIDATION_TOTAL_MEAN =",
    c8_validation_mean,
)

print()

print(
    "LAST_COMPLETED_EPOCH =",
    c8_latest_disk[
        "last_completed_epoch"
    ],
)

print(
    "NEXT_EPOCH =",
    c8_latest_disk[
        "last_completed_epoch"
    ] + 1,
)

print(
    "OPTIMIZER_STEP =",
    c8_latest_disk[
        "optimizer_step"
    ],
)

print(
    "BEST_DEFINED =",
    c8_latest_disk[
        "best_defined"
    ],
)

print(
    "BEST_EPOCH =",
    c8_latest_disk[
        "best_epoch"
    ],
)

print()

print(
    "MODEL_KNOWN_ANSWER_EXACT =",
    c8_model_known_answer_exact,
)

print(
    "OPTIMIZER_KNOWN_ANSWER_EXACT =",
    c8_optimizer_known_answer_exact,
)

print(
    "VALIDATION_KNOWN_ANSWER_EXACT =",
    c8_validation_known_answer_exact,
)

print()

print(
    "METRICS_COMMITTED_ROWS =",
    c8_metrics_audit[
        "committed_metric_rows"
    ],
)

print(
    "METRICS_NEED_RECONCILIATION =",
    c8_metrics_audit[
        "needs_reconciliation"
    ],
)

print(
    "CHECKPOINT_GATE =",
    c8_checkpoint_gate,
)

print(
    "METRICS_GATE =",
    c8_metrics_gate,
)

print(
    "DIRECTORY_GATE =",
    c8_directory_gate,
)

print()

print(
    "LOCAL_LATEST_BYTES =",
    C8_LATEST.stat().st_size,
)

print(
    "LOCAL_BEST_BYTES =",
    C8_BEST.stat().st_size,
)

print()

print(
    "OFFICIAL_DIRECTORY_SHA_BEFORE =",
    official_before_c8[
        "sha256"
    ],
)

print(
    "OFFICIAL_DIRECTORY_SHA_AFTER =",
    official_after_c8[
        "sha256"
    ],
)

print(
    "OFFICIAL_DIRECTORY_UNCHANGED =",
    official_unchanged_c8,
)

print()

print(
    "OFFICIAL_FILES_MODIFIED =",
    False,
)

print(
    "OFFICIAL_TRAINING_STARTED =",
    False,
)

print()

print(
    "NOTEBOOK_03_LOCAL_EPOCH_TRANSACTION_PASS =",
    (
        c8_checkpoint_gate
        and c8_metrics_gate
        and c8_directory_gate
        and c8_model_known_answer_exact
        and c8_optimizer_known_answer_exact
        and c8_validation_known_answer_exact
        and official_unchanged_c8
    ),
)


# ============================================================
# RELEASE MODEL / OPTIMIZER AND REMOVE LOCAL RUN COPY
# ============================================================

del c8_model
del c8_optimizer

gc.collect()
torch.cuda.empty_cache()


shutil.rmtree(
    C8_TEST_ROOT
)


print(
    "LOCAL_TEST_DIRECTORY_REMOVED =",
    not C8_TEST_ROOT.exists(),
)

### STEP 03-C8B — Correzione del known-answer della validation

Il test C8 ha già verificato l'identità esatta dello stato del modello e dell'optimizer rispetto all'esecuzione indipendente C0.

Il precedente confronto diretto della metrica di validation era però impropriamente bitwise perché C0 e C8 utilizzavano due ordini differenti di aggregazione floating-point.

Per il controllo corretto viene quindi:

1. ricostruito un modello indipendente dal checkpoint C0;
2. eseguita nuovamente la validation con **la stessa funzione di aggregazione ufficiale usata da C8**;
3. confrontato esattamente il risultato con C8.

Il vecchio valore C0 viene mantenuto esclusivamente come controllo numerico di prossimità.

Questo è ancora un test locale. La directory ufficiale non viene modificata.

In [24]:
# ============================================================
# NOTEBOOK 03 — C8 VALIDATION KNOWN-ANSWER REPAIR
# STEP 03-C8B
#
# DO NOT rerun the whole C8 epoch.
#
# Reuses the already completed LOCAL epoch-0 state.
#
# NO official training.
# NO official file modification.
# ============================================================

import gc
import shutil
import torch


# ============================================================
# PRECONDITIONS FROM THE INTERRUPTED C8 TEST
# ============================================================

assert C8_TEST_ROOT.is_dir()

assert C8_LATEST.is_file()
assert C8_BEST.is_file()

assert c8_model_known_answer_exact is True
assert c8_optimizer_known_answer_exact is True

assert (
    c8_latest_disk[
        "last_completed_epoch"
    ] == 0
)

assert (
    c8_latest_disk[
        "optimizer_step"
    ] == 54
)


# ============================================================
# EXPLAIN THE PREVIOUS FAILURE NUMERICALLY
# ============================================================

legacy_c0_validation_mean = (
    KNOWN_EPOCH0_VALIDATION_TOTAL_MEAN
)

official_c8_validation_mean = (
    c8_validation_mean
)

legacy_aggregation_abs_delta = abs(
    official_c8_validation_mean
    - legacy_c0_validation_mean
)


print(
    "=== C8 VALIDATION AGGREGATION DIAGNOSTIC ==="
)

print(
    "C0_LEGACY_AGGREGATION =",
    legacy_c0_validation_mean,
)

print(
    "C8_OFFICIAL_AGGREGATION =",
    official_c8_validation_mean,
)

print(
    "ABS_DELTA =",
    legacy_aggregation_abs_delta,
)


# ============================================================
# REBUILD AN INDEPENDENT C0 REFERENCE MODEL
#
# Same checkpoint/model state, but evaluate it using the SAME
# official aggregation routine used by C8.
# ============================================================

c8_reference_model = (
    Sakai224FullVAE(
        SPEC
    )
)


c8_reference_model.load_state_dict(
    known_epoch0_payload[
        "model_state_dict"
    ],
    strict=True,
)


reference_state_hash_cpu = (
    canonical_tree_sha256(
        c8_reference_model.state_dict()
    )
)


assert (
    reference_state_hash_cpu
    == KNOWN_EPOCH0_MODEL_SHA256
)


c8_reference_model = (
    c8_reference_model.to(
        device=device,
        dtype=torch.float32,
    )
)


# ============================================================
# SAME OFFICIAL VALIDATION AGGREGATION
# ============================================================

print()
print(
    "Running independent C0 checkpoint validation "
    "with the C8 official aggregator..."
)


c8_reference_validation_aggregate = (
    run_official_validation_aggregate(
        c8_reference_model,
        mode="stochastic",
    )
)


c8_reference_validation_mean = (
    c8_reference_validation_aggregate[
        "total_mean_per_sample"
    ]
)


# ============================================================
# CORRECT EXACT KNOWN-ANSWER GATE
# ============================================================

validation_official_aggregator_exact = (

    c8_validation_mean
    == c8_reference_validation_mean
)


validation_reconstruction_exact = (

    c8_validation_aggregate[
        "reconstruction_sum"
    ]
    == c8_reference_validation_aggregate[
        "reconstruction_sum"
    ]
)


validation_kl_exact = (

    c8_validation_aggregate[
        "kl_sum"
    ]
    == c8_reference_validation_aggregate[
        "kl_sum"
    ]
)


assert validation_official_aggregator_exact
assert validation_reconstruction_exact
assert validation_kl_exact


# Old C0 scalar should nevertheless be numerically extremely
# close. It is no longer used as a bitwise gate.
legacy_aggregation_close = (
    legacy_aggregation_abs_delta
    <= 1.0e-6
)


assert legacy_aggregation_close


# ============================================================
# COMPLETE THE C8 DIRECTORY AUDIT THAT THE PREVIOUS ASSERT
# PREVENTED FROM RUNNING
# ============================================================

c8_names = sorted(
    path.name

    for path
    in C8_TEST_ROOT.iterdir()
)


expected_c8_names = sorted([
    "best.pt",
    "latest.pt",
    "metrics.csv",
    "run_config.json",
    "runtime_sessions.jsonl",
])


c8_directory_gate = (
    c8_names
    == expected_c8_names
)


assert c8_directory_gate
assert not C8_FINAL.exists()


# ============================================================
# OFFICIAL PERSISTENT DIRECTORY MUST STILL BE BYTE-IDENTICAL
# ============================================================

official_after_c8b = (
    official_directory_fingerprint()
)


official_unchanged_c8b = (

    official_before_c8[
        "sha256"
    ]
    == official_after_c8b[
        "sha256"
    ]

    and official_before_c8[
        "entries"
    ]
    == official_after_c8b[
        "entries"
    ]
)


assert official_unchanged_c8b


# ============================================================
# FINAL CORRECTED C8 GATE
# ============================================================

c8_corrected_pass = (

    c8_checkpoint_gate

    and c8_metrics_gate

    and c8_directory_gate

    and c8_model_known_answer_exact

    and c8_optimizer_known_answer_exact

    and validation_official_aggregator_exact

    and validation_reconstruction_exact

    and validation_kl_exact

    and legacy_aggregation_close

    and official_unchanged_c8b
)


assert c8_corrected_pass


# ============================================================
# REPORT
# ============================================================

print()
print(
    "=== CORRECTED LOCAL EPOCH-0 TRANSACTION AUDIT ==="
)

print(
    "MODEL_KNOWN_ANSWER_EXACT =",
    c8_model_known_answer_exact,
)

print(
    "OPTIMIZER_KNOWN_ANSWER_EXACT =",
    c8_optimizer_known_answer_exact,
)

print()

print(
    "C8_VALIDATION_TOTAL_MEAN =",
    c8_validation_mean,
)

print(
    "REFERENCE_VALIDATION_TOTAL_MEAN =",
    c8_reference_validation_mean,
)

print(
    "VALIDATION_OFFICIAL_AGGREGATOR_EXACT =",
    validation_official_aggregator_exact,
)

print(
    "VALIDATION_RECONSTRUCTION_EXACT =",
    validation_reconstruction_exact,
)

print(
    "VALIDATION_KL_EXACT =",
    validation_kl_exact,
)

print()

print(
    "C0_LEGACY_AGGREGATION_ABS_DELTA =",
    legacy_aggregation_abs_delta,
)

print(
    "C0_LEGACY_AGGREGATION_CLOSE =",
    legacy_aggregation_close,
)

print()

print(
    "LAST_COMPLETED_EPOCH =",
    c8_latest_disk[
        "last_completed_epoch"
    ],
)

print(
    "NEXT_EPOCH =",
    c8_latest_disk[
        "last_completed_epoch"
    ] + 1,
)

print(
    "OPTIMIZER_STEP =",
    c8_latest_disk[
        "optimizer_step"
    ],
)

print(
    "BEST_EPOCH =",
    c8_latest_disk[
        "best_epoch"
    ],
)

print()

print(
    "METRICS_GATE =",
    c8_metrics_gate,
)

print(
    "CHECKPOINT_GATE =",
    c8_checkpoint_gate,
)

print(
    "DIRECTORY_GATE =",
    c8_directory_gate,
)

print()

print(
    "OFFICIAL_DIRECTORY_SHA_BEFORE =",
    official_before_c8[
        "sha256"
    ],
)

print(
    "OFFICIAL_DIRECTORY_SHA_AFTER =",
    official_after_c8b[
        "sha256"
    ],
)

print(
    "OFFICIAL_DIRECTORY_UNCHANGED =",
    official_unchanged_c8b,
)

print()

print(
    "OFFICIAL_FILES_MODIFIED =",
    False,
)

print(
    "OFFICIAL_TRAINING_STARTED =",
    False,
)

print()

print(
    "NOTEBOOK_03_LOCAL_EPOCH_TRANSACTION_PASS =",
    c8_corrected_pass,
)


# ============================================================
# CLEAN UP LOCAL C8 TEST STATE
# ============================================================

del c8_reference_model
del c8_model
del c8_optimizer

gc.collect()
torch.cuda.empty_cache()


shutil.rmtree(
    C8_TEST_ROOT
)


print(
    "LOCAL_TEST_DIRECTORY_REMOVED =",
    not C8_TEST_ROOT.exists(),
)

In [25]:
# ============================================================
# NOTEBOOK 03 — COMPLETE C8 AFTER AGGREGATION-TOLERANCE FIX
# STEP 03-C8C
#
# Continues from the already completed C8/C8B state.
#
# DOES NOT rerun epoch 0.
# DOES NOT rerun validation.
# DOES NOT modify official persistent artifacts.
# ============================================================

import gc
import math
import shutil
import torch


# ============================================================
# CONFIRM THE IMPORTANT EXACT GATES ALREADY PASSED
# ============================================================

assert validation_official_aggregator_exact is True
assert validation_reconstruction_exact is True
assert validation_kl_exact is True

assert c8_model_known_answer_exact is True
assert c8_optimizer_known_answer_exact is True


# ============================================================
# LEGACY AGGREGATION IS ONLY A NUMERIC SANITY CHECK
#
# C0:
#   sum float32(reconstruction + KL) batch by batch
#
# C8:
#   accumulate reconstruction and KL separately in Python
#   double precision and combine at the end.
#
# They are not expected to be bitwise identical.
# ============================================================

legacy_aggregation_relative_delta = (
    legacy_aggregation_abs_delta
    / abs(
        legacy_c0_validation_mean
    )
)


legacy_aggregation_close = math.isclose(
    official_c8_validation_mean,
    legacy_c0_validation_mean,

    # Diagnostic tolerance only.
    # The real correctness gate is the exact same-aggregator
    # comparison above.
    rel_tol=1.0e-7,
    abs_tol=1.0e-3,
)


assert legacy_aggregation_close


# ============================================================
# COMPLETE LOCAL DIRECTORY AUDIT
# ============================================================

assert C8_TEST_ROOT.is_dir()


c8_names = sorted(
    path.name
    for path
    in C8_TEST_ROOT.iterdir()
)


expected_c8_names = sorted([
    "best.pt",
    "latest.pt",
    "metrics.csv",
    "run_config.json",
    "runtime_sessions.jsonl",
])


c8_directory_gate = (
    c8_names
    == expected_c8_names
)


assert c8_directory_gate
assert not C8_FINAL.exists()


# ============================================================
# OFFICIAL PERSISTENT DIRECTORY MUST STILL BE BYTE-IDENTICAL
# ============================================================

official_after_c8c = (
    official_directory_fingerprint()
)


official_unchanged_c8c = (

    official_before_c8[
        "sha256"
    ]
    == official_after_c8c[
        "sha256"
    ]

    and official_before_c8[
        "entries"
    ]
    == official_after_c8c[
        "entries"
    ]
)


assert official_unchanged_c8c


# ============================================================
# FINAL CORRECTED C8 GATE
# ============================================================

c8_corrected_pass = (

    c8_checkpoint_gate

    and c8_metrics_gate

    and c8_directory_gate

    and c8_model_known_answer_exact

    and c8_optimizer_known_answer_exact

    and validation_official_aggregator_exact

    and validation_reconstruction_exact

    and validation_kl_exact

    and legacy_aggregation_close

    and official_unchanged_c8c
)


assert c8_corrected_pass


# ============================================================
# REPORT
# ============================================================

print(
    "=== FINAL CORRECTED LOCAL EPOCH-0 TRANSACTION AUDIT ==="
)

print()

print(
    "MODEL_KNOWN_ANSWER_EXACT =",
    c8_model_known_answer_exact,
)

print(
    "OPTIMIZER_KNOWN_ANSWER_EXACT =",
    c8_optimizer_known_answer_exact,
)

print()

print(
    "VALIDATION_OFFICIAL_AGGREGATOR_EXACT =",
    validation_official_aggregator_exact,
)

print(
    "VALIDATION_RECONSTRUCTION_EXACT =",
    validation_reconstruction_exact,
)

print(
    "VALIDATION_KL_EXACT =",
    validation_kl_exact,
)

print()

print(
    "C0_LEGACY_AGGREGATION =",
    legacy_c0_validation_mean,
)

print(
    "C8_OFFICIAL_AGGREGATION =",
    official_c8_validation_mean,
)

print(
    "ABS_DELTA =",
    legacy_aggregation_abs_delta,
)

print(
    "RELATIVE_DELTA =",
    legacy_aggregation_relative_delta,
)

print(
    "LEGACY_AGGREGATION_CLOSE =",
    legacy_aggregation_close,
)

print()

print(
    "LAST_COMPLETED_EPOCH =",
    c8_latest_disk[
        "last_completed_epoch"
    ],
)

print(
    "NEXT_EPOCH =",
    c8_latest_disk[
        "last_completed_epoch"
    ] + 1,
)

print(
    "OPTIMIZER_STEP =",
    c8_latest_disk[
        "optimizer_step"
    ],
)

print(
    "BEST_EPOCH =",
    c8_latest_disk[
        "best_epoch"
    ],
)

print()

print(
    "CHECKPOINT_GATE =",
    c8_checkpoint_gate,
)

print(
    "METRICS_GATE =",
    c8_metrics_gate,
)

print(
    "DIRECTORY_GATE =",
    c8_directory_gate,
)

print()

print(
    "OFFICIAL_DIRECTORY_UNCHANGED =",
    official_unchanged_c8c,
)

print(
    "OFFICIAL_FILES_MODIFIED =",
    False,
)

print(
    "OFFICIAL_TRAINING_STARTED =",
    False,
)

print()

print(
    "NOTEBOOK_03_LOCAL_EPOCH_TRANSACTION_PASS =",
    c8_corrected_pass,
)


# ============================================================
# CLEAN UP LOCAL TEST STATE
# ============================================================

del c8_reference_model
del c8_model
del c8_optimizer

gc.collect()
torch.cuda.empty_cache()


shutil.rmtree(
    C8_TEST_ROOT
)


print(
    "LOCAL_TEST_DIRECTORY_REMOVED =",
    not C8_TEST_ROOT.exists(),
)

=== FINAL CORRECTED LOCAL EPOCH-0 TRANSACTION AUDIT ===

MODEL_KNOWN_ANSWER_EXACT = True
OPTIMIZER_KNOWN_ANSWER_EXACT = True

VALIDATION_OFFICIAL_AGGREGATOR_EXACT = True
VALIDATION_RECONSTRUCTION_EXACT = True
VALIDATION_KL_EXACT = True

C0_LEGACY_AGGREGATION = 93118.81974373908
C8_OFFICIAL_AGGREGATION = 93118.81888803828
ABS_DELTA = 0.000855700796819292
RELATIVE_DELTA = 9.189343240970639e-09
LEGACY_AGGREGATION_CLOSE = True

LAST_COMPLETED_EPOCH = 0
NEXT_EPOCH = 1
OPTIMIZER_STEP = 54
BEST_EPOCH = 0

CHECKPOINT_GATE = True
METRICS_GATE = True
DIRECTORY_GATE = True

OFFICIAL_DIRECTORY_UNCHANGED = True
OFFICIAL_FILES_MODIFIED = False
OFFICIAL_TRAINING_STARTED = False

NOTEBOOK_03_LOCAL_EPOCH_TRANSACTION_PASS = True
LOCAL_TEST_DIRECTORY_REMOVED = True


## STEP 03-C9 — Final go/no-go preflight della run ufficiale

Questo è l'ultimo controllo prima del training scientifico.

Il preflight ricostruisce la situazione esclusivamente dagli artefatti persistenti e verifica:

- identità del contratto scientifico;
- identità del `run_config`;
- integrità di `LATEST`;
- coerenza `metrics.csv ↔ LATEST`;
- assenza di BEST e FINAL nello stato pre-training;
- assenza di file temporanei;
- configurazione numerica FP32/TF32/AMP;
- batch e loader congelati;
- full split `6866/1717`;
- prossimo epoch `0`;
- optimizer step iniziale `0`;
- primo epoch previsto `0 → 54`;
- stato terminale previsto `epoch 99 → step 5400`;
- directory ufficiale invariata durante l'audit.

Viene inoltre effettuato un caricamento completo di `LATEST` in un modello e optimizer nuovi, simulando l'avvio da un runtime appena ricostruito.

**Non viene eseguito alcun forward, backward o optimizer step.**

Se tutti i gate passano, la run è tecnicamente pronta per essere avviata, ma il training ufficiale non parte ancora in questa cella.

In [26]:
# ============================================================
# NOTEBOOK 03 — FINAL OFFICIAL TRAINING GO/NO-GO PREFLIGHT
# STEP 03-C9
#
# FINAL READ-ONLY gate before the scientific training run.
#
# NO forward.
# NO backward.
# NO optimizer step.
# NO official file write.
# ============================================================

import csv
import gc
import hashlib
import json
from pathlib import Path

import torch


# ============================================================
# CURRENT OFFICIAL PRE-TRAIN FINGERPRINTS
# ============================================================

EXPECTED_PRETRAIN_LATEST_SHA256 = (
    "de9609a7065acf4e52a3a80312576706"
    "c045e4a250b029b2d0deb2bc109b1650"
)

EXPECTED_PRETRAIN_METRICS_SHA256 = (
    "3172643ba06a11abb9d2956a143018c7"
    "b96c03ffeccdab21abb8ea07ce5a9f92"
)

EXPECTED_RUN_CONFIG_SHA256 = (
    "cb2160294fa2ebe82a84d689ebd0cb9f"
    "2fea0264713011eee65413d0d1a9a879"
)

EXPECTED_SCIENTIFIC_CONTRACT_SHA256 = (
    "c895b626454278f2293e217776c4604d"
    "24e03fa78dbaa7ba5049f86f917af949"
)


# ============================================================
# SNAPSHOT DIRECTORY BEFORE PREFLIGHT
# ============================================================

c9_directory_before = (
    official_directory_fingerprint()
)


# ============================================================
# DIRECTORY STATE
# ============================================================

assert OFFICIAL_PARTIAL_DIR.is_dir()

assert not OFFICIAL_FINAL_DIR.exists()


c9_names = sorted(
    path.name
    for path
    in OFFICIAL_PARTIAL_DIR.iterdir()
)


expected_c9_names = sorted([
    "latest.pt",
    "metrics.csv",
    "run_config.json",
    "runtime_sessions.jsonl",
])


directory_gate = (
    c9_names
    == expected_c9_names
)


assert directory_gate


# No BEST before first completed epoch.
assert not OFFICIAL_BEST_PATH.exists()

# FINAL is forbidden before epoch 100.
assert not OFFICIAL_FINAL_CHECKPOINT_PATH.exists()


# ============================================================
# NO TRANSACTION REMNANTS
# ============================================================

c9_stale_files = sorted(
    path.name

    for path
    in OFFICIAL_PARTIAL_DIR.iterdir()

    if (
        path.name.endswith(".tmp")
        or path.name.endswith(".reconcile.tmp")
    )
)


stale_file_gate = (
    c9_stale_files == []
)


assert stale_file_gate


# ============================================================
# RUN CONFIG FROM DISK
# ============================================================

with OFFICIAL_RUN_CONFIG_PATH.open(
    "r",
    encoding="utf-8",
) as handle:

    c9_run_config = json.load(
        handle
    )


c9_stored_run_config_sha = (
    c9_run_config[
        "run_config_sha256"
    ]
)


c9_config_without_sha = dict(
    c9_run_config
)

del c9_config_without_sha[
    "run_config_sha256"
]


c9_reconstructed_run_config_sha = (
    hashlib.sha256(
        json.dumps(
            c9_config_without_sha,
            sort_keys=True,
            separators=(",", ":"),
            ensure_ascii=True,
        ).encode(
            "utf-8"
        )
    ).hexdigest()
)


run_config_gate = (
    c9_stored_run_config_sha
        == EXPECTED_RUN_CONFIG_SHA256

    and c9_reconstructed_run_config_sha
        == EXPECTED_RUN_CONFIG_SHA256

    and RUN_CONFIG_SHA256
        == EXPECTED_RUN_CONFIG_SHA256

    and c9_run_config[
        "run_id"
    ] == RUN_ID
)


assert run_config_gate


# ============================================================
# SCIENTIFIC CONTRACT FROM DISK
# ============================================================

c9_disk_contract_json = json.dumps(
    c9_run_config[
        "scientific_contract"
    ],
    sort_keys=True,
    separators=(",", ":"),
    ensure_ascii=True,
)


c9_disk_contract_sha = (
    hashlib.sha256(
        c9_disk_contract_json.encode(
            "utf-8"
        )
    ).hexdigest()
)


scientific_contract_gate = (
    c9_disk_contract_sha
        == EXPECTED_SCIENTIFIC_CONTRACT_SHA256

    and c9_run_config[
        "scientific_contract_sha256"
    ] == EXPECTED_SCIENTIFIC_CONTRACT_SHA256

    and SCIENTIFIC_CONTRACT_SHA256
        == EXPECTED_SCIENTIFIC_CONTRACT_SHA256
)


assert scientific_contract_gate


# ============================================================
# NUMERICAL POLICY
# ============================================================

numerical_policy_gate = (

    FROZEN_DTYPE == "float32"

    and FROZEN_AMP_ENABLED is False

    and FROZEN_TF32_ENABLED is False

    and torch.backends.cuda.matmul.allow_tf32
        is False

    and torch.backends.cudnn.allow_tf32
        is False

    and torch.backends.cudnn.deterministic
        is True
)


assert numerical_policy_gate


# ============================================================
# CURRENT GPU / RUNTIME
#
# Hardware is provenance rather than scientific identity,
# but this first official run is intentionally started in the
# already benchmarked A100 environment.
# ============================================================

assert torch.cuda.is_available()


c9_gpu_name = (
    torch.cuda.get_device_name(
        0
    )
)


c9_gpu_capability = (
    torch.cuda.get_device_capability(
        0
    )
)


gpu_gate = (
    "A100"
    in c9_gpu_name.upper()
)


assert gpu_gate


# ============================================================
# DATA / SPLIT / LOADER CONTRACT
# ============================================================

data_contract_gate = (

    len(train_dataset)
        == FROZEN_TRAIN_SAMPLES
        == 6866

    and len(val_dataset)
        == FROZEN_VALIDATION_SAMPLES
        == 1717

    and FROZEN_BATCH_SIZE
        == 128

    and FROZEN_NUM_WORKERS
        == 8

    and FROZEN_PIN_MEMORY
        is True

    and FROZEN_PERSISTENT_WORKERS
        is False

    and FROZEN_TRAIN_BATCHES
        == 54

    and FROZEN_VALIDATION_BATCHES
        == 14

    and FROZEN_TRAIN_TAIL
        == 82

    and FROZEN_VALIDATION_TAIL
        == 53
)


assert data_contract_gate


# ============================================================
# FIRST-EPOCH DETERMINISTIC PLAN
# ============================================================

c9_epoch0_positions = (
    project_shuffle_permutation(
        FROZEN_TRAIN_SAMPLES,
        FROZEN_SHUFFLE_SEED,
        0,
    )[:4]
)


c9_epoch0_manifests = [
    int(
        train_rows[
            position
        ][
            "manifest_index"
        ]
    )

    for position
    in c9_epoch0_positions
]


c9_epoch0_crops = [
    gs_augment_crop_left_for_event_mode(
        FROZEN_AUGMENTATION_SEED,
        0,
        manifest_index,
        GS_AUGMENT_MODE_SYMMETRIC_FIGURE,
    )

    for manifest_index
    in c9_epoch0_manifests
]


epoch0_identity_gate = (

    c9_epoch0_positions
    == [
        752,
        2659,
        4819,
        4649,
    ]

    and c9_epoch0_manifests
    == [
        2676,
        6250,
        2396,
        5602,
    ]

    and c9_epoch0_crops
    == [
        20,
        15,
        19,
        12,
    ]
)


assert epoch0_identity_gate


# ============================================================
# PERSISTENT FILE HASHES
# ============================================================

c9_latest_file_sha = (
    sha256_file_streaming(
        OFFICIAL_LATEST_PATH
    )
)


c9_metrics_file_sha = (
    sha256_file_streaming(
        OFFICIAL_METRICS_PATH
    )
)


persistent_hash_gate = (

    c9_latest_file_sha
        == EXPECTED_PRETRAIN_LATEST_SHA256

    and c9_metrics_file_sha
        == EXPECTED_PRETRAIN_METRICS_SHA256
)


assert persistent_hash_gate


# ============================================================
# FRESH MODEL + OPTIMIZER
#
# Simulates the actual recovery path used after a disconnect.
# ============================================================

c9_model = Sakai224FullVAE(
    SPEC
)


initialize_project_c_exact_v1(
    c9_model,
    FROZEN_INITIALIZATION_SEED,
)


c9_model = c9_model.to(
    device=device,
    dtype=torch.float32,
)


c9_optimizer = torch.optim.Adam(
    c9_model.parameters(),

    lr=FROZEN_LEARNING_RATE,

    betas=(
        FROZEN_ADAM_BETA1,
        FROZEN_ADAM_BETA2,
    ),

    eps=FROZEN_ADAM_EPSILON,

    weight_decay=FROZEN_WEIGHT_DECAY,

    amsgrad=FROZEN_AMSGRAD,

    foreach=False,

    fused=False,
)


c9_latest_payload = (
    load_checkpoint_into(
        OFFICIAL_LATEST_PATH,

        c9_model,

        c9_optimizer,

        expected_kind="LATEST",

        device=device,
    )
)


# ============================================================
# CHECKPOINT POSITION
# ============================================================

c9_last_completed_epoch = int(
    c9_latest_payload[
        "last_completed_epoch"
    ]
)


c9_next_epoch = (
    c9_last_completed_epoch
    + 1
)


c9_optimizer_step = (
    get_adam_global_step(
        c9_optimizer
    )
)


checkpoint_position_gate = (

    c9_last_completed_epoch == -1

    and c9_next_epoch == 0

    and c9_optimizer_step == 0

    and c9_latest_payload[
        "optimizer_step"
    ] == 0

    and c9_latest_payload[
        "best_defined"
    ] is False

    and c9_latest_payload[
        "best_epoch"
    ] == -1

    and c9_latest_payload[
        "model_state_sha256"
    ] == OFFICIAL_INITIAL_MODEL_STATE_SHA256

    and c9_latest_payload[
        "optimizer_state_sha256"
    ] == OFFICIAL_INITIAL_OPTIMIZER_STATE_SHA256
)


assert checkpoint_position_gate


# ============================================================
# METRICS ↔ LATEST
# ============================================================

c9_metrics_raw = (
    OFFICIAL_METRICS_PATH
    .read_bytes()
)


c9_metrics_audit = (
    inspect_metrics_against_latest(
        c9_metrics_raw,
        c9_latest_payload,
    )
)


metrics_gate = (

    c9_metrics_audit[
        "committed_metric_rows"
    ] == 2

    and c9_metrics_audit[
        "needs_reconciliation"
    ] is False

    and c9_metrics_audit[
        "future_full_rows"
    ] == 0

    and c9_metrics_audit[
        "partial_tail_dropped"
    ] is False

    and c9_metrics_audit[
        "best_defined"
    ] is False

    and c9_metrics_audit[
        "best_epoch"
    ] == -1
)


assert metrics_gate


# ============================================================
# READ METRICS DIRECTLY AS A SECOND INDEPENDENT CHECK
# ============================================================

with OFFICIAL_METRICS_PATH.open(
    "r",
    newline="",
    encoding="utf-8",
) as handle:

    c9_metric_rows = list(
        csv.DictReader(
            handle
        )
    )


metric_rows_gate = (

    len(c9_metric_rows) == 2

    and c9_metric_rows[0][
        "phase"
    ] == "validation_stochastic_initial"

    and c9_metric_rows[1][
        "phase"
    ] == "validation_mean_initial"

    and int(
        c9_metric_rows[0][
            "optimizer_step"
        ]
    ) == 0

    and int(
        c9_metric_rows[1][
            "optimizer_step"
        ]
    ) == 0
)


assert metric_rows_gate


# ============================================================
# RUNTIME PROVENANCE FILE
# ============================================================

c9_runtime_lines = (
    OFFICIAL_RUNTIME_SESSIONS_PATH
    .read_text(
        encoding="utf-8"
    )
    .splitlines()
)


assert len(
    c9_runtime_lines
) >= 1


c9_runtime_first = json.loads(
    c9_runtime_lines[0]
)


runtime_provenance_gate = (

    c9_runtime_first[
        "scientific_contract_sha256"
    ] == SCIENTIFIC_CONTRACT_SHA256

    and c9_runtime_first[
        "run_config_sha256"
    ] == RUN_CONFIG_SHA256
)


assert runtime_provenance_gate


# ============================================================
# PLANNED FIRST TRANSACTION
# ============================================================

c9_planned_epoch = (
    c9_next_epoch
)


c9_planned_start_step = (
    c9_optimizer_step
)


c9_planned_end_step = (
    c9_planned_start_step
    + FROZEN_TRAIN_BATCHES
)


c9_planned_mean_diagnostic = (
    (
        c9_planned_epoch + 1
    )
    % MEAN_DIAGNOSTIC_EVERY_EPOCHS
    == 0
)


first_transaction_gate = (

    c9_planned_epoch == 0

    and c9_planned_start_step == 0

    and c9_planned_end_step == 54

    and c9_planned_mean_diagnostic
        is False
)


assert first_transaction_gate


# ============================================================
# TERMINAL PLAN
# ============================================================

terminal_plan_gate = (

    FROZEN_MAX_EPOCHS == 100

    and FROZEN_FINAL_EPOCH_INDEX == 99

    and FROZEN_TOTAL_OPTIMIZER_STEPS
        == 5400

    and OFFICIAL_MODEL_SELECTION
        == "EPOCH_100"

    and EARLY_STOPPING_ENABLED
        is False

    and len(
        MEAN_DIAGNOSTIC_EPOCH_INDICES
    ) == 20

    and MEAN_DIAGNOSTIC_EPOCH_INDICES[-1]
        == 99
)


assert terminal_plan_gate


# ============================================================
# MODEL / OPTIMIZER ARE ONLY LOADED, NEVER EXECUTED
# ============================================================

c9_loaded_model_hash = (
    canonical_tree_sha256(
        c9_model.state_dict()
    )
)


c9_loaded_optimizer_hash = (
    canonical_tree_sha256(
        c9_optimizer.state_dict()
    )
)


loaded_state_gate = (

    c9_loaded_model_hash
        == OFFICIAL_INITIAL_MODEL_STATE_SHA256

    and c9_loaded_optimizer_hash
        == OFFICIAL_INITIAL_OPTIMIZER_STATE_SHA256
)


assert loaded_state_gate


# ============================================================
# DIRECTORY MUST REMAIN BITWISE IDENTICAL
# ============================================================

c9_directory_after = (
    official_directory_fingerprint()
)


official_unchanged_c9 = (

    c9_directory_before[
        "sha256"
    ]
    == c9_directory_after[
        "sha256"
    ]

    and c9_directory_before[
        "entries"
    ]
    == c9_directory_after[
        "entries"
    ]
)


assert official_unchanged_c9


# ============================================================
# FINAL GO/NO-GO
# ============================================================

c9_ready_to_start = (

    directory_gate

    and stale_file_gate

    and run_config_gate

    and scientific_contract_gate

    and numerical_policy_gate

    and gpu_gate

    and data_contract_gate

    and epoch0_identity_gate

    and persistent_hash_gate

    and checkpoint_position_gate

    and metrics_gate

    and metric_rows_gate

    and runtime_provenance_gate

    and first_transaction_gate

    and terminal_plan_gate

    and loaded_state_gate

    and official_unchanged_c9
)


assert c9_ready_to_start


# ============================================================
# REPORT
# ============================================================

print(
    "=== FINAL OFFICIAL TRAINING GO/NO-GO PREFLIGHT ==="
)

print()

print(
    "GPU =",
    c9_gpu_name,
)

print(
    "GPU_COMPUTE_CAPABILITY =",
    c9_gpu_capability,
)

print()

print(
    "SCIENTIFIC_CONTRACT_SHA256 =",
    c9_disk_contract_sha,
)

print(
    "RUN_CONFIG_SHA256 =",
    c9_reconstructed_run_config_sha,
)

print(
    "LATEST_SHA256 =",
    c9_latest_file_sha,
)

print(
    "METRICS_SHA256 =",
    c9_metrics_file_sha,
)

print()

print(
    "LAST_COMPLETED_EPOCH =",
    c9_last_completed_epoch,
)

print(
    "NEXT_EPOCH =",
    c9_next_epoch,
)

print(
    "CURRENT_OPTIMIZER_STEP =",
    c9_optimizer_step,
)

print()

print(
    "FIRST_EPOCH_START_STEP =",
    c9_planned_start_step,
)

print(
    "FIRST_EPOCH_END_STEP =",
    c9_planned_end_step,
)

print(
    "FIRST_EPOCH_MEAN_DIAGNOSTIC =",
    c9_planned_mean_diagnostic,
)

print()

print(
    "TRAIN_SAMPLES =",
    FROZEN_TRAIN_SAMPLES,
)

print(
    "VALIDATION_SAMPLES =",
    FROZEN_VALIDATION_SAMPLES,
)

print(
    "BATCH_SIZE =",
    FROZEN_BATCH_SIZE,
)

print(
    "NUM_WORKERS =",
    FROZEN_NUM_WORKERS,
)

print()

print(
    "FINAL_EPOCH_INDEX =",
    FROZEN_FINAL_EPOCH_INDEX,
)

print(
    "FINAL_OPTIMIZER_STEP =",
    FROZEN_TOTAL_OPTIMIZER_STEPS,
)

print(
    "OFFICIAL_MODEL_SELECTION =",
    OFFICIAL_MODEL_SELECTION,
)

print()

print(
    "METRIC_ROWS =",
    len(
        c9_metric_rows
    ),
)

print(
    "METRICS_NEED_RECONCILIATION =",
    c9_metrics_audit[
        "needs_reconciliation"
    ],
)

print(
    "STALE_TEMP_FILES =",
    c9_stale_files,
)

print()

print(
    "DIRECTORY_GATE =",
    directory_gate,
)

print(
    "SCIENTIFIC_CONTRACT_GATE =",
    scientific_contract_gate,
)

print(
    "NUMERICAL_POLICY_GATE =",
    numerical_policy_gate,
)

print(
    "DATA_CONTRACT_GATE =",
    data_contract_gate,
)

print(
    "CHECKPOINT_POSITION_GATE =",
    checkpoint_position_gate,
)

print(
    "METRICS_GATE =",
    metrics_gate,
)

print(
    "TERMINAL_PLAN_GATE =",
    terminal_plan_gate,
)

print(
    "OFFICIAL_DIRECTORY_UNCHANGED =",
    official_unchanged_c9,
)

print()

print(
    "MODEL_FORWARD_EXECUTED =",
    False,
)

print(
    "BACKWARD_EXECUTED =",
    False,
)

print(
    "OPTIMIZER_STEP_EXECUTED =",
    False,
)

print(
    "OFFICIAL_FILES_MODIFIED =",
    False,
)

print(
    "OFFICIAL_TRAINING_STARTED =",
    False,
)

print()

print(
    "READY_TO_START_OFFICIAL_TRAINING =",
    c9_ready_to_start,
)

print(
    "NOTEBOOK_03_FINAL_PREFLIGHT_PASS =",
    c9_ready_to_start,
)


# ============================================================
# RELEASE READ-ONLY AUDIT OBJECTS
# ============================================================

del c9_model
del c9_optimizer

gc.collect()
torch.cuda.empty_cache()

=== FINAL OFFICIAL TRAINING GO/NO-GO PREFLIGHT ===

GPU = NVIDIA A100-SXM4-80GB
GPU_COMPUTE_CAPABILITY = (8, 0)

SCIENTIFIC_CONTRACT_SHA256 = c895b626454278f2293e217776c4604d24e03fa78dbaa7ba5049f86f917af949
RUN_CONFIG_SHA256 = cb2160294fa2ebe82a84d689ebd0cb9f2fea0264713011eee65413d0d1a9a879
LATEST_SHA256 = de9609a7065acf4e52a3a80312576706c045e4a250b029b2d0deb2bc109b1650
METRICS_SHA256 = 3172643ba06a11abb9d2956a143018c7b96c03ffeccdab21abb8ea07ce5a9f92

LAST_COMPLETED_EPOCH = -1
NEXT_EPOCH = 0
CURRENT_OPTIMIZER_STEP = 0

FIRST_EPOCH_START_STEP = 0
FIRST_EPOCH_END_STEP = 54
FIRST_EPOCH_MEAN_DIAGNOSTIC = False

TRAIN_SAMPLES = 6866
VALIDATION_SAMPLES = 1717
BATCH_SIZE = 128
NUM_WORKERS = 8

FINAL_EPOCH_INDEX = 99
FINAL_OPTIMIZER_STEP = 5400
OFFICIAL_MODEL_SELECTION = EPOCH_100

METRIC_ROWS = 2
METRICS_NEED_RECONCILIATION = False
STALE_TEMP_FILES = []

DIRECTORY_GATE = True
SCIENTIFIC_CONTRACT_GATE = True
NUMERICAL_POLICY_GATE = True
DATA_CONTRACT_GATE = True
CHECKPOINT_POSITION_GATE = True

# STEP 03-D0 — OFFICIAL `sakai224_full_v1` TRAINING

⚠️ **OFFICIAL SCIENTIFIC RUN**

Questa cella avvia o riprende il training ufficiale `sakai224_full_v1`.

Contratto congelato:

- full split: `6866 train / 1717 validation`;
- input `4×224×224`;
- latent dimension `512`;
- batch `128`;
- Adam `lr=5e-4`, `β1=0.9`, `β2=0.999`, `eps=1e-8`;
- 100 epoche complete;
- nessun early stopping;
- modello scientifico ufficiale = stato dopo epoch 99, cioè 100 epoche;
- validation stocastica dopo ogni epoch;
- MEAN diagnostic ogni 5 epoche;
- `LATEST` a ogni epoch;
- `BEST` diagnostico con strict minimum della validation;
- `FINAL` solo dopo epoch 99 / optimizer step 5400.

Da questo punto gli artefatti prodotti appartengono alla run scientifica ufficiale.

In [27]:
# ============================================================
# NOTEBOOK 03 — OFFICIAL SAKAI224_FULL TRAINING
# STEP 03-D0
#
# ⚠️ THIS CELL STARTS / RESUMES THE OFFICIAL SCIENTIFIC RUN.
#
# Persistent transaction order per epoch:
#
#   TRAIN
#   -> STOCHASTIC VALIDATION
#   -> optional MEAN diagnostic
#   -> metrics.csv
#   -> LATEST
#   -> optional BEST
#
# Terminal:
#
#   epoch 99
#   -> FINAL MEAN
#   -> final.pt
#   -> .partial directory renamed to final directory
# ============================================================

import csv
import datetime as dt
import gc
import json
import math
import os
import platform
import statistics
import time

import numpy as np
import torch

from torch.utils.data import (
    Dataset,
    DataLoader,
)


print(
    "============================================================"
)
print(
    "⚠️ OFFICIAL TRAINING SAKAI224_FULL_V1"
)
print(
    "THIS IS THE REAL SCIENTIFIC RUN"
)
print(
    "============================================================"
)


# ============================================================
# HARD AUTHORIZATION
# ============================================================

START_OFFICIAL_TRAINING = True

assert START_OFFICIAL_TRAINING is True


# ============================================================
# FINAL FROZEN IDENTITY
# ============================================================

EXPECTED_SCIENTIFIC_CONTRACT_SHA256 = (
    "c895b626454278f2293e217776c4604d"
    "24e03fa78dbaa7ba5049f86f917af949"
)

EXPECTED_RUN_CONFIG_SHA256 = (
    "cb2160294fa2ebe82a84d689ebd0cb9f"
    "2fea0264713011eee65413d0d1a9a879"
)

EXPECTED_PRETRAIN_LATEST_SHA256 = (
    "de9609a7065acf4e52a3a80312576706"
    "c045e4a250b029b2d0deb2bc109b1650"
)

EXPECTED_PRETRAIN_METRICS_SHA256 = (
    "3172643ba06a11abb9d2956a143018c7"
    "b96c03ffeccdab21abb8ea07ce5a9f92"
)


assert (
    SCIENTIFIC_CONTRACT_SHA256
    == EXPECTED_SCIENTIFIC_CONTRACT_SHA256
)

assert (
    RUN_CONFIG_SHA256
    == EXPECTED_RUN_CONFIG_SHA256
)


# ============================================================
# NUMERICAL POLICY
# ============================================================

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

torch.set_float32_matmul_precision(
    "highest"
)


assert (
    torch.backends.cuda.matmul.allow_tf32
    is False
)

assert (
    torch.backends.cudnn.allow_tf32
    is False
)

assert FROZEN_AMP_ENABLED is False


# ============================================================
# HARDWARE GATE
# ============================================================

if not torch.cuda.is_available():

    raise RuntimeError(
        "CUDA unavailable: official training cannot start"
    )


official_gpu_name = (
    torch.cuda.get_device_name(0)
)

official_gpu_capability = (
    torch.cuda.get_device_capability(0)
)


if "A100" not in official_gpu_name.upper():

    raise RuntimeError(
        "STOP: this official run was frozen for the "
        "benchmarked A100 environment. "
        f"Current GPU: {official_gpu_name}"
    )


device = torch.device(
    "cuda:0"
)


print(
    "GPU =",
    official_gpu_name,
)

print(
    "GPU_COMPUTE_CAPABILITY =",
    official_gpu_capability,
)


# ============================================================
# OFFICIAL DIRECTORY STATE
# ============================================================

if (
    OFFICIAL_FINAL_DIR.exists()
    and OFFICIAL_PARTIAL_DIR.exists()
):

    raise RuntimeError(
        "STOP: both FINAL and PARTIAL directories exist"
    )


if OFFICIAL_FINAL_DIR.exists():

    print()
    print(
        "TRAINING_ALREADY_COMPLETE = True"
    )

    print(
        "FINAL_DIRECTORY =",
        OFFICIAL_FINAL_DIR,
    )

    raise RuntimeError(
        "Official run is already finalized. "
        "No training was executed."
    )


if not OFFICIAL_PARTIAL_DIR.is_dir():

    raise RuntimeError(
        "Official .partial directory is missing"
    )


# ============================================================
# VERIFY RUN CONFIG FROM PERSISTENT STORAGE
# ============================================================

with OFFICIAL_RUN_CONFIG_PATH.open(
    "r",
    encoding="utf-8",
) as handle:

    official_disk_config = json.load(
        handle
    )


official_config_stored_sha = (
    official_disk_config[
        "run_config_sha256"
    ]
)


official_config_without_sha = dict(
    official_disk_config
)

del official_config_without_sha[
    "run_config_sha256"
]


official_config_recomputed_sha = (
    __import__(
        "hashlib"
    )
    .sha256(
        json.dumps(
            official_config_without_sha,
            sort_keys=True,
            separators=(",", ":"),
            ensure_ascii=True,
        ).encode(
            "utf-8"
        )
    )
    .hexdigest()
)


assert (
    official_config_stored_sha
    == EXPECTED_RUN_CONFIG_SHA256
)

assert (
    official_config_recomputed_sha
    == EXPECTED_RUN_CONFIG_SHA256
)

assert (
    official_disk_config[
        "scientific_contract_sha256"
    ]
    == EXPECTED_SCIENTIFIC_CONTRACT_SHA256
)


# ============================================================
# DATA CONTRACT
# ============================================================

assert len(train_dataset) == 6866
assert len(val_dataset) == 1717

assert FROZEN_BATCH_SIZE == 128
assert FROZEN_NUM_WORKERS == 8

assert FROZEN_TRAIN_BATCHES == 54
assert FROZEN_VALIDATION_BATCHES == 14

assert FROZEN_TRAIN_TAIL == 82
assert FROZEN_VALIDATION_TAIL == 53

assert FROZEN_MAX_EPOCHS == 100
assert FROZEN_FINAL_EPOCH_INDEX == 99
assert FROZEN_TOTAL_OPTIMIZER_STEPS == 5400


# ============================================================
# FREE OLD DISPOSABLE GPU OBJECTS
#
# Persistent checkpoints are already authoritative.
# ============================================================

for variable_name in [

    "official_model",
    "official_optimizer",

    "checkpoint_test_model",
    "checkpoint_test_optimizer",

    "checkpoint_loaded_model",
    "checkpoint_loaded_optimizer",

    "uninterrupted_model",
    "uninterrupted_optimizer",

    "resumed_model",
    "resumed_optimizer",
]:

    globals().pop(
        variable_name,
        None,
    )


gc.collect()
torch.cuda.empty_cache()


# ============================================================
# RECOVER CHECKPOINT PUBLICATION WINDOWS
#
# May repair a missing BEST or remove an unpublished .tmp.
#
# Published LATEST remains authoritative.
# ============================================================

publication_recovery = (
    recover_checkpoint_publication_state(
        OFFICIAL_PARTIAL_DIR
    )
)


print()
print(
    "PUBLICATION_RECOVERY_LATEST_EPOCH =",
    publication_recovery[
        "latest_epoch"
    ],
)

print(
    "PUBLICATION_RECOVERY_BEST_STATUS =",
    publication_recovery[
        "best_status"
    ],
)


# ============================================================
# LOAD PUBLISHED LATEST METADATA ON CPU
# ============================================================

official_latest_cpu = (
    load_valid_checkpoint_cpu(
        OFFICIAL_LATEST_PATH,
        expected_kind="LATEST",
    )
)


# ============================================================
# RECONCILE METRICS TO LATEST
#
# This is a no-op in the normal case.
# If a previous runtime died after metrics but before LATEST,
# future rows are removed atomically.
# ============================================================

metrics_recovery = (
    reconcile_metrics_to_latest_atomic(
        OFFICIAL_METRICS_PATH,
        official_latest_cpu,
    )
)


print(
    "METRICS_RECONCILED =",
    metrics_recovery[
        "reconciled"
    ],
)


# Reload after recovery.
official_latest_cpu = (
    load_valid_checkpoint_cpu(
        OFFICIAL_LATEST_PATH,
        expected_kind="LATEST",
    )
)


# ============================================================
# CLEAN STALE FINAL TEMP IF PRESENT
#
# final.pt.tmp is unpublished terminal state.
# LATEST has already been validated above.
# ============================================================

official_final_tmp = (
    OFFICIAL_PARTIAL_DIR
    / "final.pt.tmp"
)


if official_final_tmp.exists():

    official_final_tmp.unlink()

    print(
        "STALE_FINAL_TMP_REMOVED = True"
    )


# ============================================================
# BUILD FRESH OFFICIAL MODEL / ADAM FROM PERSISTENT LATEST
# ============================================================

training_model = Sakai224FullVAE(
    SPEC
)


initialize_project_c_exact_v1(
    training_model,
    FROZEN_INITIALIZATION_SEED,
)


training_model = training_model.to(
    device=device,
    dtype=torch.float32,
)


training_optimizer = torch.optim.Adam(
    training_model.parameters(),

    lr=FROZEN_LEARNING_RATE,

    betas=(
        FROZEN_ADAM_BETA1,
        FROZEN_ADAM_BETA2,
    ),

    eps=FROZEN_ADAM_EPSILON,

    weight_decay=FROZEN_WEIGHT_DECAY,

    amsgrad=FROZEN_AMSGRAD,

    foreach=False,

    fused=False,
)


training_latest = (
    load_checkpoint_into(
        OFFICIAL_LATEST_PATH,

        training_model,

        training_optimizer,

        expected_kind="LATEST",

        device=device,
    )
)


current_last_completed_epoch = int(
    training_latest[
        "last_completed_epoch"
    ]
)


next_epoch = (
    current_last_completed_epoch
    + 1
)


current_optimizer_step = (
    get_adam_global_step(
        training_optimizer
    )
)


assert (
    current_optimizer_step
    == next_epoch
    * FROZEN_TRAIN_BATCHES
)


# Strong pristine-state identity on the first launch.
if current_last_completed_epoch == -1:

    assert (
        sha256_file_streaming(
            OFFICIAL_LATEST_PATH
        )
        == EXPECTED_PRETRAIN_LATEST_SHA256
    )

    assert (
        sha256_file_streaming(
            OFFICIAL_METRICS_PATH
        )
        == EXPECTED_PRETRAIN_METRICS_SHA256
    )


print()
print(
    "RESUME_LAST_COMPLETED_EPOCH =",
    current_last_completed_epoch,
)

print(
    "RESUME_NEXT_EPOCH =",
    next_epoch,
)

print(
    "RESUME_OPTIMIZER_STEP =",
    current_optimizer_step,
)


# ============================================================
# RUNTIME ENVIRONMENT MUST MATCH INITIAL SESSION
# ============================================================

runtime_lines_before_training = (
    OFFICIAL_RUNTIME_SESSIONS_PATH
    .read_text(
        encoding="utf-8"
    )
    .splitlines()
)


if not runtime_lines_before_training:

    raise RuntimeError(
        "runtime_sessions.jsonl is empty"
    )


initial_runtime = json.loads(
    runtime_lines_before_training[0]
)


environment_gate = (

    initial_runtime[
        "gpu_name"
    ] == official_gpu_name

    and initial_runtime[
        "torch_version"
    ] == str(
        torch.__version__
    )

    and initial_runtime[
        "cuda_version"
    ] == str(
        torch.version.cuda
    )

    and initial_runtime[
        "cudnn_version"
    ] == int(
        torch.backends.cudnn.version()
    )
)


if not environment_gate:

    raise RuntimeError(
        "STOP: runtime environment differs from the "
        "environment frozen at run preparation"
    )


# ============================================================
# APPEND OFFICIAL TRAINING SESSION PROVENANCE
# ============================================================

training_session_record = {

    "timestamp_utc":
        dt.datetime.now(
            dt.timezone.utc
        ).isoformat(),

    "purpose":
        "OFFICIAL_TRAINING_START_OR_RESUME",

    "resume_last_completed_epoch":
        current_last_completed_epoch,

    "resume_next_epoch":
        next_epoch,

    "optimizer_step":
        current_optimizer_step,

    "python_version":
        platform.python_version(),

    "torch_version":
        str(
            torch.__version__
        ),

    "cuda_version":
        str(
            torch.version.cuda
        ),

    "cudnn_version":
        int(
            torch.backends.cudnn.version()
        ),

    "gpu_name":
        official_gpu_name,

    "gpu_compute_capability":
        list(
            official_gpu_capability
        ),

    "scientific_contract_sha256":
        SCIENTIFIC_CONTRACT_SHA256,

    "run_config_sha256":
        RUN_CONFIG_SHA256,
}


with OFFICIAL_RUNTIME_SESSIONS_PATH.open(
    "a",
    encoding="utf-8",
) as handle:

    handle.write(
        json.dumps(
            training_session_record,
            sort_keys=True,
            ensure_ascii=True,
        )
        + "\n"
    )

    handle.flush()

    os.fsync(
        handle.fileno()
    )


# ============================================================
# OFFICIAL TRAIN DATASET WITH STATELESS EPSILON
# ============================================================

official_train_dataset = (
    GravitySpy224TrainWithEpsilon(
        train_dataset,
        epoch=max(
            next_epoch,
            0,
        ),
    )
)


def make_official_train_loader(
    epoch,
):

    epoch = int(
        epoch
    )


    train_sampler.set_epoch(
        epoch
    )

    official_train_dataset.set_epoch(
        epoch
    )


    return DataLoader(
        official_train_dataset,

        batch_size=FROZEN_BATCH_SIZE,

        sampler=train_sampler,

        num_workers=FROZEN_NUM_WORKERS,

        pin_memory=FROZEN_PIN_MEMORY,

        drop_last=False,

        persistent_workers=False,

        prefetch_factor=2,
    )


# ============================================================
# ONE OFFICIAL TRAIN EPOCH
# ============================================================

def run_one_official_train_epoch(
    model,
    optimizer,
    epoch,
):

    model.train()


    loader = (
        make_official_train_loader(
            epoch
        )
    )


    reconstruction_sum = 0.0
    kl_sum = 0.0

    sample_count = 0
    batch_count = 0
    tail_size = None


    # --------------------------------------------------------
    # Expected first-batch identity from PROJECT shuffle
    # --------------------------------------------------------

    expected_positions = (
        project_shuffle_permutation(
            FROZEN_TRAIN_SAMPLES,
            FROZEN_SHUFFLE_SEED,
            epoch,
        )[:4]
    )


    expected_manifests = [

        int(
            train_rows[
                position
            ][
                "manifest_index"
            ]
        )

        for position
        in expected_positions
    ]


    expected_crops = [

        gs_augment_crop_left_for_event_mode(
            FROZEN_AUGMENTATION_SEED,
            epoch,
            manifest_index,
            GS_AUGMENT_MODE_SYMMETRIC_FIGURE,
        )

        for manifest_index
        in expected_manifests
    ]


    first_batch_gate = False


    torch.cuda.synchronize()

    train_start = (
        time.perf_counter()
    )


    for batch_index, batch in enumerate(
        loader
    ):

        if batch_index == 0:

            actual_manifests = (
                batch[
                    "manifest_index"
                ][:4].tolist()
            )

            actual_crops = (
                batch[
                    "crop_left"
                ][:4].tolist()
            )


            first_batch_gate = (

                actual_manifests
                == expected_manifests

                and actual_crops
                == expected_crops
            )


            if not first_batch_gate:

                raise RuntimeError(
                    f"Epoch {epoch}: first-batch identity mismatch"
                )


        x_batch = (
            batch["x"]
            .to(
                device=device,
                dtype=torch.float32,
                non_blocking=True,
            )
        )


        epsilon_batch = (
            batch["epsilon"]
            .to(
                device=device,
                dtype=torch.float32,
                non_blocking=True,
            )
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        (
            logits,
            mu,
            logvar,
            z,
        ) = model(
            x_batch,
            epsilon=epsilon_batch,
            mean_mode=False,
        )


        stats = sakai_vae_loss(
            logits,
            x_batch,
            mu,
            logvar,
            beta_kl=SPEC.beta_kl,
        )


        loss = stats[
            "loss"
        ]


        if not torch.isfinite(
            loss
        ).item():

            raise RuntimeError(
                f"Epoch {epoch}: non-finite train loss"
            )


        reconstruction_sum += float(
            stats[
                "reconstruction_sum"
            ].detach()
        )

        kl_sum += float(
            stats[
                "kl_sum"
            ].detach()
        )


        loss.backward()

        optimizer.step()


        current_batch_size = int(
            x_batch.shape[0]
        )


        sample_count += (
            current_batch_size
        )

        batch_count += 1

        tail_size = (
            current_batch_size
        )


    torch.cuda.synchronize()


    wall_seconds = (
        time.perf_counter()
        - train_start
    )


    # --------------------------------------------------------
    # Full-epoch integrity
    # --------------------------------------------------------

    assert sample_count == 6866
    assert batch_count == 54
    assert tail_size == 82
    assert first_batch_gate


    expected_global_step = (
        (epoch + 1)
        * FROZEN_TRAIN_BATCHES
    )


    if (
        get_adam_global_step(
            optimizer
        )
        != expected_global_step
    ):

        raise RuntimeError(
            f"Epoch {epoch}: Adam step mismatch"
        )


    if not all(

        int(
            bn.num_batches_tracked.item()
        )
        == expected_global_step

        for bn
        in get_project_batchnorm_layers(
            model
        )
    ):

        raise RuntimeError(
            f"Epoch {epoch}: BatchNorm counter mismatch"
        )


    if not all(

        torch.isfinite(
            parameter
        ).all().item()

        for parameter
        in model.parameters()
    ):

        raise RuntimeError(
            f"Epoch {epoch}: non-finite model parameter"
        )


    total_sum = (
        reconstruction_sum
        + kl_sum
    )


    return {

        "sample_count":
            sample_count,

        "batch_count":
            batch_count,

        "total_sum":
            total_sum,

        "total_mean_per_sample":
            total_sum
            / sample_count,

        "reconstruction_sum":
            reconstruction_sum,

        "reconstruction_mean_per_sample":
            reconstruction_sum
            / sample_count,

        "reconstruction_mean_per_input_value":
            reconstruction_sum
            / (
                sample_count
                * 4
                * 224
                * 224
            ),

        "kl_sum":
            kl_sum,

        "kl_mean_per_sample":
            kl_sum
            / sample_count,

        "kl_mean_per_latent_value":
            kl_sum
            / (
                sample_count
                * SPEC.latent_dim
            ),

        "wall_seconds":
            wall_seconds,

        "first_manifests":
            expected_manifests,

        "first_crops":
            expected_crops,
    }


# ============================================================
# CURRENT BEST FROM LATEST
# ============================================================

best_defined = bool(
    training_latest[
        "best_defined"
    ]
)


best_epoch = int(
    training_latest[
        "best_epoch"
    ]
)


best_validation = float(
    training_latest[
        "best_validation_total_mean_per_sample"
    ]
)


# ============================================================
# ACTUAL OFFICIAL TRAINING
# ============================================================

print()
print(
    "============================================================"
)

print(
    "OFFICIAL_TRAINING_STARTED = True"
)

print(
    "STARTING_FROM_EPOCH =",
    next_epoch,
)

print(
    "STARTING_OPTIMIZER_STEP =",
    current_optimizer_step,
)

print(
    "TARGET_FINAL_EPOCH = 99"
)

print(
    "TARGET_FINAL_OPTIMIZER_STEP = 5400"
)

print(
    "============================================================"
)

print()


official_epoch_times = []


for epoch in range(
    next_epoch,
    FROZEN_MAX_EPOCHS,
):

    transaction_start = (
        time.perf_counter()
    )


    # ========================================================
    # TRAIN
    # ========================================================

    train_aggregate = (
        run_one_official_train_epoch(
            training_model,
            training_optimizer,
            epoch,
        )
    )


    # ========================================================
    # STOCHASTIC VALIDATION
    # ========================================================

    validation_aggregate = (
        run_official_validation_aggregate(
            training_model,
            mode="stochastic",
        )
    )


    validation_mean = float(
        validation_aggregate[
            "total_mean_per_sample"
        ]
    )


    # ========================================================
    # STRICT BEST POLICY
    # ========================================================

    improved = (

        not best_defined

        or validation_mean
        < best_validation
    )


    if improved:

        best_defined = True
        best_epoch = epoch
        best_validation = (
            validation_mean
        )


    # ========================================================
    # PERIODIC MEAN DIAGNOSTIC
    # ========================================================

    mean_defined = (

        (epoch + 1)
        % MEAN_DIAGNOSTIC_EVERY_EPOCHS
        == 0
    )


    mean_aggregate = None


    if mean_defined:

        mean_aggregate = (
            run_official_validation_aggregate(
                training_model,
                mode="mean",
            )
        )


    optimizer_step = (
        get_adam_global_step(
            training_optimizer
        )
    )


    expected_step = (
        (epoch + 1)
        * FROZEN_TRAIN_BATCHES
    )


    assert (
        optimizer_step
        == expected_step
    )


    # ========================================================
    # METRICS — COMMIT BEFORE LATEST
    # ========================================================

    train_row = (
        make_official_metric_row(
            "train_stochastic",

            epoch=epoch,

            completed_epochs=(
                epoch + 1
            ),

            optimizer_step=(
                optimizer_step
            ),

            aggregate=(
                train_aggregate
            ),
        )
    )


    validation_row = (
        make_official_metric_row(
            "validation_stochastic_post_epoch",

            epoch=epoch,

            completed_epochs=(
                epoch + 1
            ),

            optimizer_step=(
                optimizer_step
            ),

            aggregate=(
                validation_aggregate
            ),
        )
    )


    append_official_metric_row(
        train_row
    )

    append_official_metric_row(
        validation_row
    )


    if mean_defined:

        mean_row = (
            make_official_metric_row(
                "validation_mean_diagnostic",

                epoch=epoch,

                completed_epochs=(
                    epoch + 1
                ),

                optimizer_step=(
                    optimizer_step
                ),

                aggregate=(
                    mean_aggregate
                ),
            )
        )


        append_official_metric_row(
            mean_row
        )


    # ========================================================
    # PUBLISH LATEST
    # ========================================================

    latest_payload = (
        build_checkpoint_payload(
            training_model,
            training_optimizer,

            kind="LATEST",

            last_completed_epoch=(
                epoch
            ),

            best_defined=(
                best_defined
            ),

            best_epoch=(
                best_epoch
            ),

            best_validation_total_mean_per_sample=(
                best_validation
            ),
        )
    )


    save_checkpoint_atomic(
        OFFICIAL_LATEST_PATH,
        latest_payload,
    )


    # ========================================================
    # PUBLISH BEST ONLY WHEN IMPROVED
    # ========================================================

    if improved:

        best_payload = (
            build_checkpoint_payload(
                training_model,
                training_optimizer,

                kind="BEST",

                last_completed_epoch=(
                    epoch
                ),

                best_defined=True,

                best_epoch=(
                    best_epoch
                ),

                best_validation_total_mean_per_sample=(
                    best_validation
                ),
            )
        )


        save_checkpoint_atomic(
            OFFICIAL_BEST_PATH,
            best_payload,
        )


        del best_payload


    # ========================================================
    # POST-COMMIT AUDIT
    # ========================================================

    latest_disk = (
        load_valid_checkpoint_cpu(
            OFFICIAL_LATEST_PATH,
            expected_kind="LATEST",
        )
    )


    assert (
        latest_disk[
            "last_completed_epoch"
        ] == epoch
    )

    assert (
        latest_disk[
            "optimizer_step"
        ] == optimizer_step
    )


    metrics_post_commit = (
        inspect_metrics_against_latest(
            OFFICIAL_METRICS_PATH.read_bytes(),
            latest_disk,
        )
    )


    assert (
        metrics_post_commit[
            "needs_reconciliation"
        ] is False
    )

    assert (
        metrics_post_commit[
            "future_full_rows"
        ] == 0
    )

    assert (
        metrics_post_commit[
            "partial_tail_dropped"
        ] is False
    )


    # BEST must agree with LATEST.
    if best_defined:

        best_disk = (
            load_valid_checkpoint_cpu(
                OFFICIAL_BEST_PATH,
                expected_kind="BEST",
            )
        )


        assert (
            best_disk[
                "best_epoch"
            ] == best_epoch
        )

        assert (
            best_disk[
                "best_validation_total_mean_per_sample"
            ] == best_validation
        )


    # ========================================================
    # PROGRESS / ETA
    # ========================================================

    transaction_seconds = (
        time.perf_counter()
        - transaction_start
    )


    official_epoch_times.append(
        transaction_seconds
    )


    median_transaction_seconds = (
        statistics.median(
            official_epoch_times
        )
    )


    remaining_after_epoch = (
        FROZEN_MAX_EPOCHS
        - epoch
        - 1
    )


    eta_minutes = (
        remaining_after_epoch
        * median_transaction_seconds
        / 60.0
    )


    mean_text = (

        "NA"

        if mean_aggregate is None

        else (
            f"{mean_aggregate['total_mean_per_sample']:.6f}"
        )
    )


    print(
        f"EPOCH {epoch:02d}/99"
        f" | step={optimizer_step}/5400"
        f" | train={train_aggregate['total_mean_per_sample']:.6f}"
        f" | val={validation_mean:.6f}"
        f" | mean={mean_text}"
        f" | best_epoch={best_epoch}"
        f" | best={best_validation:.6f}"
        f" | improved={improved}"
        f" | epoch_transaction={transaction_seconds:.2f}s"
        f" | ETA≈{eta_minutes:.1f}min"
    )


    # Release CPU checkpoint snapshots promptly.
    del latest_payload
    del latest_disk

    gc.collect()


# ============================================================
# TRAINING LOOP MUST NOW BE COMPLETE
# ============================================================

assert (
    get_adam_global_step(
        training_optimizer
    )
    == FROZEN_TOTAL_OPTIMIZER_STEPS
)


terminal_latest = (
    load_valid_checkpoint_cpu(
        OFFICIAL_LATEST_PATH,
        expected_kind="LATEST",
    )
)


assert (
    terminal_latest[
        "last_completed_epoch"
    ]
    == 99
)

assert (
    terminal_latest[
        "optimizer_step"
    ]
    == 5400
)


# ============================================================
# TERMINAL METRICS STATE
# ============================================================

terminal_metrics_audit = (
    inspect_metrics_against_latest(
        OFFICIAL_METRICS_PATH.read_bytes(),
        terminal_latest,
    )
)


# ============================================================
# FINAL MEAN VALIDATION
#
# Separate terminal evaluation, even though epoch 99 already
# contains the 20th periodic MEAN diagnostic.
# ============================================================

if not terminal_metrics_audit[
    "final_row_present"
]:

    print()
    print(
        "Running terminal FINAL MEAN validation..."
    )


    final_mean_aggregate = (
        run_official_validation_aggregate(
            training_model,
            mode="mean",
        )
    )


    final_mean_row = (
        make_official_metric_row(
            "validation_mean_final",

            epoch=99,

            completed_epochs=100,

            optimizer_step=5400,

            aggregate=(
                final_mean_aggregate
            ),
        )
    )


    append_official_metric_row(
        final_mean_row
    )


else:

    final_mean_aggregate = None


# Independent audit after optional append.
terminal_metrics_audit = (
    inspect_metrics_against_latest(
        OFFICIAL_METRICS_PATH.read_bytes(),
        terminal_latest,
    )
)


assert (
    terminal_metrics_audit[
        "final_row_present"
    ]
)


# ============================================================
# FINAL CHECKPOINT
# ============================================================

if OFFICIAL_FINAL_CHECKPOINT_PATH.exists():

    final_disk = (
        load_valid_checkpoint_cpu(
            OFFICIAL_FINAL_CHECKPOINT_PATH,
            expected_kind="FINAL",
        )
    )


    final_checkpoint_gate = (

        final_disk[
            "last_completed_epoch"
        ] == 99

        and final_disk[
            "optimizer_step"
        ] == 5400

        and final_disk[
            "model_state_sha256"
        ] == terminal_latest[
            "model_state_sha256"
        ]

        and final_disk[
            "optimizer_state_sha256"
        ] == terminal_latest[
            "optimizer_state_sha256"
        ]
    )


    if not final_checkpoint_gate:

        raise RuntimeError(
            "Existing final.pt is incompatible"
        )


else:

    final_payload = (
        build_checkpoint_payload(
            training_model,
            training_optimizer,

            kind="FINAL",

            last_completed_epoch=99,

            best_defined=True,

            best_epoch=(
                best_epoch
            ),

            best_validation_total_mean_per_sample=(
                best_validation
            ),
        )
    )


    save_checkpoint_atomic(
        OFFICIAL_FINAL_CHECKPOINT_PATH,
        final_payload,
    )


    del final_payload


# ============================================================
# TERMINAL CHECKPOINT AUDIT
# ============================================================

final_disk = (
    load_valid_checkpoint_cpu(
        OFFICIAL_FINAL_CHECKPOINT_PATH,
        expected_kind="FINAL",
    )
)


final_checkpoint_gate = (

    final_disk[
        "last_completed_epoch"
    ] == 99

    and final_disk[
        "optimizer_step"
    ] == 5400

    and final_disk[
        "model_state_sha256"
    ] == terminal_latest[
        "model_state_sha256"
    ]

    and final_disk[
        "optimizer_state_sha256"
    ] == terminal_latest[
        "optimizer_state_sha256"
    ]

    and final_disk[
        "best_epoch"
    ] == best_epoch

    and final_disk[
        "best_validation_total_mean_per_sample"
    ] == best_validation
)


assert final_checkpoint_gate


# ============================================================
# FINAL BEST AUDIT
# ============================================================

final_best = (
    load_valid_checkpoint_cpu(
        OFFICIAL_BEST_PATH,
        expected_kind="BEST",
    )
)


assert (
    final_best[
        "best_epoch"
    ]
    == best_epoch
)

assert (
    final_best[
        "best_validation_total_mean_per_sample"
    ]
    == best_validation
)


# ============================================================
# RECORD COMPLETION PROVENANCE
# ============================================================

completion_record = {

    "timestamp_utc":
        dt.datetime.now(
            dt.timezone.utc
        ).isoformat(),

    "purpose":
        "OFFICIAL_TRAINING_COMPLETED",

    "last_completed_epoch":
        99,

    "completed_epochs":
        100,

    "optimizer_step":
        5400,

    "best_epoch":
        best_epoch,

    "best_validation_total_mean_per_sample":
        best_validation,

    "gpu_name":
        official_gpu_name,

    "torch_version":
        str(
            torch.__version__
        ),

    "cuda_version":
        str(
            torch.version.cuda
        ),

    "scientific_contract_sha256":
        SCIENTIFIC_CONTRACT_SHA256,

    "run_config_sha256":
        RUN_CONFIG_SHA256,
}


with OFFICIAL_RUNTIME_SESSIONS_PATH.open(
    "a",
    encoding="utf-8",
) as handle:

    handle.write(
        json.dumps(
            completion_record,
            sort_keys=True,
            ensure_ascii=True,
        )
        + "\n"
    )

    handle.flush()

    os.fsync(
        handle.fileno()
    )


# ============================================================
# FINAL DIRECTORY ALLOWLIST
# ============================================================

terminal_names = sorted(
    path.name

    for path
    in OFFICIAL_PARTIAL_DIR.iterdir()
)


expected_terminal_names = sorted([
    "best.pt",
    "final.pt",
    "latest.pt",
    "metrics.csv",
    "run_config.json",
    "runtime_sessions.jsonl",
])


assert (
    terminal_names
    == expected_terminal_names
)


# ============================================================
# PUBLISH FINAL DIRECTORY
# ============================================================

if OFFICIAL_FINAL_DIR.exists():

    raise RuntimeError(
        "Cannot finalize: destination directory already exists"
    )


os.rename(
    OFFICIAL_PARTIAL_DIR,
    OFFICIAL_FINAL_DIR,
)


assert OFFICIAL_FINAL_DIR.is_dir()
assert not OFFICIAL_PARTIAL_DIR.exists()


# ============================================================
# FINAL FILE PATHS AFTER RENAME
# ============================================================

FINAL_LATEST_PATH = (
    OFFICIAL_FINAL_DIR
    / "latest.pt"
)

FINAL_BEST_PATH = (
    OFFICIAL_FINAL_DIR
    / "best.pt"
)

FINAL_FINAL_PATH = (
    OFFICIAL_FINAL_DIR
    / "final.pt"
)

FINAL_METRICS_PATH = (
    OFFICIAL_FINAL_DIR
    / "metrics.csv"
)


# ============================================================
# FINAL REPORT
# ============================================================

print()
print(
    "============================================================"
)

print(
    "OFFICIAL TRAINING COMPLETED"
)

print(
    "============================================================"
)

print()

print(
    "TRAINING_COMPLETED = True"
)

print(
    "COMPLETED_EPOCHS = 100"
)

print(
    "LAST_COMPLETED_EPOCH = 99"
)

print(
    "FINAL_OPTIMIZER_STEP = 5400"
)

print()

print(
    "BEST_EPOCH =",
    best_epoch,
)

print(
    "BEST_VALIDATION_TOTAL_MEAN =",
    best_validation,
)

print()

print(
    "LATEST_SHA256 =",
    sha256_file_streaming(
        FINAL_LATEST_PATH
    ),
)

print(
    "BEST_SHA256 =",
    sha256_file_streaming(
        FINAL_BEST_PATH
    ),
)

print(
    "FINAL_SHA256 =",
    sha256_file_streaming(
        FINAL_FINAL_PATH
    ),
)

print(
    "METRICS_SHA256 =",
    sha256_file_streaming(
        FINAL_METRICS_PATH
    ),
)

print()

print(
    "FINAL_MODEL_STATE_SHA256 =",
    final_disk[
        "model_state_sha256"
    ],
)

print(
    "FINAL_OPTIMIZER_STATE_SHA256 =",
    final_disk[
        "optimizer_state_sha256"
    ],
)

print()

print(
    "OFFICIAL_MODEL_SELECTION =",
    OFFICIAL_MODEL_SELECTION,
)

print(
    "OFFICIAL_OUTPUT_DIRECTORY =",
    OFFICIAL_FINAL_DIR,
)

print()

print(
    "NOTEBOOK_03_OFFICIAL_TRAINING_PASS =",
    True,
)

⚠️ OFFICIAL TRAINING SAKAI224_FULL_V1
THIS IS THE REAL SCIENTIFIC RUN
GPU = NVIDIA A100-SXM4-80GB
GPU_COMPUTE_CAPABILITY = (8, 0)

PUBLICATION_RECOVERY_LATEST_EPOCH = -1
PUBLICATION_RECOVERY_BEST_STATUS = NOT_DEFINED
METRICS_RECONCILED = False

RESUME_LAST_COMPLETED_EPOCH = -1
RESUME_NEXT_EPOCH = 0
RESUME_OPTIMIZER_STEP = 0

OFFICIAL_TRAINING_STARTED = True
STARTING_FROM_EPOCH = 0
STARTING_OPTIMIZER_STEP = 0
TARGET_FINAL_EPOCH = 99
TARGET_FINAL_OPTIMIZER_STEP = 5400

EPOCH 00/99 | step=54/5400 | train=102114.511079 | val=93118.818888 | mean=NA | best_epoch=0 | best=93118.818888 | improved=True | epoch_transaction=18.38s | ETA≈30.3min
EPOCH 01/99 | step=108/5400 | train=92505.708584 | val=91962.255169 | mean=NA | best_epoch=1 | best=91962.255169 | improved=True | epoch_transaction=19.86s | ETA≈31.2min
EPOCH 02/99 | step=162/5400 | train=91919.360442 | val=91592.409953 | mean=NA | best_epoch=2 | best=91592.409953 | improved=True | epoch_transaction=25.09s | ETA≈32.1min
EPOCH 03/99 | step